<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/DL-2026/Lecture_4/Lecture_4_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Глава 1. Введение: постановка задачи и роль рекуррентных архитектур

## 1.1. Предмет и структура главы

Настоящая глава открывает учебный курс, посвящённый рекуррентным нейронным сетям (recurrent neural networks, RNN) и алгоритму обратного распространения ошибки во времени (backpropagation through time, BPTT). Её задача — сформулировать **предмет исследования**, **мотивировать** выбор рекуррентных архитектур и задать **общую схему**, которой будут следовать все последующие главы.

Глава не содержит вычислительных формул сложнее композиции функций. Её назначение — методологическое: читатель должен уяснить, **какую задачу** решают нейронные сети, **почему** для класса последовательностных задач недостаточно сетей прямого распространения, **из каких этапов** состоит обучение любой нейронной сети и **каким образом** материал курса разбит на главы. Без этого введения последующее изложение — в частности, детальный разбор архитектуры $1 \to 1 \to 1$ в главе 4 и рекуррентных архитектур в главах 5–14 — будет лишено смысловой рамки.

Изложение ведётся в следующем порядке. В §1.2 формулируется общая задача обучения с учителем. В §1.3 выделяется класс последовательностных задач и обосновывается его специфика. В §1.4 показано, почему сети прямого распространения не приспособлены к этому классу. В §1.5 вводится идея рекуррентности. В §1.6 описываются три фазы обучения, единые для всех архитектур. В §1.7 вводится понятие графа вычислений. В §1.8 обосновывается прогрессия глав. В §1.9 определяется структура каждой главы. В §1.10 перечисляются предварительные требования. В §1.11 формулируются результаты, которые читатель получит по завершении курса. В §1.12 очерчиваются границы курса.

## 1.2. Постановка задачи обучения с учителем

В наиболее общей форме задача обучения с учителем (supervised learning) формулируется следующим образом.

> **Определение 1.1 (задача обучения с учителем).** Пусть задано множество примеров $\mathcal{D} = \{(x^{(i)}, y^{(i)})\}_{i=1}^{n}$, где $x^{(i)} \in \mathcal{X}$ — входной объект, $y^{(i)} \in \mathcal{Y}$ — правильный ответ (метка, целевое значение). Требуется найти функцию $\hat y = f(x; \theta)$, параметризованную вектором параметров $\theta \in \Theta$, которая хорошо приближает $y$ не только на известных примерах из $\mathcal{D}$, но и на **новых** объектах $x \notin \mathcal{D}$.

Существенны два момента этой формулировки. Во-первых, речь идёт о **параметризованном** семействе функций $\{f(\cdot; \theta)\}_{\theta \in \Theta}$, а не о произвольной функции. Выбор семейства определяется архитектурой — в нашем курсе это будут нейронные сети различных конфигураций. Во-вторых, критерий качества формулируется не только на обучающей выборке $\mathcal{D}$, но и на новых объектах; это требование **обобщения** (generalization) и оно принципиально отделяет обучение от простого запоминания.

Для формализации критерия вводится функция потерь.

> **Определение 1.2 (функция потерь).** Функцией потерь называется отображение $L: \mathcal{Y} \times \mathcal{Y} \to \mathbb{R}_{\geq 0}$, сопоставляющее предсказанию $\hat y$ и истинному ответу $y$ неотрицательное число $L(\hat y, y)$, интерпретируемое как «мера ошибки». Требуется, чтобы $L(\hat y, y) = 0$ при $\hat y = y$ и $L(\hat y, y)$ возрастала при отклонении $\hat y$ от $y$.

Пример, который будет использоваться на протяжении первых глав, — **квадратичная потеря**:
$$
L(\hat y, y) = \frac{1}{2}(\hat y - y)^2.
$$
Множитель $1/2$ введён исключительно для удобства дифференцирования: при вычислении производной он сокращается с двойкой, возникающей из показателя степени. Этот приём детально разбирается в главе 3.

Задача обучения, таким образом, сводится к отысканию такого вектора параметров $\theta$, при котором суммарная (или средняя) потеря на обучающей выборке минимальна, а на новых данных — достаточно мала. Формально,
$$
\theta^{*} = \arg\min_{\theta \in \Theta} \; \mathbb{E}_{(x,y) \sim p_{\text{data}}}\bigl[L(f(x; \theta), y)\bigr],
$$
где $p_{\text{data}}$ — (неизвестное) распределение, из которого порождены примеры. Поскольку распределение неизвестно, минимизация ведётся по эмпирической оценке на $\mathcal{D}$.

> **Замечание 1.1.** В общем случае задача (1.1) не имеет аналитического решения: функционал потерь, как правило, невыпуклый, а размерность пространства параметров — большая. Поэтому обучение проводится **итеративно**, методами градиентной оптимизации. Схема итеративного обучения, единая для всех архитектур курса, описана в §1.6.

## 1.3. Конкретизация: три класса задач

Формулировка §1.2 абстрактна. Чтобы придать ей содержательный смысл, рассмотрим три конкретных класса задач, принципиально различающихся структурой входных данных.

**Класс A. Задачи на объектах фиксированной размерности.** Вход $x \in \mathbb{R}^d$ — вектор признаков фиксированной размерности $d$. Пример: распознавание изображений $28 \times 28$ пикселей ($d = 784$); предсказание стоимости объекта недвижимости по набору характеристик ($d$ — число характеристик). В задачах этого класса длина входа постоянна и не зависит от объекта.

**Класс B. Задачи на множествах.** Вход — неупорядоченное множество объектов, например облако точек или набор признаков. Здесь важна только принадлежность элементов, но не их порядок.

**Класс C. Задачи на последовательностях.** Вход — упорядоченная последовательность $x_1, x_2, \dots, x_T$, где $T$ — длина последовательности. Примеры: предложение как последовательность слов; временной ряд как последовательность значений; звук как последовательность кадров; последовательность ДНК как последовательность нуклеотидов.

Курс посвящён **классу C**, и причина этого — две особенности, которые отличают его от классов A и B и делают стандартные методы неприменимыми в исходном виде.

**Особенность 1: переменная длина.** В задачах класса C длина $T$ может быть различной для разных объектов. Одно предложение содержит три слова, другое — двадцать; один временной ряд имеет 30 наблюдений, другой — 3000. Архитектура, разработанная для класса A, где $x \in \mathbb{R}^d$ с фиксированным $d$, не приспособлена к такому варьированию.

**Особенность 2: значимость порядка.** В задачах класса C порядок элементов **несёт информацию**. Фразы «собака укусила человека» и «человек укусил собаку» состоят из одних и тех же слов, но описывают противоположные ситуации. Временной ряд $1, 2, 3$ и $3, 2, 1$ — не одно и то же. Функция, зависящая от множества элементов без учёта порядка, принципиально не способна различить такие примеры.

Эти две особенности определяют требования к архитектуре, пригодной для класса C:

1. она должна обрабатывать входы **переменной длины**;
2. она должна **учитывать порядок** элементов;
3. она должна **обобщать** на последовательности, не встречавшиеся при обучении.

Как мы увидим в §1.4, сети прямого распространения (feedforward networks), к классу которых принадлежит и архитектура $1 \to 1 \to 1$ из главы 4, не удовлетворяют первым двум требованиям одновременно. Именно это обстоятельство мотивирует введение рекуррентных архитектур.

## 1.4. Почему сети прямого распространения непригодны для последовательностей

Рассмотрим сеть прямого распространения общего вида, отображающую вектор фиксированной размерности $x \in \mathbb{R}^d$ в вектор $\hat y \in \mathbb{R}^k$:
$$
\hat y = f(x; \theta), \qquad f: \mathbb{R}^d \to \mathbb{R}^k.
$$
К этому классу относится и архитектура $1 \to 1 \to 1$, детально разобранная в главе 4. Попытаемся применить её к задаче класса C. Возможны два подхода, и оба приводят к непреодолимым затруднениям.

**Подход 1: конкатенация всех элементов в один вектор.** Положим $x = [x_1; x_2; \dots; x_T] \in \mathbb{R}^{dT}$ (при условии, что каждый $x_t$ имеет размерность $d$). Тогда формально сеть прямого распространения применима.

*Недостаток 1.1.* Размерность входа $dT$ зависит от $T$. Сеть, обученная для $T = 10$, не применима к $T = 7$ или $T = 13$ — она ожидает на входе ровно $10d$ чисел. Пришлось бы строить отдельную сеть для каждой длины $T$, что абсурдно.

*Недостаток 1.2.* Порядок элементов не различается в общем случае. Если архитектура симметрична относительно перестановки компонент входа — а для многих стандартных архитектур это так, — то $f([x_1; x_2]) = f([x_2; x_1])$. В задаче класса C такое поведение неприемлемо.

**Подход 2: обработка «по одному элементу» с потерей памяти.** Будем подавать на вход $x_1$, затем $x_2$, и так далее, каждый раз обрабатывая только текущий элемент.

*Недостаток 1.3.* Сеть **не хранит** информацию о предыдущих входах. Обрабатывая $x_t$, она «не знает», что было в $x_1, \dots, x_{t-1}$. Между тем в задаче класса C смысл $x_t$ часто определяется контекстом, то есть именно предыдущими элементами. Фразы «я **не** пошёл в кино» и «я пошёл в кино» отличаются одним словом, но смысл противоположен.

Таким образом, сети прямого распространения **непригодны** для класса C в силу архитектурных ограничений: они либо требуют фиксированной длины входа, либо теряют информацию о контексте, либо не различают порядок. Требуется иной класс архитектур.

## 1.5. Идея рекуррентности

Рекуррентная нейронная сеть решает задачу класса C посредством следующей идеи.

> **Определение 1.3 (рекуррентная нейронная сеть).** Рекуррентной нейронной сетью называется архитектура, в которой на каждом шаге $t \in \{1, \dots, T\}$ вычисляется **скрытое состояние** $h_t$ по правилу
> $$
> h_t = f(x_t,\, h_{t-1};\, \theta),
> $$
> где $f$ — некоторая функция, параметризованная вектором $\theta$, а $h_{t-1}$ — скрытое состояние с предыдущего шага. Начальное состояние $h_0$ задаётся (обычно $h_0 = 0$).

Схематически,
$$
x_1 \to h_1 \to h_2 \to \dots \to h_T,
$$
где каждый переход $h_{t-1} \to h_t$ происходит с использованием $x_t$ и одних и тех же параметров $\theta$.

В этой конструкции заключены две ключевые идеи.

**Идея 1: память.** Скрытое состояние $h_t$ аккумулирует информацию о входных элементах $x_1, \dots, x_t$. Стрелка $h_{t-1} \to h_t$ — это и есть **механизм памяти**: через неё информация передаётся из прошлого в настоящее.

**Идея 2: разделение весов (weight sharing).** Параметры $\theta$ **общие** для всех шагов: при переходе от $h_1$ к $h_2$ используются те же $\theta$, что и при переходе от $h_{T-1}$ к $h_T$. Число параметров не зависит от $T$.

Эти две идеи дают три архитектурных преимущества.

**Преимущество 1: переменная длина.** Функция $f$ применяется по одному элементу за раз; число шагов определяется длиной последовательности. Сеть, обученная на $T = 10$, применима к $T = 7$ или $T = 13$ — с теми же параметрами.

**Преимущество 2: учёт порядка.** Состояние $h_t$ зависит от $x_t$ и $h_{t-1}$, а последнее — от $x_{t-1}$ и $h_{t-2}$, и так далее. Информация о $x_1$ передаётся через всю цепочку до $h_T$. Порядок элементов, таким образом, влияет на конечное состояние.

**Преимущество 3: обобщение.** Поскольку параметры $\theta$ общие для всех шагов, сеть способна применять один и тот же «навык» к разным позициям последовательности. Это ключевое свойство для обобщения на новые последовательности.

> **Замечание 1.2.** Разделение весов — это одновременно и достоинство, и источник принципиальной трудности RNN. Именно из-за того, что на каждом шаге используется одна и та же матрица весов, при обучении возникает произведение матриц вдоль всей цепочки времени. Это произведение ведёт себя экспоненциально по длине последовательности — либо убывая (затухание градиентов), либо возрастая (взрыв градиентов). Формальный анализ этого явления — предмет главы 12; архитектурные решения, устраняющие его, — предмет глав 13–14.

## 1.6. Три фазы обучения

Обучение любой нейронной сети — как прямого распространения, так и рекуррентной — состоит из **трёх фаз**, повторяющихся итеративно. Эта схема будет воспроизводиться в каждой последующей главе курса, и читателю следует уяснить её раз и навсегда.

### 1.6.1. Фаза 1: прямой проход

> **Определение 1.4 (прямой проход).** Прямым проходом (forward pass) называется вычисление предсказания $\hat y = f(x; \theta)$ по заданному входу $x$ и текущим параметрам $\theta$, а также вычисление потери $L(\hat y, y)$.

Прямой проход выполняется **последовательно**: каждый следующий шаг опирается на результат предыдущего. В рекуррентной сети это означает вычисление $h_1, h_2, \dots, h_T$ поочерёдно, поскольку $h_t$ требует $h_{t-1}$.

Результат фазы 1 — значение потери $L$ (одно число), интерпретируемое как мера ошибки на данном примере.

### 1.6.2. Фаза 2: обратный проход

> **Определение 1.5 (обратный проход).** Обратным проходом (backward pass, backpropagation) называется вычисление производных потери по каждому параметру:
> $$
> \frac{\partial L}{\partial \theta_i}, \qquad i = 1, \dots, |\theta|.
> $$
> Эти производные образуют градиент $\nabla_\theta L \in \mathbb{R}^{|\theta|}$.

Вычисление проводится по цепному правилу дифференцирования сложной функции, при этом порядок обхода **противоположен** порядку прямого прохода: от выхода к входу. В рекуррентной сети это означает обход справа налево, от $h_T$ к $h_1$.

Содержательно градиент указывает, **в каком направлении** и **насколько сильно** следует изменить каждый параметр, чтобы потеря уменьшилась. Чем больше модуль производной, тем сильнее влияние параметра на потерю.

Обратный проход — наиболее сложная часть обучения с вычислительной точки зрения, и именно ей посвящена бо́льшая часть курса. В главе 4 (сеть $1 \to 1 \to 1$) это четыре цепочки цепного правила. В главах 5–6 (рекуррентная сеть) — развёртывание суммы по путям и её свёртка в рекуррентное соотношение. В главах 7–9 — переход к векторно-матричной записи. В главах 10–11 — расширения на стек слоёв и двунаправленность. В главах 13–14 — рекуррентные соотношения для LSTM и GRU.

### 1.6.3. Фаза 3: обновление параметров

> **Определение 1.6 (обновление параметров).** Обновлением параметров называется сдвиг каждого параметра в направлении, противоположном градиенту:
> $$
> \theta_i \leftarrow \theta_i - \eta\,\frac{\partial L}{\partial \theta_i},
> $$
> где $\eta > 0$ — **шаг обучения** (learning rate).

Знак «минус» принципиален: градиент указывает направление **наискорейшего роста** функции; чтобы уменьшить потерю, нужно двигаться в противоположном направлении. Описанный метод называется **градиентным спуском** (gradient descent).

Величина $\eta$ называется шагом обучения. Слишком малый шаг приводит к медленной сходимости; слишком большой — к колебаниям и возможному расхождению. Типичные значения: $10^{-1}, 10^{-2}, 10^{-3}$.

### 1.6.4. Итеративная природа обучения

Три фазы повторяются многократно. Один цикл «прямой проход → обратный проход → обновление» называется **шагом обучения** (training step). Число шагов может составлять тысячи, десятки тысяч или миллионы — в зависимости от задачи и объёма данных.

Существенно, что параметры $\theta$ **не вычисляются** аналитически, а **итеративно подстраиваются**. Именно поэтому необходимы все три фазы: прямой проход даёт ошибку, обратный — направление её уменьшения, обновление — сам шаг.

### 1.6.5. Сводная таблица

Схема трёх фаз единообразна для всех архитектур курса; меняется лишь структура прямого прохода.

| Глава | Прямой проход | Обратный проход: ключевое отличие |
|---|---|---|
| 4 | Сеть $1 \to 1 \to 1$ без памяти | 4 цепочки цепного правила |
| 5 | Та же сеть, но с памятью, $T=2$ | Пути в развёрнутой сети |
| 6 | То же, $T=3$ | Три пути для $a_1$, первое $w_h^2$ |
| 7 | То же, общее $T$ | Рекуррентное соотношение BPTT |
| 8 | Векторы $h_t \in \mathbb{R}^N$, $T=2$ | Якобианы, транспонирование |
| 9 | Общее $T$, векторы | Матричный BPTT |
| 10 | Вектор выхода $\hat y_t \in \mathbb{R}^K$ | $W_y^\top \delta^y_t$ |
| 11 | Стек слоёв | Рекурсия по времени и по слоям |
| 12 | Две рекурсии (прямая/обратная) | Две дельты на шаг |
| 13 | — | Формальный анализ затухания |
| 14 | LSTM: 3 вентиля | 6 дельт, ключевая формула для $c_t$ |
| 15 | GRU: 2 вентиля | 4 дельты |

Таблица служит ориентиром на протяжении всего курса: где бы читатель ни находился, три фазы остаются неизменными.

## 1.7. Граф вычислений

Прежде чем переходить к дифференцированию, полезно представить модель в виде **графа вычислений**.

> **Определение 1.7 (граф вычислений).** Графом вычислений называется ориентированный ациклический граф, вершины которого соответствуют переменным модели (скалярам, векторам, матрицам), а дуги — операциям, преобразующим одни переменные в другие.

Пример для сети $1 \to 1 \to 1$ из главы 4:
$$
x \xrightarrow{\;w_1 x + b_1\;} z_1 \xrightarrow{\;\tanh\;} a_1 \xrightarrow{\;w_2 a_1 + b_2\;} z_2 \xrightarrow{\;\sigma\;} \hat y \xrightarrow{\;\frac{1}{2}(\hat y - y)^2\;} L.
$$
Граф содержит шесть вершин и пять дуг.

**Прямой проход** по графу — движение **слева направо**, от входа к выходу: на каждой вершине вычисляется значение, передаваемое по дугам.

**Обратный проход** — движение **справа налево**, от потери к параметрам. На каждой вершине вычисляется произведение пришедшего градиента на **локальную производную** данной вершины; на каждом разветвлении (когда от вершины отходит несколько дуг) градиенты от разных ветвей **суммируются**.

> **Замечание 1.3.** Формально правило обратного прохода по графу вычислений есть не что иное, как цепное правило дифференцирования сложной функции, записанное в алгоритмической форме. Настоящее замечание — ключевое для понимания всего курса: все формулы вида «дельта слоя», «путь в развёрнутой сети», «рекуррентное соотношение BPTT» — суть применения этого правила к конкретным графам.

По мере продвижения по курсу граф усложняется, но его природа не меняется:

| Глава | Что добавляется в граф |
|---|---|
| 4 | Базовая сеть $1 \to 1 \to 1$ |
| 5 | Дуга «назад по времени»: $h_1 \to a_2$ |
| 7 | Многоточие $\cdots$ между $h_2$ и $h_{T-1}$ |
| 8 | Векторные вершины, матричные дуги |
| 11 | Вертикальные дуги между слоями |
| 12 | Две параллельные цепочки |
| 14 | Вентили $f_t, i_t, o_t$ между $h_t$ и $c_t$ |

## 1.8. Архитектура курса

Курс построен по принципу **инкрементального усложнения**: каждая глава вводит одну новую идею, опираясь на предыдущие. Прогрессия следующая.

**Глава 4.** Сеть $1 \to 1 \to 1$ — простейшая сеть прямого распространения с двумя слоями. Здесь закладывается базовая схема трёх фаз и цепного правила.

**Главы 5–7.** Рекуррентная сеть с одним нейроном, но с памятью. Три уровня детализации:
- глава 5: $T = 2$ — все пути выписаны в явном виде;
- глава 6: $T = 3$ — показано, как растёт число путей и появляется произведение $w_h^2$;
- глава 7: общее $T$ — выведено рекуррентное соотношение BPTT.

**Главы 8–10.** Переход к векторам и матрицам: $N$ нейронов вместо одного, $K$ выходов вместо одного.

**Главы 11–13.** Расширения архитектуры: стек слоёв, двунаправленность, формальный анализ затухания градиентов.

**Главы 14–15.** Архитектуры, решающие проблему затухания: LSTM и GRU.

> **Обоснование прогрессии.** Каждый шаг мотивирован предыдущим.
> - $T = 2$ вводится после сети без памяти, потому что при $T = 1$ памяти нет, а при $T = 2$ появляется ровно одна дуга «назад по времени» — минимальный пример рекуррентности.
> - $T = 3$ вводится после $T = 2$, потому что при $T = 3$ число путей от $a_1$ к $L$ равно трём (а не двум), и впервые появляется произведение $w_h^2$ — именно с этого момента становится очевидной необходимость рекуррентной формулы.
> - Общее $T$ вводится после $T = 3$, потому что при $T = 100$ число путей равно $5050$ — выписать их явно невозможно, требуется свёртка в рекурсию.
> - Векторы вводятся после скаляров, потому что при $N = 1$ все формулы понятны и обозримы; матрицы добавляют новую нотацию, но не новую идею.
> - LSTM вводится после анализа затухания, потому что LSTM — это **конкретный ответ** на конкретную проблему, сформулированную в главе 13. Без этой главы LSTM выглядит как набор вентилей, а не как точечное исправление формулы BPTT.

## 1.9. Методическая структура глав

Каждая глава курса содержит **четыре уровня** изложения, согласованных между собой.

**Уровень 1: интуиция.** Словесное описание идеи без формул. Например: «у переменной $h_1$ два потребителя, поэтому в её производной два слагаемых».

**Уровень 2: формулы.** Полные выкладки с выписанным каждым множителем. Например:
$$
\delta^a_1 = \underbrace{\delta^y_1\,w_y\,(1-h_1^2)}_{\text{путь A}} + \underbrace{\delta^a_2\,w_h\,(1-h_1^2)}_{\text{путь B}}.
$$

**Уровень 3: численный пример.** Полная подстановка чисел — от прямого прохода до обновления. Например, для главы 5: $a_1 = 0.6$, $h_1 \approx 0.53705$, $\hat y_1 \approx 0.55716$, $L \approx 0.25919$. Численный пример служит средством самопроверки: если расчёт читателя расходится с приведённым, значит, в вычислениях допущена ошибка.

**Уровень 4: программная реализация.** Полный код на Python (NumPy), воспроизводящий в точности числа уровня 3.

**Рекомендации по чтению.** При первом прочтении достаточно уровней 1 и 3; при втором — уровней 1 и 2; при третьем — всех четырёх. Все уровни **согласованы**: числа уровня 3 совпадают с результатом кода уровня 4; расхождение свидетельствует об ошибке в формулах уровня 2.

## 1.10. Предварительные требования

**Математика.**
- **Обязательно:** производные функций одного переменного, цепное правило дифференцирования.
- **Желательно:** частные производные, понятие градиента.
- **Не требуется:** матричное исчисление, теория вероятностей, линейная алгебра в полном объёме. Минимально необходимый аппарат изложен в главе 2.

**Программирование.**
- **Обязательно:** Python на уровне переменных, циклов, функций, списков.
- **Желательно:** NumPy на уровне массивов и матричного умножения.
- **Не требуется:** PyTorch, TensorFlow, Keras.

> **Замечание 1.4.** Весь код в курсе реализован на NumPy «с нуля». Это методологическое решение: использование готовых фреймворков скрывает от читателя внутреннее устройство BPTT. Когда читатель пишет `dh[t] = (dy[t] * wy + Wh.T @ dh_next) * (1 - h[t]**2)`, он видит каждую операцию, и формула перестаёт быть абстракцией.

**Машинное обучение.** Не требуется. Курс изложен «с нуля».

## 1.11. Результаты обучения

По завершении курса читатель сможет:

1. **Вывести BPTT** самостоятельно — не принять его на веру, а получить из цепного правила для произвольного $T$.

2. **Объяснить затухание градиентов.** В формуле для $\delta^a_t$ присутствует произведение матриц $W_h$; показать, что это произведение ведёт себя экспоненциально по $T$ — задача главы 13.

3. **Понять, как LSTM и GRU решают проблему затухания.** Ячейка $c_t$ в LSTM и «прямой путь» $h_t$ в GRU заменяют произведение матриц $W_h$ на произведение диагональных вентилей, которые обучаемы и могут быть близки к единице.

4. **Реализовать** простую RNN, LSTM и GRU на NumPy.

5. **Усвоить общую схему** нейронных сетей: три фазы, граф вычислений, обратный проход по цепному правилу. Эта схема применима не только к RNN, но и к любым другим архитектурам.

## 1.12. Границы курса

Курс сознательно ограничен одной темой: **BPTT и её эволюция от простой RNN до LSTM и GRU**. За его пределами остаются:

- **свёрточные сети (CNN)** — отдельная тема, идея свёртки не связана с рекуррентностью;
- **трансформеры и механизм внимания (attention)** — следующий этап развития рекуррентных архитектур; структура BPTT из глав 5–15 применима и к ним, но их детальный разбор выходит за рамки курса;
- **обучение с подкреплением**;
- **генеративные модели** (VAE, GAN, диффузионные модели);
- **методы оптимизации** (Adam, RMSProp) — используется простой градиентный спуск, достаточный для понимания BPTT;
- **регуляризация** (dropout, weight decay) — не меняет структуру обратного прохода;
- **реализация на промышленных фреймворках**.

Такое ограничение продиктовано соображениями методической ясности: расширение до CNN, трансформеров и оптимизаторов увеличило бы объём в несколько раз и размыло бы фокус. Освоив BPTT, читатель сможет самостоятельно перейти к любой другой архитектуре — структура обратного прохода в них та же.

## 1.13. Резюме главы

Сформулируем три ключевых положения, которые следует запомнить перед переходом к главе 2.

**Положение 1.** Обучение любой нейронной сети состоит из трёх фаз — прямого прохода, обратного прохода, обновления параметров. Схема единообразна; меняется лишь структура прямого прохода. Все главы курса — вариации на эту тему.

**Положение 2.** Обратный проход есть алгоритмическая форма цепного правила, применяемого к графу вычислений: на каждой вершине — умножение на локальную производную, на каждом разветвлении — суммирование вкладов ветвей.

**Положение 3.** Курс построен по принципу инкрементального усложнения: от сети без памяти ($1 \to 1 \to 1$) через рекуррентную сеть с одним нейроном ($T = 2$, $T = 3$, общее $T$) к векторным RNN, стеку слоёв, двунаправленности, анализу затухания, LSTM и GRU. Каждое усложнение вводится только тогда, когда предыдущее понято.

> **Главный тезис курса.** Вся рекуррентная нейронная сеть — от простой RNN до LSTM — устроена по одной схеме: прямой проход по графу вычислений, обратный проход по цепному правилу, обновление параметров. Меняется только структура прямого прохода. Поняв эту схему один раз, читатель сможет разобрать любую рекуррентную архитектуру.


# Глава 2. Математический фундамент

## 2.1. Предмет и структура главы

Настоящая глава излагает математический аппарат, необходимый для понимания всех последующих глав. Её задача — дать читателю **ровно тот инструментарий**, который понадобится при выводе прямого и обратного проходов, и **не более того**. Каждый раздел сопровождается указанием, в какой главе курса соответствующий аппарат применяется; это позволяет читателю видеть, что изложение не является самоцелью.

Глава построена по принципу «от простого к сложному»: от производной функции одного переменного — к цепному правилу, от него — к частным производным и градиенту, от них — к якобиану и матричному исчислению, и наконец — к нормам, которые понадобятся при анализе затухания градиентов в главе 13.

Структура главы:
- §2.2 — функции и композиция;
- §2.3 — производная функции одного переменного;
- §2.4 — цепное правило для одной ветви;
- §2.5 — цепное правило для нескольких ветвей (**ключевой раздел**);
- §2.6 — частные производные;
- §2.7 — градиент;
- §2.8 — вектор-функции и якобиан;
- §2.9 — транспонирование в обратном проходе (**ключевой раздел**);
- §2.10 — внешнее произведение;
- §2.11 — поэлементные операции и произведение Адамара;
- §2.12 — нормы;
- §2.13 — правило «сумма по путям»;
- §2.14 — сводная таблица;
- §2.15 — типичные ошибки;
- §2.16 — резюме.

Разделы §2.5 и §2.9 отмечены как ключевые: без них невозможно понять структуру формул обратного прохода в главах 5–15.

## 2.2. Функции и композиция

### 2.2.1. Определение функции

> **Определение 2.1 (функция).** Функцией $f: \mathcal{X} \to \mathcal{Y}$ называется правило, сопоставляющее каждому элементу $x \in \mathcal{X}$ единственный элемент $y = f(x) \in \mathcal{Y}$. Множество $\mathcal{X}$ называется **областью определения**, $\mathcal{Y}$ — **областью значений**.

В курсе будут встречаться функции следующих видов:
- $f: \mathbb{R} \to \mathbb{R}$ — скалярная функция скалярного аргумента;
- $f: \mathbb{R}^n \to \mathbb{R}$ — скалярная функция векторного аргумента;
- $f: \mathbb{R}^n \to \mathbb{R}^m$ — вектор-функция.

### 2.2.2. Композиция функций

> **Определение 2.2 (композиция).** Композицией функций $f$ и $g$ называется функция $h = g \circ f$, определяемая равенством
> $$
> h(x) = g\bigl(f(x)\bigr).
> $$
> Иными словами, сначала применяется $f$, затем к результату — $g$.

**Пример 2.1.** Пусть $f(x) = x^2$, $g(u) = \sin u$. Тогда
$$
h(x) = g(f(x)) = \sin(x^2).
$$

> **Замечание 2.1.** Нейронная сеть, сколь бы сложной она ни казалась, есть **композиция** элементарных функций: линейных преобразований и нелинейных активаций. Сеть $1 \to 1 \to 1$ из главы 4 — это композиция четырёх функций:
> $$
> \hat y = \sigma\bigl(w_2 \cdot \tanh(w_1 x + b_1) + b_2\bigr).
> $$
> Рекуррентная сеть из главы 7 — композиция $4T$ функций, применяемых последовательно. Именно поэтому понимание композиции и правил её дифференцирования является предпосылкой для всего дальнейшего.

### 2.2.3. Граф вычислений

Композицию удобно изображать в виде графа.

> **Определение 2.3 (граф вычислений).** Графом вычислений функции $h = g \circ f$ называется ориентированный граф с вершинами $x, u = f(x), y = g(u)$ и дугами $x \to u$, $u \to y$.

**Пример 2.2.** Для $h(x) = \sin(x^2)$ граф имеет вид
$$
x \xrightarrow{\;(\cdot)^2\;} u \xrightarrow{\;\sin\;} y.
$$

Для сети $1 \to 1 \to 1$ (глава 4) граф содержит шесть вершин и пять дуг (см. §1.7).

> **Замечание 2.2.** Понятие графа вычислений будет систематически использоваться начиная с главы 4. Прямой проход — движение по графу слева направо; обратный проход — справа налево. Понимание графа позволяет «увидеть» структуру формул обратного прохода до того, как они выписаны аналитически.

## 2.3. Производная функции одного переменного

### 2.3.1. Определение

> **Определение 2.4 (производная).** Производной функции $f: \mathbb{R} \to \mathbb{R}$ в точке $x$ называется предел
> $$
> f'(x) = \lim_{\Delta x \to 0} \frac{f(x + \Delta x) - f(x)}{\Delta x},
> $$
> если этот предел существует. Функция, имеющая производную в точке $x$, называется **дифференцируемой** в этой точке.

Содержательно $f'(x)$ — это **скорость изменения** $f$ при изменении аргумента; геометрически — тангенс угла наклона касательной к графику $f$ в точке $x$.

### 2.3.2. Таблица производных

Приведём производные функций, которые встретятся в курсе.

| Функция $f(x)$ | Производная $f'(x)$ |
|---|---|
| $x^n$ | $n x^{n-1}$ |
| $e^x$ | $e^x$ |
| $\ln x$ | $1/x$ |
| $\sin x$ | $\cos x$ |
| $\cos x$ | $-\sin x$ |
| $\sigma(x) = \dfrac{1}{1 + e^{-x}}$ | $\sigma(x)\bigl(1 - \sigma(x)\bigr)$ |
| $\tanh(x)$ | $1 - \tanh^2(x)$ |
| $\dfrac{1}{2}(x - y)^2$ (по $x$) | $x - y$ |

Производные сигмоиды и гиперболического тангенса заслуживают отдельного пояснения, поскольку именно эти функции используются в качестве активаций в нейронных сетях.

**Сигмоида.** Пусть $\sigma(x) = (1 + e^{-x})^{-1}$. Тогда
$$
\sigma'(x) = -(1 + e^{-x})^{-2} \cdot (-e^{-x}) = \frac{e^{-x}}{(1 + e^{-x})^2} = \sigma(x) \cdot \frac{e^{-x}}{1 + e^{-x}} = \sigma(x)\bigl(1 - \sigma(x)\bigr).
$$
Здесь использовано тождество $\dfrac{e^{-x}}{1 + e^{-x}} = 1 - \dfrac{1}{1 + e^{-x}} = 1 - \sigma(x)$.

**Гиперболический тангенс.** Пусть $\tanh(x) = \dfrac{e^x - e^{-x}}{e^x + e^{-x}}$. Тогда
$$
\tanh'(x) = 1 - \tanh^2(x).
$$
Это тождество будет систематически использоваться начиная с главы 5, поскольку позволяет вычислить производную $\tanh$ через уже вычисленное значение самой функции — без повторного вычисления экспонент.

> **Замечание 2.3.** Удобство сигмоиды и $\tanh$ в нейронных сетях состоит именно в том, что их производные выражаются через значения самих функций. Это позволяет при обратном проходе избежать повторных вычислений.

### 2.3.3. Правила дифференцирования

Приведём без доказательства правила дифференцирования, которые будут использоваться постоянно.

**Линейность.** Для любых функций $f, g$ и констант $\alpha, \beta$:
$$
(\alpha f + \beta g)' = \alpha f' + \beta g'.
$$

**Произведение:**
$$
(f \cdot g)' = f' g + f g'.
$$

**Частное:**
$$
\left(\frac{f}{g}\right)' = \frac{f' g - f g'}{g^2}.
$$

**Сложная функция (цепное правило):** см. §2.4.

## 2.4. Цепное правило для одной ветви

Это первая из двух формул, на которых держится весь обратный проход.

> **Теорема 2.1 (цепное правило, одна ветвь).** Пусть $y = g(u)$, $u = f(x)$, обе функции дифференцируемы. Тогда
> $$
> \frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx} = g'(f(x)) \cdot f'(x).
> $$

**Мнемоника.** «Производная внешней функции, умноженная на производную внутренней». Геометрически это означает, что скорости изменения перемножаются вдоль цепочки.

**Пример 2.3.** Пусть $y = \sin(x^2)$, $x = 1.2$. Тогда
$$
\frac{dy}{dx} = \cos(x^2) \cdot 2x = \cos(1.44) \cdot 2.4 \approx 0.1304 \cdot 2.4 \approx 0.3129.
$$
Проверка через численное дифференцирование:
$$
\frac{\sin((1.2 + 0.001)^2) - \sin(1.2^2)}{0.001} \approx 0.3129.
$$

**Пример 2.4 (для главы 4).** Пусть $y = \sigma(z)$, $z = w \cdot a$. Тогда
$$
\frac{dy}{dw} = \sigma'(z) \cdot a.
$$
В главе 4 эта формула даст $\partial L/\partial w_2 = \delta_2 \cdot a_1$ (с учётом того, что $\delta_2 = \partial L/\partial z_2 = (\hat y - y)\sigma'(z_2)$).

> **Замечание 2.4.** Цепное правило для одной ветви — это случай, когда от каждой переменной к следующей идёт **ровно один** путь. В сети без разветвлений (например, перцептрон или сеть $1 \to 1 \to 1$, рассматриваемая как цепочка) этого достаточно. Но в рекуррентной сети разветвления неизбежны: одна и та же переменная $h_t$ используется в двух местах (выход шага $t$ и следующий шаг $t+1$). Поэтому необходимо обобщение — §2.5.

## 2.5. Цепное правило для нескольких ветвей

Этот раздел — **ключевой для всего курса**. Без него невозможно понять, откуда в формулах обратного прохода появляются суммы.

### 2.5.1. Формулировка

> **Теорема 2.2 (цепное правило, несколько ветвей).** Пусть $z = g(y_1, y_2, \dots, y_n)$ — функция $n$ переменных, и каждая $y_i = f_i(x)$ — функция от $x$. Тогда
> $$
> \frac{dz}{dx} = \sum_{i=1}^{n} \frac{\partial z}{\partial y_i} \cdot \frac{dy_i}{dx}.
> $$

Содержательно: если от $x$ к $z$ ведёт **несколько путей** (по одному через каждую переменную $y_i$), полная производная есть **сумма** вкладов по всем путям.

**Мнемоника.** «Сумма по всем путям от $x$ до $z$, в каждом пути — произведение локальных производных вдоль пути».

### 2.5.2. Пример с числами

**Пример 2.5.** Пусть $z = y_1 \cdot y_2$, $y_1 = x^2$, $y_2 = \sin x$, $x = 1.2$.

По теореме 2.2:
$$
\frac{dz}{dx} = \frac{\partial z}{\partial y_1} \cdot \frac{dy_1}{dx} + \frac{\partial z}{\partial y_2} \cdot \frac{dy_2}{dx} = y_2 \cdot 2x + y_1 \cdot \cos x.
$$

Подставим числа:
- $y_1 = x^2 = 1.44$;
- $y_2 = \sin(1.2) \approx 0.9320$;
- $\cos(1.2) \approx 0.3624$.

$$
\frac{dz}{dx} \approx 0.9320 \cdot 2.4 + 1.44 \cdot 0.3624 \approx 2.2369 + 0.5218 \approx 2.7587.
$$

**Пример 2.6 (простая проверка).** Пусть $z = y_1 + y_2$, $y_1 = w^2$, $y_2 = \sin w$. Тогда
$$
\frac{dz}{dw} = 1 \cdot 2w + 1 \cdot \cos w = 2w + \cos w.
$$
Действительно, $z = w^2 + \sin w$, и производная равна $2w + \cos w$. Совпадение подтверждает теорему.

### 2.5.3. Обобщение на произвольную структуру графа

Формула §2.5.1 обобщается на произвольный ациклический граф вычислений.

> **Правило (обратный проход по графу).** Пусть $L$ — скалярная функция вершин графа вычислений. Для каждой вершины $v$ определим **градиент** $\partial L/\partial v$. Тогда:
> 1. на **каждой дуге** $u \to v$ градиент «течёт» от $v$ к $u$ с множителем $\partial v/\partial u$;
> 2. в каждой **вершине** $u$ вклады от всех **входящих** дуг (в которые градиент «течёт» от последующих вершин) **суммируются**.

Это и есть алгоритм обратного распространения ошибки в общем виде. Он будет применяться к конкретным графам в главах 4–15.

### 2.5.4. Связь с главами курса

| Глава | Сколько путей от $a_1$ до $L$ | Как это выглядит в формуле |
|---|---|---|
| 4 | 2 (через $\hat y_1$ и через $a_2$) | $\delta^a_1 = (\delta^y_1 w_y + \delta^a_2 w_h)(1-h_1^2)$ |
| 5 | 3 (через $\hat y_1$, $a_2$, $a_3$) | добавляется путь через $a_3$, появляется $w_h^2$ |
| 6 | $T$ (общий случай) | суммирование сворачивается в рекуррентное соотношение |
| 13 | 2 (через $h_t$ и через $c_{t+1}$) | $\delta^c_t = \delta^h_t o_t(1-\tanh^2 c_t) + f_{t+1}\delta^c_{t+1}$ |

> **Замечание 2.5.** Теорема 2.2 — это **главная формула курса**. Все «дельты», «пути» и «рекуррентные соотношения» в последующих главах суть различные формы записи суммы по путям из этой теоремы. Когда в главе 7 будет выведена формула $\delta^a_t = (\delta^y_t w_y + W_h^\top \delta^a_{t+1}) \odot (1 - h_t^2)$, читатель должен видеть в ней **два слагаемых — два пути** из теоремы 2.2.

## 2.6. Частные производные

### 2.6.1. Определение

> **Определение 2.5 (частная производная).** Пусть $f: \mathbb{R}^n \to \mathbb{R}$. Частной производной $f$ по переменной $x_i$ в точке $(x_1, \dots, x_n)$ называется
> $$
> \frac{\partial f}{\partial x_i} = \lim_{\Delta \to 0} \frac{f(x_1, \dots, x_i + \Delta, \dots, x_n) - f(x_1, \dots, x_n)}{\Delta}.
> $$
> Иными словами, дифференцируем по $x_i$, считая **все остальные переменные константами**.

### 2.6.2. Примеры

**Пример 2.7.** Пусть $f(x, y) = x^2 y + \sin y$. Тогда
$$
\frac{\partial f}{\partial x} = 2xy, \qquad \frac{\partial f}{\partial y} = x^2 + \cos y.
$$
В первом случае $\sin y$ дифференцируется по $x$ как константа (даёт 0); во втором случае $x^2 y$ дифференцируется по $y$ как линейная функция (даёт $x^2$).

**Пример 2.8 (для главы 4).** В сети $1 \to 1 \to 1$ преактивация $z_1 = w_1 x + b_1$ зависит от двух параметров — $w_1$ и $b_1$:
$$
\frac{\partial z_1}{\partial w_1} = x, \qquad \frac{\partial z_1}{\partial b_1} = 1.
$$
Именно эти частные производные появятся в §4.5 как «локальные производные» уравнения $z_1$.

### 2.6.3. Связь с полной производной

Если $f = f(y_1, \dots, y_n)$, и каждая $y_i = y_i(x)$, то полная производная по $x$ (в одномерном случае) выражается через частные производные формулой §2.5.1. Это — проявление цепного правила для нескольких ветвей.

## 2.7. Градиент

### 2.7.1. Определение

> **Определение 2.6 (градиент).** Градиентом скалярной функции $f: \mathbb{R}^n \to \mathbb{R}$ в точке $x \in \mathbb{R}^n$ называется вектор частных производных:
> $$
> \nabla f(x) = \left(\frac{\partial f}{\partial x_1}(x), \frac{\partial f}{\partial x_2}(x), \dots, \frac{\partial f}{\partial x_n}(x)\right)^\top \in \mathbb{R}^n.
> $$

### 2.7.2. Геометрический смысл

Градиент указывает направление **наискорейшего возрастания** $f$ в точке $x$. Его длина $\|\nabla f\|$ равна скорости этого возрастания.

Отсюда — правило градиентного спуска: чтобы уменьшить $f$, нужно двигаться в направлении $-\nabla f$:
$$
x \leftarrow x - \eta \nabla f(x).
$$

### 2.7.3. Применение в курсе

Правило обновления параметров из §1.6.3 главы 1 — это в точности правило градиентного спуска:
$$
\theta_i \leftarrow \theta_i - \eta \frac{\partial L}{\partial \theta_i}.
$$
В главе 4 впервые появятся производные $\partial L/\partial w_1, \partial L/\partial b_1, \partial L/\partial w_2, \partial L/\partial b_2$, которые образуют четырёхмерный градиент. В главе 8 — градиенты по матрицам (см. §2.10).

## 2.8. Вектор-функции и якобиан

### 2.8.1. Определение

> **Определение 2.7 (вектор-функция).** Вектор-функцией называется отображение $f: \mathbb{R}^n \to \mathbb{R}^m$, сопоставляющее вектору $x \in \mathbb{R}^n$ вектор $y = f(x) \in \mathbb{R}^m$, компоненты которого $y_i = f_i(x_1, \dots, x_n)$.

> **Определение 2.8 (якобиан).** Матрицей Якоби (якобианом) вектор-функции $f$ в точке $x$ называется матрица
> $$
> J_f(x) = \begin{pmatrix} \dfrac{\partial f_1}{\partial x_1} & \cdots & \dfrac{\partial f_1}{\partial x_n} \\[6pt] \vdots & \ddots & \vdots \\[6pt] \dfrac{\partial f_m}{\partial x_1} & \cdots & \dfrac{\partial f_m}{\partial x_n} \end{pmatrix} \in \mathbb{R}^{m \times n}.
> $$
> Её элемент $(i, j)$ есть $J_{ij} = \partial f_i / \partial x_j$.

### 2.8.2. Пример: линейное отображение

**Пример 2.9.** Пусть $y = W x$, где $W \in \mathbb{R}^{m \times n}$, $x \in \mathbb{R}^n$. Тогда $y_i = \sum_{j} W_{ij} x_j$, и
$$
\frac{\partial y_i}{\partial x_j} = W_{ij}.
$$
Значит, $J_f = W$ — якобиан линейного отображения совпадает с его матрицей.

Этот пример — центральный для глав 8–10: преактивация $a_t = W_x x_t + W_h h_{t-1} + b_h$ — это композиция двух линейных отображений, и её якобианы по $x_t$ и по $h_{t-1}$ равны $W_x$ и $W_h$ соответственно.

### 2.8.3. Якобиан композиции

> **Теорема 2.3 (якобиан композиции).** Если $y = f(x)$ и $z = g(y)$, то
> $$
> J_{g \circ f}(x) = J_g(y) \cdot J_f(x).
> $$
> Иными словами, якобиан композиции равен **произведению якобианов**.

Это — матричная форма цепного правила. Для скалярных функций она сводится к обычному цепному правилу.

### 2.8.4. Связь с главами курса

В главах 8–15 рекуррентная сеть будет записываться в векторно-матричной форме:
$$
a_t = W_x x_t + W_h h_{t-1} + b_h, \qquad h_t = \tanh(a_t).
$$
Якобиан отображения $h_{t-1} \mapsto a_t$ равен $W_h$; якобиан отображения $a_t \mapsto h_t$ диагональный (см. §2.11). Произведение этих якобианов — «локальная производная» одного шага — и есть то, что войдёт в формулы обратного прохода.

## 2.9. Транспонирование в обратном проходе

Этот раздел — **ключевой** для глав 8–15. Его непонимание — источник большинства ошибок при реализации BPTT.

### 2.9.1. Постановка задачи

Пусть $y = W x$, где $W \in \mathbb{R}^{m \times n}$, $x \in \mathbb{R}^n$. Прямой проход: $x \mapsto y$, размерности $n \to m$.

Предположим, что при обратном проходе у нас уже есть градиент $\partial L/\partial y \in \mathbb{R}^m$ — вектор-столбец. Требуется найти $\partial L/\partial x \in \mathbb{R}^n$.

### 2.9.2. Формула

> **Теорема 2.4 (обратный проход через линейное отображение).** Если $y = W x$, то
> $$
> \frac{\partial L}{\partial x} = W^\top \frac{\partial L}{\partial y}.
> $$

### 2.9.3. Доказательство

По определению частной производной и цепному правилу для нескольких ветвей:
$$
\frac{\partial L}{\partial x_j} = \sum_{i=1}^{m} \frac{\partial L}{\partial y_i} \cdot \frac{\partial y_i}{\partial x_j} = \sum_{i=1}^{m} \frac{\partial L}{\partial y_i} \, W_{ij}.
$$
Сумма $\sum_i \frac{\partial L}{\partial y_i} W_{ij}$ есть $j$-я компонента вектора $W^\top \nabla_y L$. Значит, $\partial L/\partial x = W^\top \nabla_y L$. $\blacksquare$

### 2.9.4. Проверка размерностей

| Величина | Размерность |
|---|---|
| $W$ | $m \times n$ |
| $x$ | $n$ (столбец) |
| $y = W x$ | $m$ |
| $\partial L/\partial y$ | $m$ |
| $W^\top$ | $n \times m$ |
| $W^\top \partial L/\partial y$ | $n$ |
| $\partial L/\partial x$ | $n$ |

Размерности совпадают. Это — стандартная проверка корректности формулы.

### 2.9.5. Пример с числами

**Пример 2.10.** Пусть
$$
W = \begin{pmatrix} 0.2 & -0.1 \\ 0.4 & 0.3 \end{pmatrix}, \quad x = \begin{pmatrix} 1 \\ 2 \end{pmatrix}.
$$
Тогда
$$
y = W x = \begin{pmatrix} 0.2 \cdot 1 - 0.1 \cdot 2 \\ 0.4 \cdot 1 + 0.3 \cdot 2 \end{pmatrix} = \begin{pmatrix} 0 \\ 1 \end{pmatrix}.
$$
Если $\partial L/\partial y = (0.5, -0.3)^\top$, то
$$
\frac{\partial L}{\partial x} = W^\top \begin{pmatrix} 0.5 \\ -0.3 \end{pmatrix} = \begin{pmatrix} 0.2 & 0.4 \\ -0.1 & 0.3 \end{pmatrix}\begin{pmatrix} 0.5 \\ -0.3 \end{pmatrix} = \begin{pmatrix} 0.1 - 0.12 \\ -0.05 - 0.09 \end{pmatrix} = \begin{pmatrix} -0.02 \\ -0.14 \end{pmatrix}.
$$

### 2.9.6. Мнемоника и типичная ошибка

> **Мнемоника.** «Прямой проход — $W$; обратный — $W^\top$».

**Типичная ошибка.** Написать $\partial L/\partial x = W \cdot \partial L/\partial y$ — это неверно по размерности ($W$ — $m \times n$, $\partial L/\partial y$ — $m$, произведение не определено). Или написать $\partial L/\partial x = \partial L/\partial y \cdot W$ — это тоже неверно ($m \times m \times n$, произведение не даёт $n$-вектор). Единственный корректный вариант — $W^\top \partial L/\partial y$.

### 2.9.7. Связь с главами курса

Формула $W^\top$ появляется в каждой главе от 8-й:

| Глава | Формула | Где $W^\top$ |
|---|---|---|
| 8 | $\delta^a_1 = (\delta^y_1 w_y + W_h^\top \delta^a_2) \odot (1 - h_1 \odot h_1)$ | $W_h^\top \delta^a_2$ |
| 9 | $W_y^\top \delta^y_t$ | обратный проход через выходной слой |
| 11 | $W_x^{(\ell+1)\top} \delta^{a,(\ell+1)}_t$ | обратный проход между слоями |
| 12 | $\overrightarrow{W}_y^\top \delta^y_t$ | обратный проход прямой рекурсии |
| 14 | $W_f^\top \delta^{af}_{t+1}, \dots$ | обратный проход через вентили LSTM |
| 15 | $W_z^\top \delta^{az}_{t+1}, \dots$ | обратный проход через вентили GRU |

**Без усвоения этого раздела дальнейшее изложение невозможно.**

## 2.10. Внешнее произведение

### 2.10.1. Определение

> **Определение 2.9 (внешнее произведение).** Внешним произведением векторов $u \in \mathbb{R}^m$ и $v \in \mathbb{R}^n$ называется матрица
> $$
> u v^\top \in \mathbb{R}^{m \times n}, \qquad (u v^\top)_{ij} = u_i v_j.
> $$

В отличие от скалярного произведения $u^\top v \in \mathbb{R}$ (свёртка в число), внешнее произведение даёт **матрицу**.

### 2.10.2. Пример

**Пример 2.11.** Пусть $u = (1, 2, 3)^\top$, $v = (4, 5)^\top$. Тогда
$$
u v^\top = \begin{pmatrix} 1 \\ 2 \\ 3 \end{pmatrix}\begin{pmatrix} 4 & 5 \end{pmatrix} = \begin{pmatrix} 4 & 5 \\ 8 & 10 \\ 12 & 15 \end{pmatrix} \in \mathbb{R}^{3 \times 2}.
$$

### 2.10.3. Градиент по матрице

> **Теорема 2.5 (обратный проход через матрицу).** Если $y = W x$, где $W \in \mathbb{R}^{m \times n}$, $x \in \mathbb{R}^n$, $y \in \mathbb{R}^m$, то
> $$
> \frac{\partial L}{\partial W} = \frac{\partial L}{\partial y} \, x^\top \in \mathbb{R}^{m \times n}.
> $$

### 2.10.4. Доказательство

Покомпонентно:
$$
\frac{\partial L}{\partial W_{ij}} = \sum_{k=1}^{m} \frac{\partial L}{\partial y_k} \cdot \frac{\partial y_k}{\partial W_{ij}}.
$$
Так как $y_k = \sum_l W_{kl} x_l$, имеем $\partial y_k/\partial W_{ij} = \delta_{ki} x_j$. Значит,
$$
\frac{\partial L}{\partial W_{ij}} = \frac{\partial L}{\partial y_i} \, x_j = \left(\frac{\partial L}{\partial y} \, x^\top\right)_{ij}. \quad \blacksquare
$$

### 2.10.5. Проверка размерностей

$m \times 1$ (столбец) умножить на $1 \times n$ (строка) = $m \times n$ — как $W$. Корректно.

### 2.10.6. Связь с главами курса

В главе 8 формула $\partial L/\partial W_h = \sum_t \delta^a_t h_{t-1}^\top$ — это сумма внешних произведений. В главе 9 — $\partial L/\partial W_y = \sum_t \delta^y_t h_t^\top$. В главах 14–15 — аналогично для всех вентилей. Без понимания внешнего произведения эти формулы будут нечитаемы.

## 2.11. Поэлементные операции и произведение Адамара

### 2.11.1. Поэлементное применение функции

> **Определение 2.10 (поэлементное применение).** Если $f: \mathbb{R} \to \mathbb{R}$ и $a \in \mathbb{R}^n$, то $f(a) \in \mathbb{R}^n$ определяется покомпонентно:
> $$
> (f(a))_i = f(a_i).
> $$

Например, $\tanh(a)$ для вектора $a$ — это вектор из $\tanh(a_1), \dots, \tanh(a_n)$. Именно так $\tanh$ применяется к преактивации в нейронных сетях начиная с главы 8.

### 2.11.2. Произведение Адамара

> **Определение 2.11 (произведение Адамара).** Произведением Адамара (поэлементным произведением) двух векторов $u, v \in \mathbb{R}^n$ называется вектор
> $$
> (u \odot v)_i = u_i v_i.
> $$

Обозначение $\odot$ используется начиная с главы 8; в главах 14–15 через него записываются все вентили LSTM и GRU.

### 2.11.3. Якобиан поэлементной функции

> **Теорема 2.6.** Если $h = f(a)$ применяется поэлементно, то якобиан $J = \partial h/\partial a$ **диагонален**:
> $$
> J_{ij} = f'(a_i) \delta_{ij}, \qquad \text{т.е. } J = \operatorname{diag}\bigl(f'(a_1), \dots, f'(a_n)\bigr).
> $$

### 2.11.4. Обратный проход через поэлементную функцию

> **Теорема 2.7.** Если $h = f(a)$ применяется поэлементно и $\partial L/\partial h \in \mathbb{R}^n$, то
> $$
> \frac{\partial L}{\partial a} = \frac{\partial L}{\partial h} \odot f'(a).
> $$

**Доказательство.** По цепному правилу,
$$
\frac{\partial L}{\partial a_i} = \sum_j \frac{\partial L}{\partial h_j} \cdot \frac{\partial h_j}{\partial a_i} = \sum_j \frac{\partial L}{\partial h_j} \cdot f'(a_i) \delta_{ij} = \frac{\partial L}{\partial h_i} f'(a_i).
$$
Это в точности $i$-я компонента $\partial L/\partial h \odot f'(a)$. $\blacksquare$

### 2.11.5. Применение: производная $\tanh$

Для $h = \tanh(a)$:
$$
f'(a) = 1 - \tanh^2(a) = 1 - h \odot h.
$$
Значит, обратный проход:
$$
\frac{\partial L}{\partial a} = \frac{\partial L}{\partial h} \odot (1 - h \odot h).
$$

Именно эта формула появится в главе 8 в виде
$$
\delta^a_t = (\dots) \odot (1 - h_t \odot h_t),
$$
и во всех последующих главах — как «умножение на производную $\tanh$».

### 2.11.6. Ключевое различие

> **Замечание 2.6.** Поэлементное умножение $\odot$ **не то же самое**, что матричное умножение. Если в формуле стоит $\delta \odot (1 - h \odot h)$, это **не** $\delta \cdot (1 - h h^\top)$ и не $\delta \cdot \operatorname{diag}(1-h^2)$, хотя последнее эквивалентно по результату. Типичная ошибка — перепутать $\odot$ и матричное умножение при реализации.

## 2.12. Нормы

Нормы понадобятся в главе 13 при анализе затухания градиентов. Приведём минимально необходимые сведения.

### 2.12.1. Норма вектора

> **Определение 2.12 (евклидова норма).** Нормой вектора $v \in \mathbb{R}^n$ называется
> $$
> \|v\|_2 = \sqrt{\sum_{i=1}^{n} v_i^2}.
> $$

Геометрически — длина вектора.

### 2.12.2. Норма матрицы

> **Определение 2.13 (спектральная норма).** Спектральной нормой матрицы $A \in \mathbb{R}^{m \times n}$ называется
> $$
> \|A\|_2 = \max_{\|v\|_2 = 1} \|A v\|_2.
> $$
> Иными словами, максимальное растяжение, которое $A$ может применить к вектору единичной длины.

> **Определение 2.14 (норма Фробениуса).** Нормой Фробениуса матрицы $A$ называется
> $$
> \|A\|_F = \sqrt{\sum_{i,j} A_{ij}^2}.
> $$

### 2.12.3. Субмультипликативность

> **Теорема 2.8 (субмультипликативность).** Для любых матриц $A, B$ подходящих размеров
> $$
> \|A B\| \leq \|A\| \cdot \|B\|.
> $$

**Следствие.** Для произведения $k$ матриц:
$$
\|A_1 A_2 \cdots A_k\| \leq \|A_1\| \cdot \|A_2\| \cdots \|A_k\|.
$$

### 2.12.4. Применение в главе 13

В главе 13 будет показано, что норма «ядра» пути из $t$ в $s$ ограничена как
$$
\|\pi_{t,s}\| \leq \bigl(\|W_h\|_2 \cdot \bar D\bigr)^{s-t},
$$
где $\bar D = \max_k \|D_k\|_2 \leq 1$. Это прямое следствие субмультипликативности. Без неё аргумент о затухании градиентов не проходит.

## 2.13. Правило «сумма по путям»

Сформулируем правило, которое будет использоваться при выводе каждой формулы обратного прохода.

> **Правило 2.1 (сумма по путям).** Пусть $L$ — скалярная функция, $v$ — переменная графа вычислений. Тогда
> $$
> \frac{\partial L}{\partial v} = \sum_{\text{путей } p: v \to L} \prod_{\text{дуг } (u \to w) \in p} \frac{\partial w}{\partial u}.
> $$
> Иными словами, для каждой переменной **сложите произведения локальных производных по всем путям** от этой переменной до $L$.

Это — переформулировка теоремы 2.2 для произвольного графа.

### 2.13.1. Пример: два пути

Пусть $z = y_1 y_2$, $y_1 = x^2$, $y_2 = \sin x$. Пути от $x$ к $z$:
- путь через $y_1$: $x \xrightarrow{2x} y_1 \xrightarrow{y_2} z$;
- путь через $y_2$: $x \xrightarrow{\cos x} y_2 \xrightarrow{y_1} z$.

Произведения:
- путь 1: $2x \cdot y_2$;
- путь 2: $\cos x \cdot y_1$.

Сумма: $2x y_2 + y_1 \cos x$ — та же формула, что в примере 2.5.

### 2.13.2. Связь с главами курса

| Глава | Сколько путей | Формула |
|---|---|---|
| 4 | 2 пути от $a_1$ | два слагаемых в $\delta^a_1$ |
| 5 | 3 пути от $a_1$ | три слагаемых, произведение $w_h^2$ |
| 6 | $T$ путей | свёртка в рекурсию |
| 14 (LSTM) | 2 пути от $c_t$ | сумма двух слагаемых в $\delta^c_t$ |

> **Замечание 2.7.** В главе 5 при $T=3$ путей от $a_1$ к $L$ ровно три, и это видно «руками». В главе 6 при произвольном $T$ число путей равно $T(T+1)/2$ — выписать невозможно, поэтому требуется рекуррентная формула, которая **вычисляет сумму, не выписывая её**. Принцип (теорема 2.2) при этом не меняется; меняется только способ вычисления.

## 2.14. Сводная таблица: какой аппарат где понадобится

| Раздел | Где применяется |
|---|---|
| §2.2 (функции, композиция) | Главы 4, 5 |
| §2.3 (производные, $\sigma$, $\tanh$) | Главы 4–15 |
| §2.4 (цепное правило, одна ветвь) | Главы 4, 5 |
| §2.5 (**несколько ветвей**) | Главы 4–15 (**ключевой**) |
| §2.6 (частные производные) | Главы 4, 8 |
| §2.7 (градиент) | Главы 4, 8 |
| §2.8 (якобиан) | Главы 8, 9 |
| §2.9 (**транспонирование**) | Главы 8–15 (**ключевой**) |
| §2.10 (внешнее произведение) | Главы 8, 9, 14, 15 |
| §2.11 (Адамар) | Главы 8, 9, 14, 15 |
| §2.12 (нормы) | Глава 13 |
| §2.13 (сумма по путям) | Главы 4–7, 14–15 |

## 2.15. Типичные ошибки и как их избежать

В заключение приведём пять типичных ошибок, которые возникают при реализации обратного прохода. Знание их заранее избавит читателя от многих часов отладки.

**Ошибка 1: забыть транспонирование.** При обратном проходе через линейное отображение $y = W x$ **всегда** $W^\top$, никогда $W$. См. §2.9.

**Ошибка 2: перепутать $\odot$ и матричное умножение.** Поэлементное умножение $\odot$ используется только когда якобиан диагональный (поэлементная функция). Во всех остальных случаях — матричное умножение. См. §2.11.

**Ошибка 3: пропустить путь.** Если у переменной $n$ потребителей, в $\partial L/\partial v$ будет $n$ слагаемых. Перед каждой формулой **полезно перечислить потребителей** — это лучший способ проверить себя.

**Ошибка 4: перепутать размерности.** Перед каждой формулой **выпишите размерности** всех объектов. Если $W_h^\top \delta^a_{t+1}$ имеет размерность $(N,)$, а $h_t \odot h_t$ — тоже $(N,)$, их можно складывать. Если нет — где-то ошибка.

**Ошибка 5: забыть локальную производную.** Каждый узел графа вносит свой множитель. Если в формуле отсутствует $\sigma'$ или $\tanh'$ — вероятно, потерян узел графа. См. §2.5.3.

## 2.16. Резюме главы

Сформулируем три ключевых положения.

**Положение 1.** Цепное правило для одной ветви ($\S2.4$) достаточно для простых композиций (главы 4, 5). Цепное правило для нескольких ветвей ($\S2.5$) — это **главная формула курса**: все «пути», «дельты» и «рекуррентные соотношения» в главах 4–15 суть применения этого правила к конкретным графам.

**Положение 2.** Транспонирование в обратном проходе через линейное отображение ($\S2.9$) — правило «прямой проход — $W$, обратный — $W^\top$» — систематически используется начиная с главы 8. Непонимание этого правила — источник большинства ошибок при реализации.

**Положение 3.** Поэлементные операции и произведение Адамара ($\S2.11$) — инструмент для работы с диагональными якобианами. Производная $\tanh$ записывается как $1 - h \odot h$; все вентили LSTM и GRU работают через $\odot$.

> **Главный тезис главы.** Всё, что понадобится для понимания обратного прохода, — это (а) цепное правило для одной и нескольких ветвей, (б) транспонирование при обратном проходе через линейные отображения, (в) произведение Адамара для поэлементных операций. Всё остальное — следствия.


# Глава 3. Сеть прямого распространения $1 \to 1 \to 1$: три фазы обучения на минимальном примере

## 3.1. Предмет и структура главы

Настоящая глава посвящена **сети прямого распространения** (feedforward network) с одним входом, одним скрытым нейроном и одним выходом. Эта архитектура содержит всего **четыре обучаемых параметра** — минимальное число, при котором сеть уже имеет **два слоя** и, следовательно, демонстрирует все три фазы обучения в полном объёме: прямой проход, обратный проход по цепному правилу, обновление параметров.

Глава выполняет **двойную функцию**. С одной стороны, она служит первым **содержательным приложением** математического аппарата главы 2: цепное правило для одной ветви (§2.4), таблица производных (§2.3.2) и правило обновления через градиент (§2.7.3) здесь впервые применяются к реальной модели. С другой стороны, она является **непосредственной подготовкой** к главе 4, где та же архитектура будет развёрнута во времени и превратится в рекуррентную сеть при $T = 2$.

> **Замечание 3.1 (почему не перцептрон).** Читатель, знакомый с перцептроном — простейшим нейроном с одним параметром $w$, — может спросить, зачем нужна эта глава. Ответ состоит в том, что перцептрон имеет **лишь одну цепочку** в обратном проходе и **не содержит промежуточной переменной** между входом и выходом. В сети $1 \to 1 \to 1$ появляется **скрытый слой**: переменная $a_1$ (активация скрытого нейрона) стоит между входом и выходом, и именно она станет в главе 4 «скрытым состоянием» $h_t$. Без этой ступени переход к рекуррентным сетям был бы слишком резким.

**Структура главы:**
- §3.2 — архитектура и обозначения;
- §3.3 — прямой проход: четыре шага;
- §3.4 — функция потерь;
- §3.5 — таблица локальных производных («кирпичи»);
- §3.6 — обратный проход: шесть шагов с полными цепочками;
- §3.7 — численный пример с подстановкой каждого числа;
- §3.8 — программная реализация;
- §3.9 — резюме;
- §3.10 — мостик к главе 4.

## 3.2. Архитектура и обозначения

### 3.2.1. Структура сети

Сеть имеет **один вход**, **один скрытый нейрон** и **один выход**. Это две последовательные композиции линейного преобразования и нелинейности:
$$
\hat y = \sigma\bigl(w_2 \cdot \tanh(w_1 x + b_1) + b_2\bigr).
\tag{3.1}
$$

> **Определение 3.1 (сеть $1 \to 1 \to 1$).** Сетью $1 \to 1 \to 1$ называется композиция двух линейных преобразований и двух нелинейностей:
> 1. линейное преобразование скрытого нейрона: $z_1 = w_1 x + b_1$;
> 2. нелинейность скрытого нейрона: $a_1 = \tanh(z_1)$;
> 3. линейное преобразование выхода: $z_2 = w_2 a_1 + b_2$;
> 4. нелинейность выхода: $\hat y = \sigma(z_2)$.

Обозначение $1 \to 1 \to 1$ отражает число нейронов в каждом слое: один входной, один скрытый, один выходной.

### 3.2.2. Обозначения

| Символ | Смысл |
|---|---|
| $x$ | входной сигнал (вещественное число) |
| $w_1$ | вес связи «вход → скрытый нейрон» |
| $b_1$ | смещение скрытого нейрона |
| $w_2$ | вес связи «скрытый → выходной нейрон» |
| $b_2$ | смещение выходного нейрона |
| $\sigma$ | функция активации выхода (сигмоида) |
| $\tanh$ | функция активации скрытого нейрона |
| $z_1$ | преактивация скрытого нейрона |
| $a_1$ | активация скрытого нейрона |
| $z_2$ | преактивация выхода |
| $\hat y$ | предсказание сети |
| $y$ | истинный ответ |
| $L$ | функция потерь |
| $\eta$ | шаг обучения |

Итого **четыре обучаемых параметра**: $w_1$, $b_1$, $w_2$, $b_2$.

> **Замечание 3.2 (сравнение с перцептроном).** Перцептрон имеет один параметр $w$ и уравнение $\hat y = \sigma(w x)$. В сети $1 \to 1 \to 1$ параметров четыре, и между входом и выходом появляется **промежуточная переменная** $a_1 = \tanh(z_1)$. Эта переменная — ключевое отличие; именно она станет «скрытым состоянием» $h_1$ в главе 4.

### 3.2.3. Граф вычислений

В соответствии с §2.2.3 главы 2, представим сеть в виде графа вычислений:
$$
x \xrightarrow{\;w_1 x + b_1\;} z_1 \xrightarrow{\;\tanh\;} a_1 \xrightarrow{\;w_2 a_1 + b_2\;} z_2 \xrightarrow{\;\sigma\;} \hat y \xrightarrow{\;\frac{1}{2}(\hat y - y)^2\;} L.
\tag{3.2}
$$

Граф содержит **шесть вершин** ($x$, $z_1$, $a_1$, $z_2$, $\hat y$, $L$) и **пять дуг**. Каждая дуга помечена операцией, преобразующей одну переменную в другую. Прямой проход — движение по графу слева направо; обратный проход — справа налево.

## 3.3. Прямой проход

Прямой проход (§1.6.1 главы 1) для сети $1 \to 1 \to 1$ состоит из **четырёх последовательных шагов**. Каждый следующий шаг опирается на результат предыдущего.

### 3.3.1. Шаг 1: линейная комбинация скрытого нейрона

Скрытый нейрон вычисляет взвешенную сумму входа и смещения:
$$
z_1 = w_1 x + b_1.
\tag{3.3}
$$

Величина $z_1$ называется **преактивацией** скрытого нейрона — значением до применения нелинейности.

### 3.3.2. Шаг 2: активация скрытого нейрона

К $z_1$ применяется нелинейная функция активации — гиперболический тангенс:
$$
a_1 = \tanh(z_1).
\tag{3.4}
$$

Величина $a_1$ — это **выход скрытого нейрона**, который поступает на выходной слой. В главе 4 эта переменная превратится в скрытое состояние $h_1$, передаваемое на следующий шаг времени.

### 3.3.3. Шаг 3: линейная комбинация выходного нейрона

Выходной нейрон принимает $a_1$ и вычисляет взвешенную сумму:
$$
z_2 = w_2 a_1 + b_2.
\tag{3.5}
$$

### 3.3.4. Шаг 4: выход сети

Наконец, к $z_2$ применяется вторая активация — сигмоида:
$$
\hat y = \sigma(z_2) = \frac{1}{1 + e^{-z_2}}.
\tag{3.6}
$$

Величина $\hat y$ — предсказание сети для данного $x$.

> **Замечание 3.3 (зачем две разные активации).** Скрытый слой использует $\tanh$, выходной — $\sigma$. Причины: $\tanh$ центрирован относительно нуля (выход в $(-1, 1)$), что улучшает обусловленность промежуточных вычислений; $\sigma$ даёт выход в $(0, 1)$, что интерпретируется как вероятность. В более общем случае активации могут быть любыми дифференцируемыми функциями; конкретный выбор диктуется задачей.

### 3.3.5. Сводка прямого прохода

$$
\boxed{
\begin{aligned}
z_1 &= w_1 x + b_1, \\
a_1 &= \tanh(z_1), \\
z_2 &= w_2 a_1 + b_2, \\
\hat y &= \sigma(z_2).
\end{aligned}}
\tag{3.7}
$$

## 3.4. Функция потерь

После прямого прохода у нас есть предсказание $\hat y$. Для оценки его близости к истинному ответу $y$ введём функцию потерь.

> **Определение 3.2 (квадратичная потеря).** Функцией потерь для одного примера называется
> $$
> L = \frac{1}{2}\,(\hat y - y)^2 \in \mathbb{R}_{\geq 0}.
> \tag{3.8}
> $$

Множитель $1/2$ введён исключительно для удобства дифференцирования. При вычислении производной:
$$
\frac{\partial L}{\partial \hat y} = \frac{\partial}{\partial \hat y}\left[\frac{1}{2}(\hat y - y)^2\right] = (\hat y - y).
\tag{3.9}
$$
Без множителя $1/2$ получилось бы $2(\hat y - y)$ — на множитель $2$ больше, что загромождало бы все последующие формулы. Выбор $1/2$ — это **соглашение**, а не принципиальная особенность; в главе 9 при рассмотрении векторного выхода будет использована та же нормировка.

> **Замечание 3.4 (потеря — скаляр).** Функция $L$ возвращает одно число. Это существенно: именно по этому скаляру берётся градиент по всем параметрам. В главе 5 для последовательности из $T$ шагов потеря будет суммой $T$ слагаемых; в главе 9 — суммой по $T$ шагам и по $K$ выходам. Но принцип — «одно число, по нему дифференцируем» — сохраняется.

## 3.5. Таблица локальных производных

Прежде чем выписывать полные цепочки обратного прохода, составим **таблицу локальных производных** — производных каждого уравнения прямого прохода по каждой переменной, при фиксированных остальных. Это «кирпичи», из которых в §3.6 собираются полные цепочки.

### 3.5.1. Производные преактивации скрытого нейрона (уравнение 3.3)

$$
\frac{\partial z_1}{\partial w_1} = x, \qquad \frac{\partial z_1}{\partial b_1} = 1, \qquad \frac{\partial z_1}{\partial x} = w_1.
\tag{3.10}
$$

### 3.5.2. Производная активации скрытого нейрона (уравнение 3.4)

В соответствии с таблицей §2.3.2:
$$
\frac{\partial a_1}{\partial z_1} = \tanh'(z_1) = 1 - \tanh^2(z_1) = 1 - a_1^2.
\tag{3.11}
$$

Последнее равенство существенно: производная $\tanh$ выражается через **уже вычисленное значение** $a_1 = \tanh(z_1)$. Повторно вычислять экспоненты не требуется.

### 3.5.3. Производные преактивации выхода (уравнение 3.5)

$$
\frac{\partial z_2}{\partial a_1} = w_2, \qquad \frac{\partial z_2}{\partial w_2} = a_1, \qquad \frac{\partial z_2}{\partial b_2} = 1.
\tag{3.12}
$$

### 3.5.4. Производная выхода (уравнение 3.6)

$$
\frac{\partial \hat y}{\partial z_2} = \sigma'(z_2) = \sigma(z_2)\bigl(1 - \sigma(z_2)\bigr) = \hat y (1 - \hat y).
\tag{3.13}
$$

### 3.5.5. Производная потери по выходу (уравнение 3.9)

$$
\frac{\partial L}{\partial \hat y} = \hat y - y.
\tag{3.14}
$$

> **Замечание 3.5 (зачем нужна таблица).** Каждый множитель в полных цепочках §3.6 — это **ровно одна ячейка** таблицы §3.5. Составление таблицы заранее избавляет от повторного дифференцирования и позволяет проверить, что ни один множитель не потерян: если в итоговой формуле отсутствует какой-либо множитель из таблицы, это сигнал об ошибке.

## 3.6. Обратный проход

Обратный проход (§1.6.2 главы 1) состоит в вычислении производных $L$ по всем четырём параметрам. В соответствии с принципом §2.5.3 главы 2, мы движемся по графу вычислений (3.2) **справа налево**, на каждой дуге умножая на локальную производную.

В этой главе впервые появляется приём, который будет систематически использоваться во всех последующих: **сначала выписываем полную цепочку производных от $L$ до параметра, и только затем сворачиваем повторяющиеся фрагменты в обозначения** $\delta_2, \delta_1$. Так видно, откуда берётся каждая «дельта», и ни один множитель не теряется.

### 3.6.1. Шаг 1: производная по выходу сети

В соответствии с (3.14):
$$
\frac{\partial L}{\partial \hat y} = \hat y - y.
\tag{3.15}
$$

Это «сигнал ошибки» — насколько и в какую сторону мы ошиблись. Если $\hat y > y$, ошибка положительна, и параметры должны сдвинуться так, чтобы уменьшить $\hat y$. Если $\hat y < y$ — наоборот.

### 3.6.2. Шаг 2: производная выхода по преактивации $z_2$

В соответствии с (3.13):
$$
\frac{\partial \hat y}{\partial z_2} = \hat y(1 - \hat y).
\tag{3.16}
$$

### 3.6.3. Шаг 3: дельта выходного слоя

Собираем два предыдущих множителя по цепному правилу для одной ветви (§2.4 главы 2):
$$
\frac{\partial L}{\partial z_2} = \frac{\partial L}{\partial \hat y}\cdot\frac{\partial \hat y}{\partial z_2} = (\hat y - y)\,\hat y(1 - \hat y).
\tag{3.17}
$$

> **Определение 3.3 (дельта выходного слоя).** Величиной
> $$
> \delta_2 := \frac{\partial L}{\partial z_2} = (\hat y - y)\,\hat y(1 - \hat y)
> \tag{3.18}
> $$
> назовём **дельту выходного слоя**.

С этого момента $\delta_2$ — удобное сокращение для $\partial L/\partial z_2$. Оно полностью вычисляется из величин прямого прохода ($\hat y$ и $y$), поэтому дальше мы работаем с ним как с известной константой.

> **Замечание 3.6 (обозначение $\delta$).** Обозначение $\delta$ для $\partial L/\partial z$ — стандартное в литературе по обратному распространению. Оно не имеет отношения к «приращению» в смысле $\Delta$; это именно **градиент** по преактивации. В главе 5 появятся $\delta^y_t$ (дельта выхода на шаге $t$) и $\delta^a_t$ (дельта скрытого слоя на шаге $t$); в главах 14–15 — $\delta^{af}_t, \delta^{ai}_t, \dots$ для вентилей LSTM и GRU. Все они — частные случаи этого обозначения.

### 3.6.4. Шаг 4: градиенты параметров выходного слоя

**Для $w_2$.** Выпишем полную цепочку от $L$ до $w_2$, проходящую через $\hat y$ и $z_2$:
$$
\frac{\partial L}{\partial w_2} = \frac{\partial L}{\partial \hat y}\cdot\frac{\partial \hat y}{\partial z_2}\cdot\frac{\partial z_2}{\partial w_2}.
\tag{3.19}
$$

Из (3.12): $\partial z_2/\partial w_2 = a_1$. Подставляем:
$$
\frac{\partial L}{\partial w_2} = (\hat y - y)\cdot\hat y(1-\hat y)\cdot a_1.
\tag{3.20}
$$

Первые два множителя — это ровно $\delta_2$ (3.18), поэтому сворачиваем:
$$
\boxed{\;\frac{\partial L}{\partial w_2} = \delta_2 \cdot a_1.\;}
\tag{3.21}
$$

**Для $b_2$.** Цепочка та же, но последний множитель другой. Из (3.12): $\partial z_2/\partial b_2 = 1$. Значит,
$$
\frac{\partial L}{\partial b_2} = (\hat y - y)\cdot\hat y(1-\hat y)\cdot 1 = \delta_2.
\tag{3.22}
$$
$$
\boxed{\;\frac{\partial L}{\partial b_2} = \delta_2.\;}
\tag{3.23}
$$

> **Замечание 3.7 (сравнение (3.21) и (3.23)).** Градиент по $w_2$ отличается от градиента по $b_2$ множителем $a_1$ — потому что $w_2$ входит в $z_2$ умноженным на $a_1$, а $b_2$ — с коэффициентом 1. Это общее правило: градиент по параметру равен $\delta$ соответствующего слоя, умноженной на то, на что этот параметр умножается в прямом проходе.

### 3.6.5. Шаг 5: дельта скрытого слоя

Аналогично вводим дельту скрытого слоя.

> **Определение 3.4 (дельта скрытого слоя).** Величиной
> $$
> \delta_1 := \frac{\partial L}{\partial z_1}
> \tag{3.24}
> $$
> назовём **дельту скрытого слоя**.

Цепочка от $L$ до $z_1$ проходит через $\hat y$, $z_2$ и $a_1$:
$$
\delta_1 = \frac{\partial L}{\partial \hat y}\cdot\frac{\partial \hat y}{\partial z_2}\cdot\frac{\partial z_2}{\partial a_1}\cdot\frac{\partial a_1}{\partial z_1}.
\tag{3.25}
$$

Промежуточные множители:
- $\partial L/\partial \hat y \cdot \partial \hat y/\partial z_2 = \delta_2$ — по определению (3.18);
- $\partial z_2/\partial a_1 = w_2$ — из (3.12);
- $\partial a_1/\partial z_1 = 1 - a_1^2$ — из (3.11).

Сворачиваем:
$$
\boxed{\;\delta_1 = \delta_2 \cdot w_2 \cdot (1 - a_1^2).\;}
\tag{3.26}
$$

> **Замечание 3.8 (смысл формулы).** Формула (3.26) «протаскивает» ошибку назад через выходной слой: дельта выходного слоя $\delta_2$ умножается на вес связи $w_2$, по которой шла информация, и на производную активации скрытого нейрона $(1 - a_1^2)$ — потому что при обратном проходе через $\tanh$ нужно умножить на её производную. Оба множителя — **обязательны**: без $w_2$ ошибка «не дошла бы» до скрытого слоя, без $(1 - a_1^2)$ — не прошла бы через нелинейность.

### 3.6.6. Шаг 6: градиенты параметров скрытого слоя

**Для $w_1$.** Цепочка длиннее — она проходит через все узлы графа:
$$
\frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial \hat y}\cdot\frac{\partial \hat y}{\partial z_2}\cdot\frac{\partial z_2}{\partial a_1}\cdot\frac{\partial a_1}{\partial z_1}\cdot\frac{\partial z_1}{\partial w_1}.
\tag{3.27}
$$

Первый сомножитель после $\hat y$ — это $\delta_2$; три следующих — в точности $\delta_1$ (см. шаг 5). Из (3.10): $\partial z_1/\partial w_1 = x$. Подставляем:
$$
\frac{\partial L}{\partial w_1} = \delta_2 \cdot w_2 \cdot (1 - a_1^2) \cdot x = \delta_1 \cdot x.
\tag{3.28}
$$
$$
\boxed{\;\frac{\partial L}{\partial w_1} = \delta_1 \cdot x.\;}
\tag{3.29}
$$

**Для $b_1$.** Цепочка та же, но из (3.10): $\partial z_1/\partial b_1 = 1$. Значит,
$$
\frac{\partial L}{\partial b_1} = \delta_2 \cdot w_2 \cdot (1 - a_1^2) \cdot 1 = \delta_1.
\tag{3.30}
$$
$$
\boxed{\;\frac{\partial L}{\partial b_1} = \delta_1.\;}
\tag{3.31}
$$

### 3.6.7. Сводка обратного прохода

$$
\boxed{
\begin{aligned}
\delta_2 &= (\hat y - y)\,\hat y(1 - \hat y), \\
\delta_1 &= \delta_2 \cdot w_2 \cdot (1 - a_1^2); \\[4pt]
\frac{\partial L}{\partial w_2} &= \delta_2 \cdot a_1, & \frac{\partial L}{\partial b_2} &= \delta_2, \\
\frac{\partial L}{\partial w_1} &= \delta_1 \cdot x, & \frac{\partial L}{\partial b_1} &= \delta_1.
\end{aligned}}
\tag{3.32}
$$

**Проверка числа цепочек.** Четыре параметра — четыре формулы. Каждая цепочка содержит от 3 до 5 множителей в развёрнутом виде; после свёртки через дельты — 1–2 множителя.

## 3.7. Численный пример

Проделаем один полный шаг обучения на числах. Возьмём:
- вход $x = 1.0$;
- истинный ответ $y = 0.0$ (сеть должна выдавать «0», а пока выдаёт около 0.5);
- начальные параметры $w_1 = 0.5$, $b_1 = 0.1$, $w_2 = 0.8$, $b_2 = -0.2$;
- шаг обучения $\eta = 0.1$;
- $\sigma(z) = 1/(1 + e^{-z})$, $\sigma'(z) = \sigma(z)(1 - \sigma(z))$; $\tanh'(z) = 1 - \tanh^2(z)$.

### 3.7.1. Прямой проход

**Шаг 1:** $z_1 = 0.5 \cdot 1.0 + 0.1 = 0.6$.

**Шаг 2:** $a_1 = \tanh(0.6) \approx 0.53705$.

**Шаг 3:** $z_2 = 0.8 \cdot 0.53705 + (-0.2) = 0.42964 - 0.2 = 0.22964$.

**Шаг 4:** $\hat y = \sigma(0.22964) = \dfrac{1}{1 + e^{-0.22964}} \approx 0.55716$.

**Потеря:**
$$
L = \tfrac12(0.55716 - 0)^2 \approx \tfrac12 \cdot 0.31043 = 0.15521.
$$

### 3.7.2. Обратный проход

**Дельта выходного слоя (3.18):**

Сначала производная сигмоиды:
$$
\sigma'(z_2) = \hat y(1 - \hat y) = 0.55716 \cdot (1 - 0.55716) = 0.55716 \cdot 0.44284 \approx 0.24672.
$$
Затем сама дельта:
$$
\delta_2 = (\hat y - y)\cdot\sigma'(z_2) = (0.55716 - 0) \cdot 0.24672 \approx 0.13747.
$$

**Дельта скрытого слоя (3.26):**

Сначала производная $\tanh$:
$$
1 - a_1^2 = 1 - 0.53705^2 = 1 - 0.28842 \approx 0.71158.
$$
Затем:
$$
\delta_1 = \delta_2 \cdot w_2 \cdot (1 - a_1^2) = 0.13747 \cdot 0.8 \cdot 0.71158 \approx 0.07826.
$$

> **Замечание 3.9 (проверка «на глаз»).** $\delta_1 \approx 0.0783$ — примерно вдвое меньше $\delta_2 \approx 0.1375$. Это ожидаемо: при обратном проходе через выходной слой ошибка умножается на $w_2 \cdot \tanh'(z_1) = 0.8 \cdot 0.71158 \approx 0.569$ — множитель меньше 1, поэтому дельта уменьшается. В главе 13 будет показано, что при многократном умножении на такие множители дельта «затухает» — это и есть проблема затухающих градиентов.

**Градиенты параметров (3.21, 3.23, 3.29, 3.31):**

$$
\frac{\partial L}{\partial w_2} = \delta_2 \cdot a_1 = 0.13747 \cdot 0.53705 \approx 0.07382,
$$

$$
\frac{\partial L}{\partial b_2} = \delta_2 = 0.13747,
$$

$$
\frac{\partial L}{\partial w_1} = \delta_1 \cdot x = 0.07826 \cdot 1.0 = 0.07826,
$$

$$
\frac{\partial L}{\partial b_1} = \delta_1 = 0.07826.
$$

### 3.7.3. Обновление параметров

В соответствии с §1.6.3 главы 1:
$$
\theta_i \leftarrow \theta_i - \eta\,\frac{\partial L}{\partial \theta_i}.
$$

При $\eta = 0.1$:
$$
w_1 \leftarrow 0.5 - 0.1 \cdot 0.07826 = 0.49217,
$$
$$
b_1 \leftarrow 0.1 - 0.1 \cdot 0.07826 = 0.09217,
$$
$$
w_2 \leftarrow 0.8 - 0.1 \cdot 0.07382 = 0.79262,
$$
$$
b_2 \leftarrow -0.2 - 0.1 \cdot 0.13747 = -0.21375.
$$

> **Замечание 3.10 (проверка направления).** Все четыре градиента **положительны**. Это значит, что все четыре параметра нужно **уменьшить**, чтобы уменьшить потерю. Содержательно: $\hat y \approx 0.557 > y = 0$, то есть сеть предсказывает слишком большое значение. Чтобы уменьшить $\hat y$, нужно уменьшить $z_2$ (уменьшить $w_2$ и $b_2$), а также уменьшить $a_1$ (уменьшить $w_1$ и $b_1$). Знаки градиентов совпадают с этим рассуждением — формула работает корректно.

## 3.8. Программная реализация

Приведём полную программу, воспроизводящую числа §3.7. Реализация использует NumPy; все операции — базовые.

```python
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

class TinyFeedforward:
    """Сеть 1 -> 1 -> 1 (feedforward, без памяти)."""

    def __init__(self, w1=0.5, b1=0.1, w2=0.8, b2=-0.2, eta=0.1):
        self.w1, self.b1 = w1, b1
        self.w2, self.b2 = w2, b2
        self.eta = eta

    def forward(self, x):
        z1 = self.w1 * x + self.b1           # (3.3)
        a1 = np.tanh(z1)                     # (3.4)
        z2 = self.w2 * a1 + self.b2          # (3.5)
        yhat = sigmoid(z2)                   # (3.6)
        return z1, a1, z2, yhat

    def loss(self, yhat, y):
        return 0.5 * (yhat - y) ** 2         # (3.8)

    def backward(self, x, y, z1, a1, z2, yhat):
        # (3.18)
        dy = yhat - y
        dz2 = dy * yhat * (1 - yhat)
        # (3.26)
        da1 = dz2 * self.w2
        dz1 = da1 * (1 - a1 ** 2)
        # (3.21), (3.23), (3.29), (3.31)
        return dict(
            w1 = dz1 * x,
            b1 = dz1,
            w2 = dz2 * a1,
            b2 = dz2,
        )

    def step(self, x, y):
        z1, a1, z2, yhat = self.forward(x)
        L = self.loss(yhat, y)
        g = self.backward(x, y, z1, a1, z2, yhat)
        for k, v in g.items():
            setattr(self, k, getattr(self, k) - self.eta * v)
        return L, yhat, g


def demo():
    """Воспроизводит числа §3.7."""
    net = TinyFeedforward()
    x, y = 1.0, 0.0
    z1, a1, z2, yhat = net.forward(x)
    print("Прямой проход:")
    print(f"  z1 = {z1:.5f}")
    print(f"  a1 = {a1:.5f}")
    print(f"  z2 = {z2:.5f}")
    print(f"  y^ = {yhat:.5f}")
    L, yhat, g = net.step(x, y)
    print(f"\nПотеря: L = {L:.5f}")
    print("\nГрадиенты:")
    for k, v in g.items():
        print(f"  dL/d{k} = {v:+.5f}")
    print("\nОбновлённые параметры:")
    print(f"  w1 = {net.w1:.5f}")
    print(f"  b1 = {net.b1:.5f}")
    print(f"  w2 = {net.w2:.5f}")
    print(f"  b2 = {net.b2:.5f}")


if __name__ == "__main__":
    demo()
```

**Вывод программы:**
```
Прямой проход:
  z1 = 0.60000
  a1 = 0.53705
  z2 = 0.22964
  y^ = 0.55716

Потеря: L = 0.15521

Градиенты:
  dL/dw1 = +0.07826
  dL/db1 = +0.07826
  dL/dw2 = +0.07382
  dL/db2 = +0.13747

Обновлённые параметры:
  w1 = 0.49217
  b1 = 0.09217
  w2 = 0.79262
  b2 = -0.21375
```

Все числа совпадают с §3.7 до последнего знака. Это — программная проверка формул.

> **Замечание 3.11 (зачем писать с нуля).** Когда читатель пишет `dz1 = da1 * (1 - a1**2)` своими руками, он видит **каждую** операцию формулы (3.26). При использовании фреймворка эта деталь скрыта. Именно поэтому весь код в курсе — на NumPy «с нуля».

## 3.9. Резюме главы

Сформулируем, что мы получили в этой главе.

**Прямой проход** (§3.3) — четыре последовательные операции (3.3)–(3.6), порождающие $z_1, a_1, z_2, \hat y$.

**Функция потерь** (§3.4) — квадратичная (3.8), с множителем $1/2$ для удобства дифференцирования.

**Таблица локальных производных** (§3.5) — девять формул (3.10)–(3.14), по одной на каждую пару (переменная, переменная), связанную дугой графа.

**Обратный проход** (§3.6) — шесть шагов, дающих две дельты (3.18), (3.26) и четыре градиента (3.21), (3.23), (3.29), (3.31).

**Численный пример** (§3.7) — полная подстановка: $\hat y \approx 0.55716$, $L \approx 0.15521$, $\delta_2 \approx 0.13747$, $\delta_1 \approx 0.07826$.

**Программная реализация** (§3.8) — воспроизводит числа §3.7.

**Три ключевых положения главы.**

1. **Обратный проход — это последовательное умножение на локальные производные вдоль цепочек графа.** Четыре цепочки, четыре градиента. Каждый множитель — из таблицы §3.5.

2. **Дельта $\delta_\ell$ — удобное сокращение для $\partial L/\partial z_\ell$.** Она позволяет записать градиенты параметров компактно: $\partial L/\partial w_\ell = \delta_\ell \cdot (\text{вход слоя})$, $\partial L/\partial b_\ell = \delta_\ell$. Это правило сохранится и в последующих главах.

3. **Обновление — это сдвиг каждого параметра против его градиента.** Все четыре параметра обновляются **одновременно**, по градиентам, вычисленным в одной и той же точке $\theta$.

## 3.10. Мостик к главе 4

Сеть $1 \to 1 \to 1$ обрабатывает **один** вход $x$ и выдаёт **один** выход $\hat y$. В ней **нет памяти**: если подать на вход $x = 1.0$ дважды, получим один и тот же $\hat y$ дважды. Это ограничение принципиально: для задач на последовательностях (глава 1, §1.3) такая сеть непригодна.

Что нужно добавить, чтобы перейти к рекуррентной сети?

**Первое: развернуть сеть на несколько шагов времени.** Вместо одного столбца нейронов — $T$ столбцов, по одному на каждый шаг.

**Второе: добавить связь $h_{t-1} \to h_t$.** Скрытое состояние с предыдущего шага подаётся на текущий. Это и есть **память**.

**Третье: использовать одни и те же параметры на всех шагах.** Параметры $w_x, w_h, b_h, w_y, b_y$ общие для всех $t$.


# Глава 4. Пример 2: один вход, один скрытый нейрон, один выход, развёрнутая по времени ($T = 2$)

Прежде чем разбирать рекуррентные сети в общем виде, полезно взять **минимальную последовательность — всего два шага**. При $T = 2$ сеть перестаёт быть «по-настоящему» рекуррентной: она разворачивается в обычную сеть прямого распространения, и к ней применим ровно тот же аппарат, что и в главе 3. Но у каждого параметра появляется **по два пути** к функции потерь, а у скрытого состояния $h_1$ — **два потребителя**. Именно здесь видно, откуда берутся все множители в формулах BPTT. Эта глава — «эталонный пример» для всего рекуррентного раздела; дальше мы будем ссылаться на её формулы как на базовые. К каждой части главы прилагается код на Python, который порождает соответствующую схему.

## 4.1. Зачем нужен этот пример

Сеть из главы 3 была без памяти: подайте на вход $x = 1$ дважды подряд — получите один и тот же $\hat y$ дважды. RNN отличается от неё **одной стрелкой** — скрытое состояние с предыдущего шага подаётся на текущий. При $T = 2$ эта стрелка всего одна, и вся сеть состоит из восьми уравнений. Их так мало, что **каждый путь от параметра до $L$ можно выписать в явном виде** и проследить каждую стрелку на графе вычислений. Именно поэтому этот пример хорош как первая иллюстрация рекуррентности:

- **прямой проход** — четыре арифметические операции на шаге $t = 1$ и ещё четыре на шаге $t = 2$;
- **обратный проход** — цепное правило, применённое к каждому пути отдельно; у $h_1$ путей два, у $h_2$ — один;
- **обновление** — пять вычитаний, потому что параметров теперь пять: $w_x$, $w_h$, $b_h$, $w_y$, $b_y$.

Здесь появляется одна принципиально новая идея, которой не было в главе 3:

> **Веса разделяются между шагами времени.** Параметры $w_x$, $w_h$, $b_h$, $w_y$, $b_y$ — одни и те же на обоих шагах. Поэтому при обратном проходе производная по каждому из них — это **сумма вкладов от обоих шагов**.

Когда $T$ вырастет до сотен, путей станет экспоненциально много, и их придётся сворачивать в рекурсию через «дельты». Но при $T = 2$ всё ещё помещается на бумаге — и потому именно здесь видно, **что именно** эта рекурсия сворачивает. Разобравшись в этом, вы без труда перенесёте ту же логику на произвольное $T$: изменится число слагаемых, но не структура вычислений.

## 4.2. Архитектура и обозначения

Сеть имеет **один вход**, **один скрытый нейрон** и **один выход**, но разворачивается на два шага времени: на каждом шаге $t \in \{1, 2\}$ — свой столбец нейронов, а веса между столбцами **общие**. Обозначим:

| Символ | Смысл |
|---|---|
| $x_t$ | вход на шаге $t$ (вещественное число) |
| $a_t$ | преактивация скрытого нейрона: $a_t = w_x x_t + w_h h_{t-1} + b_h$ |
| $h_t$ | скрытое состояние: $h_t = \tanh(a_t)$ |
| $z_t$ | преактивация выхода: $z_t = w_y h_t + b_y$ |
| $\hat{y}_t$ | предсказание: $\hat{y}_t = \sigma(z_t)$ |
| $y_t$ | истинный ответ на шаге $t$ |
| $w_x$ | вес связи «вход → скрытый нейрон» (общий для обоих шагов) |
| $w_h$ | вес рекуррентной связи «$h_{t-1} \to h_t$» (общий для обоих шагов) |
| $b_h$ | смещение скрытого нейрона (общее) |
| $w_y$ | вес связи «скрытый → выходной нейрон» (общий) |
| $b_y$ | смещение выходного нейрона (общее) |
| $h_0$ | начальное состояние, $h_0 = 0$ |
| $\sigma$ | функция активации выхода (сигмоида) |
| $\tanh$ | функция активации скрытого нейрона |
| $L$ | функция потерь |
| $\eta$ | шаг обучения (learning rate) |

**Замечание о нотации.** По сравнению с главой 3 мы сменили обозначения: там $z_1$, $a_1$ обозначали преактивацию и активацию скрытого нейрона, а $z_2$ — преактивацию выхода. Здесь мы следуем стандартной нотации RNN: $a_t$ — преактивация скрытого, $h_t$ — его активация, $z_t$ — преактивация выхода. Соответствие между главами: $z_1 \leftrightarrow a_t$, $a_1 \leftrightarrow h_t$, $z_2 \leftrightarrow z_t$. Индекс $t$ здесь — **шаг времени**, а не номер слоя.

Итого **пять обучаемых параметров**: $w_x$, $w_h$, $b_h$, $w_y$, $b_y$. На рис. 4.1 сеть изображена развёрнутой на два шага: красные стрелки — веса, синие — смещения, фиолетовые — рекуррентная связь $w_h$.




In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch

def draw_architecture_T2():
    """Рис. 4.1: архитектура 1 → 1 → 1, развёрнутая на два шага."""
    fig, ax = plt.subplots(figsize=(13, 7))
    ax.set_xlim(0, 13); ax.set_ylim(0, 8)
    ax.set_aspect('equal'); ax.axis('off')
    r = 0.5
    cols = [2.5, 10.5]
    y_x, y_h, y_y = 2.2, 4.6, 6.5

    def arrow(p1, p2, color='black'):
        ax.add_patch(FancyArrowPatch(
            p1, p2, arrowstyle='-|>', mutation_scale=18,
            lw=2, color=color, shrinkA=0, shrinkB=0, zorder=2))

    for t, cx in enumerate(cols, start=1):
        for cy, label, color in [
            (y_x, rf'$x_{t}$',          '#AEDFF7'),
            (y_h, rf'$h_{t}$',          '#FFE29A'),
            (y_y, rf'$\hat{{y}}_{t}$',  '#B8E0B8'),
        ]:
            ax.add_patch(Circle((cx, cy), r, facecolor=color,
                                edgecolor='black', lw=1.8, zorder=3))
            ax.text(cx, cy, label, ha='center', va='center',
                    fontsize=15, zorder=4)
        arrow((cx, y_x + r), (cx, y_h - r))
        ax.text(cx + 0.18, (y_x + y_h) / 2, r'$w_x$',
                fontsize=13, color='#B22222')
        arrow((cx, y_h + r), (cx, y_y - r))
        ax.text(cx + 0.18, (y_h + y_y) / 2, r'$w_y$',
                fontsize=13, color='#B22222')
        arrow((cx - r - 0.7, y_h), (cx - r, y_h), color='#1F6FB2')
        ax.text(cx - r - 1.1, y_h + 0.2, r'$b_h$',
                fontsize=12, color='#1F6FB2')
        arrow((cx - r - 0.7, y_y), (cx - r, y_y), color='#1F6FB2')
        ax.text(cx - r - 1.1, y_y + 0.2, r'$b_y$',
                fontsize=12, color='#1F6FB2')

    arrow((cols[0] + r, y_h), (cols[1] - r, y_h), color='#7B2D8E')
    ax.text((cols[0] + cols[1]) / 2, y_h + 0.22, r'$w_h$',
            fontsize=14, color='#7B2D8E', ha='center')
    arrow((cols[0] - r - 0.7, y_h), (cols[0] - r, y_h), color='#7B2D8E')
    ax.text(cols[0] - r - 1.1, y_h + 0.2, r'$h_0 = 0$',
            fontsize=11, color='#7B2D8E')

    ax.text(6.5, 7.7, 'Рис. 4.1. Архитектура 1 → 1 → 1, развёрнутая на два шага',
            ha='center', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

draw_architecture_T2()



## 4.3. Прямой проход

Прямой проход (**forward pass**) — это вычисление выходов сети по заданной последовательности $x_1, x_2$. Оно выполняется строго последовательно: результат шага $t = 1$ нужен на шаге $t = 2$. На каждом шаге — те же четыре операции, что и в главе 3, только к линейной комбинации скрытого нейрона добавляется ещё одно слагаемое — память.

### Шаг 1. Линейная комбинация скрытого нейрона

Скрытый нейрон на шаге $t$ складывает три части — вход «сейчас», память «раньше» и смещение:

$$a_t = w_x\,x_t + w_h\,h_{t-1} + b_h. \tag{4.1}$$

Здесь $a_t$ — это **преактивация** скрытого нейрона на шаге $t$: значение до применения нелинейности. Величина $h_{t-1}$ — скрытое состояние с предыдущего шага.

Для $t = 1$ предыдущего шага нет, и используется начальное состояние $h_0 = 0$. Поэтому

$$a_1 = w_x\,x_1 + w_h\cdot 0 + b_h = w_x\,x_1 + b_h. \tag{4.1a}$$

Для $t = 2$ в правой части стоит уже вычисленное $h_1$:

$$a_2 = w_x\,x_2 + w_h\,h_1 + b_h. \tag{4.1b}$$

Именно из-за $h_1$ в (4.1b) цикл по $t$ **нельзя распараллелить**: $a_2$ можно вычислить только после того, как найден $h_1$.

### Шаг 2. Активация скрытого нейрона

К $a_t$ применяется нелинейная функция активации — гиперболический тангенс:

$$h_t = \tanh(a_t). \tag{4.2}$$

Для $\tanh$ производная выражается через сам выход:

$$\tanh'(a) = 1 - \tanh^2(a) = 1 - h_t^2. \tag{4.2a}$$

Это пригодится в обратном проходе — не придётся заново считать экспоненты. Величина $h_t$ — это **выход скрытого нейрона**: он идёт и на выходной нейрон того же шага, и на скрытый нейрон следующего шага.

### Шаг 3. Линейная комбинация выходного нейрона

Выходной нейрон принимает $h_t$ и снова считает взвешенную сумму:

$$z_t = w_y\,h_t + b_y. \tag{4.3}$$

### Шаг 4. Выход сети

Наконец, применяется вторая активация — сигмоида:

$$\hat{y}_t = \sigma(z_t) = \frac{1}{1 + e^{-z_t}}. \tag{4.4}$$

Её производная:

$$\sigma'(z_t) = \sigma(z_t)\bigl(1 - \sigma(z_t)\bigr) = \hat{y}_t\,(1 - \hat{y}_t). \tag{4.4a}$$

Величина $\hat{y}_t$ — это предсказание сети на шаге $t$.

### Подстановка чисел: шаг $t = 1$

Возьмём $x_1 = 1.0$ и параметры $w_x = 0.5$, $w_h = 0.3$, $b_h = 0.1$, $w_y = 0.8$, $b_y = -0.2$ (те же, что в §4.7). Подставляем в (4.1a)–(4.4):

$$a_1 = 0.5 \cdot 1.0 + 0.1 = 0.6,$$
$$h_1 = \tanh(0.6) \approx 0.53705,$$
$$z_1 = 0.8 \cdot 0.53705 - 0.2 = 0.22964,$$
$$\hat{y}_1 = \sigma(0.22964) \approx 0.55716.$$

### Подстановка чисел: шаг $t = 2$

Возьмём $x_2 = 0.5$. В (4.1b) подставляем **только что найденное** $h_1$:

$$a_2 = 0.5 \cdot 0.5 + 0.3 \cdot 0.53705 + 0.1 = 0.25 + 0.16112 + 0.1 = 0.51112,$$
$$h_2 = \tanh(0.51112) \approx 0.47077,$$
$$z_2 = 0.8 \cdot 0.47077 - 0.2 = 0.17662,$$
$$\hat{y}_2 = \sigma(0.17662) \approx 0.54404.$$

### Таблица локальных производных

Прежде чем переходить к обратному проходу, выпишем все **локальные производные** — производные каждого уравнения прямого прохода по каждой переменной, при фиксированных остальных. Это «кирпичи», из которых в §4.5 собираются полные цепочки. Каждая строка получается дифференцированием одного уравнения из (4.1)–(4.4) по одной переменной.

**Производные преактивации скрытого нейрона** из (4.1):

$$\frac{\partial a_t}{\partial w_x} = x_t, \qquad \frac{\partial a_t}{\partial w_h} = h_{t-1}, \qquad \frac{\partial a_t}{\partial b_h} = 1, \qquad \frac{\partial a_t}{\partial h_{t-1}} = w_h. \tag{4.5}$$

**Производная активации скрытого нейрона** из (4.2) и (4.2a):

$$\frac{\partial h_t}{\partial a_t} = \tanh'(a_t) = 1 - h_t^2. \tag{4.6}$$

**Производные преактивации выхода** из (4.3):

$$\frac{\partial z_t}{\partial h_t} = w_y, \qquad \frac{\partial z_t}{\partial w_y} = h_t, \qquad \frac{\partial z_t}{\partial b_y} = 1. \tag{4.7}$$

**Производная выхода** из (4.4) и (4.4a):

$$\frac{\partial \hat{y}_t}{\partial z_t} = \sigma'(z_t) = \hat{y}_t(1 - \hat{y}_t). \tag{4.8}$$

**Производная потери** по выходу (см. §4.4):

$$\frac{\partial L}{\partial \hat{y}_t} = \hat{y}_t - y_t. \tag{4.9}$$

**Конкретно для $T = 2$** (с учётом $h_0 = 0$, поэтому $\partial a_1/\partial w_h = 0$):

| Уравнение | по $w_x$ | по $w_h$ | по $b_h$ | по $h_{t-1}$ |
|---|---|---|---|---|
| $a_1$ (4.1a) | $x_1$ | $0$ | $1$ | — |
| $a_2$ (4.1b) | $x_2$ | $h_1$ | $1$ | $w_h$ |

| Уравнение | по $h_t$ | по $w_y$ | по $b_y$ |
|---|---|---|---|
| $z_1$ (4.3) | $w_y$ | $h_1$ | $1$ |
| $z_2$ (4.3) | $w_y$ | $h_2$ | $1$ |

Именно эти «кирпичи» и перемножаются в §4.5. Каждый множитель в полных цепочках — это ровно одна ячейка этой таблицы.

### Схема прямого прохода

На рис. 4.2 те же четыре шага подписаны прямо над стрелками каждого столбца: слева направо внутри шага $x_t$ превращается в $\hat y_t$, а фиолетовая стрелка переносит $h_t$ на следующий шаг. Видно, что на шаге $t = 2$ в $a_2$ входят **две** стрелки ($x_2$ и $h_1$), а на шаге $t = 1$ — одна ($h_0 = 0$).




In [ ]:
def draw_forward_T2():
    """Рис. 4.2: прямой проход при T=2 — по четыре шага на каждом столбце."""
    fig, ax = plt.subplots(figsize=(13, 7.5))
    ax.set_xlim(0, 13); ax.set_ylim(0, 8)
    ax.axis('off'); ax.set_aspect('equal')

    cols = [2.5, 10.5]
    y_x, y_a, y_h, y_z, y_y = 1.2, 2.6, 4.0, 5.4, 6.8
    r = 0.38

    def node(cx, cy, label, color):
        ax.add_patch(Circle((cx, cy), r, facecolor=color,
                            edgecolor='black', lw=1.4, zorder=3))
        ax.text(cx, cy, label, ha='center', va='center',
                fontsize=11, zorder=4)

    def arrow(p1, p2, color='black', ls='-'):
        ax.add_patch(FancyArrowPatch(
            p1, p2, arrowstyle='-|>', mutation_scale=14,
            lw=1.6, color=color, linestyle=ls,
            shrinkA=0, shrinkB=0, zorder=2))

    for t, cx in enumerate(cols, start=1):
        node(cx, y_x, rf'$x_{t}$',      '#AEDFF7')
        node(cx, y_a, rf'$a_{t}$',      '#F5C6A5')
        node(cx, y_h, rf'$h_{t}$',      '#FFE29A')
        node(cx, y_z, rf'$z_{t}$',      '#D9C2E9')
        node(cx, y_y, rf'$\hat{{y}}_{t}$','#B8E0B8')
        arrow((cx, y_x + r), (cx, y_a - r))
        ax.text(cx + 0.13, (y_x + y_a)/2, r'$w_x$', fontsize=10, color='#B22222')
        arrow((cx - 0.7, y_a), (cx - r, y_a), color='#1F6FB2')
        ax.text(cx - 1.15, y_a + 0.15, r'$b_h$', fontsize=10, color='#1F6FB2')
        arrow((cx, y_a + r), (cx, y_h - r))
        ax.text(cx + 0.13, (y_a + y_h)/2, r'$\tanh$', fontsize=10)
        arrow((cx, y_h + r), (cx, y_z - r))
        ax.text(cx + 0.13, (y_h + y_z)/2, r'$w_y$', fontsize=10, color='#B22222')
        arrow((cx - 0.7, y_z), (cx - r, y_z), color='#1F6FB2')
        ax.text(cx - 1.15, y_z + 0.15, r'$b_y$', fontsize=10, color='#1F6FB2')
        arrow((cx, y_z + r), (cx, y_y - r))
        ax.text(cx + 0.13, (y_z + y_y)/2, r'$\sigma$', fontsize=10)

    arrow((cols[0] + r, y_h), (cols[1] - r, y_a), color='#7B2D8E')
    ax.text((cols[0] + cols[1])/2, (y_h + y_a)/2 + 0.1, r'$w_h$',
            fontsize=12, color='#7B2D8E', ha='center')
    arrow((cols[0] - 1.4, y_h), (cols[0] - r, y_h), color='#7B2D8E')
    ax.text(cols[0] - 1.55, y_h + 0.15, r'$h_0=0$', fontsize=10, color='#7B2D8E')

    ax.text(6.5, 7.7, 'Рис. 4.2. Прямой проход при T = 2',
            ha='center', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

draw_forward_T2()


## 4.4. Функция потерь

После прямого прохода у нас есть два предсказания $\hat y_1$, $\hat y_2$. Сравним их с истинными ответами $y_1$, $y_2$ и сложим потери по шагам (half mean squared error, по одному слагаемому на шаг):

$$L = \frac{1}{2}\,(\hat{y}_1 - y_1)^2 + \frac{1}{2}\,(\hat{y}_2 - y_2)^2. \tag{4.10}$$

Множитель $1/2$ введён по той же причине, что и в главе 3: при дифференцировании он сокращается с двойкой из производной квадрата. Обратите внимание: $L$ — **сумма** двух слагаемых, и каждое $\hat y_t$ входит только в своё слагаемое. Поэтому

$$\frac{\partial L}{\partial \hat y_1} = \frac{\partial}{\partial \hat y_1}\left[\tfrac12(\hat y_1 - y_1)^2\right] = \hat y_1 - y_1, \tag{4.11a}$$

$$\frac{\partial L}{\partial \hat y_2} = \frac{\partial}{\partial \hat y_2}\left[\tfrac12(\hat y_2 - y_2)^2\right] = \hat y_2 - y_2, \tag{4.11b}$$

и никаких перекрёстных вкладов здесь нет: $\hat y_1$ не входит во второе слагаемое, а $\hat y_2$ — в первое.

## 4.5. Обратный проход

Обратный проход (**backward pass**, или backpropagation through time, BPTT) — это вычисление производных $L$ по всем пяти параметрам. При $T = 2$ у каждого параметра **два пути** к $L$ (потому что он входит в оба шага), а у скрытого состояния $h_1$ — **два потребителя**: выход шага 1 и скрытый нейрон шага 2. Мы каждый раз сначала **выписываем полную цепочку по каждому пути**, и только затем сворачиваем повторяющиеся фрагменты в обозначения $\delta^y_t$, $\delta^a_t$. Так видно, откуда берётся каждый множитель, и ни один из них не теряется.

### Таблица числа путей

Прежде чем выписывать цепочки, полезно посчитать, **сколько путей** у каждого параметра к $L$. Это сразу объясняет, почему одни градиенты — суммы двух слагаемых, а другие — одного.

| Параметр | Входит в | Число путей к $L$ | Почему |
|---|---|---|---|
| $w_y$ | $z_1$, $z_2$ | **2** | через $\hat y_1$ и через $\hat y_2$ |
| $b_y$ | $z_1$, $z_2$ | **2** | через $\hat y_1$ и через $\hat y_2$ |
| $w_x$ | $a_1$, $a_2$ | **2** | через $a_1$ и через $a_2$ |
| $w_h$ | $a_2$ (в $a_1$ его нет: $h_0 = 0$) | **1** | только через $a_2$ |
| $b_h$ | $a_1$, $a_2$ | **2** | через $a_1$ и через $a_2$ |

И **число путей от скрытых переменных** к $L$:

| Переменная | Потребители | Число путей к $L$ |
|---|---|---|
| $a_2$ | $\hat y_2$ | **1** |
| $a_1$ | $\hat y_1$ и $a_2$ | **2** |

Именно отсюда берутся два слагаемых в $\delta^a_1$ и одно — в $\delta^a_2$. При $T = 3$ у $a_1$ будет уже **три** пути, у $a_2$ — два, у $a_3$ — один; суммарно $1 + 2 + 3 = 6$. При произвольном $T$ путей от $a_1$ будет $T$, и выписывать их станет невозможно — тогда и понадобится рекуррентная формула.

### Шаг 1. Производная по выходу на шаге $t$

Дифференцируем $L$ по $\hat y_t$. Сумма (4.10) даёт вклад только от одного слагаемого (см. (4.11)):

$$\frac{\partial L}{\partial \hat y_t} = \hat y_t - y_t, \qquad t = 1, 2. \tag{4.12}$$

Это «сигнал ошибки» на шаге $t$ — насколько и в какую сторону мы ошиблись. Заметьте: $\partial L/\partial \hat y_1$ и $\partial L/\partial \hat y_2$ **независимы** — потеря аддитивна.

### Шаг 2. Производная выхода по линейной комбинации $z_t$

Так как $\hat y_t = \sigma(z_t)$, применяем производную сигмоиды (4.4a):

$$\frac{\partial \hat y_t}{\partial z_t} = \sigma'(z_t) = \hat y_t\,(1 - \hat y_t). \tag{4.13}$$

### Шаг 3. Дельта выхода на шаге $t$

Собираем два предыдущих множителя и вводим обозначение для «дельты выхода»:

$$\delta^y_t := \frac{\partial L}{\partial z_t} = \frac{\partial L}{\partial \hat y_t}\cdot\frac{\partial \hat y_t}{\partial z_t} = (\hat y_t - y_t)\,\hat y_t\,(1 - \hat y_t). \tag{4.14}$$

С этого момента $\delta^y_t$ — удобное сокращение для $\partial L/\partial z_t$. Оно полностью вычисляется из величин прямого прохода (а именно из $\hat y_t$ и $y_t$), поэтому дальше мы работаем с ним как с известной константой. Заметьте: $\delta^y_1$ и $\delta^y_2$ **независимы** — как и сигналы ошибки из шага 1.

### Шаг 4. Градиенты параметров выходного слоя

Параметры $w_y$, $b_y$ общие для обоих шагов, поэтому производная по ним — **сумма двух путей**: через $z_1$ и через $z_2$.

**Для $w_y$.** Выпишем полную цепочку по каждому пути. Путь через $z_1$:

$$\frac{\partial L}{\partial w_y}\bigg|_{\text{через } z_1} = \frac{\partial L}{\partial \hat y_1}\cdot\frac{\partial \hat y_1}{\partial z_1}\cdot\frac{\partial z_1}{\partial w_y}.$$

Путь через $z_2$:

$$\frac{\partial L}{\partial w_y}\bigg|_{\text{через } z_2} = \frac{\partial L}{\partial \hat y_2}\cdot\frac{\partial \hat y_2}{\partial z_2}\cdot\frac{\partial z_2}{\partial w_y}.$$

Складываем и подставляем локальные производные из (4.5)–(4.9): $\partial L/\partial \hat y_t \cdot \partial \hat y_t/\partial z_t = \delta^y_t$ (по определению), $\partial z_t/\partial w_y = h_t$:

$$\frac{\partial L}{\partial w_y} = \delta^y_1\,h_1 + \delta^y_2\,h_2. \tag{4.15}$$

В развёрнутом виде (без сокращения $\delta^y_t$):

$$\frac{\partial L}{\partial w_y} = (\hat y_1 - y_1)\,\hat y_1(1-\hat y_1)\,h_1 + (\hat y_2 - y_2)\,\hat y_2(1-\hat y_2)\,h_2. \tag{4.15'}$$

**Для $b_y$.** Структура путей та же, но $\partial z_t/\partial b_y = 1$:

$$\frac{\partial L}{\partial b_y} = \delta^y_1\cdot 1 + \delta^y_2\cdot 1 = \delta^y_1 + \delta^y_2. \tag{4.16}$$

В развёрнутом виде:

$$\frac{\partial L}{\partial b_y} = (\hat y_1 - y_1)\,\hat y_1(1-\hat y_1) + (\hat y_2 - y_2)\,\hat y_2(1-\hat y_2). \tag{4.16'}$$

### Шаг 5. Переход к скрытому слою

Здесь начинается главное отличие от главы 3. У $h_t$ **разное число потребителей** в зависимости от $t$:

- $h_2$ влияет на $L$ только через $z_2$ — **один путь**;
- $h_1$ влияет на $L$ **двумя путями**: через $z_1$ (свой шаг) и через $a_2 \to h_2 \to z_2$ (будущий шаг).

Поэтому и дельты $\delta^a_t := \partial L/\partial a_t$ устроены по-разному. Вычислим их справа налево — начиная с $t = 2$.

### Шаг 6. Дельта $a_2$ — один путь

Цепочка от $L$ до $a_2$ проходит через $\hat y_2$, $z_2$, $h_2$:

$$\delta^a_2 := \frac{\partial L}{\partial a_2} = \frac{\partial L}{\partial \hat y_2}\cdot\frac{\partial \hat y_2}{\partial z_2}\cdot\frac{\partial z_2}{\partial h_2}\cdot\frac{\partial h_2}{\partial a_2}.$$

Подставляем локальные производные: $\partial L/\partial \hat y_2 \cdot \partial \hat y_2/\partial z_2 = \delta^y_2$; $\partial z_2/\partial h_2 = w_y$ (из (4.7)); $\partial h_2/\partial a_2 = 1 - h_2^2$ (из (4.6)). Значит,

$$\boxed{\;\delta^a_2 = \delta^y_2\,w_y\,(1 - h_2^2).\;} \tag{4.17}$$

В развёрнутом виде (без сокращения $\delta^y_2$):

$$\delta^a_2 = (\hat y_2 - y_2)\,\hat y_2(1-\hat y_2)\cdot w_y\cdot(1 - h_2^2). \tag{4.17'}$$

### Шаг 7. Дельта $a_1$ — два пути

Цепочка от $L$ до $a_1$ идёт через $h_1$, а от $h_1$ — **двумя ветками**:

$$
\frac{\partial L}{\partial a_1} = \underbrace{\frac{\partial L}{\partial \hat y_1}\cdot\frac{\partial \hat y_1}{\partial z_1}\cdot\frac{\partial z_1}{\partial h_1}\cdot\frac{\partial h_1}{\partial a_1}}_{\text{путь A: через } \hat y_1} + \underbrace{\frac{\partial L}{\partial a_2}\cdot\frac{\partial a_2}{\partial h_1}\cdot\frac{\partial h_1}{\partial a_1}}_{\text{путь B: через } a_2}.
$$

Разберём каждый множитель:

**Путь A** (свой шаг): $\partial L/\partial \hat y_1 \cdot \partial \hat y_1/\partial z_1 = \delta^y_1$; $\partial z_1/\partial h_1 = w_y$; $\partial h_1/\partial a_1 = 1 - h_1^2$. Итого

$$\text{путь A} = \delta^y_1\,w_y\,(1 - h_1^2).$$

**Путь B** (будущий шаг): $\partial L/\partial a_2 = \delta^a_2$ (только что вычислили в (4.17)); $\partial a_2/\partial h_1 = w_h$ (из (4.5)); $\partial h_1/\partial a_1 = 1 - h_1^2$. Итого

$$\text{путь B} = \delta^a_2\,w_h\,(1 - h_1^2).$$

Складываем и выносим общий множитель $(1 - h_1^2)$:

$$\boxed{\;\delta^a_1 = \bigl(\delta^y_1\,w_y + \delta^a_2\,w_h\bigr)\,(1 - h_1^2).\;} \tag{4.18}$$

В развёрнутом виде (без сокращений):

$$
\delta^a_1 = \Bigl[(\hat y_1 - y_1)\,\hat y_1(1-\hat y_1)\cdot w_y + \delta^a_2\cdot w_h\Bigr]\,(1 - h_1^2). \tag{4.18'}
$$

Сравните с главой 3: там было $\delta_1 = \delta_2\,w_2\,\sigma'(z_1)$ — один множитель. Здесь в скобках **два слагаемых**, и это ровно два пути, которыми $a_1$ влияет на $L$.

### Шаг 8. Градиенты параметров скрытого слоя

Параметры $w_x$, $w_h$, $b_h$ входят в оба шага (кроме $w_h$ на первом шаге), поэтому снова суммируем по путям.

**Для $w_x$.** $w_x$ входит в $a_1$ и $a_2$. Два пути:

$$\frac{\partial L}{\partial w_x} = \frac{\partial L}{\partial a_1}\cdot\frac{\partial a_1}{\partial w_x} + \frac{\partial L}{\partial a_2}\cdot\frac{\partial a_2}{\partial w_x}.$$

Из (4.5): $\partial a_t/\partial w_x = x_t$. Значит,

$$\frac{\partial L}{\partial w_x} = \delta^a_1\,x_1 + \delta^a_2\,x_2. \tag{4.19}$$

**Для $w_h$.** $w_h$ входит в $a_1$ и $a_2$, но $\partial a_1/\partial w_h = h_0 = 0$ — на первом шаге памяти ещё нет. Поэтому остаётся только путь через $a_2$:

$$\frac{\partial L}{\partial w_h} = \frac{\partial L}{\partial a_1}\cdot 0 + \frac{\partial L}{\partial a_2}\cdot\frac{\partial a_2}{\partial w_h}, \qquad \frac{\partial a_2}{\partial w_h} = h_1.$$

$$\frac{\partial L}{\partial w_h} = \delta^a_2\,h_1. \tag{4.20}$$

**Для $b_h$.** $b_h$ входит в оба шага, и $\partial a_t/\partial b_h = 1$:

$$\frac{\partial L}{\partial b_h} = \frac{\partial L}{\partial a_1}\cdot 1 + \frac{\partial L}{\partial a_2}\cdot 1 = \delta^a_1 + \delta^a_2. \tag{4.21}$$

### Сводка обратного прохода при $T = 2$

$$
\boxed{
\begin{aligned}
\delta^y_t &= (\hat y_t - y_t)\,\hat y_t(1-\hat y_t), && t = 1, 2; \\[2pt]
\delta^a_2 &= \delta^y_2\,w_y\,(1 - h_2^2); \\[2pt]
\delta^a_1 &= \bigl(\delta^y_1\,w_y + \delta^a_2\,w_h\bigr)\,(1 - h_1^2); \\[4pt]
\frac{\partial L}{\partial w_y} &= \delta^y_1\,h_1 + \delta^y_2\,h_2, & \frac{\partial L}{\partial b_y} &= \delta^y_1 + \delta^y_2, \\[2pt]
\frac{\partial L}{\partial w_x} &= \delta^a_1\,x_1 + \delta^a_2\,x_2, & \frac{\partial L}{\partial w_h} &= \delta^a_2\,h_1, & \frac{\partial L}{\partial b_h} &= \delta^a_1 + \delta^a_2.
\end{aligned}}
$$

### Схема обратного прохода

Рис. 4.3 объединяет обе фазы: сверху сплошными синими стрелками идёт прямой проход, снизу пунктирными красными — обратный. Видно, что $\delta^a_1$ получает вклад из двух мест — от $\delta^y_1$ (свой шаг) и от $\delta^a_2$ (будущий шаг).




In [ ]:
def draw_backward_T2():
    """Рис. 4.3: обратный проход при T=2 — два пути в δ^a_1."""
    fig, ax = plt.subplots(figsize=(13, 6.5))
    ax.set_xlim(0, 13); ax.set_ylim(0, 8)
    ax.axis('off'); ax.set_aspect('equal')

    x1, x2 = 3.0, 10.0
    y_out, y_a = 6.0, 3.0
    r = 0.42

    def node(cx, cy, label, color):
        ax.add_patch(Circle((cx, cy), r, facecolor=color,
                            edgecolor='black', lw=1.4, zorder=3))
        ax.text(cx, cy, label, ha='center', va='center',
                fontsize=11, zorder=4)

    node(x1, y_out, r'$\hat{y}_1$', '#B8E0B8')
    node(x2, y_out, r'$\hat{y}_2$', '#B8E0B8')
    node(x1, y_a,   r'$a_1$',       '#F5C6A5')
    node(x2, y_a,   r'$a_2$',       '#F5C6A5')

    def arrow(p1, p2, color='#C0392B', ls='--', lw=1.6):
        ax.add_patch(FancyArrowPatch(
            p1, p2, arrowstyle='-|>', mutation_scale=14,
            lw=lw, color=color, linestyle=ls,
            shrinkA=0, shrinkB=0, zorder=2))

    # прямой проход (синие стрелки сверху вниз)
    ax.annotate('', xy=(x1, y_out - r), xytext=(x1, y_a + r),
                arrowprops=dict(arrowstyle='-|>', lw=1.6, color='#1F6FB2'))
    ax.annotate('', xy=(x2, y_out - r), xytext=(x2, y_a + r),
                arrowprops=dict(arrowstyle='-|>', lw=1.6, color='#1F6FB2'))
    ax.text(x1 + 0.12, (y_out + y_a) / 2, r'$\delta^y_1$',
            fontsize=11, color='#C0392B')
    ax.text(x2 + 0.12, (y_out + y_a) / 2, r'$\delta^y_2$',
            fontsize=11, color='#C0392B')

    # обратный проход a_2 → a_1
    arrow((x2 - r, y_a), (x1 + r, y_a))
    ax.text((x1 + x2) / 2, y_a + 0.25,
            r'$\delta^a_1 \leftarrow \delta^a_2\,w_h$',
            ha='center', fontsize=11, color='#C0392B')

    # --- формулы снизу: без \underbrace и без \text{...} ---
    ax.text(6.5, 1.9,
            r'$\delta^a_2 \;=\; \delta^y_2\,w_y\,(1-h_2^2)$'
            '   — один путь (свой шаг)',
            ha='center', fontsize=11)

    ax.text(6.5, 1.1,
            r'$\delta^a_1 \;=\; \delta^y_1\,w_y\,(1-h_1^2)'
            r' \;+\; \delta^a_2\,w_h\,(1-h_1^2)$',
            ha='center', fontsize=11)

    # подписи «путь A» и «путь B» под соответствующими слагаемыми
    ax.text(4.7, 0.6, 'путь A (свой шаг)',
            ha='center', fontsize=10, color='#444')
    ax.text(8.5, 0.6, 'путь B (будущий шаг)',
            ha='center', fontsize=10, color='#444')
    # дуги-стрелки от подписей к слагаемым
    ax.annotate('', xy=(4.7, 1.0), xytext=(4.7, 0.75),
                arrowprops=dict(arrowstyle='-', lw=1.0, color='#444'))
    ax.annotate('', xy=(8.5, 1.0), xytext=(8.5, 0.75),
                arrowprops=dict(arrowstyle='-', lw=1.0, color='#444'))

    ax.text(6.5, 7.5, 'Рис. 4.3. Обратный проход при T = 2',
            ha='center', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

draw_backward_T2()


## 4.6. Обновление параметров

Заключительный шаг — собственно обучение. Все пять параметров обновляются по правилу градиентного спуска с шагом обучения $\eta$. Ключевой момент, как и в главе 3: **производные берутся в той самой точке пространства параметров, в которой сеть сейчас находится**, то есть после прямого прохода с текущими весами. Обозначим текущее состояние параметров через

$$\theta^{(t)} = \bigl(w_x^{(t)},\, w_h^{(t)},\, b_h^{(t)},\, w_y^{(t)},\, b_y^{(t)}\bigr), \tag{4.22}$$

где индекс $t$ — **номер шага обучения** (не путать с шагом времени!). Тогда правило обновления:

$$
\begin{aligned}
w_x^{(t+1)} &:= w_x^{(t)} - \eta \left.\frac{\partial L}{\partial w_x}\right|_{\theta^{(t)}}, & b_h^{(t+1)} &:= b_h^{(t)} - \eta \left.\frac{\partial L}{\partial b_h}\right|_{\theta^{(t)}}, \\
w_h^{(t+1)} &:= w_h^{(t)} - \eta \left.\frac{\partial L}{\partial w_h}\right|_{\theta^{(t)}}, & w_y^{(t+1)} &:= w_y^{(t)} - \eta \left.\frac{\partial L}{\partial w_y}\right|_{\theta^{(t)}}, \\
&& b_y^{(t+1)} &:= b_y^{(t)} - \eta \left.\frac{\partial L}{\partial b_y}\right|_{\theta^{(t)}}.
\end{aligned}
\tag{4.23}
$$

### Что означает «в точке $\theta^{(t)}$»

Когда мы пишем, например,

$$\left.\frac{\partial L}{\partial w_x}\right|_{\theta^{(t)}} = \delta^{a,(t)}_1\,x_1 + \delta^{a,(t)}_2\,x_2,$$

все величины в правой части — $\delta^{a,(t)}_1$, $\delta^{a,(t)}_2$ — **вычислены при текущих значениях параметров** $\theta^{(t)}$, то есть взяты из последнего прямого и обратного прохода. Поэтому обучение идёт итеративно:

1. **Прямой проход** с текущими $\theta^{(t)}$ по всей последовательности даёт $a_1$, $h_1$, $z_1$, $\hat y_1$, $a_2$, $h_2$, $z_2$, $\hat y_2$.
2. **Обратный проход** с этими величинами (справа налево: сначала $\delta^y_2$, потом $\delta^a_2$, потом $\delta^y_1$, $\delta^a_1$) даёт все пять градиентов — **в точке $\theta^{(t)}$**.
3. **Шаг обновления** по (4.23) сдвигает все пять параметров **одновременно** — и получается $\theta^{(t+1)}$.
4. Цикл повторяется; после сдвига весов старые $h_1$, $h_2$, $\hat y_1$, $\hat y_2$ недействительны — нужен новый прямой проход.

Иными словами, формула $w := w - \eta\,\partial L/\partial w$ — это не «одно и то же вычисление», а семейство вычислений: на каждом шаге $t$ в ней фигурирует своя точка $\theta^{(t)}$, и её нужно понимать как «здесь и сейчас».

## 4.7. Один полный шаг обучения на числах

Проделаем один шаг вручную. Возьмём:

- вход $x_1 = 1.0$, $x_2 = 0.5$;
- истинные ответы $y_1 = 0.0$, $y_2 = 1.0$;
- начальные параметры $w_x = 0.5$, $w_h = 0.3$, $b_h = 0.1$, $w_y = 0.8$, $b_y = -0.2$;
- $h_0 = 0$, $\eta = 0.1$;
- $\sigma(z) = 1/(1+e^{-z})$, $\sigma'(z) = \sigma(z)(1-\sigma(z))$; $\tanh'(a) = 1 - \tanh^2(a)$.

### Прямой проход

**Шаг $t = 1$:**

$$a_1 = 0.5 \cdot 1.0 + 0.3 \cdot 0 + 0.1 = 0.6,$$
$$h_1 = \tanh(0.6) \approx 0.53705,$$
$$z_1 = 0.8 \cdot 0.53705 + (-0.2) = 0.22964,$$
$$\hat y_1 = \sigma(0.22964) \approx 0.55716.$$

**Шаг $t = 2$:**

$$a_2 = 0.5 \cdot 0.5 + 0.3 \cdot 0.53705 + 0.1 = 0.25 + 0.16112 + 0.1 = 0.51112,$$
$$h_2 = \tanh(0.51112) \approx 0.47077,$$
$$z_2 = 0.8 \cdot 0.47077 + (-0.2) = 0.17662,$$
$$\hat y_2 = \sigma(0.17662) \approx 0.54404.$$

**Потеря** по (4.10):

$$L = \tfrac12(0.55716 - 0)^2 + \tfrac12(0.54404 - 1)^2 = 0.15521 + 0.10397 = 0.25919.$$

### Обратный проход

**Дельты выходов** по (4.14). Сначала производные сигмоиды:

$$\sigma'(z_1) = 0.55716\cdot(1 - 0.55716) = 0.55716 \cdot 0.44284 \approx 0.24672,$$
$$\sigma'(z_2) = 0.54404\cdot(1 - 0.54404) = 0.54404 \cdot 0.45596 \approx 0.24806.$$

Теперь сами дельты:

$$\delta^y_1 = (\hat y_1 - y_1)\cdot\sigma'(z_1) = (0.55716 - 0)\cdot 0.24672 \approx 0.13747,$$
$$\delta^y_2 = (\hat y_2 - y_2)\cdot\sigma'(z_2) = (0.54404 - 1)\cdot 0.24806 \approx -0.11310.$$

**Производные $\tanh$** по (4.2a):

$$\tanh'(a_1) = 1 - h_1^2 = 1 - 0.53705^2 = 1 - 0.28842 \approx 0.71158,$$
$$\tanh'(a_2) = 1 - h_2^2 = 1 - 0.47077^2 = 1 - 0.22162 \approx 0.77838.$$

**Дельта $a_2$** по (4.17) — один путь:

$$\delta^a_2 = \delta^y_2\,w_y\,\tanh'(a_2) = (-0.11310)\cdot 0.8\cdot 0.77838 \approx -0.07043.$$

**Дельта $a_1$** по (4.18) — два пути, разберём их по отдельности:

$$\text{путь A} = \delta^y_1\,w_y\,\tanh'(a_1) = 0.13747 \cdot 0.8 \cdot 0.71158 \approx 0.07826,$$
$$\text{путь B} = \delta^a_2\,w_h\,\tanh'(a_1) = (-0.07043)\cdot 0.3\cdot 0.71158 \approx -0.01503,$$
$$\delta^a_1 = \text{путь A} + \text{путь B} = 0.07826 - 0.01503 = 0.06323.$$

**Пять градиентов** по (4.15)–(4.21):

$$\frac{\partial L}{\partial w_y} = \delta^y_1\,h_1 + \delta^y_2\,h_2 = 0.13747\cdot 0.53705 + (-0.11310)\cdot 0.47077 \approx 0.07382 - 0.05324 = 0.02058,$$

$$\frac{\partial L}{\partial b_y} = \delta^y_1 + \delta^y_2 = 0.13747 - 0.11310 = 0.02437,$$

$$\frac{\partial L}{\partial w_x} = \delta^a_1\,x_1 + \delta^a_2\,x_2 = 0.06323\cdot 1.0 + (-0.07043)\cdot 0.5 \approx 0.06323 - 0.03522 = 0.02802,$$

$$\frac{\partial L}{\partial w_h} = \delta^a_2\,h_1 = (-0.07043)\cdot 0.53705 \approx -0.03783,$$

$$\frac{\partial L}{\partial b_h} = \delta^a_1 + \delta^a_2 = 0.06323 - 0.07043 = -0.00720.$$

### Обновление

$$w_x \leftarrow 0.5 - 0.1\cdot 0.02802 = 0.49720,$$
$$w_h \leftarrow 0.3 - 0.1\cdot(-0.03783) = 0.30378,$$
$$b_h \leftarrow 0.1 - 0.1\cdot(-0.00720) = 0.10072,$$
$$w_y \leftarrow 0.8 - 0.1\cdot 0.02058 = 0.79794,$$
$$b_y \leftarrow -0.2 - 0.1\cdot 0.02437 = -0.20244.$$

Если теперь выполнить прямой проход с новыми весами, окажется, что $\hat y_1$ немного уменьшилось, а $\hat y_2$ — увеличилось: сеть сделала шаг в сторону правильных ответов $y_1 = 0$, $y_2 = 1$. Повторяя такие шаги много раз, мы получим $\hat y_1 \to 0$, $\hat y_2 \to 1$.

Обратите внимание: $\delta^y_1 > 0$, а $\delta^y_2 < 0$ — вклады разных шагов в $\partial L/\partial b_y$ имеют **разные знаки**. Именно поэтому суммы по путям в §4.5 — не формальность: без них градиент был бы просто неверным.

Все эти вычисления удобно проделать в коде — и заодно проверить, что ручные выкладки совпадают с программными.

## 4.8. Полная программа

Соберём всё вместе: класс `TinyRNN` реализует прямой проход, обратный проход и один шаг обучения; функция `demo_one_step` прогоняет один шаг и печатает все промежуточные величины; функция `train` — несколько итераций.




In [ ]:
import numpy as np

def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))

class TinyRNN:
    def __init__(self, wx=0.5, wh=0.3, bh=0.1, wy=0.8, by=-0.2, eta=0.1):
        self.wx, self.wh, self.bh = wx, wh, bh
        self.wy, self.by, self.eta = wy, by, eta

    def forward(self, xs, h0=0.0):
        hs, ys, h = [], [], h0
        for x in xs:
            a = self.wx * x + self.wh * h + self.bh   # (4.1)
            h = np.tanh(a)                            # (4.2)
            z = self.wy * h + self.by                 # (4.3)
            ys.append(sigmoid(z))                     # (4.4)
            hs.append(h)
        return np.array(hs), np.array(ys)

    def loss(self, ys, targets):
        return 0.5 * np.sum((ys - targets) ** 2)       # (4.10)

    def backward(self, xs, hs, ys, targets, h0=0.0):
        T = len(xs)
        dy = (ys - targets) * ys * (1.0 - ys)         # (4.14) δ^y_t
        dh = np.zeros(T); dh_next = 0.0
        for t in reversed(range(T)):
            dh[t] = (dy[t]*self.wy + dh_next*self.wh) * (1.0 - hs[t]**2)
            dh_next = dh[t]
        h_prev = np.concatenate(([h0], hs[:-1]))
        return dict(
            wx=np.sum(dh * np.array(xs)),              # (4.19)
            wh=np.sum(dh * h_prev),                    # (4.20)
            bh=np.sum(dh),                             # (4.21)
            wy=np.sum(dy * hs),                        # (4.15)
            by=np.sum(dy),                             # (4.16)
        )

    def step(self, xs, targets, h0=0.0):
        hs, ys = self.forward(xs, h0)
        L = self.loss(ys, targets)
        g = self.backward(xs, hs, ys, targets, h0)
        for k, v in g.items():
            setattr(self, k, getattr(self, k) - self.eta * v)   # (4.23)
        return L, ys, g


def demo_one_step():
    net = TinyRNN()
    xs = np.array([1.0, 0.5])
    targets = np.array([0.0, 1.0])
    hs, ys = net.forward(xs)
    print("h  =", hs)
    print("y^ =", ys)
    L, ys, g = net.step(xs, targets)
    print("L  =", L)
    for k, v in g.items():
        print(f"dL/d{k} = {v:+.5f}")


def train(steps=2000, xs=(1.0, 0.5), ys=(0.0, 1.0), eta=0.5):
    net = TinyRNN(eta=eta)
    for _ in range(steps):
        net.step(np.array(xs), np.array(ys))
    hs, ys = net.forward(np.array(xs))
    print("after training: y^ =", ys)

demo_one_step()


Если запустить `demo_one_step()`, выводятся ровно те числа, что мы посчитали вручную в §4.7. Это лучший способ поймать потерянный множитель: расхождение с ручным счётом почти наверняка означает ошибку в обратном проходе — чаще всего забытое слагаемое $\delta^a_2\,w_h$ в формуле для $\delta^a_1$ или забытую сумму по путям в градиентах $w_y$, $b_y$, $w_x$, $b_h$.

## 4.9. Что мы получили

В этой главе на минимальном примере — последовательности из двух шагов — мы разобрали все три фазы обучения RNN:

1. **Прямой проход** — четыре шага на $t = 1$, затем четыре шага на $t = 2$; единственное новое по сравнению с главой 3 — слагаемое $w_h h_{t-1}$ в преактивации скрытого нейрона.
2. **Обратный проход** — цепное правило, применённое к каждому пути отдельно. У $w_y$, $b_y$, $w_x$, $b_h$ — по два пути, у $w_h$ — один (потому что $h_0 = 0$). У $a_1$ — два пути ($\delta^y_1 w_y$ и $\delta^a_2 w_h$), у $a_2$ — один.
3. **Обновление** — пять вычитаний с шагом $\eta$, все одновременно; производные берутся **в текущей точке** $\theta^{(t)}$.

Главное, что нужно запомнить из этой главы: **два слагаемых в скобках формулы (4.18) для $\delta^a_1$ — это два пути, которыми $a_1$ влияет на $L$**. Когда $T$ вырастет, путей станет $T(T+1)/2$, и выписывать их все будет невозможно. Тогда-то и понадобится рекуррентное соотношение $\delta^a_t = (\delta^y_t w_y + \delta^a_{t+1} w_h)(1-h_t^2)$ — оно просто сворачивает эти пути в одну строку. Но при $T = 2$ всё ещё видно, **что именно** оно сворачивает — и это лучшая подготовка к главе про общее $T$.

В следующей главе мы возьмём $T = 3$: там у $a_1$ будет уже **три** пути, появится первое произведение $w_h^2$, и станет окончательно ясно, зачем нужна рекурсия. В главе про общее $T$ мы свернём её в одно рекуррентное соотношение и покажем, как из него получаются все пять градиентов. Но структура останется той же, что и здесь: двигаемся по графу вычислений справа налево, перемножая локальные производные из таблицы §4.3 и суммируя вклады всех путей. Формулы этой главы — тот строительный блок, из которого складывается обучение более длинных последовательностей.

# Глава 5. Пример 3: тот же один нейрон, но $T = 3$ — как растёт число путей

В главе 4 мы разобрали $T = 2$: там у каждого параметра было ровно два пути к $L$, а у $a_1$ — ровно два потребителя ($\hat y_1$ и $a_2$). Теперь добавим **ещё один шаг**. Ничего принципиально нового в архитектуре не появится — те же пять параметров, те же уравнения, — но у $a_1$ будет уже **три потребителя** ($\hat y_1$, $a_2$ и — косвенно — $a_3$), а в цепочках впервые возникнет произведение $w_h^2$. Эта глава — **переходная**: её задача — показать, что число путей растёт линейно по глубине и квадратично суммарно, и что выписывать их с каждым шагом всё труднее. Именно отсюда в главе 6 родится рекуррентная формула для $\delta^a_t$.

## 5.1. Зачем нужен этот пример

При $T = 2$ у $a_1$ было два пути к $L$: «свой» (через $\hat y_1$) и «будущий» (через $a_2$). При $T = 3$ добавляется **третий** путь — через $a_3$. Схематически:

$$
a_1 \;\to\; \hat y_1, \qquad a_1 \;\to\; a_2 \;\to\; \hat y_2, \qquad a_1 \;\to\; a_2 \;\to\; a_3 \;\to\; \hat y_3.
$$

Третий путь — самый длинный: в нём производная $\partial L/\partial a_1$ содержит **произведение двух** весов $w_h$ (по одному на каждый рекуррентный переход). При $T = 4$ таких переходов будет уже три, при $T = 100$ — девяносто девять. Эта глава показывает, как это произведение возникает и почему оно важно.

Формально:
- **прямой проход** — те же четыре операции, повторённые три раза;
- **обратный проход** — цепное правило, но теперь у $a_1$ **три пути**, у $a_2$ — два, у $a_3$ — один;
- **обновление** — те же пять вычитаний, что и в главе 4; меняется только состав градиентов.

Мы не будем повторять всё, что уже было в главе 4, — только то, что нового. Всё, что относится к локальным производным (уравнения (4.5)–(4.9) главы 4), остаётся в силе без изменений: это «кирпичи», которые не зависят от $T$.

## 5.2. Архитектура и обозначения

Те же пять параметров: $w_x$, $w_h$, $b_h$, $w_y$, $b_y$. Те же уравнения прямого прохода для каждого $t = 1, 2, 3$:

$$
a_t = w_x\,x_t + w_h\,h_{t-1} + b_h, \qquad h_t = \tanh(a_t), \qquad z_t = w_y\,h_t + b_y, \qquad \hat y_t = \sigma(z_t). \tag{5.1}
$$

Начальное состояние $h_0 = 0$, как и в главе 4. Всё отличие — ещё один шаг $t = 3$. На рис. 5.1 — архитектура, развёрнутая на три шага.

```python
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch

def draw_architecture_T3():
    """Рис. 5.1: архитектура 1 → 1 → 1, развёрнутая на три шага."""
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.set_xlim(0, 14); ax.set_ylim(0, 8)
    ax.set_aspect('equal'); ax.axis('off')
    r = 0.5
    cols = [2.0, 7.0, 12.0]
    y_x, y_h, y_y = 2.2, 4.6, 6.5

    def arrow(p1, p2, color='black'):
        ax.add_patch(FancyArrowPatch(
            p1, p2, arrowstyle='-|>', mutation_scale=18,
            lw=2, color=color, shrinkA=0, shrinkB=0, zorder=2))

    for t, cx in enumerate(cols, start=1):
        for cy, label, color in [
            (y_x, rf'$x_{t}$',          '#AEDFF7'),
            (y_h, rf'$h_{t}$',          '#FFE29A'),
            (y_y, rf'$\hat{{y}}_{t}$',  '#B8E0B8'),
        ]:
            ax.add_patch(Circle((cx, cy), r, facecolor=color,
                                edgecolor='black', lw=1.8, zorder=3))
            ax.text(cx, cy, label, ha='center', va='center',
                    fontsize=15, zorder=4)
        arrow((cx, y_x + r), (cx, y_h - r))
        ax.text(cx + 0.18, (y_x + y_h) / 2, r'$w_x$',
                fontsize=12, color='#B22222')
        arrow((cx, y_h + r), (cx, y_y - r))
        ax.text(cx + 0.18, (y_h + y_y) / 2, r'$w_y$',
                fontsize=12, color='#B22222')
        arrow((cx - r - 0.7, y_h), (cx - r, y_h), color='#1F6FB2')
        ax.text(cx - r - 1.1, y_h + 0.2, r'$b_h$',
                fontsize=11, color='#1F6FB2')
        arrow((cx - r - 0.7, y_y), (cx - r, y_y), color='#1F6FB2')
        ax.text(cx - r - 1.1, y_y + 0.2, r'$b_y$',
                fontsize=11, color='#1F6FB2')

    for i in range(2):
        arrow((cols[i] + r, y_h), (cols[i+1] - r, y_h), color='#7B2D8E')
        ax.text((cols[i] + cols[i+1]) / 2, y_h + 0.22, r'$w_h$',
                fontsize=13, color='#7B2D8E', ha='center')
    arrow((cols[0] - r - 0.7, y_h), (cols[0] - r, y_h), color='#7B2D8E')
    ax.text(cols[0] - r - 1.1, y_h + 0.2, r'$h_0 = 0$',
            fontsize=10, color='#7B2D8E')

    ax.text(7.0, 7.7, 'Рис. 5.1. Архитектура 1 → 1 → 1, развёрнутая на три шага',
            ha='center', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

draw_architecture_T3()
```

## 5.3. Прямой проход

Прямой проход — те же уравнения (5.1), применённые трижды. Выпишем их явно для $T = 3$:

$$
\begin{aligned}
&\textbf{Шаг } t = 1: & a_1 &= w_x x_1 + b_h, & h_1 &= \tanh(a_1), & z_1 &= w_y h_1 + b_y, & \hat y_1 &= \sigma(z_1); \\
&\textbf{Шаг } t = 2: & a_2 &= w_x x_2 + w_h h_1 + b_h, & h_2 &= \tanh(a_2), & z_2 &= w_y h_2 + b_y, & \hat y_2 &= \sigma(z_2); \\
&\textbf{Шаг } t = 3: & a_3 &= w_x x_3 + w_h h_2 + b_h, & h_3 &= \tanh(a_3), & z_3 &= w_y h_3 + b_y, & \hat y_3 &= \sigma(z_3).
\end{aligned}
$$

Обратите внимание: $a_1$ не содержит $w_h h_0$ (потому что $h_0 = 0$), а $a_2$ и $a_3$ — содержат. Поэтому $w_h$ входит в два шага из трёх, а $w_x$ — во все три.

Подстановка чисел ($x_1 = 1.0$, $x_2 = 0.5$, $x_3 = -0.3$; параметры — как в §4.7):

$$
\begin{aligned}
a_1 &= 0.5\cdot 1.0 + 0.1 = 0.6, & h_1 &= \tanh(0.6) \approx 0.53705, \\
z_1 &= 0.8\cdot 0.53705 - 0.2 = 0.22964, & \hat y_1 &\approx 0.55716; \\[4pt]
a_2 &= 0.5\cdot 0.5 + 0.3\cdot 0.53705 + 0.1 = 0.51112, & h_2 &= \tanh(0.51112) \approx 0.47077, \\
z_2 &= 0.8\cdot 0.47077 - 0.2 = 0.17662, & \hat y_2 &\approx 0.54404; \\[4pt]
a_3 &= 0.5\cdot (-0.3) + 0.3\cdot 0.47077 + 0.1 = 0.09123, & h_3 &= \tanh(0.09123) \approx 0.09097, \\
z_3 &= 0.8\cdot 0.09097 - 0.2 = -0.12723, & \hat y_3 &\approx 0.46823.
\end{aligned}
$$

## 5.4. Функция потерь

$$
L = \tfrac12(\hat y_1 - y_1)^2 + \tfrac12(\hat y_2 - y_2)^2 + \tfrac12(\hat y_3 - y_3)^2. \tag{5.2}
$$

Каждое $\hat y_t$ входит только в своё слагаемое, поэтому

$$
\frac{\partial L}{\partial \hat y_t} = \hat y_t - y_t, \qquad t = 1, 2, 3, \tag{5.3}
$$

и никаких перекрёстных вкладов нет — как и в главе 4.

## 5.5. Обратный проход: считаем пути

Прежде чем выписывать цепочки, посчитаем **число путей** — это главный результат главы. От $a_t$ до $L$ ведёт столько путей, сколько шагов осталось до конца последовательности плюс один (путь через «свой» выход):

$$
\#\text{путей от } a_t \text{ до } L \;=\; T - t + 1. \tag{5.4}
$$

Для $T = 3$:

| Переменная | Потребители | Число путей к $L$ | Список путей |
|---|---|---|---|
| $a_3$ | $\hat y_3$ | **1** | через $\hat y_3$ |
| $a_2$ | $\hat y_2$, $a_3$ | **2** | через $\hat y_2$; через $a_3 \to \hat y_3$ |
| $a_1$ | $\hat y_1$, $a_2$ | **3** | через $\hat y_1$; через $a_2 \to \hat y_2$; через $a_2 \to a_3 \to \hat y_3$ |

**Всего путей от скрытых переменных** к $L$: $1 + 2 + 3 = 6 = \dfrac{3\cdot 4}{2}$. Сравните с $T = 2$: там было $1 + 2 = 3$ пути. При $T = 100$ будет $1 + 2 + \dots + 100 = 5050$ путей.

**Число путей у параметров** (для $T = 3$):

| Параметр | Входит в | Число путей к $L$ |
|---|---|---|
| $w_y$ | $z_1, z_2, z_3$ | **3** |
| $b_y$ | $z_1, z_2, z_3$ | **3** |
| $w_x$ | $a_1, a_2, a_3$ | **3** |
| $w_h$ | $a_2, a_3$ (в $a_1$ его нет) | **2** |
| $b_h$ | $a_1, a_2, a_3$ | **3** |

Именно из этой таблицы видно, почему при $T = 100$ выписывать цепочки вручную бессмысленно: у $a_1$ будет 100 путей, у $w_h$ — 99.

### Шаг 1. Дельты выходов $\delta^y_t$

Как и в главе 4, начинаем с выходов. Определение $\delta^y_t := \partial L/\partial z_t$ от $T$ не зависит:

$$
\delta^y_t = (\hat y_t - y_t)\,\hat y_t\,(1-\hat y_t), \qquad t = 1, 2, 3. \tag{5.5}
$$

Три независимых значения — по одному на каждый шаг.

### Шаг 2. Градиенты $w_y$ и $b_y$ — три пути

$w_y$ и $b_y$ входят во все три шага, поэтому их градиенты — **сумма трёх путей**:

$$
\frac{\partial L}{\partial w_y} = \delta^y_1\,h_1 + \delta^y_2\,h_2 + \delta^y_3\,h_3, \tag{5.6}
$$

$$
\frac{\partial L}{\partial b_y} = \delta^y_1 + \delta^y_2 + \delta^y_3. \tag{5.7}
$$

Здесь структура та же, что в главе 4, просто слагаемых стало три.

### Шаг 3. Дельта $a_3$ — один путь

Самая дальняя переменная — $a_3$. У неё только один потребитель: $\hat y_3$. Цепочка:

$$
\delta^a_3 := \frac{\partial L}{\partial a_3} = \frac{\partial L}{\partial \hat y_3}\cdot\frac{\partial \hat y_3}{\partial z_3}\cdot\frac{\partial z_3}{\partial h_3}\cdot\frac{\partial h_3}{\partial a_3} = \delta^y_3\,w_y\,(1 - h_3^2). \tag{5.8}
$$

Как и $\delta^a_2$ в главе 4 — один путь.

### Шаг 4. Дельта $a_2$ — два пути

У $a_2$ два потребителя: $\hat y_2$ (свой выход) и $a_3$ (следующий шаг). Значит, **два пути**:

**Путь через $\hat y_2$:** $\delta^y_2\,w_y\,(1 - h_2^2)$.

**Путь через $a_3$:** $\delta^a_3\,w_h\,(1 - h_2^2)$.

Складываем и выносим общий множитель $(1 - h_2^2)$:

$$
\delta^a_2 = \bigl(\delta^y_2\,w_y + \delta^a_3\,w_h\bigr)\,(1 - h_2^2). \tag{5.9}
$$

Это в точности формула (4.18) из главы 4, только с индексом 2 вместо 1 и с $\delta^a_3$ вместо $\delta^a_2$. Структура та же.

### Шаг 5. Дельта $a_1$ — три пути, впервые появляется $w_h^2$

Вот главное. У $a_1$ **три** потребителя, и три пути к $L$. Выпишем их **все полностью**:

**Путь A (свой шаг):** $a_1 \to h_1 \to z_1 \to \hat y_1$:

$$
\text{путь A} = \delta^y_1\,w_y\,(1 - h_1^2). \tag{5.10a}
$$

**Путь B (один шаг назад по времени):** $a_1 \to h_1 \to a_2 \to h_2 \to z_2 \to \hat y_2$:

$$
\text{путь B} = \delta^y_2\,w_y\,(1 - h_2^2)\cdot w_h\cdot(1 - h_1^2). \tag{5.10b}
$$

**Путь C (два шага назад):** $a_1 \to h_1 \to a_2 \to h_2 \to a_3 \to h_3 \to z_3 \to \hat y_3$:

$$
\text{путь C} = \delta^y_3\,w_y\,(1 - h_3^2)\cdot w_h\cdot(1 - h_2^2)\cdot w_h\cdot(1 - h_1^2). \tag{5.10c}
$$

Обратите внимание: путь C содержит **произведение $w_h^2$** — по одному множителю на каждый рекуррентный переход. Именно здесь впервые видно, откуда возьмётся затухание: при $|w_h| < 1$ множитель $w_h^2$ существенно уменьшает вклад дальнего шага.

Складываем все три и группируем по $(1 - h_1^2)$:

$$
\delta^a_1 = \text{путь A} + \text{путь B} + \text{путь C}. \tag{5.11}
$$

Заметим, что путь B — это ровно $\delta^a_2$ (без множителя $(1 - h_1^2)$), а путь C — это $\delta^a_3\,w_h\,(1 - h_2^2)$ (снова без $(1 - h_1^2)$). Поэтому (5.11) можно записать короче:

$$
\boxed{\;\delta^a_1 = \bigl(\delta^y_1\,w_y + \delta^a_2\,w_h\bigr)\,(1 - h_1^2).\;} \tag{5.12}
$$

Сравните с (4.18): формула **символьно совпадает**. Это и есть момент озарения главы 5: **при переходе от $T = 2$ к $T = 3$ формула не изменилась** — изменилось только то, что в правой части теперь стоит $\delta^a_2$, которое само содержит вклад пути C. Иными словами, рекуррентная формула $\delta^a_t = (\delta^y_t w_y + \delta^a_{t+1} w_h)(1 - h_t^2)$ **уже видна** в (5.9) и (5.12) — она просто применяется дважды. В главе 6 мы её запишем в общем виде и покажем, что она не меняется при любом $T$.

### Шаг 6. Градиенты $w_x$, $w_h$, $b_h$

Параметры скрытого слоя — снова суммы по путям:

$$
\frac{\partial L}{\partial w_x} = \delta^a_1\,x_1 + \delta^a_2\,x_2 + \delta^a_3\,x_3, \tag{5.13}
$$

$$
\frac{\partial L}{\partial w_h} = \delta^a_1\cdot h_0 + \delta^a_2\cdot h_1 + \delta^a_3\cdot h_2 = \delta^a_2\,h_1 + \delta^a_3\,h_2, \tag{5.14}
$$

$$
\frac{\partial L}{\partial b_h} = \delta^a_1 + \delta^a_2 + \delta^a_3. \tag{5.15}
$$

В (5.14) первое слагаемое равно нулю, потому что $h_0 = 0$ — как и в главе 4. У $w_h$ два пути (через $a_2$ и через $a_3$), у $w_x$ и $b_h$ — три.

### Сводка обратного прохода при $T = 3$

$$
\boxed{
\begin{aligned}
\delta^y_t &= (\hat y_t - y_t)\,\hat y_t(1-\hat y_t), && t = 1, 2, 3; \\[2pt]
\delta^a_3 &= \delta^y_3\,w_y\,(1 - h_3^2); \\[2pt]
\delta^a_2 &= \bigl(\delta^y_2\,w_y + \delta^a_3\,w_h\bigr)\,(1 - h_2^2); \\[2pt]
\delta^a_1 &= \bigl(\delta^y_1\,w_y + \delta^a_2\,w_h\bigr)\,(1 - h_1^2); \\[4pt]
\frac{\partial L}{\partial w_y} &= \delta^y_1\,h_1 + \delta^y_2\,h_2 + \delta^y_3\,h_3, & \frac{\partial L}{\partial b_y} &= \delta^y_1 + \delta^y_2 + \delta^y_3, \\[2pt]
\frac{\partial L}{\partial w_x} &= \delta^a_1\,x_1 + \delta^a_2\,x_2 + \delta^a_3\,x_3, & \frac{\partial L}{\partial w_h} &= \delta^a_2\,h_1 + \delta^a_3\,h_2, \\[2pt]
&& \frac{\partial L}{\partial b_h} &= \delta^a_1 + \delta^a_2 + \delta^a_3.
\end{aligned}}
$$

### Схема обратного прохода

Рис. 5.2 показывает три пути от $a_1$ к $L$ и подчёркивает, что путь C — самый длинный, с произведением $w_h^2$.

```python
def draw_backward_T3():
    """Рис. 5.2: три пути от a_1 к L — и как они сворачиваются в δ^a_1."""
    fig, ax = plt.subplots(figsize=(13, 6))
    ax.set_xlim(0, 13); ax.set_ylim(0, 8)
    ax.axis('off'); ax.set_aspect('equal')
    x1, x2, x3 = 2.0, 6.5, 11.0
    y_out, y_a = 6.5, 2.5
    r = 0.42

    def node(cx, cy, label, color):
        ax.add_patch(Circle((cx, cy), r, facecolor=color,
                            edgecolor='black', lw=1.4, zorder=3))
        ax.text(cx, cy, label, ha='center', va='center',
                fontsize=11, zorder=4)

    for cx, lab in [(x1, r'$\hat y_1$'), (x2, r'$\hat y_2$'), (x3, r'$\hat y_3$')]:
        node(cx, y_out, lab, '#B8E0B8')
    for cx, lab in [(x1, r'$a_1$'), (x2, r'$a_2$'), (x3, r'$a_3$')]:
        node(cx, y_a, lab, '#F5C6A5')

    # три «петли» — три пути от a_1 к L
    ax.annotate('', xy=(x1, y_out - r), xytext=(x1, y_a + r),
                arrowprops=dict(arrowstyle='-|>', lw=2, color='#C0392B',
                                connectionstyle="arc3,rad=0.0"))
    ax.annotate('', xy=(x2, y_out - r), xytext=(x1, y_a + r),
                arrowprops=dict(arrowstyle='-|>', lw=2, color='#C0392B',
                                connectionstyle="arc3,rad=-0.25"))
    ax.annotate('', xy=(x3, y_out - r), xytext=(x1, y_a + r),
                arrowprops=dict(arrowstyle='-|>', lw=2, color='#C0392B',
                                connectionstyle="arc3,rad=-0.45"))

    ax.text(x1 - 0.15, (y_out + y_a)/2, r'путь A', fontsize=10,
            color='#C0392B', ha='right')
    ax.text(3.5, 4.7, r'путь B $(w_h)$', fontsize=10, color='#C0392B')
    ax.text(6.5, 5.7, r'путь C $(w_h^2)$', fontsize=10, color='#C0392B')

    ax.text(6.5, 1.0,
            r'$\delta^a_1 = \underbrace{\delta^y_1 w_y (1-h_1^2)}_{A} + \underbrace{\delta^a_2 w_h (1-h_1^2)}_{B + C}$',
            ha='center', fontsize=11)
    ax.text(6.5, 0.2,
            r'где $\delta^a_2 = \delta^y_2 w_y (1-h_2^2) + \delta^a_3 w_h (1-h_2^2)$  — уже два пути',
            ha='center', fontsize=10)

    ax.text(6.5, 7.5, 'Рис. 5.2. Обратный проход при T = 3',
            ha='center', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

draw_backward_T3()
```

## 5.6. Обновление параметров

Правило то же, что и в главе 4 — по (4.23), с теми же пятью параметрами. Единственное отличие — величины $\delta^{a,(t)}_1, \delta^{a,(t)}_2, \delta^{a,(t)}_3$ в правых частях теперь учитывают три шага, а не два. То есть один пример — это уже последовательность длины 3; один шаг обучения = один прямой проход по всем трём шагам + один обратный проход + одно обновление пяти параметров.

## 5.7. Один полный шаг обучения на числах

Возьмём те же параметры, что и в §4.7: $w_x = 0.5$, $w_h = 0.3$, $b_h = 0.1$, $w_y = 0.8$, $b_y = -0.2$; $h_0 = 0$, $\eta = 0.1$. Входы: $x_1 = 1.0$, $x_2 = 0.5$, $x_3 = -0.3$. Истинные ответы: $y_1 = 0.0$, $y_2 = 1.0$, $y_3 = 0.5$.

### Прямой проход (числа — из §5.3)

$$h_1 \approx 0.53705, \quad \hat y_1 \approx 0.55716, \qquad h_2 \approx 0.47077, \quad \hat y_2 \approx 0.54404, \qquad h_3 \approx 0.09097, \quad \hat y_3 \approx 0.46823.$$

**Потеря** по (5.2):

$$
L = \tfrac12(0.55716)^2 + \tfrac12(0.54404 - 1)^2 + \tfrac12(0.46823 - 0.5)^2 = 0.15521 + 0.10397 + 0.00050 = 0.25968.
$$

### Обратный проход

**Дельты выходов** по (5.5):

$$
\delta^y_1 \approx 0.13747, \qquad \delta^y_2 \approx -0.11310, \qquad \delta^y_3 \approx -0.00791.
$$

**Производные $\tanh$:**

$$
\tanh'(a_1) \approx 0.71158, \qquad \tanh'(a_2) \approx 0.77838, \qquad \tanh'(a_3) \approx 0.99172.
$$

**Дельты скрытых (справа налево) по (5.8), (5.9), (5.12):**

$$
\delta^a_3 = (-0.00791)\cdot 0.8\cdot 0.99172 \approx -0.00628,
$$

$$
\delta^a_2 = \bigl((-0.11310)\cdot 0.8 + (-0.00628)\cdot 0.3\bigr)\cdot 0.77838 \approx (-0.09048 - 0.00188)\cdot 0.77838 \approx -0.07189,
$$

$$
\delta^a_1 = \bigl(0.13747\cdot 0.8 + (-0.07189)\cdot 0.3\bigr)\cdot 0.71158 \approx (0.10998 - 0.02157)\cdot 0.71158 \approx 0.06291.
$$

**Три пути для $\delta^a_1$** по (5.10) — проверим, что сумма совпадает с (5.12):

$$
\text{путь A} = 0.13747\cdot 0.8\cdot 0.71158 \approx 0.07826,
$$
$$
\text{путь B} = (-0.11310)\cdot 0.8\cdot 0.77838\cdot 0.3\cdot 0.71158 \approx -0.01503,
$$
$$
\text{путь C} = (-0.00791)\cdot 0.8\cdot 0.99172\cdot 0.3\cdot 0.77838\cdot 0.3\cdot 0.71158 \approx -0.00031.
$$

Сумма: $0.07826 - 0.01503 - 0.00031 = 0.06292$. Совпадает с (5.12) до последнего знака. **Обратите внимание**: вклад пути C в 50 раз меньше вклада пути A — это уже при $T = 3$ видно, как далёкий шаг «затухает» из-за $w_h^2$.

**Пять градиентов** по (5.6), (5.7), (5.13)–(5.15):

$$
\frac{\partial L}{\partial w_y} = 0.13747\cdot 0.53705 + (-0.11310)\cdot 0.47077 + (-0.00791)\cdot 0.09097 \approx 0.01986,
$$

$$
\frac{\partial L}{\partial b_y} = 0.13747 - 0.11310 - 0.00791 \approx 0.01646,
$$

$$
\frac{\partial L}{\partial w_x} = 0.06291\cdot 1.0 + (-0.07189)\cdot 0.5 + (-0.00628)\cdot (-0.3) \approx 0.02885,
$$

$$
\frac{\partial L}{\partial w_h} = 0 + (-0.07189)\cdot 0.53705 + (-0.00628)\cdot 0.47077 \approx -0.04156,
$$

$$
\frac{\partial L}{\partial b_h} = 0.06291 - 0.07189 - 0.00628 \approx -0.01526.
$$

### Обновление

$$
\begin{aligned}
w_x &\leftarrow 0.5 - 0.1\cdot 0.02885 = 0.49712, & b_h &\leftarrow 0.1 - 0.1\cdot(-0.01526) = 0.10153, \\
w_h &\leftarrow 0.3 - 0.1\cdot(-0.04156) = 0.30416, & w_y &\leftarrow 0.8 - 0.1\cdot 0.01986 = 0.79801, \\
&& b_y &\leftarrow -0.2 - 0.1\cdot 0.01646 = -0.20165.
\end{aligned}
$$

Сравните с главой 4: значения близки, но не совпадают — третий шаг вносит свой вклад во все пять градиентов.

## 5.8. Полная программа

Код — тот же, что в главе 4, просто длина `xs` и `targets` равна 3. Ничего в классе `TinyRNN` менять не нужно — циклы по $t$ уже написаны в общем виде.

```python
import numpy as np

def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))

class TinyRNN:
    def __init__(self, wx=0.5, wh=0.3, bh=0.1, wy=0.8, by=-0.2, eta=0.1):
        self.wx, self.wh, self.bh = wx, wh, bh
        self.wy, self.by, self.eta = wy, by, eta

    def forward(self, xs, h0=0.0):
        hs, ys, h = [], [], h0
        for x in xs:
            a = self.wx * x + self.wh * h + self.bh   # (5.1)
            h = np.tanh(a)
            z = self.wy * h + self.by
            ys.append(sigmoid(z))
            hs.append(h)
        return np.array(hs), np.array(ys)

    def loss(self, ys, targets):
        return 0.5 * np.sum((ys - targets) ** 2)       # (5.2)

    def backward(self, xs, hs, ys, targets, h0=0.0):
        T = len(xs)
        dy = (ys - targets) * ys * (1.0 - ys)         # (5.5) δ^y_t
        dh = np.zeros(T); dh_next = 0.0
        for t in reversed(range(T)):                   # (5.8)–(5.12)
            dh[t] = (dy[t]*self.wy + dh_next*self.wh) * (1.0 - hs[t]**2)
            dh_next = dh[t]
        h_prev = np.concatenate(([h0], hs[:-1]))
        return dict(
            wx=np.sum(dh * np.array(xs)),              # (5.13)
            wh=np.sum(dh * h_prev),                    # (5.14)
            bh=np.sum(dh),                             # (5.15)
            wy=np.sum(dy * hs),                        # (5.6)
            by=np.sum(dy),                             # (5.7)
        )

    def step(self, xs, targets, h0=0.0):
        hs, ys = self.forward(xs, h0)
        L = self.loss(ys, targets)
        g = self.backward(xs, hs, ys, targets, h0)
        for k, v in g.items():
            setattr(self, k, getattr(self, k) - self.eta * v)
        return L, ys, g


def demo_T3():
    net = TinyRNN()
    xs = np.array([1.0, 0.5, -0.3])
    targets = np.array([0.0, 1.0, 0.5])
    hs, ys = net.forward(xs)
    print("h  =", hs)      # [0.53705 0.47077 0.09097]
    print("y^ =", ys)      # [0.55716 0.54404 0.46823]
    L, ys, g = net.step(xs, targets)
    print("L  =", L)       # 0.25968
    for k, v in g.items():
        print(f"dL/d{k} = {v:+.5f}")
    # dL/dwx = +0.02885, dL/dwh = -0.04156, dL/dbh = -0.01526,
    # dL/dwy = +0.01986, dL/dby = +0.01646

demo_T3()
```

Вывод совпадает с ручным расчётом §5.7. Обратите внимание: **программа не изменилась** — это аргумент в пользу того, что формулы главы 5 и главы 4 — разные записи одного и того же; рекуррентная формула для $\delta^a_t$ уже работает при любом $T$, и код это подтверждает.

## 5.9. Что мы получили и что дальше

Итог главы 5:

1. **Число путей растёт линейно по глубине и квадратично суммарно**: у $a_3$ — 1, у $a_2$ — 2, у $a_1$ — 3; всего 6. При $T=100$ будет 5050.
2. **Первое произведение $w_h^2$** появляется в пути C — самом длинном. Именно этот множитель и отвечает за затухание дальних вкладов: уже при $T=3$ путь C в 50 раз меньше пути A.
3. **Рекуррентная формула $\delta^a_t = (\delta^y_t w_y + \delta^a_{t+1} w_h)(1 - h_t^2)$ уже видна**: в (5.9) и (5.12) она применена дважды. При переходе от $T=2$ к $T=3$ она **символьно не изменилась** — изменилось только то, что в правой части стоит $\delta^a_{t+1}$, само содержащее дальние вклады.
4. **Код не изменился** — это подтверждает, что формула уже в общем виде.

В главе 6 мы запишем эту формулу **один раз** и покажем, что при любом $T$ она даёт все пять градиентов. Собственно, весь BPTT — это:

$$
\delta^a_t = \bigl(\delta^y_t\,w_y + \delta^a_{t+1}\,w_h\bigr)(1 - h_t^2), \qquad t = T, T-1, \dots, 1,
$$

плюс пять сумм по $t$. Всё остальное — детали. Но запомнить эту формулу и понять, **что именно** она сворачивает, помогает именно случай $T = 3$: три пути для $a_1$ — это и есть «содержимое» одного рекуррентного шага.

# Глава 6. Общий случай: $T$ произвольно — обратный проход во времени (BPTT)

В главе 4 мы разобрали $T = 2$: там всё выписывалось руками, и было видно каждый путь. В главе 5 — $T = 3$: путей стало шесть, впервые появилось произведение $w_h^2$, и обнаружилось, что формула для $\delta^a_1$ **символьно не изменилась** по сравнению с $T = 2$. Эта глава — обобщение. Её задача: записать формулы **один раз** для произвольного $T$, доказать, что они верны, и показать, что главы 4 и 5 — просто частные случаи. Всё «почему» уже разобрано в главах 4–5; здесь — «как в общем виде».

## 6.1. Зачем нужна эта глава

При $T = 2$ число путей от скрытых переменных к $L$ равно $3$; при $T = 3$ — $6$; при $T = 100$ — $5050$. Выписывать столько путей невозможно. Нужен способ вычислять все пять градиентов, не выписывая пути явно.

Таким способом служит **рекуррентное соотношение** для дельт. Оно позволяет найти $\delta^a_T$, затем $\delta^a_{T-1}$ из него, затем $\delta^a_{T-2}$ — и так до $\delta^a_1$. Каждый шаг требует $O(1)$ операций, всего — $O(T)$. Именно это и есть **backpropagation through time (BPTT)**.

В главе 4 мы видели зачаток этой идеи: $\delta^a_2$ вычислялась первой, потом $\delta^a_1$ через неё. В главе 5 то же самое повторилось дважды: сначала $\delta^a_3$, потом $\delta^a_2$ через неё, потом $\delta^a_1$ через $\delta^a_2$. Здесь мы просто **запишем этот шаблон в общем виде** и докажем, что он работает для любого $T$.

## 6.2. Архитектура и обозначения

Те же пять параметров, те же уравнения на каждом шаге $t = 1, 2, \dots, T$:

$$
a_t = w_x\,x_t + w_h\,h_{t-1} + b_h, \qquad h_t = \tanh(a_t), \qquad z_t = w_y\,h_t + b_y, \qquad \hat y_t = \sigma(z_t). \tag{6.1}
$$

Начальное состояние $h_0 = 0$. Обозначения — как в главах 4–5; напомним:

| Символ | Смысл |
|---|---|
| $x_t$ | вход на шаге $t$ |
| $a_t$ | преактивация скрытого: $a_t = w_x x_t + w_h h_{t-1} + b_h$ |
| $h_t$ | скрытое состояние: $h_t = \tanh(a_t)$ |
| $z_t$ | преактивация выхода: $z_t = w_y h_t + b_y$ |
| $\hat y_t$ | предсказание: $\hat y_t = \sigma(z_t)$ |
| $y_t$ | истинный ответ на шаге $t$ |
| $w_x$, $w_h$, $b_h$, $w_y$, $b_y$ | пять обучаемых параметров, **общие для всех** $t$ |
| $h_0 = 0$ | начальное состояние |
| $T$ | длина последовательности |
| $\eta$ | шаг обучения |

На рис. 6.1 — схема, развёрнутая на произвольное $T$ (нарисованы $T-1$, $T$ и «$\dots$»).

```python
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch

def draw_architecture_TN():
    """Рис. 6.1: архитектура 1 → 1 → 1, развёрнутая на произвольное T."""
    fig, ax = plt.subplots(figsize=(14, 6.5))
    ax.set_xlim(0, 14); ax.set_ylim(0, 7.5)
    ax.set_aspect('equal'); ax.axis('off')
    r = 0.45
    cols = [1.5, 4.5, 10.0, 13.0]
    y_x, y_h, y_y = 2.2, 4.4, 6.2

    def arrow(p1, p2, color='black'):
        ax.add_patch(FancyArrowPatch(
            p1, p2, arrowstyle='-|>', mutation_scale=16,
            lw=1.8, color=color, shrinkA=0, shrinkB=0, zorder=2))

    labels_h = [r'$h_1$', r'$h_2$', r'$h_{T-1}$', r'$h_T$']
    labels_x = [r'$x_1$', r'$x_2$', r'$x_{T-1}$', r'$x_T$']
    labels_y = [r'$\hat y_1$', r'$\hat y_2$', r'$\hat y_{T-1}$', r'$\hat y_T$']
    colors = ['#FFE29A'] * 4
    colors_in = ['#AEDFF7'] * 4
    colors_out = ['#B8E0B8'] * 4

    for i, cx in enumerate(cols):
        for cy, lab, col in [(y_x, labels_x[i], colors_in[i]),
                              (y_h, labels_h[i], colors[i]),
                              (y_y, labels_y[i], colors_out[i])]:
            ax.add_patch(Circle((cx, cy), r, facecolor=col,
                                edgecolor='black', lw=1.6, zorder=3))
            ax.text(cx, cy, lab, ha='center', va='center',
                    fontsize=13, zorder=4)
        arrow((cx, y_x + r), (cx, y_h - r))
        ax.text(cx + 0.15, (y_x + y_h)/2, r'$w_x$', fontsize=10, color='#B22222')
        arrow((cx, y_h + r), (cx, y_y - r))
        ax.text(cx + 0.15, (y_h + y_y)/2, r'$w_y$', fontsize=10, color='#B22222')
        arrow((cx - r - 0.5, y_h), (cx - r, y_h), color='#1F6FB2')
        ax.text(cx - r - 0.85, y_h + 0.15, r'$b_h$', fontsize=9, color='#1F6FB2')
        arrow((cx - r - 0.5, y_y), (cx - r, y_y), color='#1F6FB2')
        ax.text(cx - r - 0.85, y_y + 0.15, r'$b_y$', fontsize=9, color='#1F6FB2')

    # рекуррентные связи
    for i, j in [(0, 1), (2, 3)]:
        arrow((cols[i] + r, y_h), (cols[j] - r, y_h), color='#7B2D8E')
        ax.text((cols[i] + cols[j])/2, y_h + 0.2, r'$w_h$',
                fontsize=11, color='#7B2D8E', ha='center')

    # многоточие между h_2 и h_{T-1}
    ax.text((cols[1] + cols[2])/2, y_h, r'$\cdots$', ha='center', va='center',
            fontsize=20, color='#7B2D8E')

    # h_0
    arrow((cols[0] - r - 0.5, y_h), (cols[0] - r, y_h), color='#7B2D8E')
    ax.text(cols[0] - r - 0.9, y_h + 0.15, r'$h_0=0$', fontsize=9, color='#7B2D8E')

    ax.text(7.0, 7.1, 'Рис. 6.1. Архитектура 1 → 1 → 1, развёрнутая на произвольное T',
            ha='center', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

draw_architecture_TN()
```

## 6.3. Прямой проход

Прямой проход — это применение (6.1) последовательно для $t = 1, \dots, T$. Цикл **нельзя распараллелить**: $h_t$ нужен для $a_{t+1}$.

$$
\begin{aligned}
t = 1: & \quad a_1 = w_x x_1 + b_h, & h_1 &= \tanh(a_1), & z_1 &= w_y h_1 + b_y, & \hat y_1 &= \sigma(z_1), \\
t = 2: & \quad a_2 = w_x x_2 + w_h h_1 + b_h, & h_2 &= \tanh(a_2), & z_2 &= w_y h_2 + b_y, & \hat y_2 &= \sigma(z_2), \\
\cdots & \\
t = T: & \quad a_T = w_x x_T + w_h h_{T-1} + b_h, & h_T &= \tanh(a_T), & z_T &= w_y h_T + b_y, & \hat y_T &= \sigma(z_T).
\end{aligned}
$$

**Результат прямого прохода** — набор величин $\{(a_t, h_t, z_t, \hat y_t)\}_{t=1}^{T}$, которые понадобятся в обратном проходе.

Локальные производные — те же, что в §4.3 главы 4; приведём их ещё раз для полноты, потому что в обратном проходе мы будем ссылаться на них по номерам (6.2):

$$
\frac{\partial a_t}{\partial w_x} = x_t, \qquad \frac{\partial a_t}{\partial w_h} = h_{t-1}, \qquad \frac{\partial a_t}{\partial b_h} = 1, \qquad \frac{\partial a_t}{\partial h_{t-1}} = w_h, \tag{6.2a}
$$

$$
\frac{\partial h_t}{\partial a_t} = \tanh'(a_t) = 1 - h_t^2, \tag{6.2b}
$$

$$
\frac{\partial z_t}{\partial h_t} = w_y, \qquad \frac{\partial z_t}{\partial w_y} = h_t, \qquad \frac{\partial z_t}{\partial b_y} = 1, \tag{6.2c}
$$

$$
\frac{\partial \hat y_t}{\partial z_t} = \sigma'(z_t) = \hat y_t(1-\hat y_t). \tag{6.2d}
$$

Эти «кирпичи» **не зависят от $T$** — важно для дальнейшего.

## 6.4. Функция потерь

$$
L = \sum_{t=1}^{T} \frac{1}{2}\,(\hat y_t - y_t)^2. \tag{6.3}
$$

Дифференцируем по $\hat y_t$: сумма даёт вклад только от $t$-го слагаемого (все остальные от $\hat y_t$ не зависят):

$$
\frac{\partial L}{\partial \hat y_t} = \hat y_t - y_t. \tag{6.4}
$$

## 6.5. Обратный проход (BPTT)

### 6.5.1. Сколько путей — общая формула

В главах 4 и 5 мы считали пути для $T=2$ и $T=3$. Общая формула:

$$
\#\text{путей от } a_t \text{ до } L = T - t + 1. \tag{6.5}
$$

Доказательство — прямым подсчётом. Из $a_t$ можно дойти до $L$ через:
- $\hat y_t$ — свой шаг ($1$ путь);
- $a_{t+1} \to \hat y_{t+1}$ — один шаг вперёд ($1$ путь);
- $a_{t+1} \to a_{t+2} \to \hat y_{t+2}$ — два шага вперёд ($1$ путь);
- $\dots$
- $a_{t+1} \to \dots \to a_T \to \hat y_T$ — $T - t$ шагов вперёд ($1$ путь).

Итого $1 + (T - t) = T - t + 1$ путей. Суммарно по всем $t$:

$$
\sum_{t=1}^{T} (T - t + 1) = T + (T-1) + \dots + 1 = \frac{T(T+1)}{2}. \tag{6.6}
$$

При $T = 2$ — $3$ пути, при $T = 3$ — $6$, при $T = 100$ — $5050$. Видно, что выписывать все пути вручную бессмысленно: нужна рекурсия.

**Таблица числа путей** (проверка):

| $T$ | $\#\{$путей$\}$ | Сравнение с главами 4–5 |
|---|---|---|
| 2 | $1 + 2 = 3$ | гл. 4: $a_2$ — 1, $a_1$ — 2 |
| 3 | $1 + 2 + 3 = 6$ | гл. 5: $a_3$ — 1, $a_2$ — 2, $a_1$ — 3 |
| 10 | $55$ | — |
| 100 | $5050$ | — |

### 6.5.2. Ключевая рекуррентная формула

**Утверждение.** Определим $\delta^a_t := \partial L/\partial a_t$. Тогда

$$
\boxed{\;
\delta^a_t = \bigl(\delta^y_t\,w_y + \delta^a_{t+1}\,w_h\bigr)\,(1 - h_t^2), \quad t = 1, \dots, T-1;
\qquad
\delta^a_T = \delta^y_T\,w_y\,(1 - h_T^2).
\;}
\tag{6.7}
$$

**Доказательство.** Разберём переменную $a_t$. Она входит ровно в одно уравнение прямого прохода: $h_t = \tanh(a_t)$. Значит,

$$
\delta^a_t = \frac{\partial L}{\partial a_t} = \frac{\partial L}{\partial h_t}\cdot\frac{\partial h_t}{\partial a_t} = \frac{\partial L}{\partial h_t}\,(1 - h_t^2). \tag{6.8}
$$

Осталось найти $\partial L/\partial h_t$. **Сколько уравнений содержат $h_t$?** Ровно два:

- $z_t = w_y h_t + b_y$ — выход **своего** шага $t$;
- $a_{t+1} = w_x x_{t+1} + w_h h_t + b_h$ — преактивация скрытого **следующего** шага $t+1$.

При $t = T$ второго уравнения нет — «дальше времени нет». Поэтому при $t < T$ полная производная $\partial L/\partial h_t$ — сумма двух вкладов:

$$
\frac{\partial L}{\partial h_t} = \underbrace{\frac{\partial L}{\partial z_t}\cdot\frac{\partial z_t}{\partial h_t}}_{\text{вклад через шаг } t} + \underbrace{\frac{\partial L}{\partial a_{t+1}}\cdot\frac{\partial a_{t+1}}{\partial h_t}}_{\text{вклад через шаг } t+1}. \tag{6.9}
$$

Подставим локальные производные из (6.2):

$$
\frac{\partial z_t}{\partial h_t} = w_y, \qquad \frac{\partial a_{t+1}}{\partial h_t} = w_h.
$$

И учтём определения $\delta^y_t := \partial L/\partial z_t$ и $\delta^a_{t+1} := \partial L/\partial a_{t+1}$:

$$
\frac{\partial L}{\partial h_t} = \delta^y_t\,w_y + \delta^a_{t+1}\,w_h. \tag{6.10}
$$

Подставляем (6.10) в (6.8) — получаем (6.7) для $t < T$. Для $t = T$ второго слагаемого в (6.9) нет, поэтому $\partial L/\partial h_T = \delta^y_T\,w_y$, и (6.7) даёт $\delta^a_T = \delta^y_T\,w_y\,(1 - h_T^2)$. $\blacksquare$

**Проверка на $T = 2$** (глава 4):

$$
\delta^a_2 = \delta^y_2\,w_y\,(1-h_2^2) \quad \text{— совпадает с (4.17);}
$$
$$
\delta^a_1 = \bigl(\delta^y_1\,w_y + \delta^a_2\,w_h\bigr)(1-h_1^2) \quad \text{— совпадает с (4.18).}
$$

**Проверка на $T = 3$** (глава 5):

$$
\delta^a_3 = \delta^y_3\,w_y\,(1-h_3^2) \quad \text{— совпадает с (5.8);}
$$
$$
\delta^a_2 = \bigl(\delta^y_2\,w_y + \delta^a_3\,w_h\bigr)(1-h_2^2) \quad \text{— совпадает с (5.9);}
$$
$$
\delta^a_1 = \bigl(\delta^y_1\,w_y + \delta^a_2\,w_h\bigr)(1-h_1^2) \quad \text{— совпадает с (5.12).}
$$

**Смысл формулы.** Скобка в (6.7) — это два пути, которыми $a_t$ влияет на $L$: «свой» ($\delta^y_t w_y$) и «будущий» ($\delta^a_{t+1} w_h$). Множитель $(1 - h_t^2)$ — производная $\tanh$ на шаге $t$. Рекурсия работает потому, что «будущий» вклад $\delta^a_{t+1}$ **уже посчитан** на предыдущей итерации обратного прохода — именно поэтому мы идём справа налево, от $t = T$ к $t = 1$.

**Порядок вычисления:**

1. Вычисляем все $\delta^y_t$ по формуле (см. §6.5.3) — независимо на каждом шаге.
2. Полагаем $t = T$, вычисляем $\delta^a_T$ по базовому случаю (6.7).
3. Для $t = T-1, T-2, \dots, 1$: вычисляем $\delta^a_t$ по рекуррентной части (6.7), используя уже посчитанное $\delta^a_{t+1}$.

### 6.5.3. Дельты выходов

Из (6.2d) и (6.4), по определению $\delta^y_t := \partial L/\partial z_t$:

$$
\delta^y_t = \frac{\partial L}{\partial \hat y_t}\cdot\frac{\partial \hat y_t}{\partial z_t} = (\hat y_t - y_t)\,\hat y_t\,(1 - \hat y_t), \qquad t = 1, \dots, T. \tag{6.11}
$$

Все $T$ значений $\delta^y_t$ **независимы** — потеря аддитивна, $\hat y_t$ входит только в $t$-е слагаемое.

### 6.5.4. Пять градиентов — суммы по путям

Каждый параметр входит во все (или почти все) шаги, поэтому производная по нему — сумма по шагам. Выпишем полные цепочки (как в главах 4 и 5), но сразу в свёрнутом виде через $\delta$.

**Для $w_y$.** $w_y$ входит в $z_t$ на каждом шаге $t$; $\partial z_t/\partial w_y = h_t$:

$$
\frac{\partial L}{\partial w_y} = \sum_{t=1}^{T} \frac{\partial L}{\partial z_t}\cdot\frac{\partial z_t}{\partial w_y} = \sum_{t=1}^{T} \delta^y_t\,h_t. \tag{6.12}
$$

**Для $b_y$.** $\partial z_t/\partial b_y = 1$:

$$
\frac{\partial L}{\partial b_y} = \sum_{t=1}^{T} \delta^y_t. \tag{6.13}
$$

**Для $w_x$.** $w_x$ входит в $a_t$ на каждом шаге; $\partial a_t/\partial w_x = x_t$:

$$
\frac{\partial L}{\partial w_x} = \sum_{t=1}^{T} \frac{\partial L}{\partial a_t}\cdot\frac{\partial a_t}{\partial w_x} = \sum_{t=1}^{T} \delta^a_t\,x_t. \tag{6.14}
$$

**Для $w_h$.** $\partial a_t/\partial w_h = h_{t-1}$:

$$
\frac{\partial L}{\partial w_h} = \sum_{t=1}^{T} \delta^a_t\,h_{t-1}. \tag{6.15}
$$

При $t = 1$ имеем $h_0 = 0$, так что первое слагаемое равно нулю — это автоматически учитывается формулой.

**Для $b_h$.** $\partial a_t/\partial b_h = 1$:

$$
\frac{\partial L}{\partial b_h} = \sum_{t=1}^{T} \delta^a_t. \tag{6.16}
$$

### 6.5.5. Сводка обратного прохода (BPTT)

$$
\boxed{
\begin{aligned}
&\text{1. Для } t = 1, \dots, T: \quad \delta^y_t = (\hat y_t - y_t)\,\hat y_t(1-\hat y_t); \\[4pt]
&\text{2. } \delta^a_T = \delta^y_T\,w_y\,(1 - h_T^2); \\[2pt]
&\text{3. Для } t = T-1, \dots, 1: \quad \delta^a_t = \bigl(\delta^y_t\,w_y + \delta^a_{t+1}\,w_h\bigr)(1 - h_t^2); \\[4pt]
&\text{4. } \frac{\partial L}{\partial w_y} = \sum_{t} \delta^y_t\,h_t, \quad \frac{\partial L}{\partial b_y} = \sum_{t} \delta^y_t, \\
&\text{5. } \frac{\partial L}{\partial w_x} = \sum_{t} \delta^a_t\,x_t, \quad \frac{\partial L}{\partial w_h} = \sum_{t} \delta^a_t\,h_{t-1}, \quad \frac{\partial L}{\partial b_h} = \sum_{t} \delta^a_t.
\end{aligned}}
\tag{6.17}
$$

Шаги 2 и 3 — это и есть собственно BPTT: обратный проход по времени, где $\delta^a_{t+1}$ «протаскивает» ошибку из будущего в прошлое.

## 6.6. Обновление параметров

Правило — то же, что и в главах 4–5: градиентный спуск в текущей точке $\theta^{(t)}$:

$$
w_i^{(t+1)} := w_i^{(t)} - \eta\left.\frac{\partial L}{\partial w_i}\right|_{\theta^{(t)}}, \qquad b_i^{(t+1)} := b_i^{(t)} - \eta\left.\frac{\partial L}{\partial b_i}\right|_{\theta^{(t)}}, \tag{6.18}
$$

где $w_i \in \{w_x, w_h, w_y\}$, $b_i \in \{b_h, b_y\}$, а $\theta^{(t)} = (w_x^{(t)}, w_h^{(t)}, b_h^{(t)}, w_y^{(t)}, b_y^{(t)})$ — **текущая точка** в пространстве параметров. Все пять параметров обновляются одновременно, по градиентам, вычисленным в одной и той же точке.

После обновления старые $\{a_t, h_t, z_t, \hat y_t\}$ недействительны — нужен новый прямой проход. Один шаг обучения = прямой проход по всей последовательности + обратный проход по всей последовательности + одно обновление.

## 6.7. Численный пример: $T = 4$

Продолжим ряд из глав 4–5. Параметры: $w_x = 0.5$, $w_h = 0.3$, $b_h = 0.1$, $w_y = 0.8$, $b_y = -0.2$; $h_0 = 0$; $\eta = 0.1$. Входы и ответы: $x = (1.0,\, 0.5,\, -0.3,\, 0.8)$; $y = (0.0,\, 1.0,\, 0.5,\, 0.0)$.

### Прямой проход

$$
\begin{aligned}
t=1: && a_1 &= 0.6, & h_1 &= 0.53705, & z_1 &= 0.22964, & \hat y_1 &= 0.55716, \\
t=2: && a_2 &= 0.51112, & h_2 &= 0.47077, & z_2 &= 0.17662, & \hat y_2 &= 0.54404, \\
t=3: && a_3 &= 0.09123, & h_3 &= 0.09097, & z_3 &= -0.12722, & \hat y_3 &= 0.46823, \\
t=4: && a_4 &= 0.52729, & h_4 &= 0.48275, & z_4 &= 0.18620, & \hat y_4 &= 0.54642.
\end{aligned}
$$

Потеря:

$$
L = \tfrac12[(\hat y_1)^2 + (\hat y_2 - 1)^2 + (\hat y_3 - 0.5)^2 + (\hat y_4)^2] \approx 0.40895.
$$

### Обратный проход

**Дельты выходов** (6.11):

$$
\delta^y_1 \approx 0.13747, \quad \delta^y_2 \approx -0.11310, \quad \delta^y_3 \approx -0.00791, \quad \delta^y_4 \approx 0.13543.
$$

**Производные $\tanh$:**

$$
\tanh'(a_1) \approx 0.71158, \quad \tanh'(a_2) \approx 0.77838, \quad \tanh'(a_3) \approx 0.99172, \quad \tanh'(a_4) \approx 0.76695.
$$

**Дельты скрытых** по (6.7), справа налево:

$$
\delta^a_4 = \delta^y_4\,w_y\,\tanh'(a_4) = 0.13543\cdot 0.8\cdot 0.76695 \approx 0.08309,
$$
$$
\delta^a_3 = (\delta^y_3\,w_y + \delta^a_4\,w_h)\tanh'(a_3) = (-0.00633 + 0.02493)\cdot 0.99172 \approx 0.01846,
$$
$$
\delta^a_2 = (\delta^y_2\,w_y + \delta^a_3\,w_h)\tanh'(a_2) = (-0.09048 + 0.00554)\cdot 0.77838 \approx -0.06612,
$$
$$
\delta^a_1 = (\delta^y_1\,w_y + \delta^a_2\,w_h)\tanh'(a_1) = (0.10998 - 0.01984)\cdot 0.71158 \approx 0.06414.
$$

**Пять градиентов** по (6.12)–(6.16):

$$
\frac{\partial L}{\partial w_y} = \sum_t \delta^y_t\,h_t \approx 0.08525,
$$
$$
\frac{\partial L}{\partial b_y} = \sum_t \delta^y_t \approx 0.15189,
$$
$$
\frac{\partial L}{\partial w_x} = \sum_t \delta^a_t\,x_t \approx 0.09201,
$$
$$
\frac{\partial L}{\partial w_h} = \sum_t \delta^a_t\,h_{t-1} \approx 0 + (-0.06612)\cdot 0.53705 + 0.01846\cdot 0.47077 + 0.08309\cdot 0.09097 \approx -0.01926,
$$
$$
\frac{\partial L}{\partial b_h} = \sum_t \delta^a_t \approx 0.09957.
$$

**Проверка совпадения с главами 4–5.** При $T = 2$ (первые два шага, $y = (0, 1)$) формулы (6.12)–(6.16) дают ровно числа §4.7. При $T = 3$ (первые три шага, $y = (0, 1, 0.5)$) — числа §5.7. Здесь $T=4$ даёт новые числа — это ожидаемо, потому что добавился четвёртый шаг со своей ошибкой $\delta^y_4 = 0.13543$, а она входит во все $\delta^a_t$.

### Обновление

$$
w_x \leftarrow 0.5 - 0.1\cdot 0.09201 = 0.49080, \qquad w_h \leftarrow 0.3 - 0.1\cdot(-0.01926) = 0.30193,
$$
$$
b_h \leftarrow 0.1 - 0.1\cdot 0.09957 = 0.09004, \qquad w_y \leftarrow 0.8 - 0.1\cdot 0.08525 = 0.79147,
$$
$$
b_y \leftarrow -0.2 - 0.1\cdot 0.15189 = -0.21519.
$$

## 6.8. Полная программа

Код — тот же, что в главах 4–5. Меняется только длина `xs` и `targets`. Именно это и подтверждает, что формулы (6.17) общие: класс `TinyRNN` уже реализует BPTT, и никаких изменений для произвольного $T$ не требуется.

```python
import numpy as np

def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))

class TinyRNN:
    def __init__(self, wx=0.5, wh=0.3, bh=0.1, wy=0.8, by=-0.2, eta=0.1):
        self.wx, self.wh, self.bh = wx, wh, bh
        self.wy, self.by, self.eta = wy, by, eta

    def forward(self, xs, h0=0.0):
        hs, ys, h = [], [], h0
        for x in xs:
            a = self.wx * x + self.wh * h + self.bh   # (6.1)
            h = np.tanh(a)
            z = self.wy * h + self.by
            ys.append(sigmoid(z))
            hs.append(h)
        return np.array(hs), np.array(ys)

    def loss(self, ys, targets):
        return 0.5 * np.sum((ys - targets) ** 2)       # (6.3)

    def backward(self, xs, hs, ys, targets, h0=0.0):
        T = len(xs)
        dy = (ys - targets) * ys * (1.0 - ys)         # (6.11)
        dh = np.zeros(T); dh_next = 0.0
        for t in reversed(range(T)):                   # (6.7)
            dh[t] = (dy[t]*self.wy + dh_next*self.wh) * (1.0 - hs[t]**2)
            dh_next = dh[t]
        h_prev = np.concatenate(([h0], hs[:-1]))
        return dict(
            wx=np.sum(dh * np.array(xs)),              # (6.14)
            wh=np.sum(dh * h_prev),                    # (6.15)
            bh=np.sum(dh),                             # (6.16)
            wy=np.sum(dy * hs),                        # (6.12)
            by=np.sum(dy),                             # (6.13)
        )

    def step(self, xs, targets, h0=0.0):
        hs, ys = self.forward(xs, h0)
        L = self.loss(ys, targets)
        g = self.backward(xs, hs, ys, targets, h0)
        for k, v in g.items():
            setattr(self, k, getattr(self, k) - self.eta * v)   # (6.18)
        return L, ys, g


def demo_TN(xs, ys):
    net = TinyRNN()
    L, _, g = net.step(np.array(xs), np.array(ys))
    print(f"T = {len(xs)}, L = {L:.5f}")
    for k, v in g.items():
        print(f"  dL/d{k} = {v:+.5f}")


demo_TN([1.0, 0.5], [0.0, 1.0])                       # T = 2: 0.25919 и т. д.
demo_TN([1.0, 0.5, -0.3], [0.0, 1.0, 0.5])            # T = 3: 0.25968 и т. д.
demo_TN([1.0, 0.5, -0.3, 0.8], [0.0, 1.0, 0.5, 0.0])  # T = 4: 0.40895 и т. д.
```

Запустив `demo_TN` на трёх разных длинах, вы получите ровно числа из §4.7, §5.7 и §6.7. Это прямое доказательство, что рекуррентная формула (6.7) работает при любом $T$ и не требует изменений в коде.

## 6.9. Следствие: затухание и взрыв градиентов

Формула (6.7) позволяет понять, почему в простой RNN плохо обучается долгая память. Раскроем её по цепочке:

$$
\delta^a_t = \delta^y_t\,w_y\,(1-h_t^2) + \delta^a_{t+1}\,w_h\,(1-h_t^2).
$$

Подставим $\delta^a_{t+1}$ (выраженное через $\delta^a_{t+2}$), затем $\delta^a_{t+2}$ и т. д. — получим явное выражение через дельты выходов всех будущих шагов:

$$
\delta^a_t = \underbrace{\delta^y_t\,w_y\,(1-h_t^2)}_{\text{свой шаг}} \;+\; \underbrace{\delta^y_{t+1}\,w_y\,w_h\,(1-h_{t+1}^2)(1-h_t^2)}_{\text{через шаг } t+1} \;+\; \dots
$$

Общий вид вклада от шага $s \geq t$:

$$
\text{вклад от } s \;=\; \delta^y_s\,w_y\;\underbrace{w_h^{\,s-t}}_{\text{рекуррентная часть}}\;\underbrace{\prod_{k=t}^{s}(1-h_k^2)}_{\text{активации}}. \tag{6.19}
$$

Именно эту сумму сворачивает рекурсия (6.7) — но её свойства видны из (6.19). Множители:

- $w_h^{\,s-t}$ — произведение веса за $s - t$ шагов. При $|w_h| < 1$ оно **убывает экспоненциально** по $s - t$; при $|w_h| > 1$ — **растёт экспоненциально**;
- $\prod_{k=t}^{s}(1-h_k^2)$ — произведение производных $\tanh$. Каждый множитель $\leq 1$ (обычно $\ll 1$), и произведение тоже **убывает**.

**Итог.** Если $|w_h| < 1$, дальние шаги вносят в $\delta^a_t$ ничтожно малый вклад — градиент «затухает». Если $|w_h| > 1$, вклад растёт и обучение «взрывается»; спасает только произведение $(1-h_k^2) \leq 1$, но при большом $T$ оно не всегда компенсирует рост $w_h^{s-t}$.

Уже в главе 5 при $T = 3$ мы видели численный пример: вклад «пути C» был примерно в 50 раз меньше вклада «пути A», при том что $w_h = 0.3 < 1$. При $T = 100$ произведение $w_h^{99} \approx 10^{-52}$ — вклад практически обнуляется. Это и есть **проблема затухающих градиентов**; LSTM и GRU (следующие главы) появятся как способ заменить множители $w_h$ и $\tanh'$ на «вентили», которые умеют сохранять память.

**Проверка на $T = 4$** (§6.7). Сравним вклады от разных шагов в $\delta^a_1$: путь A ($s = 1$) даёт $0.07826$, путь через $s=2$ даёт $\approx -0.01503$, через $s=3$ даёт $\approx -0.00031$, через $s=4$ даёт $\approx -0.00006$. Видно, что каждый следующий шаг в 4–50 раз меньше предыдущего — это и есть экспоненциальное затухание в действии.

## 6.10. Что мы получили

1. **Прямой проход** — те же уравнения (6.1), повторённые $T$ раз. Единственное отличие от главы 3 — слагаемое $w_h h_{t-1}$ в преактивации скрытого нейрона.
2. **Обратный проход (BPTT)** — рекуррентное соотношение (6.7) для $\delta^a_t$, применяемое справа налево, плюс пять сумм по $t$ (6.12)–(6.16).
3. **Доказательство рекуррентной формулы** — прямое: у $h_t$ ровно два потребителя ($z_t$ и $a_{t+1}$), поэтому $\partial L/\partial h_t$ — сумма двух вкладов, и после умножения на $\tanh'(a_t)$ получается (6.7). Формула (6.7) **символьно совпадает** с тем, что мы видели в главах 4 и 5 для $T=2$ и $T=3$.
4. **Число путей** — $T - t + 1$ для $\delta^a_t$ и $T(T+1)/2$ суммарно; при больших $T$ пути не выписываются, а сворачиваются рекурсией.
5. **Затухание градиентов** — следствие (6.19): произведение $w_h^{\,s-t}$ и $\prod(1-h_k^2)$ делает вклад дальних шагов экспоненциально малым (или большим). Это ограничивает память простой RNN и мотивирует LSTM/GRU.

Дальше мы будем наращивать число нейронов и слоёв — но структура останется той же: двигаемся по графу вычислений справа налево, применяем на каждом шаге рекуррентное соотношение (6.7), а в конце суммируем вклады по всем шагам. Формулы этой главы — тот строительный блок, из которого складывается обучение более глубоких рекуррентных сетей.

# Глава 7. Пример 4: $N$ скрытых нейронов, $T = 2$ — переход от скаляров к векторам

В главах 4–6 все величины были скалярами: один вход, один скрытый нейрон, один выход. Это позволило выписать каждую производную в явном виде и проследить каждый путь. Но реальные сети состоят из **многих** нейронов, и уже при $N = 2$ скалярная запись становится нечитаемой: пришлось бы писать $a_{t,1}, a_{t,2}, h_{t,1}, h_{t,2}, \dots$ и отдельные уравнения для каждого нейрона. Эта глава — переход к **векторно-матричной записи**. Никакой новой математики в ней нет: всё та же цепочка, что в главе 4, но объекты становятся векторами и матрицами. При $N = 1$ глава 7 **дословно сводится** к главе 4 — это полезная проверка.

## 7.1. Зачем нужен этот пример

При $N = 1$ каждое уравнение прямого прохода — одна строка. При $N$ нейронах тех же уравнений становится $N$ на каждом шаге, и если писать их все, получится $4T \cdot N$ строк. Вместо этого мы группируем $N$ скаляров в вектор и записываем уравнения **один раз** — но уже для векторов.

Появляются три новых объекта:

- **вектор** $h_t \in \mathbb{R}^N$ — скрытое состояние, $N$ чисел;
- **матрица** $W_h \in \mathbb{R}^{N \times N}$ — веса рекуррентной связи, $N^2$ чисел;
- **вектор** $w_y \in \mathbb{R}^N$ — веса выходного нейрона.

И одно новое действие — **поэлементное умножение** $\odot$ (произведение Адамара): $(u \odot v)_i = u_i v_i$. Оно понадобится потому, что $\tanh$ применяется к вектору покомпонентно, и её производная — тоже покомпонентная.

Главная идея главы: **все формулы главы 4 сохраняются**, если заменить:

| Было (глава 4) | Стало (глава 7) |
|---|---|
| скаляр $a_t$ | вектор $a_t \in \mathbb{R}^N$ |
| произведение $w_x x_t$ | матрично-векторное $W_x x_t$ |
| произведение $w_h h_{t-1}$ | матрично-векторное $W_h h_{t-1}$ |
| произведение $w_y h_t$ | скалярное $w_y^\top h_t$ |
| $1 - h_t^2$ | $1 - h_t \odot h_t$ (поэлементно) |
| $w_2 a_1$ (скаляр × скаляр) | $W_y a_1$ или $w_y^\top h_t$ — по контексту |

## 7.2. Архитектура и обозначения

Сеть имеет **вход размерности $D$**, **$N$ скрытых нейронов** и **один скалярный выход**, развёрнута на $T = 2$ шага.

| Символ | Размерность | Смысл |
|---|---|---|
| $x_t$ | $\mathbb{R}^D$ | входной вектор на шаге $t$ |
| $a_t$ | $\mathbb{R}^N$ | преактивация скрытого слоя |
| $h_t$ | $\mathbb{R}^N$ | скрытое состояние (активация) |
| $z_t$ | $\mathbb{R}$ | преактивация выхода (скаляр) |
| $\hat y_t$ | $\mathbb{R}$ | предсказание |
| $y_t$ | $\mathbb{R}$ | истинный ответ |
| $W_x$ | $\mathbb{R}^{N \times D}$ | веса «вход → скрытый» |
| $W_h$ | $\mathbb{R}^{N \times N}$ | веса рекуррентной связи |
| $b_h$ | $\mathbb{R}^N$ | смещение скрытого слоя |
| $w_y$ | $\mathbb{R}^N$ | веса «скрытый → выход» |
| $b_y$ | $\mathbb{R}$ | смещение выхода |
| $h_0$ | $\mathbb{R}^N$ | начальное состояние, $h_0 = \mathbf{0}$ |
| $L$ | $\mathbb{R}$ | функция потерь |
| $\eta$ | $\mathbb{R}$ | шаг обучения |

**Число обучаемых параметров:**

$$
\underbrace{ND}_{W_x} + \underbrace{N^2}_{W_h} + \underbrace{N}_{b_h} + \underbrace{N}_{w_y} + \underbrace{1}_{b_y} = ND + N^2 + 2N + 1. \tag{7.1}
$$

При $N = 1, D = 1$ это $1 + 1 + 2 + 1 = 5$ — ровно пять параметров главы 4 ($w_x, w_h, b_h, w_y, b_y$). При $N = 2, D = 1$ — $2 + 4 + 4 + 1 = 11$. При $N = 128, D = 64$ — $8192 + 16384 + 256 + 1 = 24833$.

**Уравнения прямого прохода** (для каждого $t = 1, 2$):

$$
\boxed{
\begin{aligned}
a_t &= W_x\,x_t + W_h\,h_{t-1} + b_h &&\in \mathbb{R}^N, \\
h_t &= \tanh(a_t) &&\in \mathbb{R}^N \text{ (покомпонентно)}, \\
z_t &= w_y^\top\,h_t + b_y &&\in \mathbb{R}, \\
\hat y_t &= \sigma(z_t) &&\in \mathbb{R}.
\end{aligned}}
\tag{7.2}
$$

**Покомпонентная запись тех же уравнений** (полезна, чтобы увидеть, откуда берутся матрицы):

$$
a_{t,i} = \sum_{j=1}^{D} (W_x)_{ij}\,x_{t,j} + \sum_{j=1}^{N} (W_h)_{ij}\,h_{t-1,j} + (b_h)_i, \qquad i = 1, \dots, N. \tag{7.2'}
$$

$$
h_{t,i} = \tanh(a_{t,i}), \qquad z_t = \sum_{i=1}^{N} (w_y)_i\,h_{t,i} + b_y, \qquad \hat y_t = \sigma(z_t). \tag{7.2''}
$$

**Проверка при $N=1, D=1$.** Матрица $W_x \in \mathbb{R}^{1 \times 1}$ — это просто скаляр $w_x$; $W_h$ — скаляр $w_h$; $w_y$ — скаляр $w_y$; $\tanh$ от скаляра. Уравнения (7.2) превращаются в (4.1)–(4.4) главы 4. **Глава 7 — прямое обобщение главы 4.**

На рис. 7.1 — архитектура при $N = 3$, $D = 2$, $T = 2$. Каждый скрытый нейрон связан со всеми входами (стрелки $W_x$) и со всеми скрытыми нейронами предыдущего шага (стрелки $W_h$). Число стрелок быстро растёт, но структура та же, что в главе 4.

```python
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch

def draw_architecture_N_T2(N=3, D=2):
    """Рис. 7.1: N скрытых нейронов, D входов, T=2."""
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_xlim(0, 14); ax.set_ylim(0, 9)
    ax.set_aspect('equal'); ax.axis('off')
    r = 0.35
    # координаты столбцов
    cols = [2.5, 11.5]
    y_in_base = 1.2
    y_h_base = 4.2
    y_out = 7.5

    def node(cx, cy, label, color):
        ax.add_patch(Circle((cx, cy), r, facecolor=color,
                            edgecolor='black', lw=1.5, zorder=3))
        ax.text(cx, cy, label, ha='center', va='center',
                fontsize=11, zorder=4)

    def arrow(p1, p2, color='black', lw=1.0, alpha=1.0):
        ax.add_patch(FancyArrowPatch(
            p1, p2, arrowstyle='-|>', mutation_scale=12,
            lw=lw, color=color, alpha=alpha,
            shrinkA=0, shrinkB=0, zorder=2))

    # для каждого шага t
    for t, cx in enumerate(cols, start=1):
        # входные нейроны
        for j in range(D):
            node(cx, y_in_base + j*0.9, rf'$x_{t},{j+1}$', '#AEDFF7')
        # скрытые нейроны
        for i in range(N):
            node(cx, y_h_base + i*0.9, rf'$h_{t},{i+1}$', '#FFE29A')
        # выход
        node(cx, y_out, rf'$\hat y_{t}$', '#B8E0B8')

        # стрелки W_x: все входы → все скрытые
        for j in range(D):
            for i in range(N):
                arrow((cx + r*0.5, y_in_base + j*0.9),
                      (cx - r*0.5, y_h_base + i*0.9),
                      color='#B22222', lw=0.8, alpha=0.5)
        # стрелки w_y: все скрытые → выход
        for i in range(N):
            arrow((cx + r*0.5, y_h_base + i*0.9),
                  (cx - r*0.5, y_out),
                  color='#B22222', lw=0.8, alpha=0.5)

    # рекуррентные стрелки W_h (h_1 → h_2)
    for i in range(N):
        for k in range(N):
            arrow((cols[0] + r, y_h_base + i*0.9),
                  (cols[1] - r, y_h_base + k*0.9),
                  color='#7B2D8E', lw=0.8, alpha=0.6)

    # подписи
    ax.text(cols[0], 0.3, 'Шаг t=1', ha='center', fontsize=12, fontweight='bold')
    ax.text(cols[1], 0.3, 'Шаг t=2', ha='center', fontsize=12, fontweight='bold')
    ax.text((cols[0]+cols[1])/2, 8.7,
            f'Рис. 7.1. Архитектура: D={D} входа, N={N} скрытых, 1 выход, T=2',
            ha='center', fontsize=13, fontweight='bold')
    ax.text((cols[0]+cols[1])/2, 6.0, r'$W_h$', fontsize=14,
            color='#7B2D8E', ha='center')
    ax.text(cols[0] + 0.6, 2.8, r'$W_x$', fontsize=11, color='#B22222')
    ax.text(cols[0] + 0.6, 6.2, r'$w_y$', fontsize=11, color='#B22222')
    plt.tight_layout(); plt.show()

draw_architecture_N_T2(N=3, D=2)
```

## 7.3. Прямой проход

Прямой проход — это последовательное применение (7.2) для $t = 1, 2$. Выпишем в матричной записи и в компонентной форме.

### Шаг $t = 1$

**Матрично:**

$$
a_1 = W_x\,x_1 + W_h\,h_0 + b_h = W_x\,x_1 + b_h \quad (\text{так как } h_0 = \mathbf{0}),
$$
$$
h_1 = \tanh(a_1), \qquad z_1 = w_y^\top h_1 + b_y, \qquad \hat y_1 = \sigma(z_1).
$$

**Покомпонентно:**

$$
a_{1,i} = \sum_{j=1}^{D} (W_x)_{ij}\,x_{1,j} + (b_h)_i, \quad i = 1,\dots,N.
$$

### Шаг $t = 2$

**Матрично:**

$$
a_2 = W_x\,x_2 + W_h\,h_1 + b_h, \qquad h_2 = \tanh(a_2), \qquad z_2 = w_y^\top h_2 + b_y, \qquad \hat y_2 = \sigma(z_2).
$$

**Покомпонентно:**

$$
a_{2,i} = \sum_{j=1}^{D} (W_x)_{ij}\,x_{2,j} + \sum_{k=1}^{N} (W_h)_{ik}\,h_{1,k} + (b_h)_i, \quad i = 1,\dots,N.
$$

Здесь видно, откуда «растут» $N^2$ параметров $W_h$: каждый из $N$ скрытых нейронов на шаге 2 получает вклад от каждого из $N$ скрытых нейронов на шаге 1.

### Таблица форм величин (проверка корректности)

| Величина | $t=1$ | $t=2$ | Форма |
|---|---|---|---|
| $x_t$ | $x_1 \in \mathbb{R}^D$ | $x_2 \in \mathbb{R}^D$ | $(D,)$ |
| $W_x x_t$ | $\mathbb{R}^N$ | $\mathbb{R}^N$ | $(N,)$ |
| $W_h h_{t-1}$ | $W_h \cdot \mathbf{0} = \mathbf{0}$ | $W_h h_1 \in \mathbb{R}^N$ | $(N,)$ |
| $a_t$ | $\mathbb{R}^N$ | $\mathbb{R}^N$ | $(N,)$ |
| $h_t = \tanh(a_t)$ | $\mathbb{R}^N$ | $\mathbb{R}^N$ | $(N,)$ |
| $z_t = w_y^\top h_t + b_y$ | $\mathbb{R}$ | $\mathbb{R}$ | скаляр |
| $\hat y_t = \sigma(z_t)$ | $\mathbb{R}$ | $\mathbb{R}$ | скаляр |

Если хоть одна размерность не сходится — ошибка в постановке. Например, $w_y h_t$ (без транспонирования) дало бы матрицу $N \times N$, а не скаляр — это типичная ошибка при переходе от скаляров к векторам.

### Численный пример: $N=2$, $D=1$, $T=2$

Возьмём:

$$
W_x = \begin{pmatrix} 0.5 \\ 0.3 \end{pmatrix} \in \mathbb{R}^{2 \times 1}, \quad
W_h = \begin{pmatrix} 0.2 & -0.1 \\ 0.4 & 0.3 \end{pmatrix} \in \mathbb{R}^{2 \times 2}, \quad
b_h = \begin{pmatrix} 0.1 \\ -0.2 \end{pmatrix} \in \mathbb{R}^2,
$$
$$
w_y = \begin{pmatrix} 0.6 \\ -0.5 \end{pmatrix} \in \mathbb{R}^2, \quad b_y = 0.1 \in \mathbb{R}.
$$

Входы: $x_1 = 1.0$, $x_2 = 0.5$. Ответы: $y_1 = 0.0$, $y_2 = 1.0$. Начальное состояние $h_0 = (0, 0)^\top$.

**Шаг $t = 1$:**

$$
a_1 = W_x x_1 + b_h = \begin{pmatrix} 0.5 \\ 0.3 \end{pmatrix}\cdot 1.0 + \begin{pmatrix} 0.1 \\ -0.2 \end{pmatrix} = \begin{pmatrix} 0.6 \\ 0.1 \end{pmatrix},
$$
$$
h_1 = \tanh(a_1) = \begin{pmatrix} \tanh(0.6) \\ \tanh(0.1) \end{pmatrix} \approx \begin{pmatrix} 0.53705 \\ 0.09967 \end{pmatrix},
$$
$$
z_1 = w_y^\top h_1 + b_y = 0.6 \cdot 0.53705 + (-0.5) \cdot 0.09967 + 0.1 = 0.32223 - 0.04983 + 0.1 = 0.37240,
$$
$$
\hat y_1 = \sigma(0.37240) \approx 0.59206.
$$

**Шаг $t = 2$:**

$$
W_h h_1 = \begin{pmatrix} 0.2 \cdot 0.53705 + (-0.1) \cdot 0.09967 \\ 0.4 \cdot 0.53705 + 0.3 \cdot 0.09967 \end{pmatrix} = \begin{pmatrix} 0.10741 - 0.00997 \\ 0.21482 + 0.02990 \end{pmatrix} = \begin{pmatrix} 0.09744 \\ 0.24472 \end{pmatrix},
$$
$$
a_2 = W_x x_2 + W_h h_1 + b_h = \begin{pmatrix} 0.25 \\ 0.15 \end{pmatrix} + \begin{pmatrix} 0.09744 \\ 0.24472 \end{pmatrix} + \begin{pmatrix} 0.1 \\ -0.2 \end{pmatrix} = \begin{pmatrix} 0.44744 \\ 0.19472 \end{pmatrix},
$$
$$
h_2 = \tanh(a_2) = \begin{pmatrix} \tanh(0.44744) \\ \tanh(0.19472) \end{pmatrix} \approx \begin{pmatrix} 0.41978 \\ 0.19230 \end{pmatrix},
$$
$$
z_2 = 0.6 \cdot 0.41978 + (-0.5) \cdot 0.19230 + 0.1 = 0.25187 - 0.09615 + 0.1 = 0.25572,
$$
$$
\hat y_2 = \sigma(0.25572) \approx 0.56362.
$$

**Проверка при $N=1$.** Если убрать второй компонент всех векторов и вторую строку/столбец матриц, получатся в точности числа главы 4, §4.7 — с точностью до смены значений параметров (мы взяли другие, чтобы пример был «двумерным»).

## 7.4. Функция потерь

$$
L = \frac{1}{2}\,(\hat y_1 - y_1)^2 + \frac{1}{2}\,(\hat y_2 - y_2)^2 \in \mathbb{R}. \tag{7.3}
$$

Выход по-прежнему скалярный, поэтому потеря — скаляр, и производные по $\hat y_t$ те же, что в главе 4:

$$
\frac{\partial L}{\partial \hat y_t} = \hat y_t - y_t \in \mathbb{R}, \qquad t = 1, 2. \tag{7.4}
$$

## 7.5. Таблица локальных производных

Все локальные производные из главы 4 остаются, но объекты меняются. Выпишем их в векторно-матричной и компонентной форме.

**Производные $a_t$:**

| Величина | Форма | Компонентно |
|---|---|---|
| $\partial a_t / \partial x_t$ | $\mathbb{R}^{N \times D}$ | $\partial a_{t,i}/\partial x_{t,j} = (W_x)_{ij}$ |
| $\partial a_t / \partial h_{t-1}$ | $\mathbb{R}^{N \times N}$ | $\partial a_{t,i}/\partial h_{t-1,k} = (W_h)_{ik}$ |
| $\partial a_t / \partial b_h$ | $\mathbb{R}^{N \times N}$ | $\partial a_{t,i}/\partial (b_h)_k = \delta_{ik}$ |
| $\partial a_t / \partial (W_x)_{jk}$ | тензор $N \times N \times D$ | $\partial a_{t,i}/\partial (W_x)_{jk} = \delta_{ij}\,x_{t,k}$ |
| $\partial a_t / \partial (W_h)_{jk}$ | тензор $N \times N \times N$ | $\partial a_{t,i}/\partial (W_h)_{jk} = \delta_{ij}\,h_{t-1,k}$ |

Тензорные производные по матрицам **не нужно** вычислять явно: при обратном проходе они сворачиваются в **внешние произведения** $\delta^a_t x_t^\top$ и $\delta^a_t h_{t-1}^\top$. Это — главный практический трюк.

**Производная активации (покомпонентная):**

$$
\frac{\partial h_t}{\partial a_t} = \text{diag}\bigl(1 - h_t \odot h_t\bigr) \in \mathbb{R}^{N \times N}. \tag{7.5}
$$

В компонентной форме: $\partial h_{t,i}/\partial a_{t,j} = \delta_{ij}(1 - h_{t,i}^2)$. В векторной форме при умножении на вектор это даёт **поэлементное** произведение: $\partial L/\partial a_t = (1 - h_t \odot h_t) \odot \partial L/\partial h_t$.

**Производные выхода:**

| Величина | Форма | Компонентно |
|---|---|---|
| $\partial z_t / \partial h_t$ | $\mathbb{R}^N$ | $\partial z_t/\partial h_{t,i} = (w_y)_i$ |
| $\partial z_t / \partial w_y$ | $\mathbb{R}^N$ | $\partial z_t/\partial (w_y)_i = h_{t,i}$ |
| $\partial z_t / \partial b_y$ | $\mathbb{R}$ | $1$ |
| $\partial \hat y_t/\partial z_t$ | $\mathbb{R}$ | $\hat y_t(1-\hat y_t)$ |

**Производные потерь по выходам** — те же, что (7.4).

Эти «кирпичи» **не зависят от $N$** — как и в главе 4. Меняются только объекты: вместо скаляров — векторы и матрицы.

## 7.6. Обратный проход при $T = 2$

### Дельты выходов

Как и в главе 4, начинаем с выходов. Определение $\delta^y_t := \partial L/\partial z_t$ **остаётся скаляром** (потому что $z_t$ — скаляр):

$$
\delta^y_t = (\hat y_t - y_t)\,\hat y_t\,(1 - \hat y_t) \in \mathbb{R}, \qquad t = 1, 2. \tag{7.6}
$$

Численно: $\delta^y_1 \approx 0.14301$, $\delta^y_2 \approx -0.10733$.

### Дельта $a_2$ — один путь

$h_2$ влияет на $L$ только через $z_2$ (следующего шага нет). Цепочка от $L$ до $a_2$:

$$
\frac{\partial L}{\partial a_{2,i}} = \frac{\partial L}{\partial h_{2,i}}\cdot\frac{\partial h_{2,i}}{\partial a_{2,i}}, \qquad \frac{\partial L}{\partial h_{2,i}} = \delta^y_2 \cdot (w_y)_i.
$$

В векторной форме:

$$
\frac{\partial L}{\partial h_2} = \delta^y_2\,w_y \in \mathbb{R}^N, \qquad
\boxed{\;\delta^a_2 := \frac{\partial L}{\partial a_2} = \bigl(\delta^y_2\,w_y\bigr) \odot (1 - h_2 \odot h_2) \in \mathbb{R}^N.\;} \tag{7.7}
$$

Сравните с (4.17): там было $\delta^a_2 = \delta^y_2 w_y (1 - h_2^2)$. Здесь $w_y$ — вектор, а $(1 - h_2^2)$ — вектор, и они перемножаются **поэлементно**. При $N = 1$ (7.7) превращается в (4.17).

**Численно:**

$$
\delta^y_2 w_y = (-0.10733) \cdot \begin{pmatrix} 0.6 \\ -0.5 \end{pmatrix} = \begin{pmatrix} -0.06440 \\ 0.05367 \end{pmatrix},
$$
$$
1 - h_2 \odot h_2 = \begin{pmatrix} 1 - 0.41978^2 \\ 1 - 0.19230^2 \end{pmatrix} = \begin{pmatrix} 0.82378 \\ 0.96302 \end{pmatrix},
$$
$$
\delta^a_2 = \begin{pmatrix} -0.06440 \\ 0.05367 \end{pmatrix} \odot \begin{pmatrix} 0.82378 \\ 0.96302 \end{pmatrix} = \begin{pmatrix} -0.05305 \\ 0.05169 \end{pmatrix}.
$$

### Дельта $a_1$ — два пути

$h_1$ влияет на $L$ **двумя путями**: через $z_1$ (свой шаг) и через $a_2$ (будущий шаг). В векторной форме:

$$
\frac{\partial L}{\partial h_1} = \underbrace{\delta^y_1\,w_y}_{\text{путь A}} \;+\; \underbrace{W_h^\top \delta^a_2}_{\text{путь B}} \in \mathbb{R}^N. \tag{7.8}
$$

Здесь $W_h^\top \delta^a_2 \in \mathbb{R}^N$ — это **матрично-векторное** произведение: компонентно $(W_h^\top \delta^a_2)_k = \sum_i (W_h)_{ik}\,\delta^a_{2,i}$ — вклад от **всех** $N$ скрытых нейронов шага 2 в $k$-й скрытый нейрон шага 1. Именно здесь и «спрятаны» все $N^2$ путей через $W_h$; выписывать их покомпонентно не нужно — матричное умножение делает это автоматически.

Умножаем на производную $\tanh$:

$$
\boxed{\;\delta^a_1 := \frac{\partial L}{\partial a_1} = \bigl(\delta^y_1\,w_y + W_h^\top \delta^a_2\bigr) \odot (1 - h_1 \odot h_1) \in \mathbb{R}^N.\;} \tag{7.9}
$$

Сравните с (4.18): $\delta^a_1 = (\delta^y_1 w_y + \delta^a_2 w_h)(1 - h_1^2)$. При $N=1$: $W_h^\top \delta^a_2$ — это $\delta^a_2 w_h$ (скаляр), и (7.9) превращается в (4.18).

**Численно:**

$$
\delta^y_1 w_y = 0.14301 \cdot \begin{pmatrix} 0.6 \\ -0.5 \end{pmatrix} = \begin{pmatrix} 0.08581 \\ -0.07151 \end{pmatrix},
$$
$$
W_h^\top \delta^a_2 = \begin{pmatrix} 0.2 & 0.4 \\ -0.1 & 0.3 \end{pmatrix} \begin{pmatrix} -0.05305 \\ 0.05169 \end{pmatrix} = \begin{pmatrix} 0.2 \cdot (-0.05305) + 0.4 \cdot 0.05169 \\ -0.1 \cdot (-0.05305) + 0.3 \cdot 0.05169 \end{pmatrix} = \begin{pmatrix} 0.01007 \\ 0.02082 \end{pmatrix},
$$
$$
\frac{\partial L}{\partial h_1} = \begin{pmatrix} 0.08581 \\ -0.07151 \end{pmatrix} + \begin{pmatrix} 0.01007 \\ 0.02082 \end{pmatrix} = \begin{pmatrix} 0.09588 \\ -0.05069 \end{pmatrix},
$$
$$
\delta^a_1 = \begin{pmatrix} 0.09588 \\ -0.05069 \end{pmatrix} \odot \begin{pmatrix} 0.71158 \\ 0.99007 \end{pmatrix} = \begin{pmatrix} 0.06823 \\ -0.05019 \end{pmatrix}.
$$

### Пять градиентов параметров

**$w_y$.** $w_y$ входит в $z_t$ на обоих шагах. Цепочка по каждому шагу:

$$
\frac{\partial L}{\partial (w_y)_i} = \sum_{t=1}^{2} \frac{\partial L}{\partial z_t}\cdot\frac{\partial z_t}{\partial (w_y)_i} = \sum_{t=1}^{2} \delta^y_t\,h_{t,i}.
$$

В векторной форме:

$$
\boxed{\;\frac{\partial L}{\partial w_y} = \delta^y_1\,h_1 + \delta^y_2\,h_2 \in \mathbb{R}^N.\;} \tag{7.10}
$$

Сравните с (4.15): $\delta^y_1 h_1 + \delta^y_2 h_2$. При $N=1$ — то же самое. **Формула не изменилась**, потому что она была записана через произведение скаляров, а теперь — скаляра на вектор.

**$b_y$.** $\partial z_t/\partial b_y = 1$:

$$
\frac{\partial L}{\partial b_y} = \delta^y_1 + \delta^y_2 \in \mathbb{R}. \tag{7.11}
$$

**$W_x$.** $W_x$ входит в $a_t$ на обоих шагах. Покомпонентно:

$$
\frac{\partial L}{\partial (W_x)_{jk}} = \sum_{t=1}^{2} \sum_{i=1}^{N} \frac{\partial L}{\partial a_{t,i}}\cdot\frac{\partial a_{t,i}}{\partial (W_x)_{jk}} = \sum_{t=1}^{2} \sum_{i=1}^{N} \delta^a_{t,i}\,\delta_{ij}\,x_{t,k} = \sum_{t=1}^{2} \delta^a_{t,j}\,x_{t,k}.
$$

Это **внешнее произведение** $\delta^a_t \in \mathbb{R}^N$ и $x_t \in \mathbb{R}^D$:

$$
\boxed{\;\frac{\partial L}{\partial W_x} = \delta^a_1\,x_1^\top + \delta^a_2\,x_2^\top \in \mathbb{R}^{N \times D}.\;} \tag{7.12}
$$

Здесь важно: $x_t^\top$ — **строка** (форма $(1, D)$), $\delta^a_t$ — **столбец** (форма $(N, 1)$), их произведение — матрица $(N, D)$. Типичная ошибка — забыть транспонирование и получить неверную форму.

**$W_h$.** $W_h$ входит в $a_2$ (в $a_1$ его нет, потому что $h_0 = \mathbf{0}$). Покомпонентно:

$$
\frac{\partial L}{\partial (W_h)_{jk}} = \sum_{i=1}^{N} \delta^a_{2,i}\,\delta_{ij}\,h_{1,k} = \delta^a_{2,j}\,h_{1,k}.
$$

Внешнее произведение:

$$
\boxed{\;\frac{\partial L}{\partial W_h} = \delta^a_2\,h_1^\top \in \mathbb{R}^{N \times N}.\;} \tag{7.13}
$$

При $N > 1$ это уже матрица $N \times N$ — а не скаляр, как в (4.20).

**$b_h$.** $\partial a_t/\partial b_h = I_N$ (единичная матрица):

$$
\boxed{\;\frac{\partial L}{\partial b_h} = \delta^a_1 + \delta^a_2 \in \mathbb{R}^N.\;} \tag{7.14}
$$

### Сводка обратного прохода при $T = 2$

$$
\boxed{
\begin{aligned}
\delta^y_t &= (\hat y_t - y_t)\,\hat y_t(1-\hat y_t) \in \mathbb{R}, && t = 1, 2; \\[2pt]
\delta^a_2 &= (\delta^y_2\,w_y) \odot (1 - h_2 \odot h_2) \in \mathbb{R}^N; \\[2pt]
\delta^a_1 &= \bigl(\delta^y_1\,w_y + W_h^\top \delta^a_2\bigr) \odot (1 - h_1 \odot h_1) \in \mathbb{R}^N; \\[4pt]
\frac{\partial L}{\partial w_y} &= \delta^y_1\,h_1 + \delta^y_2\,h_2 \in \mathbb{R}^N, & \frac{\partial L}{\partial b_y} &= \delta^y_1 + \delta^y_2 \in \mathbb{R}, \\
\frac{\partial L}{\partial W_x} &= \delta^a_1\,x_1^\top + \delta^a_2\,x_2^\top \in \mathbb{R}^{N \times D}, & \frac{\partial L}{\partial W_h} &= \delta^a_2\,h_1^\top \in \mathbb{R}^{N \times N}, \\
&& \frac{\partial L}{\partial b_h} &= \delta^a_1 + \delta^a_2 \in \mathbb{R}^N.
\end{aligned}}
\tag{7.15}
$$

**Проверка при $N=1, D=1$.** Все $W_x, W_h, w_y$ — скаляры, $h_t, \delta^a_t$ — скаляры, $\odot$ — обычное умножение, $W_h^\top \delta^a_2$ — $\delta^a_2 w_h$, $x_t^\top$ — $x_t$. Формулы (7.15) превращаются в точности в формулы (4.15)–(4.21) главы 4. Это и есть доказательство, что глава 7 — **корректное обобщение**.

### Схема обратного прохода

Рис. 7.2 показывает те же два пути, что и рис. 4.3, но теперь $\delta^a_2$ — вектор, а $W_h^\top \delta^a_2$ — матрично-векторное произведение, которое автоматически суммирует вклады всех $N$ нейронов.

```python
def draw_backward_N_T2(N=2):
    """Рис. 7.2: обратный проход при N нейронах, T=2."""
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.set_xlim(0, 12); ax.set_ylim(0, 7)
    ax.axis('off'); ax.set_aspect('equal')
    x1, x2 = 3.0, 9.0
    y_out, y_h = 5.5, 2.5
    r = 0.45

    def node(cx, cy, label, color):
        ax.add_patch(Circle((cx, cy), r, facecolor=color,
                            edgecolor='black', lw=1.4, zorder=3))
        ax.text(cx, cy, label, ha='center', va='center',
                fontsize=11, zorder=4)

    node(x1, y_out, r'$\hat y_1$', '#B8E0B8')
    node(x2, y_out, r'$\hat y_2$', '#B8E0B8')
    # «стопки» нейронов
    ax.add_patch(plt.Rectangle((x1 - 0.9, y_h - 0.7), 1.8, 1.4,
                               facecolor='#F5C6A5', edgecolor='black'))
    ax.text(x1, y_h, r'$a_1\in\mathbb{R}^N$', ha='center', va='center', fontsize=10)
    ax.add_patch(plt.Rectangle((x2 - 0.9, y_h - 0.7), 1.8, 1.4,
                               facecolor='#F5C6A5', edgecolor='black'))
    ax.text(x2, y_h, r'$a_2\in\mathbb{R}^N$', ha='center', va='center', fontsize=10)

    ax.annotate('', xy=(x1, y_out - r), xytext=(x1, y_h + 0.7),
                arrowprops=dict(arrowstyle='-|>', lw=1.6, color='#C0392B'))
    ax.annotate('', xy=(x2, y_out - r), xytext=(x2, y_h + 0.7),
                arrowprops=dict(arrowstyle='-|>', lw=1.6, color='#C0392B'))
    ax.text(x1 + 0.15, (y_out + y_h)/2, r'$\delta^y_1\,w_y$', fontsize=10, color='#C0392B')
    ax.text(x2 + 0.15, (y_out + y_h)/2, r'$\delta^y_2\,w_y$', fontsize=10, color='#C0392B')

    ax.annotate('', xy=(x1 + 0.9, y_h), xytext=(x2 - 0.9, y_h),
                arrowprops=dict(arrowstyle='-|>', lw=2, color='#C0392B',
                                linestyle='--'))
    ax.text((x1 + x2)/2, y_h + 0.3, r'$W_h^\top \delta^a_2$',
            ha='center', fontsize=11, color='#C0392B')

    ax.text(6.0, 0.9,
            r'$\delta^a_2 = (\delta^y_2 w_y) \odot (1 - h_2 \odot h_2)$',
            ha='center', fontsize=11)
    ax.text(6.0, 0.2,
            r'$\delta^a_1 = (\delta^y_1 w_y + W_h^\top \delta^a_2) \odot (1 - h_1 \odot h_1)$',
            ha='center', fontsize=11)

    ax.text(6.0, 6.7, 'Рис. 7.2. Обратный проход при N нейронах, T = 2',
            ha='center', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

draw_backward_N_T2(N=2)
```

## 7.7. Численный пример: сводка

Продолжим пример из §7.3. Все числа — с точностью до $10^{-5}$.

### Прямой проход

$$
h_1 = \begin{pmatrix} 0.53705 \\ 0.09967 \end{pmatrix}, \quad \hat y_1 = 0.59206, \quad h_2 = \begin{pmatrix} 0.41978 \\ 0.19230 \end{pmatrix}, \quad \hat y_2 = 0.56362.
$$

### Потеря

$$
L = \tfrac12(0.59206)^2 + \tfrac12(0.56362 - 1)^2 \approx 0.17527 + 0.09521 = 0.27048.
$$

### Обратный проход

$$
\delta^y_1 = 0.14301, \qquad \delta^y_2 = -0.10733,
$$
$$
\delta^a_2 = \begin{pmatrix} -0.05305 \\ 0.05169 \end{pmatrix}, \qquad \delta^a_1 = \begin{pmatrix} 0.06823 \\ -0.05019 \end{pmatrix}.
$$

### Градиенты параметров

$$
\frac{\partial L}{\partial w_y} = \begin{pmatrix} 0.03175 \\ -0.00639 \end{pmatrix}, \qquad \frac{\partial L}{\partial b_y} = 0.03568,
$$
$$
\frac{\partial L}{\partial W_x} = \begin{pmatrix} 0.04170 \\ -0.02434 \end{pmatrix} \in \mathbb{R}^{2 \times 1}, \qquad
\frac{\partial L}{\partial W_h} = \begin{pmatrix} -0.02850 & -0.00529 \\ 0.02776 & 0.00515 \end{pmatrix} \in \mathbb{R}^{2 \times 2},
$$
$$
\frac{\partial L}{\partial b_h} = \begin{pmatrix} 0.01518 \\ 0.00150 \end{pmatrix}.
$$

### Обновление ($\eta = 0.1$)

$$
W_x \leftarrow \begin{pmatrix} 0.5 \\ 0.3 \end{pmatrix} - 0.1 \begin{pmatrix} 0.04170 \\ -0.02434 \end{pmatrix} = \begin{pmatrix} 0.49583 \\ 0.30243 \end{pmatrix},
$$
$$
W_h \leftarrow \begin{pmatrix} 0.2 & -0.1 \\ 0.4 & 0.3 \end{pmatrix} - 0.1 \begin{pmatrix} -0.02850 & -0.00529 \\ 0.02776 & 0.00515 \end{pmatrix} = \begin{pmatrix} 0.20285 & -0.09947 \\ 0.39722 & 0.29949 \end{pmatrix},
$$
$$
b_h \leftarrow \begin{pmatrix} 0.1 \\ -0.2 \end{pmatrix} - 0.1 \begin{pmatrix} 0.01518 \\ 0.00150 \end{pmatrix} = \begin{pmatrix} 0.09848 \\ -0.20015 \end{pmatrix},
$$
$$
w_y \leftarrow \begin{pmatrix} 0.6 \\ -0.5 \end{pmatrix} - 0.1 \begin{pmatrix} 0.03175 \\ -0.00639 \end{pmatrix} = \begin{pmatrix} 0.59683 \\ -0.49936 \end{pmatrix}, \qquad
b_y \leftarrow 0.1 - 0.1 \cdot 0.03568 = 0.09643.
$$

## 7.8. Полная программа

Класс `TinyRNN` обобщается до `VectorRNN`: скалярные умножения заменяются на матрично-векторные, а $1 - h_t^2$ — на $1 - h_t \odot h_t$.

```python
import numpy as np

def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))

class VectorRNN:
    def __init__(self, N, D, seed=0, eta=0.1):
        rng = np.random.default_rng(seed)
        self.N, self.D = N, D
        self.Wx = rng.normal(scale=0.5, size=(N, D))   # (N, D)
        self.Wh = rng.normal(scale=0.5, size=(N, N))   # (N, N)
        self.bh = rng.normal(scale=0.1, size=(N,))     # (N,)
        self.wy = rng.normal(scale=0.5, size=(N,))     # (N,)
        self.by = 0.0
        self.eta = eta

    def forward(self, xs, h0=None):
        """xs: список длины T, каждый элемент — вектор (D,)."""
        T = len(xs)
        h = np.zeros(self.N) if h0 is None else h0
        hs, ys = [], []
        for x in xs:
            a = self.Wx @ x + self.Wh @ h + self.bh  # (N,)
            h = np.tanh(a)                            # (N,)
            z = self.wy @ h + self.by                 # скаляр
            ys.append(sigmoid(z))
            hs.append(h)
        return np.array(hs), np.array(ys)              # (T, N), (T,)

    def loss(self, ys, targets):
        return 0.5 * np.sum((ys - targets) ** 2)

    def backward(self, xs, hs, ys, targets, h0=None):
        T = len(xs)
        dy = (ys - targets) * ys * (1.0 - ys)         # (T,)
        dh = np.zeros_like(hs)                        # (T, N)
        dh_next = np.zeros(self.N)                    # (N,)
        for t in reversed(range(T)):
            # δ^a_t = (δ^y_t w_y + W_h^T δ^a_{t+1}) ⊙ (1 - h_t ⊙ h_t)
            dh[t] = (dy[t] * self.wy + self.Wh.T @ dh_next) * (1.0 - hs[t]**2)
            dh_next = dh[t]
        h_prev = np.vstack([np.zeros(self.N), hs[:-1]])   # (T, N)
        # xs: (T, D); dh: (T, N); h_prev: (T, N)
        gWx = sum(np.outer(dh[t], xs[t]) for t in range(T))   # (N, D)
        gWh = sum(np.outer(dh[t], h_prev[t]) for t in range(T))  # (N, N)
        gbh = dh.sum(axis=0)                              # (N,)
        gwy = sum(dy[t] * hs[t] for t in range(T))        # (N,)
        gby = dy.sum()                                    # скаляр
        return dict(Wx=gWx, Wh=gWh, bh=gbh, wy=gwy, by=gby)

    def step(self, xs, targets, h0=None):
        hs, ys = self.forward(xs, h0)
        L = self.loss(ys, targets)
        g = self.backward(xs, hs, ys, targets, h0)
        for k, v in g.items():
            setattr(self, k, getattr(self, k) - self.eta * v)
        return L, ys, g


# Проверка на примере из §7.7
rnn = VectorRNN(N=2, D=1)
# параметры — явно, чтобы получить в точности числа из §7.7
rnn.Wx = np.array([[0.5], [0.3]])
rnn.Wh = np.array([[0.2, -0.1], [0.4, 0.3]])
rnn.bh = np.array([0.1, -0.2])
rnn.wy = np.array([0.6, -0.5])
rnn.by = 0.1
rnn.eta = 0.1

xs = [np.array([1.0]), np.array([0.5])]
ys = np.array([0.0, 1.0])
hs, yhat = rnn.forward(xs)
print("h  =\n", hs)         # [[0.53705, 0.09967], [0.41978, 0.19230]]
print("y^ =", yhat)         # [0.59206, 0.56362]
L, _, g = rnn.step(xs, ys)
print("L  =", L)            # 0.27048
for k, v in g.items():
    print(f"dL/d{k} =\n{v}\n")
```

Вывод совпадает с ручным расчётом §7.7 — значит, все матричные формулы (7.15) реализованы правильно. Если запустить код с $N=1, D=1$, он даст ровно числа главы 4 — это вторая проверка обобщения.

## 7.9. Что мы получили

1. **Переход от скаляров к векторам** — не новая математика, а новая **запись** той же цепочки. Все формулы главы 4 сохраняются с заменами из таблицы §7.1: $w_x x_t \to W_x x_t$, $w_h h_{t-1} \to W_h h_{t-1}$, $w_y h_t \to w_y^\top h_t$, $1 - h_t^2 \to 1 - h_t \odot h_t$.
2. **Поэлементное умножение** $\odot$ — ключевое новое действие; оно нужно потому, что $\tanh$ применяется покомпонентно.
3. **Матрично-векторные произведения** $W_h^\top \delta^a_2$ автоматически суммируют вклады всех $N$ нейронов. В скалярной записи это была бы сумма $N$ слагаемых — здесь она «спрятана» в умножении матрицы на вектор.
4. **Внешние произведения** $\delta^a_t x_t^\top$ и $\delta^a_t h_{t-1}^\top$ дают градиенты по матрицам $W_x$, $W_h$ — без явных тензоров.
5. **Проверка при $N=1, D=1$** — формулы (7.15) превращаются в (4.15)–(4.21) главы 4 **дословно**. Это доказывает корректность обобщения.
6. **Код не требует изменения структуры** — только замены скалярных операций на матричные. Класс `VectorRNN` работает при любых $N, D, T$; при $T=2$ он даёт ровно числа этой главы.

В главе 8 мы перейдём к **произвольному $T$**: рекуррентное соотношение (6.7) из главы 6 объединится с матричной записью этой главы и даст **матричный BPTT**. Структура останется той же: $\delta^a_t = (\delta^y_t w_y + W_h^\top \delta^a_{t+1}) \odot (1 - h_t \odot h_t)$, просто $t$ пробегает все значения от $T$ до $1$. Формулы этой главы — строительный блок для всего дальнейшего: LSTM и GRU (главы 13–14) отличаются от (7.2) только заменой одного уравнения $h_t = \tanh(a_t)$ на систему вентилей — но обратный проход для них строится по той же схеме, что (7.15).

# Глава 8. Общий случай: $N$ нейронов и произвольное $T$ — матричный BPTT

В главе 6 мы получили рекуррентное соотношение для $T$ произвольного, но при одном нейроне. В главе 7 — перешли к $N$ нейронам при $T = 2$. Эта глава объединяет оба обобщения: **$N$ нейронов и произвольное $T$**. Никакой новой математики в ней нет — только соединение двух ранее полученных результатов. Ключевая формула этой главы:

$$
\boxed{\;\delta^a_t = \bigl(\delta^y_t\,w_y + W_h^\top \delta^a_{t+1}\bigr) \odot (1 - h_t \odot h_t)\;}
$$

— это ровно (6.7) из главы 6, в которой скаляры заменены на векторы и матрицы по правилам таблицы §7.1. При $N=1$ она даёт (6.7); при $T=2$ — (7.15). Проверим оба предельных случая.

## 8.1. Зачем нужна эта глава

Формально главы 6 и 7 — это две проекции одного и того же объекта. Глава 6 показывает, **как** устроена рекурсия по времени (сколько путей, откуда слагаемые). Глава 7 показывает, **как** устроена матричная запись (форма величин, поэлементное умножение). Эта глава показывает **обе структуры сразу** — и именно она даёт формулу, которую можно использовать на практике.

Практические причины:
- **Реальная RNN** почти всегда имеет $N \gg 1$ и $T \gg 1$. При $N = 128$, $T = 50$ в формуле (6.7) пришлось бы писать $N$ уравнений на каждом шаге — $6400$ строк. В матричной форме — одна строка.
- **Реализация** любого фреймворка (PyTorch, TensorFlow) использует именно матричную форму: `dh = (dy @ wy + dh_next @ Wh) * (1 - h**2)` — и всё.
- **Следующие главы** (LSTM, GRU) записываются только в матричной форме; без этой главы они будут нечитаемы.

## 8.2. Архитектура и обозначения

Обозначения — те же, что в главе 7, но $T$ произвольно:

| Символ | Размерность | Смысл |
|---|---|---|
| $x_t$ | $\mathbb{R}^D$ | вход на шаге $t$ |
| $a_t$ | $\mathbb{R}^N$ | преактивация скрытого слоя |
| $h_t$ | $\mathbb{R}^N$ | скрытое состояние |
| $z_t$ | $\mathbb{R}$ | преактивация выхода (скаляр) |
| $\hat y_t$ | $\mathbb{R}$ | предсказание |
| $y_t$ | $\mathbb{R}$ | истинный ответ |
| $W_x$ | $\mathbb{R}^{N \times D}$ | веса «вход → скрытый» |
| $W_h$ | $\mathbb{R}^{N \times N}$ | веса рекуррентной связи |
| $b_h$ | $\mathbb{R}^N$ | смещение скрытого слоя |
| $w_y$ | $\mathbb{R}^N$ | веса «скрытый → выход» |
| $b_y$ | $\mathbb{R}$ | смещение выхода |
| $h_0 = \mathbf{0}$ | $\mathbb{R}^N$ | начальное состояние |
| $T$ | $\mathbb{N}$ | длина последовательности |
| $\eta$ | $\mathbb{R}$ | шаг обучения |

Уравнения прямого прохода (для каждого $t = 1, \dots, T$):

$$
\boxed{
\begin{aligned}
a_t &= W_x\,x_t + W_h\,h_{t-1} + b_h &&\in \mathbb{R}^N, \\
h_t &= \tanh(a_t) &&\in \mathbb{R}^N \text{ (покомпонентно)}, \\
z_t &= w_y^\top\,h_t + b_y &&\in \mathbb{R}, \\
\hat y_t &= \sigma(z_t) &&\in \mathbb{R}.
\end{aligned}}
\tag{8.1}
$$

**Число обучаемых параметров** — то же, что (7.1): $ND + N^2 + 2N + 1$, не зависит от $T$.

**Проверка при $N=1, D=1$.** $W_x, W_h, w_y$ — скаляры, $\tanh$ от скаляра — (8.1) превращается в (6.1) главы 6. **Глава 8 — прямое обобщение главы 6.**

## 8.3. Прямой проход

Прямой проход — это применение (8.1) для $t = 1, \dots, T$. Цикл не параллелится: $h_t$ нужен для $a_{t+1}$.

### Подстановка чисел: $N=2$, $D=1$, $T=3$

Возьмём те же параметры, что в §7.7:

$$
W_x = \begin{pmatrix} 0.5 \\ 0.3 \end{pmatrix}, \quad
W_h = \begin{pmatrix} 0.2 & -0.1 \\ 0.4 & 0.3 \end{pmatrix}, \quad
b_h = \begin{pmatrix} 0.1 \\ -0.2 \end{pmatrix}, \quad
w_y = \begin{pmatrix} 0.6 \\ -0.5 \end{pmatrix}, \quad b_y = 0.1.
$$

Входы: $x_1 = 1.0$, $x_2 = 0.5$, $x_3 = -0.3$. Ответы: $y_1 = 0.0$, $y_2 = 1.0$, $y_3 = 0.5$. Начальное состояние $h_0 = (0, 0)^\top$.

**Шаг $t = 1$:**

$$
a_1 = W_x x_1 + W_h h_0 + b_h = \begin{pmatrix} 0.5 \\ 0.3 \end{pmatrix} + \begin{pmatrix} 0 \\ 0 \end{pmatrix} + \begin{pmatrix} 0.1 \\ -0.2 \end{pmatrix} = \begin{pmatrix} 0.6 \\ 0.1 \end{pmatrix},
$$
$$
h_1 = \tanh(a_1) = \begin{pmatrix} \tanh(0.6) \\ \tanh(0.1) \end{pmatrix} \approx \begin{pmatrix} 0.53705 \\ 0.09967 \end{pmatrix},
$$
$$
z_1 = w_y^\top h_1 + b_y = 0.6\cdot 0.53705 - 0.5\cdot 0.09967 + 0.1 \approx 0.37240, \qquad \hat y_1 \approx 0.59206.
$$

**Шаг $t = 2$:**

$$
W_h h_1 = \begin{pmatrix} 0.2 & -0.1 \\ 0.4 & 0.3 \end{pmatrix}\begin{pmatrix} 0.53705 \\ 0.09967 \end{pmatrix} = \begin{pmatrix} 0.10741 - 0.00997 \\ 0.21482 + 0.02990 \end{pmatrix} = \begin{pmatrix} 0.09744 \\ 0.24472 \end{pmatrix},
$$
$$
a_2 = \begin{pmatrix} 0.25 \\ 0.15 \end{pmatrix} + \begin{pmatrix} 0.09744 \\ 0.24472 \end{pmatrix} + \begin{pmatrix} 0.1 \\ -0.2 \end{pmatrix} = \begin{pmatrix} 0.44744 \\ 0.19472 \end{pmatrix},
$$
$$
h_2 \approx \begin{pmatrix} 0.41978 \\ 0.19230 \end{pmatrix}, \qquad z_2 \approx 0.25572, \qquad \hat y_2 \approx 0.56362.
$$

**Шаг $t = 3$:**

$$
W_h h_2 = \begin{pmatrix} 0.2 & -0.1 \\ 0.4 & 0.3 \end{pmatrix}\begin{pmatrix} 0.41978 \\ 0.19230 \end{pmatrix} = \begin{pmatrix} 0.08396 - 0.01923 \\ 0.16791 + 0.05769 \end{pmatrix} = \begin{pmatrix} 0.06473 \\ 0.22560 \end{pmatrix},
$$
$$
a_3 = \begin{pmatrix} -0.15 \\ -0.09 \end{pmatrix} + \begin{pmatrix} 0.06473 \\ 0.22560 \end{pmatrix} + \begin{pmatrix} 0.1 \\ -0.2 \end{pmatrix} = \begin{pmatrix} 0.01473 \\ -0.06440 \end{pmatrix},
$$
$$
h_3 \approx \begin{pmatrix} 0.01472 \\ -0.06431 \end{pmatrix}, \qquad z_3 \approx 0.14099, \qquad \hat y_3 \approx 0.53519.
$$

### Таблица локальных производных

**Кирпичи — те же, что в §7.5**, повторяем для удобства:

$$
\frac{\partial a_t}{\partial x_t} = W_x \in \mathbb{R}^{N \times D}, \quad
\frac{\partial a_t}{\partial h_{t-1}} = W_h \in \mathbb{R}^{N \times N}, \quad
\frac{\partial a_t}{\partial b_h} = I_N, \tag{8.2a}
$$

$$
\frac{\partial h_t}{\partial a_t} = \operatorname{diag}(1 - h_t \odot h_t) \in \mathbb{R}^{N \times N}, \tag{8.2b}
$$

$$
\frac{\partial z_t}{\partial h_t} = w_y \in \mathbb{R}^N, \quad
\frac{\partial z_t}{\partial w_y} = h_t \in \mathbb{R}^N, \quad
\frac{\partial z_t}{\partial b_y} = 1, \tag{8.2c}
$$

$$
\frac{\partial \hat y_t}{\partial z_t} = \hat y_t(1-\hat y_t). \tag{8.2d}
$$

Эти формулы **не зависят от $T$** — важно для дальнейшего: именно поэтому матричный BPTT работает при любом $T$.

## 8.4. Функция потерь

$$
L = \sum_{t=1}^{T} \frac{1}{2}\,(\hat y_t - y_t)^2 \in \mathbb{R}. \tag{8.3}
$$

Потеря аддитивна, каждое $\hat y_t$ входит только в своё слагаемое:

$$
\frac{\partial L}{\partial \hat y_t} = \hat y_t - y_t, \qquad t = 1, \dots, T. \tag{8.4}
$$

## 8.5. Обратный проход (BPTT)

### 8.5.1. Сколько путей — общая формула

В главе 6 мы посчитали: от $a_t$ до $L$ ведёт **$T - t + 1$ путей** — «свой выход» плюс по одному пути на каждый будущий шаг:

| Откуда | Куда | Число путей |
|---|---|---|
| $a_T$ | $\hat y_T$ | 1 |
| $a_{T-1}$ | $\hat y_{T-1}$, $a_T \to \hat y_T$ | 2 |
| $a_{T-2}$ | $\hat y_{T-2}$, $a_{T-1}\to\hat y_{T-1}$, $a_{T-1}\to a_T\to\hat y_T$ | 3 |
| $\dots$ | $\dots$ | $\dots$ |
| $a_1$ | $\hat y_1$, $a_2 \to \hat y_2$, $\dots$, $a_2 \to \dots \to a_T \to \hat y_T$ | $T$ |

Суммарно — $T(T+1)/2$ путей. При $T = 100$ это $5050$, при $T = 1000$ — $500500$. Выписывать их руками бессмысленно, нужна рекурсия. **Матричная формула ниже сворачивает все эти пути в одну строку.**

### 8.5.2. Ключевая рекуррентная формула

**Утверждение.** Определим $\delta^a_t := \partial L/\partial a_t \in \mathbb{R}^N$ и $\delta^y_t := \partial L/\partial z_t \in \mathbb{R}$ (скаляр). Тогда

$$
\boxed{\;
\delta^a_t = \bigl(\delta^y_t\,w_y + W_h^\top \delta^a_{t+1}\bigr) \odot (1 - h_t \odot h_t), \quad t = 1, \dots, T-1;
\qquad
\delta^a_T = (\delta^y_T\,w_y) \odot (1 - h_T \odot h_T).
\;}
\tag{8.5}
$$

**Доказательство.** Рассмотрим $a_t \in \mathbb{R}^N$. Оно входит только в $h_t = \tanh(a_t)$ (покомпонентно). По цепному правилу:

$$
\frac{\partial L}{\partial a_{t,i}} = \frac{\partial L}{\partial h_{t,i}}\cdot\frac{\partial h_{t,i}}{\partial a_{t,i}} = \frac{\partial L}{\partial h_{t,i}}\,(1 - h_{t,i}^2). \tag{8.6}
$$

В векторной форме:

$$
\frac{\partial L}{\partial a_t} = \frac{\partial L}{\partial h_t} \odot (1 - h_t \odot h_t). \tag{8.7}
$$

**Считаем $\partial L/\partial h_t$.** Переменная $h_t \in \mathbb{R}^N$ входит ровно в **два** уравнения:

- $z_t = w_y^\top h_t + b_y$ — выход **своего** шага;
- $a_{t+1} = W_x x_{t+1} + W_h h_t + b_h$ — преактивация **следующего** шага.

При $t = T$ второго уравнения нет. Поэтому при $t < T$ полная производная — **сумма двух вкладов**:

$$
\frac{\partial L}{\partial h_t} = \underbrace{\frac{\partial L}{\partial z_t}\cdot\frac{\partial z_t}{\partial h_t}}_{\text{вклад через свой шаг}} + \underbrace{W_h^\top \frac{\partial L}{\partial a_{t+1}}}_{\text{вклад через шаг } t+1}. \tag{8.8}
$$

**Первый вклад.** $\partial L/\partial z_t = \delta^y_t$ — скаляр; $\partial z_t/\partial h_t = w_y$ — вектор. Значит, первый вклад — это $\delta^y_t\,w_y \in \mathbb{R}^N$.

**Второй вклад.** Здесь нужно быть аккуратным: $\partial a_{t+1}/\partial h_t = W_h \in \mathbb{R}^{N \times N}$ — это **якобиан** отображения $\mathbb{R}^N \to \mathbb{R}^N$, и при умножении на вектор $\partial L/\partial a_{t+1} \in \mathbb{R}^N$ нужно брать **транспонированный** якобиан. Покомпонентно:

$$
\left(\frac{\partial L}{\partial h_t}\right)_{\text{второй}} = \sum_{j=1}^{N} \frac{\partial L}{\partial a_{t+1,j}}\cdot\frac{\partial a_{t+1,j}}{\partial h_{t,i}} = \sum_{j=1}^{N} \delta^a_{t+1,j}\,(W_h)_{ji} = (W_h^\top \delta^a_{t+1})_i.
$$

Это и есть второй вклад — $W_h^\top \delta^a_{t+1} \in \mathbb{R}^N$.

**Складываем:**

$$
\frac{\partial L}{\partial h_t} = \delta^y_t\,w_y + W_h^\top \delta^a_{t+1}. \tag{8.9}
$$

Подставляем (8.9) в (8.7) — получаем (8.5) для $t < T$. Для $t = T$ второго вклада нет: $\partial L/\partial h_T = \delta^y_T\,w_y$, и (8.5) даёт базовый случай. $\blacksquare$

**Сравнение с предельными случаями.**

| Формула | При $N=1$, $T$ общем | При $N$ общем, $T=2$ |
|---|---|---|
| (8.5) | (6.7): $\delta^a_t = (\delta^y_t w_y + \delta^a_{t+1} w_h)(1-h_t^2)$ | (7.15): $\delta^a_1 = (\delta^y_1 w_y + W_h^\top \delta^a_2)\odot(1-h_1\odot h_1)$ |
| совпадение | $\checkmark$ | $\checkmark$ |

При $N=1$: $W_h^\top$ — скаляр $w_h$, $\odot$ — обычное умножение. При $T=2$: в (8.5) есть только $t=1$, и это ровно (7.15) из главы 7.

**Порядок вычисления** (справа налево):

1. Считаем все $\delta^y_t$ по формуле (8.10) — независимо на каждом шаге.
2. Для $t = T$: вычисляем $\delta^a_T = (\delta^y_T w_y) \odot (1 - h_T \odot h_T)$.
3. Для $t = T-1, \dots, 1$: вычисляем $\delta^a_t$ по (8.5), используя уже посчитанное $\delta^a_{t+1}$.

### 8.5.3. Дельты выходов

Из (8.2d) и (8.4), по определению $\delta^y_t := \partial L/\partial z_t$:

$$
\delta^y_t = (\hat y_t - y_t)\,\hat y_t\,(1 - \hat y_t) \in \mathbb{R}, \qquad t = 1, \dots, T. \tag{8.10}
$$

Все $T$ значений **независимы** — потеря аддитивна, $\hat y_t$ входит только в $t$-е слагаемое.

### 8.5.4. Пять градиентов — суммы по шагам

Каждый параметр входит во все (или почти все) шаги, поэтому производная по нему — **сумма по $t$**. Каждое слагаемое — это цепочка, свёрнутая в $\delta$.

**Для $w_y \in \mathbb{R}^N$.** $w_y$ входит в $z_t$ на каждом шаге; $\partial z_t/\partial w_y = h_t$:

$$
\boxed{\;\frac{\partial L}{\partial w_y} = \sum_{t=1}^{T} \delta^y_t\,h_t \in \mathbb{R}^N.\;} \tag{8.11}
$$

**Для $b_y \in \mathbb{R}$.** $\partial z_t/\partial b_y = 1$:

$$
\boxed{\;\frac{\partial L}{\partial b_y} = \sum_{t=1}^{T} \delta^y_t \in \mathbb{R}.\;} \tag{8.12}
$$

**Для $W_x \in \mathbb{R}^{N \times D}$.** Покомпонентно:

$$
\frac{\partial L}{\partial (W_x)_{jk}} = \sum_{t=1}^{T} \sum_{i=1}^{N} \delta^a_{t,i}\,\delta_{ij}\,x_{t,k} = \sum_{t=1}^{T} \delta^a_{t,j}\,x_{t,k}.
$$

Внешнее произведение $\delta^a_t \in \mathbb{R}^N$ и $x_t \in \mathbb{R}^D$:

$$
\boxed{\;\frac{\partial L}{\partial W_x} = \sum_{t=1}^{T} \delta^a_t\,x_t^\top \in \mathbb{R}^{N \times D}.\;} \tag{8.13}
$$

**Для $W_h \in \mathbb{R}^{N \times N}$.** Покомпонентно:

$$
\frac{\partial L}{\partial (W_h)_{jk}} = \sum_{t=1}^{T} \delta^a_{t,j}\,h_{t-1,k}.
$$

Внешнее произведение:

$$
\boxed{\;\frac{\partial L}{\partial W_h} = \sum_{t=1}^{T} \delta^a_t\,h_{t-1}^\top \in \mathbb{R}^{N \times N}.\;} \tag{8.14}
$$

Слагаемое $t=1$ равно нулю, так как $h_0 = \mathbf{0}$; но в сумме оно учитывается автоматически.

**Для $b_h \in \mathbb{R}^N$.** $\partial a_t/\partial b_h = I_N$:

$$
\boxed{\;\frac{\partial L}{\partial b_h} = \sum_{t=1}^{T} \delta^a_t \in \mathbb{R}^N.\;} \tag{8.15}
$$

### 8.5.5. Сводка обратного прохода (матричный BPTT)

$$
\boxed{
\begin{aligned}
&\text{1. Для } t = 1, \dots, T: \quad \delta^y_t = (\hat y_t - y_t)\,\hat y_t(1-\hat y_t) \in \mathbb{R}; \\[4pt]
&\text{2. } \delta^a_T = (\delta^y_T\,w_y) \odot (1 - h_T \odot h_T) \in \mathbb{R}^N; \\[2pt]
&\text{3. Для } t = T-1, \dots, 1: \quad \delta^a_t = \bigl(\delta^y_t\,w_y + W_h^\top \delta^a_{t+1}\bigr) \odot (1 - h_t \odot h_t); \\[4pt]
&\text{4. } \frac{\partial L}{\partial w_y} = \sum_t \delta^y_t\,h_t, \quad \frac{\partial L}{\partial b_y} = \sum_t \delta^y_t, \\
&\text{5. } \frac{\partial L}{\partial W_x} = \sum_t \delta^a_t\,x_t^\top, \quad \frac{\partial L}{\partial W_h} = \sum_t \delta^a_t\,h_{t-1}^\top, \quad \frac{\partial L}{\partial b_h} = \sum_t \delta^a_t.
\end{aligned}}
\tag{8.16}
$$

**Проверка при $N=1$, $T=3$** (глава 5): $W_h^\top \delta^a_{t+1}$ становится $\delta^a_{t+1} w_h$ (скаляр), $\odot$ — обычное умножение. Получаем (5.8)–(5.12). При $N=1$, $T=2$ (глава 4) — (4.17), (4.18). При $N$ общем, $T=2$ (глава 7) — (7.15). Все три предельных случая воспроизводятся. $\checkmark$

## 8.6. Численный пример: $T=3$, $N=2$

Продолжим пример из §8.3.

### Потеря

$$
L = \tfrac12(0.59206)^2 + \tfrac12(0.56362 - 1)^2 + \tfrac12(0.53519 - 0.5)^2 \approx 0.17527 + 0.09521 + 0.00062 = 0.27110.
$$

### Дельты выходов

$$
\delta^y_1 = 0.59206\cdot 0.59206\cdot(1 - 0.59206) \approx 0.14301,
$$
$$
\delta^y_2 = (-0.43638)\cdot 0.56362\cdot 0.43638 \approx -0.10733,
$$
$$
\delta^y_3 = 0.03519\cdot 0.53519\cdot 0.46481 \approx 0.00875.
$$

### Дельты скрытых (справа налево)

**Шаг $t=3$.** $\delta^a_3 = (\delta^y_3\,w_y) \odot (1 - h_3 \odot h_3)$:

$$
\delta^y_3\,w_y = 0.00875\cdot\begin{pmatrix} 0.6 \\ -0.5 \end{pmatrix} = \begin{pmatrix} 0.00525 \\ -0.00438 \end{pmatrix},
$$
$$
1 - h_3 \odot h_3 = \begin{pmatrix} 1 - 0.01472^2 \\ 1 - (-0.06431)^2 \end{pmatrix} = \begin{pmatrix} 0.99978 \\ 0.99586 \end{pmatrix},
$$
$$
\delta^a_3 = \begin{pmatrix} 0.00525 \\ -0.00438 \end{pmatrix}\odot\begin{pmatrix} 0.99978 \\ 0.99586 \end{pmatrix} = \begin{pmatrix} 0.00525 \\ -0.00436 \end{pmatrix}.
$$

**Шаг $t=2$.** $\delta^a_2 = (\delta^y_2\,w_y + W_h^\top \delta^a_3) \odot (1 - h_2 \odot h_2)$:

$$
\delta^y_2\,w_y = (-0.10733)\begin{pmatrix} 0.6 \\ -0.5 \end{pmatrix} = \begin{pmatrix} -0.06440 \\ 0.05366 \end{pmatrix},
$$
$$
W_h^\top \delta^a_3 = \begin{pmatrix} 0.2 & 0.4 \\ -0.1 & 0.3 \end{pmatrix}\begin{pmatrix} 0.00525 \\ -0.00436 \end{pmatrix} = \begin{pmatrix} 0.00105 - 0.00174 \\ -0.00053 - 0.00131 \end{pmatrix} = \begin{pmatrix} -0.00069 \\ -0.00184 \end{pmatrix},
$$
$$
\delta^y_2\,w_y + W_h^\top \delta^a_3 = \begin{pmatrix} -0.06509 \\ 0.05182 \end{pmatrix},
$$
$$
1 - h_2 \odot h_2 = \begin{pmatrix} 1 - 0.41978^2 \\ 1 - 0.19230^2 \end{pmatrix} = \begin{pmatrix} 0.82378 \\ 0.96302 \end{pmatrix},
$$
$$
\delta^a_2 = \begin{pmatrix} -0.06509 \\ 0.05182 \end{pmatrix}\odot\begin{pmatrix} 0.82378 \\ 0.96302 \end{pmatrix} = \begin{pmatrix} -0.05362 \\ 0.04991 \end{pmatrix}.
$$

**Шаг $t=1$.** $\delta^a_1 = (\delta^y_1\,w_y + W_h^\top \delta^a_2) \odot (1 - h_1 \odot h_1)$:

$$
\delta^y_1\,w_y = 0.14301\begin{pmatrix} 0.6 \\ -0.5 \end{pmatrix} = \begin{pmatrix} 0.08581 \\ -0.07151 \end{pmatrix},
$$
$$
W_h^\top \delta^a_2 = \begin{pmatrix} 0.2 & 0.4 \\ -0.1 & 0.3 \end{pmatrix}\begin{pmatrix} -0.05362 \\ 0.04991 \end{pmatrix} = \begin{pmatrix} -0.01072 + 0.01996 \\ 0.00536 + 0.01497 \end{pmatrix} = \begin{pmatrix} 0.00924 \\ 0.02033 \end{pmatrix},
$$
$$
\delta^y_1\,w_y + W_h^\top \delta^a_2 = \begin{pmatrix} 0.09505 \\ -0.05117 \end{pmatrix},
$$
$$
1 - h_1 \odot h_1 = \begin{pmatrix} 1 - 0.53705^2 \\ 1 - 0.09967^2 \end{pmatrix} = \begin{pmatrix} 0.71158 \\ 0.99007 \end{pmatrix},
$$
$$
\delta^a_1 = \begin{pmatrix} 0.09505 \\ -0.05117 \end{pmatrix}\odot\begin{pmatrix} 0.71158 \\ 0.99007 \end{pmatrix} = \begin{pmatrix} 0.06763 \\ -0.05066 \end{pmatrix}.
$$

### Пять градиентов

$$
\frac{\partial L}{\partial w_y} = \sum_t \delta^y_t\,h_t = \begin{pmatrix} 0.03187 \\ -0.00695 \end{pmatrix}, \qquad \frac{\partial L}{\partial b_y} = \sum_t \delta^y_t = 0.04443.
$$

$$
\frac{\partial L}{\partial W_x} = \delta^a_1 x_1^\top + \delta^a_2 x_2^\top + \delta^a_3 x_3^\top = \begin{pmatrix} 0.06763 \\ -0.05066 \end{pmatrix}\cdot 1.0 + \begin{pmatrix} -0.05362 \\ 0.04991 \end{pmatrix}\cdot 0.5 + \begin{pmatrix} 0.00525 \\ -0.00436 \end{pmatrix}\cdot(-0.3)
$$
$$
= \begin{pmatrix} 0.06763 - 0.02681 - 0.00157 \\ -0.05066 + 0.02496 + 0.00131 \end{pmatrix} = \begin{pmatrix} 0.03925 \\ -0.02440 \end{pmatrix} \in \mathbb{R}^{2\times 1}.
$$

$$
\frac{\partial L}{\partial W_h} = \delta^a_2 h_1^\top + \delta^a_3 h_2^\top = \begin{pmatrix} -0.05362 \\ 0.04991 \end{pmatrix}\begin{pmatrix} 0.53705 & 0.09967 \end{pmatrix} + \begin{pmatrix} 0.00525 \\ -0.00436 \end{pmatrix}\begin{pmatrix} 0.41978 & 0.19230 \end{pmatrix}
$$
$$
= \begin{pmatrix} -0.02880 & -0.00534 \\ 0.02681 & 0.00498 \end{pmatrix} + \begin{pmatrix} 0.00220 & 0.00101 \\ -0.00183 & -0.00084 \end{pmatrix} = \begin{pmatrix} -0.02660 & -0.00434 \\ 0.02498 & 0.00414 \end{pmatrix}.
$$

$$
\frac{\partial L}{\partial b_h} = \delta^a_1 + \delta^a_2 + \delta^a_3 = \begin{pmatrix} 0.06763 - 0.05362 + 0.00525 \\ -0.05066 + 0.04991 - 0.00436 \end{pmatrix} = \begin{pmatrix} 0.01926 \\ -0.00511 \end{pmatrix}.
$$

**Проверка при $N=1$.** Если убрать вторые компоненты и вторые строки/столбцы, числа **совпадают** с §5.7 главы 5 (при небольшой разнице в $y_3$ — там $0.5$, здесь тоже $0.5$; все параметры те же). Это доказывает, что (8.16) корректно сводится к (5.8)–(5.12).

## 8.7. Затухание и взрыв: анализ через матричную формулу

Матричная форма позволяет формально увидеть, откуда берётся затухание. Развернём (8.5) по цепочке. Обозначим $D_t := \operatorname{diag}(1 - h_t \odot h_t)$:

$$
\delta^a_t = D_t\,(\delta^y_t\,w_y) + D_t\,W_h^\top\,\delta^a_{t+1}. \tag{8.17}
$$

Подставляем $\delta^a_{t+1}$ (через $\delta^a_{t+2}$), затем $\delta^a_{t+2}$ и т. д. — получаем **явное выражение** через дельты выходов всех будущих шагов:

$$
\delta^a_t = \sum_{s=t}^{T} \Bigl(\prod_{k=t}^{s-1} D_k\,W_h^\top\Bigr)\,D_s\,(\delta^y_s\,w_y). \tag{8.18}
$$

Каждое слагаемое в сумме соответствует **одному пути** от $a_t$ до $\hat y_s$. Слагаемое $s = t$ — путь «через свой выход»; слагаемое $s = t+1$ — путь через один шаг; $s = T$ — путь через $T - t$ шагов.

**Норма вклада от шага $s$:**

$$
\|\text{вклад от } s\| \;\leq\; \underbrace{\|W_h\|^{s-t}}_{\text{рекуррентная часть}}\;\underbrace{\prod_{k=t}^{s-1}\|D_k\|}_{\leq 1}\;\|D_s\|\,|\delta^y_s|\,\|w_y\|. \tag{8.19}
$$

Множители:

- $\|W_h\|^{s-t}$ — **экспоненциально** растёт при $\|W_h\| > 1$ и убывает при $\|W_h\| < 1$;
- $\|D_k\| = \max_i |1 - h_{k,i}^2| \leq 1$ — тангенс всегда сжимает, но не растягивает.

**Следствие.** Если $\|W_h\| < 1$, вклад далёких шагов **экспоненциально мал** — градиент «затухает», и сеть не может выучить долгую зависимость. Если $\|W_h\| > 1$, вклад может **взорваться** — обучение нестабильно. Это и есть **проблема затухающих/взрывающихся градиентов**.

**Проверка на примере §8.6.** $\|W_h\|_F \approx \sqrt{0.2^2 + 0.1^2 + 0.4^2 + 0.3^2} = \sqrt{0.30} \approx 0.55 < 1$. Вклад $s=1$ в $\delta^a_1$ — это $\delta^y_1 w_y \approx (0.0858, -0.0715)$ с нормой $\approx 0.112$; вклад $s=2$ — $W_h^\top \delta^a_2 \approx (0.0092, 0.0203)$ с нормой $\approx 0.022$ (в 5 раз меньше); вклад $s=3$ — уже $0.0053$ и $-0.0044$, норма $\approx 0.007$ (ещё в 3 раза меньше). Экспоненциальное затухание видно уже при $T=3$.

**Как с этим борется LSTM** (глава 13): он заменяет произведение $D_k W_h^\top$ на произведение **вентилей**, которые могут принимать значения, близкие к $1$, сохраняя память на много шагов. Именно поэтому LSTM не «страдает» от $w_h^{s-t}$ — этого множителя в её формуле для $\partial L/\partial c_t$ просто нет.

## 8.8. Полная программа

Класс `VectorRNN` из главы 7 уже работает при любом $T$. Ничего менять не нужно: циклы `for t in reversed(range(T))` уже написаны в общем виде.

```python
import numpy as np

def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))

class VectorRNN:
    def __init__(self, N, D, seed=0, eta=0.1):
        rng = np.random.default_rng(seed)
        self.N, self.D = N, D
        self.Wx = rng.normal(scale=0.5, size=(N, D))   # (N, D)
        self.Wh = rng.normal(scale=0.5, size=(N, N))   # (N, N)
        self.bh = rng.normal(scale=0.1, size=(N,))     # (N,)
        self.wy = rng.normal(scale=0.5, size=(N,))     # (N,)
        self.by = 0.0
        self.eta = eta

    def forward(self, xs, h0=None):
        T = len(xs)
        h = np.zeros(self.N) if h0 is None else h0
        hs, ys = [], []
        for x in xs:
            a = self.Wx @ x + self.Wh @ h + self.bh  # (8.1)
            h = np.tanh(a)
            z = self.wy @ h + self.by
            ys.append(sigmoid(z))
            hs.append(h)
        return np.array(hs), np.array(ys)

    def loss(self, ys, targets):
        return 0.5 * np.sum((ys - targets) ** 2)       # (8.3)

    def backward(self, xs, hs, ys, targets, h0=None):
        T = len(xs)
        dy = (ys - targets) * ys * (1.0 - ys)         # (8.10)
        dh = np.zeros_like(hs)                        # (T, N)
        dh_next = np.zeros(self.N)                    # (N,)
        for t in reversed(range(T)):
            # (8.5): δ^a_t = (δ^y_t w_y + W_h^T δ^a_{t+1}) ⊙ (1 - h_t ⊙ h_t)
            dh[t] = (dy[t] * self.wy + self.Wh.T @ dh_next) * (1.0 - hs[t]**2)
            dh_next = dh[t]
        h_prev = np.vstack([np.zeros(self.N), hs[:-1]])   # (T, N)
        gWx = sum(np.outer(dh[t], xs[t]) for t in range(T))   # (8.13)
        gWh = sum(np.outer(dh[t], h_prev[t]) for t in range(T))  # (8.14)
        gbh = dh.sum(axis=0)                              # (8.15)
        gwy = sum(dy[t] * hs[t] for t in range(T))        # (8.11)
        gby = dy.sum()                                    # (8.12)
        return dict(Wx=gWx, Wh=gWh, bh=gbh, wy=gwy, by=gby)

    def step(self, xs, targets, h0=None):
        hs, ys = self.forward(xs, h0)
        L = self.loss(ys, targets)
        g = self.backward(xs, hs, ys, targets, h0)
        for k, v in g.items():
            setattr(self, k, getattr(self, k) - self.eta * v)
        return L, ys, g


# Проверка на примере из §8.6
rnn = VectorRNN(N=2, D=1)
rnn.Wx = np.array([[0.5], [0.3]])
rnn.Wh = np.array([[0.2, -0.1], [0.4, 0.3]])
rnn.bh = np.array([0.1, -0.2])
rnn.wy = np.array([0.6, -0.5])
rnn.by = 0.1
rnn.eta = 0.1

xs = [np.array([1.0]), np.array([0.5]), np.array([-0.3])]
ys = np.array([0.0, 1.0, 0.5])
hs, yhat = rnn.forward(xs)
print("h =", hs); print("y^ =", yhat)
L, _, g = rnn.step(xs, ys)
print("L =", L)
for k, v in g.items():
    print(f"dL/d{k} =\n{v}")
```

Вывод совпадает с §8.6. Если запустить код с $N=1, D=1$, он даст числа главы 5; если с $N=2, D=1$ и $T=2$ — числа главы 7. **Один и тот же класс** работает при любых $N, D, T$ — это прямое доказательство, что формулы (8.16) общие.

## 8.9. Что мы получили

1. **Матричный BPTT** — формулы (8.16): рекуррентное соотношение (8.5) плюс пять сумм (8.11)–(8.15). Они работают при любых $N, D, T$.
2. **Доказательство (8.5)** — прямое: у $h_t$ ровно два потребителя ($z_t$ и $a_{t+1}$), поэтому $\partial L/\partial h_t$ — сумма двух вкладов; после умножения на $D_t = \operatorname{diag}(1 - h_t \odot h_t)$ получается (8.5). Ключевой момент — транспонирование $W_h^\top$: якобиан отображения $h_t \mapsto a_{t+1}$ равен $W_h$, и при умножении на вектор-градиент его нужно транспонировать.
3. **Проверка трёх предельных случаев** — (8.16) сводится к (4.15)–(4.21) при $N=1, T=2$; к (5.8)–(5.15) при $N=1, T=3$; к (7.15) при $N$ общем, $T=2$. Это доказывает корректность обобщения.
4. **Затухание и взрыв** — формально следует из (8.18)–(8.19): вклад далёкого шага умножается на $\|W_h\|^{s-t}$ на каждом шаге. Это и есть проблема, которую решают LSTM и GRU.
5. **Код** — один класс `VectorRNN` работает при любых $N, D, T$; никаких изменений при переходе от $T=2$ к $T$ общему не требуется.

В следующих главах мы расширим схему в двух направлениях: **несколько выходов на шаге** (глава 9), **стек слоёв RNN** (глава 10), **двунаправленная RNN** (глава 11), **формальный анализ затухания** (глава 12). Но структура останется той же, что (8.16): рекуррентное соотношение для $\delta^a_t$ + пять сумм по $t$. Формулы этой главы — строительный блок для всего дальнейшего.

# Глава 9. Общий случай: $K$ выходов на шаге — от одного выхода к вектору выходов

В главах 7–8 мы перешли от скаляров к векторам для **скрытого** состояния: $h_t \in \mathbb{R}^N$. Но выход оставался скалярным: $z_t, \hat y_t, y_t \in \mathbb{R}$. Этого достаточно, когда сеть предсказывает одно число. Но во многих задачах выход — **вектор**: классификация на $K$ классов, многозадачное обучение, предсказание $K$ значений временного ряда одновременно. Эта глава — переход к $K$ выходам. Формально изменение мало: $w_y \in \mathbb{R}^N \to W_y \in \mathbb{R}^{K \times N}$, $\delta^y_t \in \mathbb{R} \to \delta^y_t \in \mathbb{R}^K$. Но именно это изменение показывает **общий шаблон** формул BPTT — $\delta^y_t w_y$ превращается в $W_y^\top \delta^y_t$, и этот шаблон «транспонирования» повторяется во всех последующих главах.

## 9.1. Зачем нужна эта глава

В главе 8 $\delta^y_t$ был **скаляром**: сеть предсказывает одно число, ошибка — одно число. В реальных задачах выходов обычно много:

- **Классификация на $K$ классов** — сеть выдаёт вектор логитов или вероятностей, потеря сравнивает его с one-hot вектором;
- **Многозадачное обучение** — один и тот же скрытый вектор $h_t$ используется для $K$ разных предсказаний;
- **Предсказание $K$ значений временного ряда** — на каждом шаге $t$ предсказывается вектор будущих значений.

Во всех этих случаях $\hat y_t, y_t \in \mathbb{R}^K$, а потеря — сумма по $K$ компонентам. Это и есть предмет главы.

**Ключевое изменение по сравнению с главой 8** — одно:

$$
\delta^y_t\,w_y \quad\to\quad W_y^\top \delta^y_t, \tag{9.1}
$$

где $w_y \in \mathbb{R}^N$ — вектор, $W_y \in \mathbb{R}^{K \times N}$ — матрица, $\delta^y_t \in \mathbb{R}$ — скаляр в главе 8, $\delta^y_t \in \mathbb{R}^K$ — вектор в этой главе. Произведение $\delta^y_t w_y$ в главе 8 — это скаляр × вектор = вектор $\mathbb{R}^N$. Произведение $W_y^\top \delta^y_t$ в этой главе — это матрица $\mathbb{R}^{N \times K}$ × вектор $\mathbb{R}^K$ = вектор $\mathbb{R}^N$. **Результат той же формы**, но структура другая: теперь каждый из $K$ выходов вносит свой вклад в каждую из $N$ компонент $\delta^a_t$, и матрица $W_y^\top$ автоматически суммирует эти вклады.

Именно этот шаблон — **замена вектор-вектора на матрицу-транспонированную** — повторится в LSTM (глава 13), где $w_y$ заменится на $W_f, W_i, W_o, W_c$.

## 9.2. Архитектура и обозначения

Сеть имеет **вход размерности $D$**, **$N$ скрытых нейронов** и **$K$ выходов**, развёрнута на $T$ шагов.

| Символ | Размерность | Смысл |
|---|---|---|
| $x_t$ | $\mathbb{R}^D$ | вход на шаге $t$ |
| $a_t$ | $\mathbb{R}^N$ | преактивация скрытого слоя |
| $h_t$ | $\mathbb{R}^N$ | скрытое состояние |
| $z_t$ | $\mathbb{R}^K$ | преактивация выходов |
| $\hat y_t$ | $\mathbb{R}^K$ | предсказание |
| $y_t$ | $\mathbb{R}^K$ | истинный ответ |
| $W_x$ | $\mathbb{R}^{N \times D}$ | веса «вход → скрытый» |
| $W_h$ | $\mathbb{R}^{N \times N}$ | веса рекуррентной связи |
| $b_h$ | $\mathbb{R}^N$ | смещение скрытого слоя |
| $W_y$ | $\mathbb{R}^{K \times N}$ | веса «скрытый → выходы» |
| $b_y$ | $\mathbb{R}^K$ | смещения выходов |
| $h_0 = \mathbf{0}$ | $\mathbb{R}^N$ | начальное состояние |
| $T$ | $\mathbb{N}$ | длина последовательности |
| $K$ | $\mathbb{N}$ | число выходов на шаге |
| $\eta$ | $\mathbb{R}$ | шаг обучения |

**Уравнения прямого прохода** (для каждого $t = 1, \dots, T$):

$$
\boxed{
\begin{aligned}
a_t &= W_x\,x_t + W_h\,h_{t-1} + b_h &&\in \mathbb{R}^N, \\
h_t &= \tanh(a_t) &&\in \mathbb{R}^N \text{ (покомпонентно)}, \\
z_t &= W_y\,h_t + b_y &&\in \mathbb{R}^K, \\
\hat y_t &= \sigma(z_t) &&\in \mathbb{R}^K \text{ (покомпонентно)}.
\end{aligned}}
\tag{9.2}
$$

**Покомпонентная запись выхода:**

$$
z_{t,k} = \sum_{i=1}^{N} (W_y)_{ki}\,h_{t,i} + (b_y)_k, \qquad \hat y_{t,k} = \sigma(z_{t,k}), \qquad k = 1, \dots, K. \tag{9.2'}
$$

**Число обучаемых параметров:**

$$
\underbrace{ND}_{W_x} + \underbrace{N^2}_{W_h} + \underbrace{N}_{b_h} + \underbrace{KN}_{W_y} + \underbrace{K}_{b_y} = ND + N^2 + N + KN + K. \tag{9.3}
$$

При $K = 1$ это $ND + N^2 + N + N + 1 = ND + N^2 + 2N + 1$ — ровно число из (7.1) и (8.1). **Глава 9 — прямое обобщение глав 7–8.**

На рис. 9.1 — архитектура с $K=3$ выходами, $N=3$ скрытыми, $D=2$, $T=2$.

```python
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch

def draw_architecture_K(N=3, D=2, K=3):
    """Рис. 9.1: N скрытых нейронов, K выходов, T=2."""
    fig, ax = plt.subplots(figsize=(14, 9))
    ax.set_xlim(0, 14); ax.set_ylim(0, 10)
    ax.set_aspect('equal'); ax.axis('off')
    r = 0.32
    cols = [2.5, 11.5]
    y_in_base = 1.2
    y_h_base = 4.2
    y_out_base = 7.8

    def node(cx, cy, label, color):
        ax.add_patch(Circle((cx, cy), r, facecolor=color,
                            edgecolor='black', lw=1.5, zorder=3))
        ax.text(cx, cy, label, ha='center', va='center',
                fontsize=10, zorder=4)

    def arrow(p1, p2, color='black', lw=0.7, alpha=0.4):
        ax.add_patch(FancyArrowPatch(
            p1, p2, arrowstyle='-|>', mutation_scale=10,
            lw=lw, color=color, alpha=alpha,
            shrinkA=0, shrinkB=0, zorder=2))

    for t, cx in enumerate(cols, start=1):
        for j in range(D):
            node(cx, y_in_base + j*0.9, rf'$x_{t},{j+1}$', '#AEDFF7')
        for i in range(N):
            node(cx, y_h_base + i*0.9, rf'$h_{t},{i+1}$', '#FFE29A')
        for k in range(K):
            node(cx, y_out_base + k*0.9, rf'$\hat y_{t},{k+1}$', '#B8E0B8')
        for j in range(D):
            for i in range(N):
                arrow((cx + r*0.5, y_in_base + j*0.9),
                      (cx - r*0.5, y_h_base + i*0.9), color='#B22222')
        for i in range(N):
            for k in range(K):
                arrow((cx + r*0.5, y_h_base + i*0.9),
                      (cx - r*0.5, y_out_base + k*0.9), color='#B22222')

    for i in range(N):
        for k in range(N):
            arrow((cols[0] + r, y_h_base + i*0.9),
                  (cols[1] - r, y_h_base + k*0.9),
                  color='#7B2D8E', lw=0.8, alpha=0.5)

    ax.text(cols[0], 0.3, 'Шаг t=1', ha='center', fontsize=12, fontweight='bold')
    ax.text(cols[1], 0.3, 'Шаг t=2', ha='center', fontsize=12, fontweight='bold')
    ax.text((cols[0]+cols[1])/2, 9.7,
            f'Рис. 9.1. Архитектура: D={D} входов, N={N} скрытых, K={K} выходов, T=2',
            ha='center', fontsize=13, fontweight='bold')
    ax.text((cols[0]+cols[1])/2, 6.0, r'$W_h$', fontsize=14, color='#7B2D8E', ha='center')
    ax.text(cols[0] + 0.6, 2.8, r'$W_x$', fontsize=11, color='#B22222')
    ax.text(cols[0] + 0.6, 7.2, r'$W_y$', fontsize=11, color='#B22222')
    plt.tight_layout(); plt.show()

draw_architecture_K(N=3, D=2, K=3)
```

## 9.3. Прямой проход

Прямой проход — это применение (9.2) для $t = 1, \dots, T$. Скрытая часть полностью совпадает с главой 8; новое — в выходных уравнениях.

### Подстановка чисел: $N=2$, $D=1$, $K=2$, $T=2$

Возьмём те же $W_x, W_h, b_h$, что в §8.3, и **новые** $W_y, b_y$:

$$
W_x = \begin{pmatrix} 0.5 \\ 0.3 \end{pmatrix}, \quad
W_h = \begin{pmatrix} 0.2 & -0.1 \\ 0.4 & 0.3 \end{pmatrix}, \quad
b_h = \begin{pmatrix} 0.1 \\ -0.2 \end{pmatrix}, \quad
W_y = \begin{pmatrix} 0.6 & -0.5 \\ 0.4 & 0.3 \end{pmatrix}, \quad
b_y = \begin{pmatrix} 0.1 \\ -0.1 \end{pmatrix}.
$$

Входы: $x_1 = 1.0$, $x_2 = 0.5$. Ответы: $y_1 = (0, 1)^\top$, $y_2 = (1, 0)^\top$. Начальное состояние $h_0 = (0, 0)^\top$.

**Шаг $t = 1$.** Скрытая часть — как в §8.3:

$$
a_1 = \begin{pmatrix} 0.6 \\ 0.1 \end{pmatrix}, \qquad h_1 = \tanh(a_1) \approx \begin{pmatrix} 0.53705 \\ 0.09967 \end{pmatrix}.
$$

Выход — **вектор**:

$$
z_1 = W_y h_1 + b_y = \begin{pmatrix} 0.6 \cdot 0.53705 - 0.5 \cdot 0.09967 + 0.1 \\ 0.4 \cdot 0.53705 + 0.3 \cdot 0.09967 - 0.1 \end{pmatrix} = \begin{pmatrix} 0.37240 \\ 0.14472 \end{pmatrix},
$$
$$
\hat y_1 = \sigma(z_1) \approx \begin{pmatrix} 0.59206 \\ 0.53612 \end{pmatrix}.
$$

**Шаг $t = 2$.** Скрытая часть — как в §8.3:

$$
a_2 = \begin{pmatrix} 0.44744 \\ 0.19472 \end{pmatrix}, \qquad h_2 = \tanh(a_2) \approx \begin{pmatrix} 0.41978 \\ 0.19230 \end{pmatrix}.
$$

Выход:

$$
z_2 = W_y h_2 + b_y = \begin{pmatrix} 0.6 \cdot 0.41978 - 0.5 \cdot 0.19230 + 0.1 \\ 0.4 \cdot 0.41978 + 0.3 \cdot 0.19230 - 0.1 \end{pmatrix} = \begin{pmatrix} 0.25572 \\ 0.12560 \end{pmatrix},
$$
$$
\hat y_2 = \sigma(z_2) \approx \begin{pmatrix} 0.56360 \\ 0.53136 \end{pmatrix}.
$$

### Таблица локальных производных

Кирпичи из §8.5 сохраняются для скрытой части; **новое** — только для выходной:

| Величина | Форма | Компонентно |
|---|---|---|
| $\partial z_t/\partial h_t$ | $\mathbb{R}^{K \times N}$ | $\partial z_{t,k}/\partial h_{t,i} = (W_y)_{ki}$ |
| $\partial z_t/\partial (W_y)_{kj}$ | тензор $K \times K \times N$ | $\delta_{kk'}\,h_{t,j}$ |
| $\partial z_t/\partial b_y$ | $\mathbb{R}^{K \times K}$ | $\delta_{kk'}$ |
| $\partial \hat y_{t,k}/\partial z_{t,j}$ | $\mathbb{R}^{K \times K}$ | $\delta_{kj}\,\hat y_{t,k}(1-\hat y_{t,k})$ |

В векторной форме:

$$
\frac{\partial z_t}{\partial h_t} = W_y \in \mathbb{R}^{K \times N}, \qquad \frac{\partial \hat y_t}{\partial z_t} = \operatorname{diag}\bigl(\hat y_t \odot (1 - \hat y_t)\bigr) \in \mathbb{R}^{K \times K}. \tag{9.4}
$$

Скрытая часть — **та же**, что (8.2a)–(8.2b): $\partial a_t/\partial h_{t-1} = W_h$, $\partial h_t/\partial a_t = \operatorname{diag}(1 - h_t \odot h_t)$. Эти «кирпичи» **не зависят ни от $T$, ни от $K$**.

## 9.4. Функция потерь

Потеря — **сумма по всем шагам и по всем выходам**:

$$
L = \sum_{t=1}^{T} \sum_{k=1}^{K} \frac{1}{2}\,(\hat y_{t,k} - y_{t,k})^2 = \sum_{t=1}^{T} \frac{1}{2}\,\|\hat y_t - y_t\|^2. \tag{9.5}
$$

Потеря аддитивна по $t$ и по $k$ — каждое $\hat y_{t,k}$ входит только в одно слагаемое. Поэтому

$$
\frac{\partial L}{\partial \hat y_{t,k}} = \hat y_{t,k} - y_{t,k}, \qquad t = 1, \dots, T, \quad k = 1, \dots, K, \tag{9.6}
$$

и в векторной форме:

$$
\frac{\partial L}{\partial \hat y_t} = \hat y_t - y_t \in \mathbb{R}^K, \qquad t = 1, \dots, T. \tag{9.6'}
$$

**Замечание о выборе функции потерь.** В этой главе используется MSE — та же, что в главах 4–8, чтобы сохранить преемственность. Для $K$-классовой классификации обычно берут **softmax** на выходе и **кросс-энтропию** — тогда $\delta^y_{t,k} = \hat y_{t,k} - y_{t,k}$ (без множителя $\sigma'$). Структура BPTT при этом **не меняется** — меняется только формула для $\delta^y_t$. Это будет важно для LSTM (глава 13).

## 9.5. Обратный проход (BPTT)

### 9.5.1. Сколько путей — та же формула

Число путей от $a_t$ до $L$ **не зависит от $K$**: по-прежнему $T - t + 1$ путей от $a_t$, суммарно $T(T+1)/2$. Появление $K$ выходов меняет не число путей, а **толщину** каждого: теперь путь от $a_t$ до $\hat y_s$ даёт вклад не в один скаляр, а в $K$ компонент. Но рекурсия (9.7) ниже сворачивает их все — как и в главе 8.

### 9.5.2. Дельты выходов — вектор $\mathbb{R}^K$

Из (9.4) и (9.6'), по определению $\delta^y_t := \partial L/\partial z_t$:

$$
\delta^y_{t,k} = \frac{\partial L}{\partial z_{t,k}} = \frac{\partial L}{\partial \hat y_{t,k}}\cdot\frac{\partial \hat y_{t,k}}{\partial z_{t,k}} = (\hat y_{t,k} - y_{t,k})\,\hat y_{t,k}\,(1 - \hat y_{t,k}),
$$

или в векторной форме:

$$
\boxed{\;\delta^y_t = (\hat y_t - y_t) \odot \hat y_t \odot (1 - \hat y_t) \in \mathbb{R}^K, \qquad t = 1, \dots, T.\;} \tag{9.7}
$$

**Ключевое отличие от главы 8:** $\delta^y_t$ — **вектор** длины $K$, а не скаляр. Все $T$ значений $\delta^y_t$ **независимы** — как и раньше.

### 9.5.3. Ключевая рекуррентная формула

**Утверждение.** Определим $\delta^a_t := \partial L/\partial a_t \in \mathbb{R}^N$. Тогда

$$
\boxed{\;
\delta^a_t = \bigl(W_y^\top \delta^y_t + W_h^\top \delta^a_{t+1}\bigr) \odot (1 - h_t \odot h_t), \quad t = 1, \dots, T-1;
\qquad
\delta^a_T = (W_y^\top \delta^y_T) \odot (1 - h_T \odot h_T).
\;}
\tag{9.8}
$$

**Доказательство.** Рассмотрим $a_t \in \mathbb{R}^N$. Оно входит только в $h_t = \tanh(a_t)$ (покомпонентно), поэтому

$$
\frac{\partial L}{\partial a_t} = \frac{\partial L}{\partial h_t} \odot (1 - h_t \odot h_t). \tag{9.9}
$$

Осталось найти $\partial L/\partial h_t \in \mathbb{R}^N$. Переменная $h_t$ входит ровно в **два** уравнения:

- $z_t = W_y h_t + b_y$ — выход **своего** шага (размерность $K$);
- $a_{t+1} = W_x x_{t+1} + W_h h_t + b_h$ — преактивация **следующего** шага (размерность $N$).

При $t = T$ второго уравнения нет. Поэтому при $t < T$:

$$
\frac{\partial L}{\partial h_t} = \underbrace{\frac{\partial L}{\partial z_t}\cdot\frac{\partial z_t}{\partial h_t}}_{\text{вклад через свой шаг}} + \underbrace{\frac{\partial L}{\partial a_{t+1}}\cdot\frac{\partial a_{t+1}}{\partial h_t}}_{\text{вклад через шаг } t+1}. \tag{9.10}
$$

**Первый вклад.** $\partial L/\partial z_t = \delta^y_t \in \mathbb{R}^K$; якобиан $\partial z_t/\partial h_t = W_y \in \mathbb{R}^{K \times N}$. При умножении **вектора-градиента** на якобиан нужно брать **транспонированный** якобиан (чтобы результат имел форму $\mathbb{R}^N$):

$$
\left(\frac{\partial L}{\partial h_t}\right)_{\text{первый}} = W_y^\top \delta^y_t \in \mathbb{R}^N. \tag{9.11}
$$

Покомпонентно:

$$
(W_y^\top \delta^y_t)_i = \sum_{k=1}^{K} (W_y)_{ki}\,\delta^y_{t,k}.
$$

Это суммирование по $K$ выходам — именно оно и делает вектор $W_y^\top \delta^y_t \in \mathbb{R}^N$. В главе 8 при $K=1$ это было $\delta^y_t w_y$ — скаляр × вектор = вектор.

**Второй вклад.** $\partial L/\partial a_{t+1} = \delta^a_{t+1} \in \mathbb{R}^N$; якобиан $\partial a_{t+1}/\partial h_t = W_h \in \mathbb{R}^{N \times N}$. Транспонирование — как в главе 8:

$$
\left(\frac{\partial L}{\partial h_t}\right)_{\text{второй}} = W_h^\top \delta^a_{t+1} \in \mathbb{R}^N. \tag{9.12}
$$

**Складываем:**

$$
\frac{\partial L}{\partial h_t} = W_y^\top \delta^y_t + W_h^\top \delta^a_{t+1}. \tag{9.13}
$$

Подставляем (9.13) в (9.9) — получаем (9.8) для $t < T$. Для $t = T$ второго вклада нет: $\partial L/\partial h_T = W_y^\top \delta^y_T$, и (9.8) даёт базовый случай. $\blacksquare$

**Сравнение с предельными случаями.**

| Формула | При $K=1$ | При $K>1$, $T=2$ |
|---|---|---|
| (9.8) | (8.5): $\delta^a_t = (\delta^y_t w_y + W_h^\top \delta^a_{t+1}) \odot (1 - h_t \odot h_t)$ | (9.8) при $T=2$ — те же две строки |
| совпадение | $\checkmark$ | $\checkmark$ |

При $K=1$: $W_y$ — это строка $(1 \times N)$, $W_y^\top$ — столбец $(N \times 1)$, $\delta^y_t$ — скаляр, $W_y^\top \delta^y_t = \delta^y_t w_y$. При $T=2$ в (9.8) есть только $t=1$, и это ровно рекуррентная часть (8.5).

**Порядок вычисления:**

1. Для $t = 1, \dots, T$: $\delta^y_t = (\hat y_t - y_t) \odot \hat y_t \odot (1 - \hat y_t) \in \mathbb{R}^K$.
2. $t = T$: $\delta^a_T = (W_y^\top \delta^y_T) \odot (1 - h_T \odot h_T) \in \mathbb{R}^N$.
3. $t = T-1, \dots, 1$: $\delta^a_t$ по (9.8), используя уже посчитанное $\delta^a_{t+1}$.

### 9.5.4. Пять градиентов

**Для $W_y \in \mathbb{R}^{K \times N}$.** $W_y$ входит в $z_t$ на каждом шаге. Покомпонентно:

$$
\frac{\partial L}{\partial (W_y)_{kj}} = \sum_{t=1}^{T} \sum_{k'=1}^{K} \frac{\partial L}{\partial z_{t,k'}}\cdot\frac{\partial z_{t,k'}}{\partial (W_y)_{kj}} = \sum_{t=1}^{T} \sum_{k'=1}^{K} \delta^y_{t,k'}\,\delta_{k'k}\,h_{t,j} = \sum_{t=1}^{T} \delta^y_{t,k}\,h_{t,j}.
$$

Это — **внешнее произведение** $\delta^y_t \in \mathbb{R}^K$ и $h_t \in \mathbb{R}^N$:

$$
\boxed{\;\frac{\partial L}{\partial W_y} = \sum_{t=1}^{T} \delta^y_t\,h_t^\top \in \mathbb{R}^{K \times N}.\;} \tag{9.14}
$$

**Для $b_y \in \mathbb{R}^K$.** $\partial z_t/\partial b_y = I_K$:

$$
\boxed{\;\frac{\partial L}{\partial b_y} = \sum_{t=1}^{T} \delta^y_t \in \mathbb{R}^K.\;} \tag{9.15}
$$

**Для $W_x, W_h, b_h$** — формулы (8.13)–(8.15) главы 8 **без изменений**, потому что скрытая часть не изменилась:

$$
\frac{\partial L}{\partial W_x} = \sum_{t=1}^{T} \delta^a_t\,x_t^\top \in \mathbb{R}^{N \times D}, \qquad \frac{\partial L}{\partial W_h} = \sum_{t=1}^{T} \delta^a_t\,h_{t-1}^\top \in \mathbb{R}^{N \times N}, \qquad \frac{\partial L}{\partial b_h} = \sum_{t=1}^{T} \delta^a_t \in \mathbb{R}^N. \tag{9.16}
$$

Здесь важно: $\delta^a_t$ **изменилась** (формула 9.8), поэтому численно градиенты $W_x, W_h, b_h$ будут **другими**, чем в главе 8 — но **формульно** они те же. Это проявление общей схемы: смена выхода меняет $\delta^y_t$, а через неё — и $\delta^a_t$, и все скрытые градиенты.

### 9.5.5. Сводка обратного прохода

$$
\boxed{
\begin{aligned}
&\text{1. Для } t = 1, \dots, T: \quad \delta^y_t = (\hat y_t - y_t) \odot \hat y_t \odot (1 - \hat y_t) \in \mathbb{R}^K; \\[4pt]
&\text{2. } \delta^a_T = (W_y^\top \delta^y_T) \odot (1 - h_T \odot h_T) \in \mathbb{R}^N; \\[2pt]
&\text{3. Для } t = T-1, \dots, 1: \quad \delta^a_t = \bigl(W_y^\top \delta^y_t + W_h^\top \delta^a_{t+1}\bigr) \odot (1 - h_t \odot h_t); \\[4pt]
&\text{4. } \frac{\partial L}{\partial W_y} = \sum_t \delta^y_t\,h_t^\top, \quad \frac{\partial L}{\partial b_y} = \sum_t \delta^y_t, \\
&\text{5. } \frac{\partial L}{\partial W_x} = \sum_t \delta^a_t\,x_t^\top, \quad \frac{\partial L}{\partial W_h} = \sum_t \delta^a_t\,h_{t-1}^\top, \quad \frac{\partial L}{\partial b_h} = \sum_t \delta^a_t.
\end{aligned}}
\tag{9.17}
$$

**Проверка при $K=1$, $T$ общем** (глава 8): $W_y^\top \delta^y_t$ становится $\delta^y_t w_y$, $\delta^y_t$ становится скаляром. Все формулы (9.17) превращаются в (8.16). $\checkmark$

## 9.6. Численный пример: $N=2$, $D=1$, $K=2$, $T=2$

Продолжим пример из §9.3.

### Потеря по (9.5)

$$
L = \tfrac12\|\hat y_1 - y_1\|^2 + \tfrac12\|\hat y_2 - y_2\|^2.
$$

$$
\hat y_1 - y_1 = \begin{pmatrix} 0.59206 \\ 0.53612 \end{pmatrix} - \begin{pmatrix} 0 \\ 1 \end{pmatrix} = \begin{pmatrix} 0.59206 \\ -0.46388 \end{pmatrix}, \qquad \|\hat y_1 - y_1\|^2 = 0.35054 + 0.21519 = 0.56573,
$$

$$
\hat y_2 - y_2 = \begin{pmatrix} 0.56360 \\ 0.53136 \end{pmatrix} - \begin{pmatrix} 1 \\ 0 \end{pmatrix} = \begin{pmatrix} -0.43640 \\ 0.53136 \end{pmatrix}, \qquad \|\hat y_2 - y_2\|^2 = 0.19045 + 0.28234 = 0.47279,
$$

$$
L = \tfrac12(0.56573) + \tfrac12(0.47279) = 0.28287 + 0.23640 = 0.51927.
$$

### Дельты выходов по (9.7)

**Шаг $t = 1$:**

$$
\hat y_1 \odot (1 - \hat y_1) = \begin{pmatrix} 0.59206 \cdot 0.40794 \\ 0.53612 \cdot 0.46388 \end{pmatrix} = \begin{pmatrix} 0.24153 \\ 0.24872 \end{pmatrix},
$$
$$
\delta^y_1 = \begin{pmatrix} 0.59206 \\ -0.46388 \end{pmatrix} \odot \begin{pmatrix} 0.24153 \\ 0.24872 \end{pmatrix} = \begin{pmatrix} 0.14301 \\ -0.11538 \end{pmatrix}.
$$

**Шаг $t = 2$:**

$$
\hat y_2 \odot (1 - \hat y_2) = \begin{pmatrix} 0.56360 \cdot 0.43640 \\ 0.53136 \cdot 0.46864 \end{pmatrix} = \begin{pmatrix} 0.24596 \\ 0.24902 \end{pmatrix},
$$
$$
\delta^y_2 = \begin{pmatrix} -0.43640 \\ 0.53136 \end{pmatrix} \odot \begin{pmatrix} 0.24596 \\ 0.24902 \end{pmatrix} = \begin{pmatrix} -0.10733 \\ 0.13232 \end{pmatrix}.
$$

### Дельты скрытых (справа налево)

**Шаг $t = 2$** по (9.8), базовый случай:

$$
W_y^\top \delta^y_2 = \begin{pmatrix} 0.6 & 0.4 \\ -0.5 & 0.3 \end{pmatrix}\begin{pmatrix} -0.10733 \\ 0.13232 \end{pmatrix} = \begin{pmatrix} -0.06440 + 0.05293 \\ 0.05367 + 0.03970 \end{pmatrix} = \begin{pmatrix} -0.01147 \\ 0.09337 \end{pmatrix},
$$
$$
1 - h_2 \odot h_2 = \begin{pmatrix} 1 - 0.41978^2 \\ 1 - 0.19230^2 \end{pmatrix} = \begin{pmatrix} 0.82379 \\ 0.96302 \end{pmatrix},
$$
$$
\delta^a_2 = \begin{pmatrix} -0.01147 \\ 0.09337 \end{pmatrix} \odot \begin{pmatrix} 0.82379 \\ 0.96302 \end{pmatrix} = \begin{pmatrix} -0.00945 \\ 0.08992 \end{pmatrix}.
$$

**Шаг $t = 1$** по (9.8), рекуррентная часть:

$$
W_y^\top \delta^y_1 = \begin{pmatrix} 0.6 & 0.4 \\ -0.5 & 0.3 \end{pmatrix}\begin{pmatrix} 0.14301 \\ -0.11538 \end{pmatrix} = \begin{pmatrix} 0.08581 - 0.04615 \\ -0.07151 - 0.03461 \end{pmatrix} = \begin{pmatrix} 0.03966 \\ -0.10612 \end{pmatrix},
$$
$$
W_h^\top \delta^a_2 = \begin{pmatrix} 0.2 & 0.4 \\ -0.1 & 0.3 \end{pmatrix}\begin{pmatrix} -0.00945 \\ 0.08992 \end{pmatrix} = \begin{pmatrix} -0.00189 + 0.03597 \\ 0.000945 + 0.02698 \end{pmatrix} = \begin{pmatrix} 0.03408 \\ 0.02792 \end{pmatrix},
$$
$$
\delta^a_1 = \left[\begin{pmatrix} 0.03966 \\ -0.10612 \end{pmatrix} + \begin{pmatrix} 0.03408 \\ 0.02792 \end{pmatrix}\right] \odot \begin{pmatrix} 0.71158 \\ 0.99007 \end{pmatrix} = \begin{pmatrix} 0.07374 \\ -0.07820 \end{pmatrix} \odot \begin{pmatrix} 0.71158 \\ 0.99007 \end{pmatrix} = \begin{pmatrix} 0.05247 \\ -0.07742 \end{pmatrix}.
$$

### Пять градиентов

**$\partial L/\partial W_y$** по (9.14):

$$
\delta^y_1 h_1^\top = \begin{pmatrix} 0.14301 \\ -0.11538 \end{pmatrix}\begin{pmatrix} 0.53705 & 0.09967 \end{pmatrix} = \begin{pmatrix} 0.07680 & 0.01426 \\ -0.06197 & -0.01150 \end{pmatrix},
$$

$$
\delta^y_2 h_2^\top = \begin{pmatrix} -0.10733 \\ 0.13232 \end{pmatrix}\begin{pmatrix} 0.41978 & 0.19230 \end{pmatrix} = \begin{pmatrix} -0.04505 & -0.02064 \\ 0.05554 & 0.02545 \end{pmatrix},
$$

$$
\frac{\partial L}{\partial W_y} = \begin{pmatrix} 0.03175 & -0.00638 \\ -0.00643 & 0.01395 \end{pmatrix} \in \mathbb{R}^{2\times 2}.
$$

**$\partial L/\partial b_y$** по (9.15):

$$
\frac{\partial L}{\partial b_y} = \delta^y_1 + \delta^y_2 = \begin{pmatrix} 0.14301 - 0.10733 \\ -0.11538 + 0.13232 \end{pmatrix} = \begin{pmatrix} 0.03568 \\ 0.01694 \end{pmatrix} \in \mathbb{R}^2.
$$

**$\partial L/\partial W_x$** по (9.16):

$$
\frac{\partial L}{\partial W_x} = \delta^a_1 x_1^\top + \delta^a_2 x_2^\top = \begin{pmatrix} 0.05247 \\ -0.07742 \end{pmatrix}\cdot 1.0 + \begin{pmatrix} -0.00945 \\ 0.08992 \end{pmatrix}\cdot 0.5 = \begin{pmatrix} 0.05247 - 0.004725 \\ -0.07742 + 0.04496 \end{pmatrix} = \begin{pmatrix} 0.04775 \\ -0.03246 \end{pmatrix}.
$$

**$\partial L/\partial W_h$** по (9.16):

$$
\frac{\partial L}{\partial W_h} = \delta^a_2 h_1^\top = \begin{pmatrix} -0.00945 \\ 0.08992 \end{pmatrix}\begin{pmatrix} 0.53705 & 0.09967 \end{pmatrix} = \begin{pmatrix} -0.00507 & -0.00094 \\ 0.04829 & 0.00896 \end{pmatrix}.
$$

**$\partial L/\partial b_h$** по (9.16):

$$
\frac{\partial L}{\partial b_h} = \delta^a_1 + \delta^a_2 = \begin{pmatrix} 0.05247 - 0.00945 \\ -0.07742 + 0.08992 \end{pmatrix} = \begin{pmatrix} 0.04302 \\ 0.01250 \end{pmatrix}.
$$

### Проверка при $K=1$

Если оставить только первую строку $W_y$ и первый компонент $b_y$, положить $y_t$ скалярами — все числа совпадут с §8.6 главы 8 (при соответствующем выборе параметров). Это доказывает, что (9.17) корректно сводится к (8.16).

## 9.7. Полная программа

Расширим `VectorRNN` из глав 7–8 до `MultiOutputRNN`: добавляются $W_y \in \mathbb{R}^{K \times N}$, $b_y \in \mathbb{R}^K$, а $\hat y_t$ становится вектором.

```python
import numpy as np

def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))

class MultiOutputRNN:
    def __init__(self, N, D, K, seed=0, eta=0.1):
        rng = np.random.default_rng(seed)
        self.N, self.D, self.K = N, D, K
        self.Wx = rng.normal(scale=0.5, size=(N, D))
        self.Wh = rng.normal(scale=0.5, size=(N, N))
        self.bh = rng.normal(scale=0.1, size=(N,))
        self.Wy = rng.normal(scale=0.5, size=(K, N))
        self.by = rng.normal(scale=0.1, size=(K,))
        self.eta = eta

    def forward(self, xs, h0=None):
        T = len(xs)
        h = np.zeros(self.N) if h0 is None else h0
        hs, ys = [], []
        for x in xs:
            a = self.Wx @ x + self.Wh @ h + self.bh  # (N,)
            h = np.tanh(a)
            z = self.Wy @ h + self.by                 # (K,)
            ys.append(sigmoid(z))
            hs.append(h)
        return np.array(hs), np.array(ys)             # (T, N), (T, K)

    def loss(self, ys, targets):
        return 0.5 * np.sum((ys - targets) ** 2)      # (9.5)

    def backward(self, xs, hs, ys, targets, h0=None):
        T = len(xs)
        dy = (ys - targets) * ys * (1.0 - ys)          # (T, K) — (9.7)
        dh = np.zeros_like(hs)                         # (T, N)
        dh_next = np.zeros(self.N)                     # (N,)
        for t in reversed(range(T)):
            # (9.8): δ^a_t = (W_y^T δ^y_t + W_h^T δ^a_{t+1}) ⊙ (1 - h_t ⊙ h_t)
            dh[t] = (self.Wy.T @ dy[t] + self.Wh.T @ dh_next) * (1.0 - hs[t]**2)
            dh_next = dh[t]
        h_prev = np.vstack([np.zeros(self.N), hs[:-1]])   # (T, N)
        gWy = sum(np.outer(dy[t], hs[t]) for t in range(T))   # (9.14)
        gby = dy.sum(axis=0)                                  # (9.15)
        gWx = sum(np.outer(dh[t], xs[t]) for t in range(T))   # (9.16)
        gWh = sum(np.outer(dh[t], h_prev[t]) for t in range(T))
        gbh = dh.sum(axis=0)
        return dict(Wx=gWx, Wh=gWh, bh=gbh, Wy=gWy, by=gby)

    def step(self, xs, targets, h0=None):
        hs, ys = self.forward(xs, h0)
        L = self.loss(ys, targets)
        g = self.backward(xs, hs, ys, targets, h0)
        for k, v in g.items():
            setattr(self, k, getattr(self, k) - self.eta * v)
        return L, ys, g


# Проверка на примере из §9.6
rnn = MultiOutputRNN(N=2, D=1, K=2, eta=0.1)
rnn.Wx = np.array([[0.5], [0.3]])
rnn.Wh = np.array([[0.2, -0.1], [0.4, 0.3]])
rnn.bh = np.array([0.1, -0.2])
rnn.Wy = np.array([[0.6, -0.5], [0.4, 0.3]])
rnn.by = np.array([0.1, -0.1])

xs = [np.array([1.0]), np.array([0.5])]
ys = np.array([[0.0, 1.0], [1.0, 0.0]])
hs, yhat = rnn.forward(xs)
print("h =", hs); print("y^ =", yhat)
L, _, g = rnn.step(xs, ys)
print("L =", L)              # 0.51927
for k, v in g.items():
    print(f"dL/d{k} =\n{v}")
```

Вывод совпадает с §9.6. Если запустить с $K=1$, получим ровно числа главы 8 (при тех же параметрах); с $K=1, N=1$ — числа главы 6.

## 9.8. Что мы получили

1. **Многомерный выход** — $z_t, \hat y_t, y_t \in \mathbb{R}^K$; $W_y \in \mathbb{R}^{K \times N}$, $b_y \in \mathbb{R}^K$. Потеря — сумма по $t$ **и** по $k$.
2. **Дельты выходов** $\delta^y_t \in \mathbb{R}^K$ — вектор, а не скаляр. Формула (9.7) — покомпонентное произведение Адамара, как (8.10), но с векторами.
3. **Ключевое изменение в BPTT** — первое слагаемое в скобке (9.8): $\delta^y_t w_y \to W_y^\top \delta^y_t$. Это **матрица-транспонированная × вектор**, и она автоматически суммирует вклады всех $K$ выходов в каждую из $N$ компонент $\delta^a_t$.
4. **Градиент по $W_y$** — внешнее произведение $\delta^y_t h_t^\top \in \mathbb{R}^{K \times N}$; при $K=1$ это $\delta^y_t h_t^\top \in \mathbb{R}^{1 \times N}$ — строка, эквивалентная вектору $\delta^y_t h_t$ из главы 8.
5. **Скрытые градиенты** ($W_x, W_h, b_h$) формульно **те же**, что (9.16), — но численно другие, потому что $\delta^a_t$ изменилась. Это проявление общей схемы: смена выхода меняет $\delta^y_t$, а через неё — все $\delta^a_t$.
6. **Проверка при $K=1$** — формулы (9.17) сводятся к (8.16) главы 8 **дословно**. Это доказывает корректность обобщения.

**Что дальше.** В главе 10 мы добавим **стек слоёв** — выход слоя $\ell$ становится входом слоя $\ell+1$. Это добавит вторую рекурсию в обратном проходе — по слоям, в дополнение к рекурсии по времени. Формула (9.8) при этом заменится на

$$
\delta^{a,(\ell)}_t = \Bigl(W_y^{(\ell+1)\top} \delta^{a,(\ell+1)}_t + W_h^{(\ell)\top} \delta^{a,(\ell)}_{t+1}\Bigr) \odot \tanh'(a^{(\ell)}_t),
$$

— то есть к «будущему шагу» добавится «следующий слой». Структура та же, что (9.8), просто два слагаемых в скобке теперь означают не «свой выход + будущий шаг», а «следующий слой + будущий шаг». Это и есть общий шаблон BPTT, который повторяется во всех последующих главах.

# Глава 10. Стек слоёв RNN: несколько слоёв и $T$ шагов — обратный проход по двум измерениям

В главах 7–9 мы работали с **одним** слоем скрытых нейронов. Но реальные архитектуры почти всегда **стопки**: выход слоя $\ell$ становится входом слоя $\ell+1$, и только верхний слой выдаёт предсказание. Это добавляет к BPTT **второе измерение** — слои. Обратный проход теперь идёт **и по времени (справа налево), и по слоям (сверху вниз)**. Формально появляется второе слагаемое в скобке рекуррентной формулы: кроме «будущего шага» $a_{t+1}^{(\ell)}$, у $h_t^{(\ell)}$ появляется «слой сверху» $a_t^{(\ell+1)}$. Эта глава показывает, как формула (9.8) обобщается на много слоёв.

## 10.1. Зачем нужен стек слоёв

Один слой RNN — это одно линейное преобразование $W_h h_{t-1} + W_x x_t$ плюс $\tanh$. Он может выучить только «одношаговую» функцию от входа и памяти. Чтобы выучить **иерархию** признаков — как в глубоких сетях прямого распространения — нужно несколько слоёв.

Ключевое отличие от главы 9 — **что нового** появляется в обратном проходе:

| В главе 9 | В этой главе |
|---|---|
| $h_t$ имеет 2 потребителя: $z_t$ и $a_{t+1}$ | $h_t^{(\ell)}$ имеет 2 потребителя: $a_t^{(\ell+1)}$ и $a_{t+1}^{(\ell)}$ |
| Одна рекурсия: по $t$ | **Две** рекурсии: по $t$ **и** по $\ell$ |
| $\delta^a_t = (W_y^\top \delta^y_t + W_h^\top \delta^a_{t+1})\odot\tanh'(a_t)$ | $\delta^{a,(\ell)}_t = \bigl(W_x^{(\ell+1)\top}\delta^{a,(\ell+1)}_t + W_h^{(\ell)\top}\delta^{a,(\ell)}_{t+1}\bigr)\odot\tanh'(a^{(\ell)}_t)$ |

**Структура та же** — два слагаемых в скобке, по одному на каждого «потребителя». Но теперь слагаемые означают «слой сверху» и «будущий шаг», а не «свой выход» и «будущий шаг».

## 10.2. Архитектура и обозначения

Стек из $L$ слоёв, каждый с $N_\ell$ нейронов, развёрнутый на $T$ шагов. Вход $x_t \in \mathbb{R}^D$, выход $\hat y_t \in \mathbb{R}^K$.

| Символ | Размерность | Смысл |
|---|---|---|
| $x_t$ | $\mathbb{R}^{D}$ | вход на шаге $t$ |
| $h_t^{(0)}$ | $\mathbb{R}^{D}$ | «входной слой»: $h_t^{(0)} := x_t$ |
| $a_t^{(\ell)}$ | $\mathbb{R}^{N_\ell}$ | преактивация слоя $\ell$, $1 \le \ell \le L$ |
| $h_t^{(\ell)}$ | $\mathbb{R}^{N_\ell}$ | скрытое состояние слоя $\ell$ |
| $z_t$ | $\mathbb{R}^{K}$ | преактивация выхода |
| $\hat y_t, y_t$ | $\mathbb{R}^K$ | предсказание, ответ |
| $W_x^{(\ell)}$ | $\mathbb{R}^{N_\ell \times N_{\ell-1}}$ | веса «слой $\ell-1$ → слой $\ell$» |
| $W_h^{(\ell)}$ | $\mathbb{R}^{N_\ell \times N_\ell}$ | рекуррентные веса слоя $\ell$ |
| $b_h^{(\ell)}$ | $\mathbb{R}^{N_\ell}$ | смещение слоя $\ell$ |
| $W_y$ | $\mathbb{R}^{K \times N_L}$ | веса выхода (из **верхнего** слоя) |
| $b_y$ | $\mathbb{R}^K$ | смещение выхода |
| $h_0^{(\ell)} = \mathbf{0}$ | $\mathbb{R}^{N_\ell}$ | начальное состояние слоя $\ell$ |

**Уравнения прямого прохода** (для $t = 1, \dots, T$, $\ell = 1, \dots, L$):

$$
\boxed{
\begin{aligned}
a_t^{(\ell)} &= W_x^{(\ell)}\,h_t^{(\ell-1)} + W_h^{(\ell)}\,h_{t-1}^{(\ell)} + b_h^{(\ell)} &&\in \mathbb{R}^{N_\ell}, \\
h_t^{(\ell)} &= \tanh\bigl(a_t^{(\ell)}\bigr) &&\in \mathbb{R}^{N_\ell}, \\
z_t &= W_y\,h_t^{(L)} + b_y &&\in \mathbb{R}^K, \\
\hat y_t &= \sigma(z_t) &&\in \mathbb{R}^K.
\end{aligned}}
\tag{10.1}
$$

Соглашение: $h_t^{(0)} := x_t$ (вход — это «нулевой слой»), $h_0^{(\ell)} := \mathbf{0}$ для всех $\ell$.

**Порядок вычислений** в прямом проходе: для каждого $t = 1, \dots, T$ — снизу вверх по слоям $\ell = 1, \dots, L$. Именно так: $h_t^{(1)}$ нужно до $a_t^{(2)}$, $h_t^{(2)}$ — до $a_t^{(3)}$, и т. д.

**Число параметров:**

$$
\sum_{\ell=1}^{L} \bigl(\underbrace{N_\ell N_{\ell-1}}_{W_x^{(\ell)}} + \underbrace{N_\ell^2}_{W_h^{(\ell)}} + \underbrace{N_\ell}_{b_h^{(\ell)}}\bigr) + \underbrace{K N_L}_{W_y} + \underbrace{K}_{b_y}. \tag{10.2}
$$

При $L = 1$ это $ND + N^2 + N + KN + K$ — ровно (9.3) главы 9. **Глава 10 — прямое обобщение главы 9.**

На рис. 10.1 — архитектура $L=2$, $T=2$, $N_1 = N_2 = 2$, $D = 1$, $K = 1$.

```python
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch

def draw_architecture_L2(N1=2, N2=2, D=1, K=1):
    """Рис. 10.1: два слоя RNN, T=2."""
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.set_xlim(0, 14); ax.set_ylim(0, 11)
    ax.set_aspect('equal'); ax.axis('off')
    r = 0.32
    cols = [2.5, 11.5]
    y_x = 1.0
    y_h1_base = 3.5
    y_h2_base = 6.5
    y_y = 9.5

    def node(cx, cy, label, color):
        ax.add_patch(Circle((cx, cy), r, facecolor=color,
                            edgecolor='black', lw=1.4, zorder=3))
        ax.text(cx, cy, label, ha='center', va='center', fontsize=9, zorder=4)

    def arrow(p1, p2, color='black', lw=1.0, alpha=0.6):
        ax.add_patch(FancyArrowPatch(
            p1, p2, arrowstyle='-|>', mutation_scale=10,
            lw=lw, color=color, alpha=alpha,
            shrinkA=0, shrinkB=0, zorder=2))

    for t, cx in enumerate(cols, start=1):
        # вход
        for j in range(D):
            node(cx, y_x + j*0.8, rf'$x_{t},{j+1}$', '#AEDFF7')
        # слой 1
        for i in range(N1):
            node(cx, y_h1_base + i*0.8, rf'$h_{t},{i+1}^{(1)}$', '#FFE29A')
        # слой 2
        for i in range(N2):
            node(cx, y_h2_base + i*0.8, rf'$h_{t},{i+1}^{(2)}$', '#F5C6A5')
        # выход
        for k in range(K):
            node(cx, y_y + k*0.8, rf'$\hat y_{t},{k+1}$', '#B8E0B8')
        # стрелки
        for j in range(D):
            for i in range(N1):
                arrow((cx + r*0.4, y_x + j*0.8), (cx - r*0.4, y_h1_base + i*0.8),
                      color='#B22222', alpha=0.4)
        for i in range(N1):
            for j in range(N2):
                arrow((cx + r*0.4, y_h1_base + i*0.8), (cx - r*0.4, y_h2_base + j*0.8),
                      color='#2E86AB', alpha=0.4)
        for i in range(N2):
            for k in range(K):
                arrow((cx + r*0.4, y_h2_base + i*0.8), (cx - r*0.4, y_y + k*0.8),
                      color='#B22222', alpha=0.4)

    # рекуррентные связи
    for i in range(N1):
        for k in range(N1):
            arrow((cols[0] + r, y_h1_base + i*0.8),
                  (cols[1] - r, y_h1_base + k*0.8),
                  color='#7B2D8E', lw=0.7, alpha=0.4)
    for i in range(N2):
        for k in range(N2):
            arrow((cols[0] + r, y_h2_base + i*0.8),
                  (cols[1] - r, y_h2_base + k*0.8),
                  color='#7B2D8E', lw=0.7, alpha=0.4)

    ax.text(cols[0], 0.3, 'Шаг t=1', ha='center', fontsize=12, fontweight='bold')
    ax.text(cols[1], 0.3, 'Шаг t=2', ha='center', fontsize=12, fontweight='bold')
    ax.text(7, 10.7,
            f'Рис. 10.1. Стек L=2: D={D}, N1={N1}, N2={N2}, K={K}, T=2',
            ha='center', fontsize=13, fontweight='bold')
    ax.text(7, 2.2, r'$W_x^{(1)}$', fontsize=11, color='#B22222', ha='center')
    ax.text(7, 5.2, r'$W_x^{(2)}$', fontsize=11, color='#2E86AB', ha='center')
    ax.text(7, 8.2, r'$W_y$', fontsize=11, color='#B22222', ha='center')
    plt.tight_layout(); plt.show()

draw_architecture_L2()
```

## 10.3. Прямой проход

Прямой проход — применение (10.1) для $t = 1, \dots, T$ и $\ell = 1, \dots, L$. **Порядок важен:** сначала все слои на шаге $t$, потом $t+1$.

### Подстановка чисел: $L=2$, $T=2$, $N_1=N_2=2$, $D=1$, $K=1$

Возьмём:

$$
W_x^{(1)} = \begin{pmatrix} 0.5 \\ 0.3 \end{pmatrix},\quad
W_h^{(1)} = \begin{pmatrix} 0.2 & -0.1 \\ 0.4 & 0.3 \end{pmatrix},\quad
b_h^{(1)} = \begin{pmatrix} 0.1 \\ -0.2 \end{pmatrix},
$$

$$
W_x^{(2)} = \begin{pmatrix} 0.3 & 0.2 \\ -0.1 & 0.4 \end{pmatrix},\quad
W_h^{(2)} = \begin{pmatrix} 0.1 & -0.2 \\ 0.3 & 0.1 \end{pmatrix},\quad
b_h^{(2)} = \begin{pmatrix} 0.05 \\ -0.1 \end{pmatrix},
$$

$$
W_y = \begin{pmatrix} 0.6 & -0.5 \end{pmatrix},\quad b_y = 0.1.
$$

Входы $x_1 = 1.0$, $x_2 = 0.5$. Ответы $y_1 = 0.0$, $y_2 = 1.0$. $h_0^{(\ell)} = (0, 0)^\top$.

**Шаг $t=1$, слой $\ell=1$:**

$$
a_1^{(1)} = W_x^{(1)} x_1 + W_h^{(1)} h_0^{(1)} + b_h^{(1)} = \begin{pmatrix} 0.5 \\ 0.3 \end{pmatrix} + \begin{pmatrix} 0 \\ 0 \end{pmatrix} + \begin{pmatrix} 0.1 \\ -0.2 \end{pmatrix} = \begin{pmatrix} 0.6 \\ 0.1 \end{pmatrix},
$$
$$
h_1^{(1)} = \tanh(a_1^{(1)}) \approx \begin{pmatrix} 0.53705 \\ 0.09967 \end{pmatrix}.
$$

**Шаг $t=1$, слой $\ell=2$:**

$$
W_x^{(2)} h_1^{(1)} = \begin{pmatrix} 0.3 & 0.2 \\ -0.1 & 0.4 \end{pmatrix}\begin{pmatrix} 0.53705 \\ 0.09967 \end{pmatrix} = \begin{pmatrix} 0.16112 + 0.01993 \\ -0.05371 + 0.03987 \end{pmatrix} = \begin{pmatrix} 0.18105 \\ -0.01384 \end{pmatrix},
$$
$$
a_1^{(2)} = \begin{pmatrix} 0.18105 \\ -0.01384 \end{pmatrix} + \begin{pmatrix} 0 \\ 0 \end{pmatrix} + \begin{pmatrix} 0.05 \\ -0.1 \end{pmatrix} = \begin{pmatrix} 0.23105 \\ -0.11384 \end{pmatrix},
$$
$$
h_1^{(2)} = \tanh(a_1^{(2)}) \approx \begin{pmatrix} 0.22705 \\ -0.11335 \end{pmatrix}.
$$

**Выход $t=1$:**

$$
z_1 = W_y h_1^{(2)} + b_y = 0.6\cdot 0.22705 + (-0.5)\cdot(-0.11335) + 0.1 = 0.29291,
$$
$$
\hat y_1 = \sigma(z_1) \approx 0.57268.
$$

**Шаг $t=2$, слой $\ell=1$:**

$$
W_h^{(1)} h_1^{(1)} = \begin{pmatrix} 0.2 & -0.1 \\ 0.4 & 0.3 \end{pmatrix}\begin{pmatrix} 0.53705 \\ 0.09967 \end{pmatrix} = \begin{pmatrix} 0.10741 - 0.00997 \\ 0.21482 + 0.02990 \end{pmatrix} = \begin{pmatrix} 0.09744 \\ 0.24472 \end{pmatrix},
$$
$$
a_2^{(1)} = \begin{pmatrix} 0.25 \\ 0.15 \end{pmatrix} + \begin{pmatrix} 0.09744 \\ 0.24472 \end{pmatrix} + \begin{pmatrix} 0.1 \\ -0.2 \end{pmatrix} = \begin{pmatrix} 0.44744 \\ 0.19472 \end{pmatrix},
$$
$$
h_2^{(1)} \approx \begin{pmatrix} 0.41978 \\ 0.19230 \end{pmatrix}.
$$

**Шаг $t=2$, слой $\ell=2$:**

$$
W_x^{(2)} h_2^{(1)} = \begin{pmatrix} 0.3 & 0.2 \\ -0.1 & 0.4 \end{pmatrix}\begin{pmatrix} 0.41978 \\ 0.19230 \end{pmatrix} = \begin{pmatrix} 0.12593 + 0.03846 \\ -0.04198 + 0.07692 \end{pmatrix} = \begin{pmatrix} 0.16439 \\ 0.03494 \end{pmatrix},
$$
$$
W_h^{(2)} h_1^{(2)} = \begin{pmatrix} 0.1 & -0.2 \\ 0.3 & 0.1 \end{pmatrix}\begin{pmatrix} 0.22705 \\ -0.11335 \end{pmatrix} = \begin{pmatrix} 0.02271 + 0.02267 \\ 0.06812 - 0.01134 \end{pmatrix} = \begin{pmatrix} 0.04538 \\ 0.05678 \end{pmatrix},
$$
$$
a_2^{(2)} = \begin{pmatrix} 0.16439 \\ 0.03494 \end{pmatrix} + \begin{pmatrix} 0.04538 \\ 0.05678 \end{pmatrix} + \begin{pmatrix} 0.05 \\ -0.1 \end{pmatrix} = \begin{pmatrix} 0.25977 \\ -0.00828 \end{pmatrix},
$$
$$
h_2^{(2)} \approx \begin{pmatrix} 0.25409 \\ -0.00828 \end{pmatrix}.
$$

**Выход $t=2$:**

$$
z_2 = 0.6\cdot 0.25409 + (-0.5)\cdot(-0.00828) + 0.1 = 0.25660, \qquad \hat y_2 \approx 0.56383.
$$

### Таблица локальных производных

**Для каждого слоя $\ell$ — те же «кирпичи», что в главе 9, но с индексом $\ell$:**

| Величина | Форма | Компонентно |
|---|---|---|
| $\partial a_t^{(\ell)}/\partial h_t^{(\ell-1)}$ | $\mathbb{R}^{N_\ell \times N_{\ell-1}}$ | $(W_x^{(\ell)})_{ij}$ |
| $\partial a_t^{(\ell)}/\partial h_{t-1}^{(\ell)}$ | $\mathbb{R}^{N_\ell \times N_\ell}$ | $(W_h^{(\ell)})_{ij}$ |
| $\partial a_t^{(\ell)}/\partial b_h^{(\ell)}$ | $\mathbb{R}^{N_\ell \times N_\ell}$ | $\delta_{ij}$ |
| $\partial h_t^{(\ell)}/\partial a_t^{(\ell)}$ | $\mathbb{R}^{N_\ell \times N_\ell}$ | $\delta_{ij}(1 - (h_{t,i}^{(\ell)})^2)$ |

**Для выхода — как в главе 9:** $\partial z_t/\partial h_t^{(L)} = W_y$, $\partial z_t/\partial W_y = h_t^{(L)}$, $\partial \hat y_t/\partial z_t = \operatorname{diag}(\hat y_t \odot (1 - \hat y_t))$.

Эти «кирпичи» **не зависят ни от $T$, ни от $L$** — как и раньше.

## 10.4. Функция потерь

$$
L = \sum_{t=1}^{T} \frac{1}{2}\,\|\hat y_t - y_t\|^2 \in \mathbb{R}. \tag{10.3}
$$

Потеря аддитивна по $t$:

$$
\frac{\partial L}{\partial \hat y_t} = \hat y_t - y_t \in \mathbb{R}^K, \qquad t = 1, \dots, T. \tag{10.4}
$$

## 10.5. Обратный проход

### 10.5.1. Кто чьи потребители

Центральный вопрос главы: **у $h_t^{(\ell)}$ сколько потребителей?** При $L = 1$ (глава 9) было два: $z_t$ (если $\ell = L$) и $a_{t+1}^{(L)}$ (следующий шаг). При $L > 1$:

- **Верхний слой $\ell = L$.** $h_t^{(L)}$ входит в $z_t$ (выход) **и** в $a_{t+1}^{(L)}$ (следующий шаг того же слоя). **Два потребителя.**
- **Внутренний слой $\ell < L$.** $h_t^{(\ell)}$ входит в $a_t^{(\ell+1)}$ (слой сверху, **тот же** шаг) **и** в $a_{t+1}^{(\ell)}$ (следующий шаг **того же** слоя). **Два потребителя.**

В обоих случаях — **два** слагаемых в $\partial L/\partial h_t^{(\ell)}$, но разных:

| Слой | Слагаемое 1 | Слагаемое 2 |
|---|---|---|
| $\ell = L$ | выход: $W_y^\top \delta^y_t$ | будущий шаг: $W_h^{(L)\top} \delta^{a,(L)}_{t+1}$ |
| $\ell < L$ | слой сверху: $W_x^{(\ell+1)\top} \delta^{a,(\ell+1)}_t$ | будущий шаг: $W_h^{(\ell)\top} \delta^{a,(\ell)}_{t+1}$ |

**Граничные случаи:** при $t = T$ слагаемое «будущий шаг» равно нулю (дальше времени нет); при $\ell = L$ слагаемое «слой сверху» заменяется выходом.

### 10.5.2. Ключевая рекуррентная формула

**Утверждение.** Определим $\delta^{a,(\ell)}_t := \partial L/\partial a_t^{(\ell)} \in \mathbb{R}^{N_\ell}$ и $\delta^y_t := \partial L/\partial z_t \in \mathbb{R}^K$. Тогда для всех $\ell = 1, \dots, L$ и $t = 1, \dots, T$:

$$
\boxed{
\begin{aligned}
\delta^{a,(\ell)}_t &= \Bigl(W_x^{(\ell+1)\top}\delta^{a,(\ell+1)}_t \;+\; W_h^{(\ell)\top}\delta^{a,(\ell)}_{t+1}\Bigr)\odot\tanh'(a_t^{(\ell)}) \quad (\ell < L), \\[2pt]
\delta^{a,(L)}_t &= \Bigl(W_y^\top \delta^y_t \;+\; W_h^{(L)\top}\delta^{a,(L)}_{t+1}\Bigr)\odot\tanh'(a_t^{(L)}),
\end{aligned}}
\tag{10.5}
$$

с граничными условиями $\delta^{a,(\ell)}_{T+1} = \mathbf{0}$ и $\tanh'(a_t^{(\ell)}) := 1 - h_t^{(\ell)}\odot h_t^{(\ell)}$.

**Доказательство.** Рассмотрим $h_t^{(\ell)}$. Сначала для $\ell < L$. Оно входит ровно в два уравнения:

- $a_t^{(\ell+1)} = W_x^{(\ell+1)} h_t^{(\ell)} + W_h^{(\ell+1)} h_{t-1}^{(\ell+1)} + b_h^{(\ell+1)}$ — слой сверху, **тот же** шаг $t$;
- $a_{t+1}^{(\ell)} = W_x^{(\ell)} h_{t+1}^{(\ell-1)} + W_h^{(\ell)} h_t^{(\ell)} + b_h^{(\ell)}$ — **следующий** шаг того же слоя.

(При $t = T$ второго уравнения нет.) Поэтому

$$
\frac{\partial L}{\partial h_t^{(\ell)}} = \underbrace{\frac{\partial L}{\partial a_t^{(\ell+1)}}\cdot\frac{\partial a_t^{(\ell+1)}}{\partial h_t^{(\ell)}}}_{\text{вклад через слой сверху}} + \underbrace{\frac{\partial L}{\partial a_{t+1}^{(\ell)}}\cdot\frac{\partial a_{t+1}^{(\ell)}}{\partial h_t^{(\ell)}}}_{\text{вклад через будущий шаг}}. \tag{10.6}
$$

Якобианы: $\partial a_t^{(\ell+1)}/\partial h_t^{(\ell)} = W_x^{(\ell+1)}$ (форма $N_{\ell+1} \times N_\ell$); $\partial a_{t+1}^{(\ell)}/\partial h_t^{(\ell)} = W_h^{(\ell)}$ (форма $N_\ell \times N_\ell$). При умножении вектор-градиента на якобиан — транспонируем:

$$
\frac{\partial L}{\partial h_t^{(\ell)}} = W_x^{(\ell+1)\top}\delta^{a,(\ell+1)}_t + W_h^{(\ell)\top}\delta^{a,(\ell)}_{t+1}. \tag{10.7}
$$

Наконец, $h_t^{(\ell)} = \tanh(a_t^{(\ell)})$, поэтому

$$
\delta^{a,(\ell)}_t = \frac{\partial L}{\partial a_t^{(\ell)}} = \frac{\partial L}{\partial h_t^{(\ell)}}\odot\tanh'(a_t^{(\ell)}). \tag{10.8}
$$

Подставляем (10.7) в (10.8) — получаем первую строку (10.5). Для $\ell = L$ второе уравнение в (10.6) заменяется на $z_t = W_y h_t^{(L)} + b_y$, и $\partial L/\partial z_t = \delta^y_t$, $\partial z_t/\partial h_t^{(L)} = W_y$; получаем вторую строку. $\blacksquare$

**Сравнение с предельными случаями.**

| Формула | При $L=1$ | При $T=2$, $L$ общем |
|---|---|---|
| (10.5) | (9.8): $\delta^{a}_t = (W_y^\top \delta^y_t + W_h^\top \delta^a_{t+1})\odot\tanh'(a_t)$ | те же две строки |
| совпадение | $\checkmark$ (слой $L=1$ — верхний, поэтому вторая строка) | $\checkmark$ |

### 10.5.3. Порядок вычисления: два измерения

Из (10.5) видно: чтобы найти $\delta^{a,(\ell)}_t$, нужны **две** уже известные величины: $\delta^{a,(\ell+1)}_t$ (слой сверху, тот же шаг) и $\delta^{a,(\ell)}_{t+1}$ (тот же слой, следующий шаг). Значит, порядок обхода:

```
для ℓ = L, L-1, ..., 1:
    для t = T, T-1, ..., 1:
        вычислить δ^{a,(ℓ)}_t по (10.5)
```

**Почему этот порядок работает:**

- Когда мы находимся в точке $(\ell, t)$, **уже посчитаны**:
  - $\delta^{a,(\ell+1)}_t$ — слой сверху, все шаги уже пройдены (внешний цикл);
  - $\delta^{a,(\ell)}_{t+1}$ — тот же слой, следующий шаг (внутренний цикл).
- **Нельзя** обойти наоборот (сначала все слои на шаге $T$, потом все слои на шаге $T-1$): для $\delta^{a,(\ell)}_t$ нужен $\delta^{a,(\ell)}_{t+1}$, а он ещё не посчитан, если мы не прошли шаг $t+1$ для слоя $\ell$.

### 10.5.4. Дельты выходов

Из (10.4) и $\hat y_t = \sigma(z_t)$:

$$
\delta^y_t = (\hat y_t - y_t)\odot\hat y_t\odot(1 - \hat y_t) \in \mathbb{R}^K, \qquad t = 1, \dots, T. \tag{10.9}
$$

Все $T$ значений $\delta^y_t$ независимы — потеря аддитивна.

### 10.5.5. Градиенты параметров

Для каждого слоя $\ell$ — свои формулы, структурно **те же**, что в главах 8–9, но с индексом $\ell$:

**Выход** ($W_y \in \mathbb{R}^{K \times N_L}$, $b_y \in \mathbb{R}^K$):

$$
\boxed{\;\frac{\partial L}{\partial W_y} = \sum_{t=1}^{T} \delta^y_t\,h_t^{(L)\top},\qquad \frac{\partial L}{\partial b_y} = \sum_{t=1}^{T} \delta^y_t.\;} \tag{10.10}
$$

**Слой $\ell$** ($W_x^{(\ell)}$, $W_h^{(\ell)}$, $b_h^{(\ell)}$):

$$
\boxed{
\begin{aligned}
\frac{\partial L}{\partial W_x^{(\ell)}} &= \sum_{t=1}^{T} \delta^{a,(\ell)}_t\,h_t^{(\ell-1)\top} \in \mathbb{R}^{N_\ell \times N_{\ell-1}}, \\
\frac{\partial L}{\partial W_h^{(\ell)}} &= \sum_{t=1}^{T} \delta^{a,(\ell)}_t\,h_{t-1}^{(\ell)\top} \in \mathbb{R}^{N_\ell \times N_\ell}, \\
\frac{\partial L}{\partial b_h^{(\ell)}} &= \sum_{t=1}^{T} \delta^{a,(\ell)}_t \in \mathbb{R}^{N_\ell}.
\end{aligned}}
\tag{10.11}
$$

**Проверка при $L=1$:** $h_t^{(0)} = x_t$, и (10.11) превращается в (9.16) главы 9. **Проверка при $T=2$:** суммы по $t$ содержат два слагаемых — как в (7.15).

### 10.5.6. Сводка обратного прохода

$$
\boxed{
\begin{aligned}
&\text{1. Для } t = 1, \dots, T: \quad \delta^y_t = (\hat y_t - y_t)\odot\hat y_t\odot(1 - \hat y_t); \\[4pt]
&\text{2. Для } \ell = L, L-1, \dots, 1 \text{ и } t = T, T-1, \dots, 1: \\
&\qquad \delta^{a,(\ell)}_t = \Bigl(\tilde{W}_t^{(\ell)} + W_h^{(\ell)\top}\delta^{a,(\ell)}_{t+1}\Bigr)\odot\tanh'(a_t^{(\ell)}), \\
&\qquad\text{где } \tilde{W}_t^{(\ell)} = \begin{cases} W_y^\top \delta^y_t, & \ell = L, \\ W_x^{(\ell+1)\top}\delta^{a,(\ell+1)}_t, & \ell < L. \end{cases} \\[4pt]
&\text{3. } \frac{\partial L}{\partial W_y} = \sum_t \delta^y_t\,h_t^{(L)\top}, \quad \frac{\partial L}{\partial b_y} = \sum_t \delta^y_t; \\
&\text{4. Для } \ell = 1, \dots, L: \\
&\qquad \frac{\partial L}{\partial W_x^{(\ell)}} = \sum_t \delta^{a,(\ell)}_t\,h_t^{(\ell-1)\top},\quad
\frac{\partial L}{\partial W_h^{(\ell)}} = \sum_t \delta^{a,(\ell)}_t\,h_{t-1}^{(\ell)\top},\quad
\frac{\partial L}{\partial b_h^{(\ell)}} = \sum_t \delta^{a,(\ell)}_t.
\end{aligned}}
\tag{10.12}
$$

## 10.6. Численный пример: $L=2$, $T=2$, $N_1=N_2=2$

Продолжим пример из §10.3.

### Потеря

$$
L = \tfrac12(0.57268 - 0)^2 + \tfrac12(0.56383 - 1)^2 = 0.16398 + 0.09512 = 0.25910.
$$

### Дельты выходов

$$
\delta^y_1 = 0.57268\cdot 0.57268\cdot(1 - 0.57268) \approx 0.14018,
$$
$$
\delta^y_2 = (-0.43617)\cdot 0.56383\cdot 0.43617 \approx -0.10725.
$$

### Слой $\ell=2$ (верхний)

**$t=2$** (базовый случай, $\delta^{a,(2)}_{3} = \mathbf{0}$):

$$
W_y^\top \delta^y_2 = \begin{pmatrix} 0.6 \\ -0.5 \end{pmatrix}\cdot(-0.10725) = \begin{pmatrix} -0.06435 \\ 0.05363 \end{pmatrix},
$$
$$
\tanh'(a_2^{(2)}) = \begin{pmatrix} 1 - 0.25409^2 \\ 1 - (-0.00828)^2 \end{pmatrix} = \begin{pmatrix} 0.93544 \\ 0.99993 \end{pmatrix},
$$
$$
\delta^{a,(2)}_2 = \begin{pmatrix} -0.06435 \\ 0.05363 \end{pmatrix}\odot\begin{pmatrix} 0.93544 \\ 0.99993 \end{pmatrix} = \begin{pmatrix} -0.06020 \\ 0.05363 \end{pmatrix}.
$$

**$t=1$:**

$$
W_y^\top \delta^y_1 = 0.14018\cdot\begin{pmatrix} 0.6 \\ -0.5 \end{pmatrix} = \begin{pmatrix} 0.08411 \\ -0.07009 \end{pmatrix},
$$
$$
W_h^{(2)\top}\delta^{a,(2)}_2 = \begin{pmatrix} 0.1 & 0.3 \\ -0.2 & 0.1 \end{pmatrix}\begin{pmatrix} -0.06020 \\ 0.05363 \end{pmatrix} = \begin{pmatrix} -0.00602 + 0.01609 \\ 0.01204 + 0.00536 \end{pmatrix} = \begin{pmatrix} 0.01007 \\ 0.01740 \end{pmatrix},
$$
$$
\tanh'(a_1^{(2)}) = \begin{pmatrix} 1 - 0.22705^2 \\ 1 - (-0.11335)^2 \end{pmatrix} = \begin{pmatrix} 0.94845 \\ 0.98715 \end{pmatrix},
$$
$$
\delta^{a,(2)}_1 = \begin{pmatrix} 0.08411 + 0.01007 \\ -0.07009 + 0.01740 \end{pmatrix}\odot\begin{pmatrix} 0.94845 \\ 0.98715 \end{pmatrix} = \begin{pmatrix} 0.08932 \\ -0.05201 \end{pmatrix}.
$$

### Слой $\ell=1$ (внутренний)

Теперь слагаемое «слой сверху» — это $W_x^{(2)\top}\delta^{a,(2)}_t$, а не $W_y^\top \delta^y_t$.

**$t=2$** (граница по времени, $\delta^{a,(1)}_{3} = \mathbf{0}$):

$$
W_x^{(2)\top}\delta^{a,(2)}_2 = \begin{pmatrix} 0.3 & -0.1 \\ 0.2 & 0.4 \end{pmatrix}\begin{pmatrix} -0.06020 \\ 0.05363 \end{pmatrix} = \begin{pmatrix} -0.01806 - 0.00536 \\ -0.01204 + 0.02145 \end{pmatrix} = \begin{pmatrix} -0.02342 \\ 0.00941 \end{pmatrix},
$$
$$
\tanh'(a_2^{(1)}) = \begin{pmatrix} 1 - 0.41978^2 \\ 1 - 0.19230^2 \end{pmatrix} = \begin{pmatrix} 0.82378 \\ 0.96302 \end{pmatrix},
$$
$$
\delta^{a,(1)}_2 = \begin{pmatrix} -0.02342 \\ 0.00941 \end{pmatrix}\odot\begin{pmatrix} 0.82378 \\ 0.96302 \end{pmatrix} = \begin{pmatrix} -0.01929 \\ 0.00906 \end{pmatrix}.
$$

**$t=1$:**

$$
W_x^{(2)\top}\delta^{a,(2)}_1 = \begin{pmatrix} 0.3 & -0.1 \\ 0.2 & 0.4 \end{pmatrix}\begin{pmatrix} 0.08932 \\ -0.05201 \end{pmatrix} = \begin{pmatrix} 0.02680 + 0.00520 \\ 0.01786 - 0.02080 \end{pmatrix} = \begin{pmatrix} 0.03200 \\ -0.00294 \end{pmatrix},
$$
$$
W_h^{(1)\top}\delta^{a,(1)}_2 = \begin{pmatrix} 0.2 & 0.4 \\ -0.1 & 0.3 \end{pmatrix}\begin{pmatrix} -0.01929 \\ 0.00906 \end{pmatrix} = \begin{pmatrix} -0.00386 + 0.00362 \\ 0.00193 + 0.00272 \end{pmatrix} = \begin{pmatrix} -0.00024 \\ 0.00465 \end{pmatrix},
$$
$$
\tanh'(a_1^{(1)}) = \begin{pmatrix} 1 - 0.53705^2 \\ 1 - 0.09967^2 \end{pmatrix} = \begin{pmatrix} 0.71158 \\ 0.99007 \end{pmatrix},
$$
$$
\delta^{a,(1)}_1 = \begin{pmatrix} 0.03200 - 0.00024 \\ -0.00294 + 0.00465 \end{pmatrix}\odot\begin{pmatrix} 0.71158 \\ 0.99007 \end{pmatrix} = \begin{pmatrix} 0.03176 \\ 0.00171 \end{pmatrix}\odot\begin{pmatrix} 0.71158 \\ 0.99007 \end{pmatrix} = \begin{pmatrix} 0.02260 \\ 0.00169 \end{pmatrix}.
$$

### Пять категорий градиентов

**Выход:**

$$
\frac{\partial L}{\partial W_y} = \delta^y_1 h_1^{(2)\top} + \delta^y_2 h_2^{(2)\top} = \begin{pmatrix} 0.00458 & -0.01500 \end{pmatrix} \in \mathbb{R}^{1 \times 2},
$$
$$
\frac{\partial L}{\partial b_y} = \delta^y_1 + \delta^y_2 = 0.03293.
$$

**Слой $\ell=2$:**

$$
\frac{\partial L}{\partial W_h^{(2)}} = \delta^{a,(2)}_2 h_1^{(2)\top} = \begin{pmatrix} -0.01367 & 0.00682 \\ 0.01218 & -0.00608 \end{pmatrix},
$$
$$
\frac{\partial L}{\partial W_x^{(2)}} = \delta^{a,(2)}_2 h_2^{(1)\top} + \delta^{a,(2)}_1 h_1^{(1)\top} = \begin{pmatrix} 0.02270 & -0.00268 \\ -0.00542 & 0.00513 \end{pmatrix},
$$
$$
\frac{\partial L}{\partial b_h^{(2)}} = \delta^{a,(2)}_1 + \delta^{a,(2)}_2 = \begin{pmatrix} 0.02912 \\ 0.00162 \end{pmatrix}.
$$

**Слой $\ell=1$:**

$$
\frac{\partial L}{\partial W_h^{(1)}} = \delta^{a,(1)}_2 h_1^{(1)\top} = \begin{pmatrix} -0.01036 & -0.00192 \\ 0.00487 & 0.00090 \end{pmatrix},
$$
$$
\frac{\partial L}{\partial W_x^{(1)}} = \delta^{a,(1)}_1 x_1 + \delta^{a,(1)}_2 x_2 = \begin{pmatrix} 0.02260 \\ 0.00169 \end{pmatrix} + 0.5\begin{pmatrix} -0.01929 \\ 0.00906 \end{pmatrix} = \begin{pmatrix} 0.01295 \\ 0.00622 \end{pmatrix},
$$
$$
\frac{\partial L}{\partial b_h^{(1)}} = \delta^{a,(1)}_1 + \delta^{a,(1)}_2 = \begin{pmatrix} 0.00331 \\ 0.01075 \end{pmatrix}.
$$

### Проверка при $L=1$

Если убрать слой 2 ($L=1$), то $\ell = L = 1$ — верхний, и формула (10.5) даёт $\delta^{a,(1)}_t = (W_y^\top \delta^y_t + W_h^{(1)\top}\delta^{a,(1)}_{t+1})\odot\tanh'(a_t^{(1)})$ — ровно (9.8). Числа сойдутся с §9.6 (при соответствующем $W_y$). **Обобщение корректно.**

## 10.7. Полная программа

Расширим класс из глав 8–9 до `StackedRNN`: список слоёв, у каждого — свои $W_x, W_h, b_h$.

```python
import numpy as np

def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))
def tanh_prime(h):  return 1.0 - h*h

class StackedRNN:
    def __init__(self, sizes, D, K, seed=0, eta=0.1):
        """sizes = [N1, N2, ..., NL] — размеры скрытых слоёв."""
        rng = np.random.default_rng(seed)
        self.L = len(sizes)
        self.sizes, self.D, self.K = sizes, D, K
        self.eta = eta
        # Wx[0] : (N1, D),  Wx[l] : (N_{l+1}, N_l)  для l >= 1
        prev = D
        self.Wx, self.Wh, self.bh = [], [], []
        for N in sizes:
            self.Wx.append(rng.normal(scale=0.5, size=(N, prev)))
            self.Wh.append(rng.normal(scale=0.5, size=(N, N)))
            self.bh.append(rng.normal(scale=0.1, size=(N,)))
            prev = N
        self.Wy = rng.normal(scale=0.5, size=(K, sizes[-1]))
        self.by = rng.normal(scale=0.1, size=(K,))

    def forward(self, xs, h0=None):
        T = len(xs)
        if h0 is None:
            h0 = [np.zeros(N) for N in self.sizes]
        h = [x.copy() for x in h0]
        # hs[l][t] — состояние слоя l на шаге t; hs[0] — входы
        hs = [[x for x in xs]]                    # «слой 0» — входы
        for l in range(self.L):
            layer_h = []
            for t in range(T):
                a = self.Wx[l] @ hs[l][t] + self.Wh[l] @ h[l] + self.bh[l]
                h[l] = np.tanh(a)
                layer_h.append(h[l].copy())
            hs.append(layer_h)
        ys = np.array([sigmoid(self.Wy @ hs[-1][t] + self.by) for t in range(T)])
        return hs, ys

    def loss(self, ys, targets):
        return 0.5 * np.sum((ys - targets)**2)

    def backward(self, hs, ys, targets):
        T = len(ys)
        # δ^y
        dy = (ys - targets) * ys * (1.0 - ys)           # (T, K)
        # δ^{a,(l)} — список слоёв, каждый (T, N_l)
        dh = [np.zeros((T, N)) for N in self.sizes]
        # градиенты
        gWx = [np.zeros_like(W) for W in self.Wx]
        gWh = [np.zeros_like(W) for W in self.Wh]
        gbh = [np.zeros_like(b) for b in self.bh]
        gWy = np.zeros_like(self.Wy); gby = np.zeros_like(self.by)

        for l in range(self.L-1, -1, -1):       # сверху вниз по слоям
            dh_next = np.zeros(self.sizes[l])   # δ^{a,(l)}_{T+1} = 0
            for t in range(T-1, -1, -1):        # справа налево по времени
                # слагаемое «слой сверху» (или выход для l = L-1)
                if l == self.L-1:
                    top = self.Wy.T @ dy[t]
                else:
                    top = self.Wx[l+1].T @ dh[l+1][t]
                # слагаемое «будущий шаг»
                fut = self.Wh[l].T @ dh_next
                # δ^{a,(l)}_t
                dh[l][t] = (top + fut) * tanh_prime(hs[l+1][t])
                dh_next = dh[l][t]
                # накопление градиентов
                gWx[l] += np.outer(dh[l][t], hs[l][t])
                gWh[l] += np.outer(dh[l][t], hs[l+1][t-1] if t > 0 else np.zeros(self.sizes[l]))
                gbh[l] += dh[l][t]

        # выход
        for t in range(T):
            gWy += np.outer(dy[t], hs[-1][t])
            gby += dy[t]
        return dict(Wx=gWx, Wh=gWh, bh=gbh, Wy=gWy, by=gby)

    def step(self, xs, targets):
        hs, ys = self.forward(xs)
        L = self.loss(ys, targets)
        g = self.backward(hs, ys, targets)
        for l in range(self.L):
            self.Wx[l] -= self.eta * g['Wx'][l]
            self.Wh[l] -= self.eta * g['Wh'][l]
            self.bh[l] -= self.eta * g['bh'][l]
        self.Wy -= self.eta * g['Wy']
        self.by -= self.eta * g['by']
        return L, ys


# Проверка на §10.6
net = StackedRNN(sizes=[2, 2], D=1, K=1, eta=0.1)
net.Wx[0] = np.array([[0.5], [0.3]])
net.Wh[0] = np.array([[0.2, -0.1], [0.4, 0.3]])
net.bh[0] = np.array([0.1, -0.2])
net.Wx[1] = np.array([[0.3, 0.2], [-0.1, 0.4]])
net.Wh[1] = np.array([[0.1, -0.2], [0.3, 0.1]])
net.bh[1] = np.array([0.05, -0.1])
net.Wy = np.array([[0.6, -0.5]])
net.by = np.array([0.1])

xs = [np.array([1.0]), np.array([0.5])]
ys = np.array([[0.0], [1.0]])
L, yhat = net.step(xs, ys)
print("L =", L)     # 0.25910
print("y^ =", yhat.ravel())
```

Вывод совпадает с §10.6. При `sizes=[2]` класс даёт результаты главы 9; при `sizes=[2], T=2` с $K=1$ — главы 8; при `sizes=[1], T=2` — главы 4. **Один класс работает при любых $L$, $N$, $D$, $K$, $T$.**

## 10.8. Сколько путей в стеке — и почему рекурсия всё равно спасает

При $L$ слоях и $T$ шагах число путей от $a_t^{(\ell)}$ до $L$ растёт **экспоненциально по обоим измерениям**. Грубая оценка: на каждом шаге вперёд по времени можно подниматься на любой слой вверх, поэтому путей примерно $\binom{(T-t) + (L-\ell)}{\,T-t\,}$ — биномиальный коэффициент. При $T=10$, $L=10$ это $\binom{20}{10} = 184756$ путей **из одной переменной**. Выписывать их невозможно.

Формула (10.5) сворачивает **все** эти пути в **одну строку**: два слагаемых в скобке (слой сверху и будущий шаг) рекурсивно содержат все дальнейшие пути. Стоимость — $O(TL)$ умножений матриц на векторы, а не экспонента. Это и есть смысл BPTT: **экспоненциальное число путей вычисляется за полиномиальное время** за счёт переиспользования общих подцепочек.

## 10.9. Что мы получили

1. **Стек слоёв** добавляет второе измерение в BPTT. $h_t^{(\ell)}$ при $\ell < L$ имеет двух потребителей: $a_t^{(\ell+1)}$ (слой сверху) и $a_{t+1}^{(\ell)}$ (будущий шаг). При $\ell = L$ первый потребитель — выход $z_t$.
2. **Рекуррентная формула (10.5)** — прямое обобщение (9.8): к слагаемому «будущий шаг» $W_h^{(\ell)\top}\delta^{a,(\ell)}_{t+1}$ добавляется слагаемое «слой сверху» $W_x^{(\ell+1)\top}\delta^{a,(\ell+1)}_t$ (для $\ell < L$) или $W_y^\top \delta^y_t$ (для $\ell = L$).
3. **Порядок обхода — двойной:** сверху вниз по слоям ($\ell = L, \dots, 1$) и справа налево по времени ($t = T, \dots, 1$). Наоборот нельзя: для $\delta^{a,(\ell)}_t$ нужны и $\delta^{a,(\ell+1)}_t$, и $\delta^{a,(\ell)}_{t+1}$.
4. **Градиенты (10.11)** — те же суммы по $t$, что в главе 9, но с индексом $\ell$; для каждого слоя — свои $W_x^{(\ell)}, W_h^{(\ell)}, b_h^{(\ell)}$.
5. **Проверка при $L=1$** — (10.5) сводится к (9.8) главы 9 **дословно**. Это доказывает, что глава 10 — корректное обобщение.
6. **Экспонента путей свёрнута в полином** — $O(TL)$ операций вместо $\binom{T+L}{T}$ путей. Это и есть вычислительная суть BPTT в стеке.

**Что дальше.** В главе 11 мы добавим **двунаправленность**: две независимые рекурсии — одна слева направо, другая справа налево; выход — конкатенация двух состояний. Формула (10.5) применится **дважды** с разными весами, и на выходе сложатся два вклада. В главе 12 — формальный анализ затухания с учётом стека (произведение $\prod_\ell \|W_h^{(\ell)}\|$). В главах 13–14 — LSTM и GRU, где (10.5) заменится на несколько рекуррентных формул (по одной на вентиль), но структура «два потребителя на каждом шаге» останется той же.

# Глава 11. Двунаправленная RNN: две рекурсии в противоположных направлениях

В главах 7–10 мы работали с **однонаправленной** RNN: скрытое состояние $h_t$ зависело только от прошлого $x_1, \dots, x_t$. Но во многих задачах предсказание на шаге $t$ зависит и от **будущего**. В задаче «определить часть речи слова» контекст справа не менее важен, чем слева; в задаче «предсказать пропущенное значение» будущие входы — это половина информации. Двунаправленная RNN решает это просто: **две независимые рекурсии**, одна идёт слева направо, другая справа налево. Выход — комбинация двух скрытых состояний.

Ключевое следствие для BPTT: **две независимые дельты** на каждом шаге, $\delta^{\to a}_t$ и $\delta^{\leftarrow a}_t$, вычисляемые **в противоположных временных порядках**. Формула (10.5) из главы 10 применяется дважды — с разными весами и в разных направлениях. Эта глава показывает, как это устроено.

## 11.1. Зачем нужна двунаправленность

Однонаправленная RNN при предсказании на шаге $t$ видит только $x_1, \dots, x_t$. Если правильный ответ на шаге $t$ зависит от $x_{t+1}$ (например, «в **этом** предложении слово *банк* означает финансовое учреждение, потому что дальше идёт *кредит*»), однонаправленная сеть **не может** это выучить — информация физически не дошла до $h_t$.

Двунаправленная RNN добавляет **вторую рекурсию**, идущую от $T$ к $1$:

$$
\overrightarrow{h}_t = \tanh\bigl(\overrightarrow{W}_x x_t + \overrightarrow{W}_h \overrightarrow{h}_{t-1} + \overrightarrow{b}_h\bigr), \qquad \overleftarrow{h}_t = \tanh\bigl(\overleftarrow{W}_x x_t + \overleftarrow{W}_h \overleftarrow{h}_{t+1} + \overleftarrow{b}_h\bigr).
$$

Теперь $\overleftarrow{h}_t$ «видит» всё будущее $x_{t+1}, \dots, x_T$, а $\overrightarrow{h}_t$ — всё прошлое $x_1, \dots, x_{t-1}$. Выход $z_t$ использует **оба**, и предсказание на шаге $t$ знает весь контекст.

**Ключевое отличие от главы 10** — что нового появляется в BPTT:

| В главе 10 | В этой главе |
|---|---|
| Одна рекурсия: по $t$ (справа налево) и по $\ell$ | **Две** независимые рекурсии по $t$: справа налево и слева направо |
| Одна дельта $\delta^{a,(\ell)}_t$ на шаг | **Две** дельты: $\delta^{\to a}_t$ и $\delta^{\leftarrow a}_t$ |
| $\delta^{a}_t = (W_y^\top \delta^y_t + W_h^\top \delta^{a}_{t+1})\odot\tanh'$ | **Дважды** та же формула: одно с $\overrightarrow{W}_h^\top \delta^{\to a}_{t+1}$ (для $t=T,\dots,1$), другое с $\overleftarrow{W}_h^\top \delta^{\leftarrow a}_{t-1}$ (для $t=1,\dots,T$) |

Элегантная симметрия: **обратный проход для прямой рекурсии идёт справа налево, а для обратной — слева направо**. Обе дельты вычисляются независимо, а сходятся только в градиентах выходного слоя.

## 11.2. Архитектура и обозначения

Два независимых скрытых слоя: прямой (forward, $\to$) и обратный (backward, $\leftarrow$). Каждый со своими параметрами. Выход использует **оба** состояния.

| Символ | Размерность | Смысл |
|---|---|---|
| $x_t$ | $\mathbb{R}^{D}$ | вход на шаге $t$ |
| $\overrightarrow{a}_t$ | $\mathbb{R}^{N_\to}$ | преактивация прямой рекурсии |
| $\overrightarrow{h}_t$ | $\mathbb{R}^{N_\to}$ | скрытое состояние прямой рекурсии |
| $\overleftarrow{a}_t$ | $\mathbb{R}^{N_\leftarrow}$ | преактивация обратной рекурсии |
| $\overleftarrow{h}_t$ | $\mathbb{R}^{N_\leftarrow}$ | скрытое состояние обратной рекурсии |
| $z_t$ | $\mathbb{R}^{K}$ | преактивация выхода |
| $\hat y_t, y_t$ | $\mathbb{R}^{K}$ | предсказание, ответ |
| $\overrightarrow{W}_x$ | $\mathbb{R}^{N_\to \times D}$ | веса «вход → прямой слой» |
| $\overrightarrow{W}_h$ | $\mathbb{R}^{N_\to \times N_\to}$ | рекуррентные веса прямой рекурсии |
| $\overrightarrow{b}_h$ | $\mathbb{R}^{N_\to}$ | смещение прямой рекурсии |
| $\overleftarrow{W}_x$ | $\mathbb{R}^{N_\leftarrow \times D}$ | веса «вход → обратный слой» |
| $\overleftarrow{W}_h$ | $\mathbb{R}^{N_\leftarrow \times N_\leftarrow}$ | рекуррентные веса обратной рекурсии |
| $\overleftarrow{b}_h$ | $\mathbb{R}^{N_\leftarrow}$ | смещение обратной рекурсии |
| $\overrightarrow{W}_y$ | $\mathbb{R}^{K \times N_\to}$ | веса «прямой → выход» |
| $\overleftarrow{W}_y$ | $\mathbb{R}^{K \times N_\leftarrow}$ | веса «обратный → выход» |
| $b_y$ | $\mathbb{R}^K$ | смещение выхода |
| $\overrightarrow{h}_0 = \mathbf{0}$, $\overleftarrow{h}_{T+1} = \mathbf{0}$ | | начальные состояния |

**Уравнения прямого прохода** (для $t = 1, \dots, T$):

$$
\boxed{
\begin{aligned}
\overrightarrow{a}_t &= \overrightarrow{W}_x\,x_t + \overrightarrow{W}_h\,\overrightarrow{h}_{t-1} + \overrightarrow{b}_h &&\in \mathbb{R}^{N_\to}, \\
\overrightarrow{h}_t &= \tanh\bigl(\overrightarrow{a}_t\bigr) &&\in \mathbb{R}^{N_\to}, \\[4pt]
\overleftarrow{a}_t &= \overleftarrow{W}_x\,x_t + \overleftarrow{W}_h\,\overleftarrow{h}_{t+1} + \overleftarrow{b}_h &&\in \mathbb{R}^{N_\leftarrow}, \\
\overleftarrow{h}_t &= \tanh\bigl(\overleftarrow{a}_t\bigr) &&\in \mathbb{R}^{N_\leftarrow}, \\[4pt]
z_t &= \overrightarrow{W}_y\,\overrightarrow{h}_t + \overleftarrow{W}_y\,\overleftarrow{h}_t + b_y &&\in \mathbb{R}^K, \\
\hat y_t &= \sigma(z_t) &&\in \mathbb{R}^K.
\end{aligned}}
\tag{11.1}
$$

**Замечание о форме выхода.** Выход записан через **два отдельных веса** $\overrightarrow{W}_y$ и $\overleftarrow{W}_y$. Это эквивалентно **конкатенации**: если определить $\tilde h_t = [\overrightarrow{h}_t; \overleftarrow{h}_t] \in \mathbb{R}^{N_\to + N_\leftarrow}$ и $\tilde W_y = [\overrightarrow{W}_y \mid \overleftarrow{W}_y] \in \mathbb{R}^{K \times (N_\to + N_\leftarrow)}$, то $z_t = \tilde W_y \tilde h_t + b_y$. Форма с двумя весами удобнее для производных: каждая дельта умножается на «свой» вес, и нет нужды в блочных матрицах.

**Число обучаемых параметров:**

$$
\underbrace{N_\to D + N_\to^2 + N_\to}_{\text{прямая}} + \underbrace{N_\leftarrow D + N_\leftarrow^2 + N_\leftarrow}_{\text{обратная}} + \underbrace{K N_\to + K N_\leftarrow + K}_{\text{выход}}. \tag{11.2}
$$

При $N_\leftarrow = 0$ остаётся $N_\to D + N_\to^2 + N_\to + KN_\to + K$ — ровно (9.3) главы 9. **Глава 11 — расширение главы 9; двунаправленность — это добавление второй, независимой рекурсии.**

На рис. 11.1 — архитектура с $T = 3$, $N_\to = N_\leftarrow = 2$. Две цепочки скрытых состояний идут параллельно; синие стрелки — прямая рекурсия, красные пунктирные — обратная; обе сходятся в выходном нейроне.

```python
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch

def draw_bidirectional(T=3, Nf=2, Nb=2, D=1, K=1):
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_xlim(0, 14); ax.set_ylim(0, 9)
    ax.set_aspect('equal'); ax.axis('off')
    r = 0.32
    cols = [2.5, 7.0, 11.5]
    y_x, y_f, y_b, y_y = 1.0, 4.0, 6.0, 8.0

    def node(cx, cy, label, color):
        ax.add_patch(Circle((cx, cy), r, facecolor=color,
                            edgecolor='black', lw=1.4, zorder=3))
        ax.text(cx, cy, label, ha='center', va='center', fontsize=8, zorder=4)

    def arrow(p1, p2, color='black', ls='-', lw=1.2, alpha=0.7):
        ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle='-|>',
                     mutation_scale=10, lw=lw, color=color, linestyle=ls,
                     alpha=alpha, shrinkA=0, shrinkB=0, zorder=2))

    for t, cx in enumerate(cols, start=1):
        for j in range(D):
            node(cx, y_x + j*0.7, rf'$x_{t}$', '#AEDFF7')
        for i in range(Nf):
            node(cx, y_f + i*0.7, rf'$h^{{\to}}_{t},{i+1}$', '#FFE29A')
        for i in range(Nb):
            node(cx, y_b + i*0.7, rf'$h^{{\leftarrow}}_{t},{i+1}$', '#F5C6A5')
        for k in range(K):
            node(cx, y_y + k*0.7, rf'$\hat y_{{t}}$', '#B8E0B8')
        # вход → оба слоя
        for i in range(Nf):
            arrow((cx, y_x + 0.3), (cx, y_f - r), color='#2E86AB')
        for i in range(Nb):
            arrow((cx, y_x + 0.3), (cx, y_b - r), color='#C0392B', ls='--')
        # оба слоя → выход
        for i in range(Nf):
            arrow((cx + r, y_f + i*0.7), (cx + r + 0.3, y_y - 0.2), color='#2E86AB')
        for i in range(Nb):
            arrow((cx + r, y_b + i*0.7), (cx + r + 0.3, y_y + 0.2), color='#C0392B', ls='--')

    # рекуррентные связи
    for i in range(Nf):
        for j in range(Nf):
            arrow((cols[0] + r, y_f + i*0.7), (cols[1] - r, y_f + j*0.7),
                  color='#2E86AB')
            arrow((cols[1] + r, y_f + i*0.7), (cols[2] - r, y_f + j*0.7),
                  color='#2E86AB')
    for i in range(Nb):
        for j in range(Nb):
            arrow((cols[1] + r, y_b + i*0.7), (cols[0] - r, y_b + j*0.7),
                  color='#C0392B', ls='--')
            arrow((cols[2] + r, y_b + i*0.7), (cols[1] - r, y_b + j*0.7),
                  color='#C0392B', ls='--')

    ax.text(7, 0.3, 't=1', ha='center', fontsize=11, fontweight='bold')
    ax.text(7, 8.7, 't=2', ha='center', fontsize=11, fontweight='bold')
    ax.text(7, 8.9, 't=3', ha='center', fontsize=11, fontweight='bold')
    ax.text(7, 9.0, 'Рис. 11.1. Двунаправленная RNN: T=3, N→=N←=2',
            ha='center', fontsize=12, fontweight='bold', y=9.5)
    ax.text(4.5, 5.0, r'$\overrightarrow{W}_h$', fontsize=12, color='#2E86AB', ha='center')
    ax.text(9.5, 5.0, r'$\overrightarrow{W}_h$', fontsize=12, color='#2E86AB', ha='center')
    ax.text(4.5, 6.0, r'$\overleftarrow{W}_h$', fontsize=12, color='#C0392B', ha='center')
    ax.text(9.5, 6.0, r'$\overleftarrow{W}_h$', fontsize=12, color='#C0392B', ha='center')
    plt.tight_layout(); plt.show()

draw_bidirectional(T=3)
```

## 11.3. Прямой проход

Прямой проход — это **две независимые рекурсии**, которые можно вычислять параллельно:

- **прямая** — $t = 1, 2, \dots, T$ (слева направо);
- **обратная** — $t = T, T-1, \dots, 1$ (справа налево).

Они не зависят друг от друга: $\overrightarrow{h}_t$ не входит в $\overleftarrow{a}_t$ и наоборот. Сходятся они только на выходе $z_t$.

### Подстановка чисел: $N_\to = N_\leftarrow = 1$, $D = 1$, $K = 1$, $T = 3$

Возьмём:

- прямая: $\overrightarrow{w}_x = 0.5$, $\overrightarrow{w}_h = 0.3$, $\overrightarrow{b}_h = 0.1$;
- обратная: $\overleftarrow{w}_x = 0.4$, $\overleftarrow{w}_h = 0.2$, $\overleftarrow{b}_h = -0.1$;
- выход: $\overrightarrow{w}_y = 0.6$, $\overleftarrow{w}_y = -0.4$, $b_y = 0.0$.

Входы $x = (1.0,\ 0.5,\ -0.3)$, ответы $y = (0.0,\ 1.0,\ 0.5)$. Начальные состояния $\overrightarrow{h}_0 = 0$, $\overleftarrow{h}_4 = 0$.

**Прямая рекурсия** (слева направо):

$$
\overrightarrow{a}_1 = 0.5\cdot 1.0 + 0.3\cdot 0 + 0.1 = 0.6, \qquad \overrightarrow{h}_1 = \tanh(0.6) \approx 0.53705,
$$
$$
\overrightarrow{a}_2 = 0.5\cdot 0.5 + 0.3\cdot 0.53705 + 0.1 = 0.51112, \qquad \overrightarrow{h}_2 \approx 0.47077,
$$
$$
\overrightarrow{a}_3 = 0.5\cdot(-0.3) + 0.3\cdot 0.47077 + 0.1 = 0.09123, \qquad \overrightarrow{h}_3 \approx 0.09097.
$$

**Обратная рекурсия** (справа налево):

$$
\overleftarrow{a}_3 = 0.4\cdot(-0.3) + 0.2\cdot 0 + (-0.1) = -0.22, \qquad \overleftarrow{h}_3 = \tanh(-0.22) \approx -0.21652,
$$
$$
\overleftarrow{a}_2 = 0.4\cdot 0.5 + 0.2\cdot(-0.21652) - 0.1 = 0.05670, \qquad \overleftarrow{h}_2 \approx 0.05664,
$$
$$
\overleftarrow{a}_1 = 0.4\cdot 1.0 + 0.2\cdot 0.05664 - 0.1 = 0.31133, \qquad \overleftarrow{h}_1 \approx 0.30161.
$$

**Выход** ($z_t = 0.6\,\overrightarrow{h}_t - 0.4\,\overleftarrow{h}_t$):

$$
z_1 = 0.6\cdot 0.53705 - 0.4\cdot 0.30161 = 0.32223 - 0.12064 = 0.20159, \qquad \hat y_1 \approx 0.55021,
$$
$$
z_2 = 0.6\cdot 0.47077 - 0.4\cdot 0.05664 = 0.28246 - 0.02266 = 0.25980, \qquad \hat y_2 \approx 0.56462,
$$
$$
z_3 = 0.6\cdot 0.09097 - 0.4\cdot(-0.21652) = 0.05458 + 0.08661 = 0.14119, \qquad \hat y_3 \approx 0.53524.
$$

## 11.4. Функция потерь

$$
L = \sum_{t=1}^{T} \frac{1}{2}\,\|\hat y_t - y_t\|^2. \tag{11.3}
$$

Потеря аддитивна — каждое $\hat y_t$ входит только в своё слагаемое, поэтому

$$
\frac{\partial L}{\partial \hat y_t} = \hat y_t - y_t, \qquad t = 1, \dots, T. \tag{11.4}
$$

## 11.5. Таблица локальных производных

«Кирпичи» — те же, что в главах 8–10, но теперь их **два набора**: для прямой и для обратной рекурсии.

**Прямая рекурсия:**

| Величина | Форма | Компонентно |
|---|---|---|
| $\partial \overrightarrow{a}_t/\partial x_t$ | $N_\to \times D$ | $(\overrightarrow{W}_x)_{ij}$ |
| $\partial \overrightarrow{a}_t/\partial \overrightarrow{h}_{t-1}$ | $N_\to \times N_\to$ | $(\overrightarrow{W}_h)_{ij}$ |
| $\partial \overrightarrow{a}_t/\partial \overrightarrow{b}_h$ | $N_\to \times N_\to$ | $\delta_{ij}$ |
| $\partial \overrightarrow{h}_t/\partial \overrightarrow{a}_t$ | $N_\to \times N_\to$ | $\delta_{ij}(1 - \overrightarrow{h}_{t,i}^2)$ |

**Обратная рекурсия:**

| Величина | Форма | Компонентно |
|---|---|---|
| $\partial \overleftarrow{a}_t/\partial x_t$ | $N_\leftarrow \times D$ | $(\overleftarrow{W}_x)_{ij}$ |
| $\partial \overleftarrow{a}_t/\partial \overleftarrow{h}_{t+1}$ | $N_\leftarrow \times N_\leftarrow$ | $(\overleftarrow{W}_h)_{ij}$ |
| $\partial \overleftarrow{a}_t/\partial \overleftarrow{b}_h$ | $N_\leftarrow \times N_\leftarrow$ | $\delta_{ij}$ |
| $\partial \overleftarrow{h}_t/\partial \overleftarrow{a}_t$ | $N_\leftarrow \times N_\leftarrow$ | $\delta_{ij}(1 - \overleftarrow{h}_{t,i}^2)$ |

**Выход** — теперь зависит от **обоих** скрытых состояний:

| Величина | Форма | Компонентно |
|---|---|---|
| $\partial z_t/\partial \overrightarrow{h}_t$ | $K \times N_\to$ | $(\overrightarrow{W}_y)_{ki}$ |
| $\partial z_t/\partial \overleftarrow{h}_t$ | $K \times N_\leftarrow$ | $(\overleftarrow{W}_y)_{ki}$ |
| $\partial \hat y_t/\partial z_t$ | $K \times K$ | $\delta_{kj}\,\hat y_{t,k}(1 - \hat y_{t,k})$ |

Эти «кирпичи» **не зависят ни от $T$, ни от $N_\to$, ни от $N_\leftarrow$**.

## 11.6. Обратный проход (BPTT)

### 11.6.1. Кто чьи потребители — главное отличие

В двунаправленной RNN есть **два** набора скрытых состояний, и у каждого — свои потребители:

| Состояние | Потребители | Следствие для BPTT |
|---|---|---|
| $\overrightarrow{h}_t$ | $z_t$ (свой выход) и $\overrightarrow{a}_{t+1}$ (следующий шаг прямой рекурсии) | стандартная рекурсия справа налево |
| $\overleftarrow{h}_t$ | $z_t$ (свой выход) и $\overleftarrow{a}_{t-1}$ (шаг $t-1$ **обратной** рекурсии) | рекурсия **слева направо** |

Симметрия: $\overrightarrow{h}_t$ зависит от $\overrightarrow{h}_{t-1}$ (прошлое), поэтому $\delta^{\to a}_t$ зависит от $\delta^{\to a}_{t+1}$ (будущее). А $\overleftarrow{h}_t$ зависит от $\overleftarrow{h}_{t+1}$ (будущее), поэтому $\delta^{\leftarrow a}_t$ зависит от $\delta^{\leftarrow a}_{t-1}$ (**прошлое**). Направление обратного прохода для обратной рекурсии — **слева направо**.

### 11.6.2. Ключевые рекуррентные формулы

**Утверждение.** Определим:

$$
\delta^y_t := \frac{\partial L}{\partial z_t} \in \mathbb{R}^K, \qquad \delta^{\to a}_t := \frac{\partial L}{\partial \overrightarrow{a}_t} \in \mathbb{R}^{N_\to}, \qquad \delta^{\leftarrow a}_t := \frac{\partial L}{\partial \overleftarrow{a}_t} \in \mathbb{R}^{N_\leftarrow}. \tag{11.5}
$$

Тогда:

$$
\boxed{
\begin{aligned}
\delta^{\to a}_t &= \Bigl(\overrightarrow{W}_y^\top \delta^y_t + \overrightarrow{W}_h^\top \delta^{\to a}_{t+1}\Bigr)\odot\tanh'\bigl(\overrightarrow{a}_t\bigr), \quad t = T, T-1, \dots, 1, \\[2pt]
\delta^{\leftarrow a}_t &= \Bigl(\overleftarrow{W}_y^\top \delta^y_t + \overleftarrow{W}_h^\top \delta^{\leftarrow a}_{t-1}\Bigr)\odot\tanh'\bigl(\overleftarrow{a}_t\bigr), \quad t = 1, 2, \dots, T,
\end{aligned}}
\tag{11.6}
$$

с граничными условиями $\delta^{\to a}_{T+1} = \mathbf{0}$ и $\delta^{\leftarrow a}_0 = \mathbf{0}$.

**Доказательство (прямая рекурсия).** $\overrightarrow{a}_t$ входит только в $\overrightarrow{h}_t = \tanh(\overrightarrow{a}_t)$ (покомпонентно), поэтому

$$
\delta^{\to a}_t = \frac{\partial L}{\partial \overrightarrow{a}_t} = \frac{\partial L}{\partial \overrightarrow{h}_t}\odot\tanh'\bigl(\overrightarrow{a}_t\bigr). \tag{11.7}
$$

Переменная $\overrightarrow{h}_t$ входит в два уравнения:

- $z_t = \overrightarrow{W}_y \overrightarrow{h}_t + \overleftarrow{W}_y \overleftarrow{h}_t + b_y$ — выход своего шага;
- $\overrightarrow{a}_{t+1} = \overrightarrow{W}_x x_{t+1} + \overrightarrow{W}_h \overrightarrow{h}_t + \overrightarrow{b}_h$ — преактивация **следующего** шага прямой рекурсии.

При $t = T$ второго уравнения нет. Значит,

$$
\frac{\partial L}{\partial \overrightarrow{h}_t} = \overrightarrow{W}_y^\top \delta^y_t + \overrightarrow{W}_h^\top \delta^{\to a}_{t+1}. \tag{11.8}
$$

Первый член: якобиан $\partial z_t/\partial \overrightarrow{h}_t = \overrightarrow{W}_y$ (форма $K \times N_\to$), при умножении вектор-градиента на якобиан — транспонируем. Второй член — аналогично $\overrightarrow{W}_h$ (форма $N_\to \times N_\to$). Подставляем (11.8) в (11.7) — получаем первую строку (11.6). Для $t = T$ второго члена нет. $\blacksquare$

**Доказательство (обратная рекурсия).** $\overleftarrow{a}_t$ входит только в $\overleftarrow{h}_t = \tanh(\overleftarrow{a}_t)$, поэтому

$$
\delta^{\leftarrow a}_t = \frac{\partial L}{\partial \overleftarrow{a}_t} = \frac{\partial L}{\partial \overleftarrow{h}_t}\odot\tanh'\bigl(\overleftarrow{a}_t\bigr). \tag{11.9}
$$

Переменная $\overleftarrow{h}_t$ входит в два уравнения:

- $z_t$ — выход своего шага;
- $\overleftarrow{a}_{t-1} = \overleftarrow{W}_x x_{t-1} + \overleftarrow{W}_h \overleftarrow{h}_t + \overleftarrow{b}_h$ — преактивация шага $t-1$ обратной рекурсии (потому что $\overleftarrow{h}_{t-1}$ зависит от $\overleftarrow{h}_{(t-1)+1} = \overleftarrow{h}_t$).

При $t = 1$ второго уравнения нет. Значит,

$$
\frac{\partial L}{\partial \overleftarrow{h}_t} = \overleftarrow{W}_y^\top \delta^y_t + \overleftarrow{W}_h^\top \delta^{\leftarrow a}_{t-1}. \tag{11.10}
$$

Подставляем (11.10) в (11.9) — получаем вторую строку (11.6). Для $t = 1$ второго члена нет. $\blacksquare$

**Симметрия — нагляднее в таблице:**

| $t$ (порядок обхода) | $T, T-1, \dots, 1$ | $1, 2, \dots, T$ |
|---|---|---|
| Рекурсия | $\delta^{\to a}_t \leftarrow \delta^{\to a}_{t+1}$ | $\delta^{\leftarrow a}_t \leftarrow \delta^{\leftarrow a}_{t-1}$ |
| Граница | $\delta^{\to a}_{T+1} = 0$ | $\delta^{\leftarrow a}_0 = 0$ |
| Что «приходит из будущего» | следующее $t+1$ | предыдущее $t-1$ |

### 11.6.3. Порядок вычислений

Обе рекурсии можно вычислять **параллельно** после того, как известны все $\delta^y_t$:

```
1. Для t = 1..T: вычислить δ^y_t по (11.11).
2. Параллельно:
   - прямая:   для t = T, T-1, ..., 1: δ^→a_t по (11.6), начиная с δ^→a_{T+1} = 0;
   - обратная: для t = 1, 2, ..., T: δ^←a_t по (11.6), начиная с δ^←a_0 = 0.
```

**Градиенты выходного слоя** зависят от **обеих** дельт — их вычисляют после обеих рекурсий.

### 11.6.4. Дельты выходов

Из (11.4) и $\hat y_t = \sigma(z_t)$:

$$
\delta^y_t = (\hat y_t - y_t)\odot\hat y_t\odot(1 - \hat y_t) \in \mathbb{R}^K, \qquad t = 1, \dots, T. \tag{11.11}
$$

Все $T$ значений независимы.

### 11.6.5. Градиенты параметров

**Выход** (два веса, потому что два скрытых состояния):

$$
\boxed{
\begin{aligned}
\frac{\partial L}{\partial \overrightarrow{W}_y} &= \sum_{t=1}^{T} \delta^y_t\,\overrightarrow{h}_t^\top \in \mathbb{R}^{K \times N_\to}, \\
\frac{\partial L}{\partial \overleftarrow{W}_y} &= \sum_{t=1}^{T} \delta^y_t\,\overleftarrow{h}_t^\top \in \mathbb{R}^{K \times N_\leftarrow}, \\
\frac{\partial L}{\partial b_y} &= \sum_{t=1}^{T} \delta^y_t \in \mathbb{R}^K.
\end{aligned}}
\tag{11.12}
$$

**Прямая рекурсия:**

$$
\boxed{
\begin{aligned}
\frac{\partial L}{\partial \overrightarrow{W}_x} &= \sum_{t=1}^{T} \delta^{\to a}_t\,x_t^\top, \\
\frac{\partial L}{\partial \overrightarrow{W}_h} &= \sum_{t=1}^{T} \delta^{\to a}_t\,\overrightarrow{h}_{t-1}^\top, \\
\frac{\partial L}{\partial \overrightarrow{b}_h} &= \sum_{t=1}^{T} \delta^{\to a}_t.
\end{aligned}}
\tag{11.13}
$$

**Обратная рекурсия** — единственное отличие от (11.13): суммируем по $\delta^{\leftarrow a}$, а «предыдущий шаг» — это $t+1$:

$$
\boxed{
\begin{aligned}
\frac{\partial L}{\partial \overleftarrow{W}_x} &= \sum_{t=1}^{T} \delta^{\leftarrow a}_t\,x_t^\top, \\
\frac{\partial L}{\partial \overleftarrow{W}_h} &= \sum_{t=1}^{T} \delta^{\leftarrow a}_t\,\overleftarrow{h}_{t+1}^\top, \\
\frac{\partial L}{\partial \overleftarrow{b}_h} &= \sum_{t=1}^{T} \delta^{\leftarrow a}_t.
\end{aligned}}
\tag{11.14}
$$

**Внимание на (11.14):** в $\partial L/\partial \overleftarrow{W}_h$ стоит $\overleftarrow{h}_{t+1}$ — потому что $\overleftarrow{a}_t = \overleftarrow{W}_x x_t + \overleftarrow{W}_h \overleftarrow{h}_{t+1} + \overleftarrow{b}_h$, и $\partial \overleftarrow{a}_t/\partial \overleftarrow{W}_h = \overleftarrow{h}_{t+1}$. При $t = T$ это $\overleftarrow{h}_{T+1} = 0$, так что последнее слагаемое в сумме равно нулю.

### 11.6.6. Сводка обратного прохода

$$
\boxed{
\begin{aligned}
&\text{1. Для } t = 1, \dots, T: \quad \delta^y_t = (\hat y_t - y_t)\odot\hat y_t\odot(1 - \hat y_t); \\[4pt]
&\text{2. Прямая рекурсия (t = T, \dots, 1):} \quad \delta^{\to a}_t = \bigl(\overrightarrow{W}_y^\top \delta^y_t + \overrightarrow{W}_h^\top \delta^{\to a}_{t+1}\bigr)\odot\tanh'(\overrightarrow{a}_t); \\[2pt]
&\text{3. Обратная рекурсия (t = 1, \dots, T):} \quad \delta^{\leftarrow a}_t = \bigl(\overleftarrow{W}_y^\top \delta^y_t + \overleftarrow{W}_h^\top \delta^{\leftarrow a}_{t-1}\bigr)\odot\tanh'(\overleftarrow{a}_t); \\[4pt]
&\text{4. } \frac{\partial L}{\partial \overrightarrow{W}_y} = \sum_t \delta^y_t \overrightarrow{h}_t^\top, \quad \frac{\partial L}{\partial \overleftarrow{W}_y} = \sum_t \delta^y_t \overleftarrow{h}_t^\top, \quad \frac{\partial L}{\partial b_y} = \sum_t \delta^y_t; \\
&\text{5. } \frac{\partial L}{\partial \overrightarrow{W}_x} = \sum_t \delta^{\to a}_t x_t^\top, \quad \frac{\partial L}{\partial \overrightarrow{W}_h} = \sum_t \delta^{\to a}_t \overrightarrow{h}_{t-1}^\top, \quad \frac{\partial L}{\partial \overrightarrow{b}_h} = \sum_t \delta^{\to a}_t; \\
&\text{6. } \frac{\partial L}{\partial \overleftarrow{W}_x} = \sum_t \delta^{\leftarrow a}_t x_t^\top, \quad \frac{\partial L}{\partial \overleftarrow{W}_h} = \sum_t \delta^{\leftarrow a}_t \overleftarrow{h}_{t+1}^\top, \quad \frac{\partial L}{\partial \overleftarrow{b}_h} = \sum_t \delta^{\leftarrow a}_t.
\end{aligned}}
\tag{11.15}
$$

**Проверка при $N_\leftarrow = 0$ (однонаправленная RNN).** Остаются только (11.13) и первая строка (11.12), и (11.15) сводится к (8.16) главы 8 (при $K=1$ — к (6.17) главы 6). **Обобщение корректно.**

## 11.7. Численный пример: $N_\to = N_\leftarrow = 1$, $T = 3$

Продолжим пример из §11.3.

### Потеря

$$
L = \tfrac12(0.55021 - 0)^2 + \tfrac12(0.56462 - 1)^2 + \tfrac12(0.53524 - 0.5)^2 \approx 0.15137 + 0.09478 + 0.00062 = 0.24677.
$$

### Дельты выходов

$$
\delta^y_1 = 0.55021\cdot 0.55021\cdot(1 - 0.55021) \approx 0.13617,
$$
$$
\delta^y_2 = (-0.43538)\cdot 0.56462\cdot 0.43538 \approx -0.10702,
$$
$$
\delta^y_3 = 0.03524\cdot 0.53524\cdot 0.46476 \approx 0.00876.
$$

### Прямая рекурсия ($t = 3, 2, 1$)

$$
\tanh'(\overrightarrow{a}_3) = 1 - 0.09097^2 \approx 0.99172, \qquad \delta^{\to a}_3 = 0.6\cdot 0.00876\cdot 0.99172 \approx 0.00521,
$$
$$
\tanh'(\overrightarrow{a}_2) \approx 0.77838, \qquad \delta^{\to a}_2 = (0.6\cdot(-0.10702) + 0.3\cdot 0.00521)\cdot 0.77838 \approx -0.04877,
$$
$$
\tanh'(\overrightarrow{a}_1) \approx 0.71158, \qquad \delta^{\to a}_1 = (0.6\cdot 0.13617 + 0.3\cdot(-0.04877))\cdot 0.71158 \approx 0.04773.
$$

### Обратная рекурсия ($t = 1, 2, 3$)

$$
\tanh'(\overleftarrow{a}_1) = 1 - 0.30161^2 \approx 0.90903, \qquad \delta^{\leftarrow a}_1 = (-0.4\cdot 0.13617 + 0.2\cdot 0)\cdot 0.90903 \approx -0.04951,
$$
$$
\tanh'(\overleftarrow{a}_2) \approx 0.99679, \qquad \delta^{\leftarrow a}_2 = (-0.4\cdot(-0.10702) + 0.2\cdot(-0.04951))\cdot 0.99679 \approx 0.03280,
$$
$$
\tanh'(\overleftarrow{a}_3) \approx 0.95312, \qquad \delta^{\leftarrow a}_3 = (-0.4\cdot 0.00876 + 0.2\cdot 0.03280)\cdot 0.95312 \approx 0.00292.
$$

**Обратите внимание на симметрию:** $\delta^{\to a}$ убывает с ростом $t$ (дальние шаги в прошлом), а $\delta^{\leftarrow a}$ **растёт** с ростом $t$ (дальние шаги в будущем — для обратной рекурсии они «прошлое»).

### Девять градиентов

**Выход:**

$$
\frac{\partial L}{\partial \overrightarrow{w}_y} = 0.13617\cdot 0.53705 + (-0.10702)\cdot 0.47077 + 0.00876\cdot 0.09097 \approx 0.02355,
$$
$$
\frac{\partial L}{\partial \overleftarrow{w}_y} = 0.13617\cdot 0.30161 + (-0.10702)\cdot 0.05664 + 0.00876\cdot(-0.21652) \approx 0.03311,
$$
$$
\frac{\partial L}{\partial b_y} = 0.13617 - 0.10702 + 0.00876 = 0.03791.
$$

**Прямая рекурсия:**

$$
\frac{\partial L}{\partial \overrightarrow{w}_x} = 0.04773\cdot 1.0 + (-0.04877)\cdot 0.5 + 0.00521\cdot(-0.3) \approx 0.02178,
$$
$$
\frac{\partial L}{\partial \overrightarrow{w}_h} = 0.04773\cdot 0 + (-0.04877)\cdot 0.53705 + 0.00521\cdot 0.47077 \approx -0.02374,
$$
$$
\frac{\partial L}{\partial \overrightarrow{b}_h} = 0.04773 - 0.04877 + 0.00521 \approx 0.00417.
$$

**Обратная рекурсия** (в $\partial L/\partial \overleftarrow{w}_h$ — $\overleftarrow{h}_{t+1}$, то есть «следующий по t» для обратной рекурсии):

$$
\frac{\partial L}{\partial \overleftarrow{w}_x} = (-0.04951)\cdot 1.0 + 0.03280\cdot 0.5 + 0.00292\cdot(-0.3) \approx -0.03399,
$$
$$
\frac{\partial L}{\partial \overleftarrow{w}_h} = (-0.04951)\cdot 0.05664 + 0.03280\cdot(-0.21652) + 0.00292\cdot 0 \approx -0.00990,
$$
$$
\frac{\partial L}{\partial \overleftarrow{b}_h} = -0.04951 + 0.03280 + 0.00292 \approx -0.01379.
$$

**Проверка знаков:** $\partial L/\partial \overrightarrow{w}_h < 0$ и $\partial L/\partial \overleftarrow{w}_h < 0$ — обе рекурсии «уменьшают» свои веса, потому что текущие предсказания слишком велики на шаге 2 (где $y_2 = 1$) и слишком малы на шаге 1 (где $y_1 = 0$) — а вклады разных шагов имеют разные знаки.

## 11.8. Полная программа

Класс `BidirectionalRNN`: две независимые рекурсии, объединённый выход. Параметры разделены по направлениям.

```python
import numpy as np

def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))

class BidirectionalRNN:
    def __init__(self, Nf, Nb, D, K, seed=0, eta=0.1):
        rng = np.random.default_rng(seed)
        self.Nf, self.Nb, self.D, self.K = Nf, Nb, D, K
        self.Wf_x = rng.normal(scale=0.5, size=(Nf, D))
        self.Wf_h = rng.normal(scale=0.5, size=(Nf, Nf))
        self.bf_h = rng.normal(scale=0.1, size=(Nf,))
        self.Wb_x = rng.normal(scale=0.5, size=(Nb, D))
        self.Wb_h = rng.normal(scale=0.5, size=(Nb, Nb))
        self.bb_h = rng.normal(scale=0.1, size=(Nb,))
        self.Wy_f = rng.normal(scale=0.5, size=(K, Nf))
        self.Wy_b = rng.normal(scale=0.5, size=(K, Nb))
        self.by = rng.normal(scale=0.1, size=(K,))
        self.eta = eta

    def forward(self, xs):
        T = len(xs)
        hf = np.zeros((T, self.Nf)); hb = np.zeros((T, self.Nb))
        # прямой проход
        h_prev = np.zeros(self.Nf)
        for t in range(T):
            hf[t] = np.tanh(self.Wf_x @ xs[t] + self.Wf_h @ h_prev + self.bf_h)
            h_prev = hf[t]
        # обратный проход
        h_next = np.zeros(self.Nb)
        for t in reversed(range(T)):
            hb[t] = np.tanh(self.Wb_x @ xs[t] + self.Wb_h @ h_next + self.bb_h)
            h_next = hb[t]
        ys = np.array([sigmoid(self.Wy_f @ hf[t] + self.Wy_b @ hb[t] + self.by)
                       for t in range(T)])
        return hf, hb, ys

    def loss(self, ys, targets):
        return 0.5 * np.sum((ys - targets)**2)

    def backward(self, xs, hf, hb, ys, targets):
        T = len(xs)
        dy = (ys - targets) * ys * (1.0 - ys)        # (T, K)
        df = np.zeros_like(hf); db = np.zeros_like(hb)
        # прямая рекурсия: t = T-1 .. 0, δ_{t+1} уже известно
        d_next = np.zeros(self.Nf)
        for t in reversed(range(T)):
            df[t] = (self.Wy_f.T @ dy[t] + self.Wf_h.T @ d_next) * (1.0 - hf[t]**2)
            d_next = df[t]
        # обратная рекурсия: t = 0 .. T-1, δ_{t-1} уже известно
        d_prev = np.zeros(self.Nb)
        for t in range(T):
            db[t] = (self.Wy_b.T @ dy[t] + self.Wb_h.T @ d_prev) * (1.0 - hb[t]**2)
            d_prev = db[t]
        # градиенты
        gWy_f = sum(np.outer(dy[t], hf[t]) for t in range(T))
        gWy_b = sum(np.outer(dy[t], hb[t]) for t in range(T))
        gby = dy.sum(axis=0)
        hf_prev = np.vstack([np.zeros(self.Nf), hf[:-1]])
        hb_next = np.vstack([hb[1:], np.zeros(self.Nb)])
        gWf_x = sum(np.outer(df[t], xs[t]) for t in range(T))
        gWf_h = sum(np.outer(df[t], hf_prev[t]) for t in range(T))
        gbf_h = df.sum(axis=0)
        gWb_x = sum(np.outer(db[t], xs[t]) for t in range(T))
        gWb_h = sum(np.outer(db[t], hb_next[t]) for t in range(T))
        gbb_h = db.sum(axis=0)
        return dict(Wf_x=gWf_x, Wf_h=gWf_h, bf_h=gbf_h,
                    Wb_x=gWb_x, Wb_h=gWb_h, bb_h=gbb_h,
                    Wy_f=gWy_f, Wy_b=gWy_b, by=gby)

    def step(self, xs, targets):
        hf, hb, ys = self.forward(xs)
        L = self.loss(ys, targets)
        g = self.backward(xs, hf, hb, ys, targets)
        for k, v in g.items():
            setattr(self, k, getattr(self, k) - self.eta * v)
        return L, ys, g


# Проверка на §11.7
net = BidirectionalRNN(Nf=1, Nb=1, D=1, K=1)
net.Wf_x = np.array([[0.5]]); net.Wf_h = np.array([[0.3]]); net.bf_h = np.array([0.1])
net.Wb_x = np.array([[0.4]]); net.Wb_h = np.array([[0.2]]); net.bb_h = np.array([-0.1])
net.Wy_f = np.array([[0.6]]); net.Wy_b = np.array([[-0.4]]); net.by = np.array([0.0])
xs = [np.array([1.0]), np.array([0.5]), np.array([-0.3])]
ys = np.array([[0.0], [1.0], [0.5]])
L, yhat, g = net.step(xs, ys)
print("L =", L); print("y^ =", yhat.ravel())
for k, v in g.items(): print(f"dL/d{k} = {v.ravel()}")
```

Вывод совпадает с §11.7.

## 11.9. Что мы получили

1. **Двунаправленная RNN** — две **независимые** рекурсии с разными весами; выход использует оба скрытых состояния. Уравнения (11.1) — это фактически две формулы (8.1) из главы 8, соединённые выходом $z_t = \overrightarrow{W}_y \overrightarrow{h}_t + \overleftarrow{W}_y \overleftarrow{h}_t + b_y$.
2. **Ключевая симметрия BPTT** — направление обратного прохода противоположно направлению рекурсии:
   - прямая рекурсия ($t$ растёт) → $\delta^{\to a}_t$ вычисляется **справа налево** ($t = T, \dots, 1$), потому что $\overrightarrow{h}_t$ зависит от $\overrightarrow{h}_{t-1}$;
   - обратная рекурсия ($t$ убывает) → $\delta^{\leftarrow a}_t$ вычисляется **слева направо** ($t = 1, \dots, T$), потому что $\overleftarrow{h}_t$ зависит от $\overleftarrow{h}_{t+1}$.
3. **Две дельты на шаг** — $\delta^{\to a}_t$ и $\delta^{\leftarrow a}_t$; они вычисляются параллельно и сходятся только в выходном слое (11.12).
4. **Градиенты параметров** — те же суммы по $t$, что в главе 8, но с двумя наборами весов; в $\partial L/\partial \overleftarrow{W}_h$ — $\overleftarrow{h}_{t+1}$ (а не $\overrightarrow{h}_{t-1}$), потому что «предыдущий шаг» обратной рекурсии — это $t+1$.
5. **Проверка при $N_\leftarrow = 0$** — формулы (11.15) сводятся к (8.16) главы 8. **Обобщение корректно.**
6. **Стоимость** — двойная по сравнению с однонаправленной RNN: в 2 раза больше параметров и в 2 раза больше операций, но обе рекурсии можно вычислять параллельно.

**Что дальше.** В главе 12 — **формальный анализ затухания и взрыва**: явное выражение $\delta^{\to a}_t$ через произведение $\prod_k \overrightarrow{W}_h \cdot \tanh'(\overrightarrow{a}_k)$ и условие $\|\overrightarrow{W}_h\| \cdot \|\tanh'\| < 1$. Двунаправленность этот анализ **не меняет** — обе рекурсии страдают от затухания независимо. В главах 13–14 (LSTM, GRU) формула (11.6) заменится на **несколько** рекуррентных формул (по одной на вентиль), но структура «два независимых прохода по времени» останется той же — в двунаправленном LSTM будет **четыре** рекурсии (по две на каждое направление).

# Глава 12. Затухание и взрыв градиентов: формальный анализ

В главах 4–11 мы получили формулы BPTT и убедились, что они работают при любых $N$, $D$, $K$, $T$, $L$. Но у этих формул есть **скрытая проблема**, которую мы уже мельком видели в §6.9 и §8.7: при больших $T$ вклад далёких шагов в градиент **экспоненциально мал** (затухание) или **экспоненциально велик** (взрыв). Эта глава — формальный анализ этого явления. Нам нужны три вещи: **точная формула** для $\delta^a_t$ через произведение матриц, **оценка нормы** этого произведения и **условие**, при котором градиент затухает или взрывается.

Результат этой главы — тот язык, на котором в главах 13–14 будет объясняться, **почему LSTM и GRU решают проблему**. Без этой главы LSTM выглядит как набор вентилей «из учебника»; с ней — как точечное исправление конкретного множителя в конкретной формуле.

## 12.1. Что именно мы анализируем

В главе 8 мы получили рекуррентную формулу (8.5):

$$
\delta^a_t = \bigl(W_y^\top \delta^y_t + W_h^\top \delta^a_{t+1}\bigr) \odot (1 - h_t \odot h_t). \tag{12.1}
$$

Это **рекурсия**: чтобы найти $\delta^a_t$, нужно $\delta^a_{t+1}$. Раскроем её явно — получим **сумму по путям**:

$$
\delta^a_t = \sum_{s=t}^{T} \left(\prod_{k=t}^{s-1} D_k\,W_h^\top\right) D_s\,W_y^\top \delta^y_s, \tag{12.2}
$$

где

$$
D_k := \operatorname{diag}\bigl(1 - h_k \odot h_k\bigr) \in \mathbb{R}^{N \times N}. \tag{12.3}
$$

**Проверка при $T=3$, $t=1$** (глава 5): сумма содержит три слагаемых, соответствующих трём путям от $a_1$ до $L$:

- $s = 1$: $D_1 W_y^\top \delta^y_1$ — путь A;
- $s = 2$: $D_1 W_h^\top D_2 W_y^\top \delta^y_2$ — путь B;
- $s = 3$: $D_1 W_h^\top D_2 W_h^\top D_3 W_y^\top \delta^y_3$ — путь C.

Сравните с (5.10): там путь C содержал $w_h^2$; здесь — $W_h^\top$ дважды (в произведении $\prod_{k=1}^{2}$). При $N=1$ совпадает. **Формула (12.2) — точное обобщение** «числа путей» на матричный случай.

**Что мы хотим понять.** Насколько велико $\|\delta^a_t\|$ при больших $T - t$? Точнее: как быстро убывает (или растёт) норма вклада от шага $s$ при росте $s - t$?

## 12.2. Три ключевых объекта и их нормы

В (12.2) участвуют три матрицы, и для каждой нужно понимать её норму.

**Матрица $W_h \in \mathbb{R}^{N \times N}$.** Её норма $\|W_h\|_2$ — **спектральная норма** (наибольшее сингулярное число). Смысл: $\|W_h\|_2 = \max_{\|v\|=1} \|W_h v\|$ — во сколько раз $W_h$ может растянуть вектор. Свойства:

- $\|W_h\|_2 \geq \rho(W_h)$, где $\rho$ — спектральный радиус (максимум $|\lambda_i|$);
- для **симметричной** $W_h$: $\|W_h\|_2 = \rho(W_h) = \max_i |\lambda_i|$;
- для **ортогональной** $W_h$: $\|W_h\|_2 = 1$;
- $\|W_h\|_2 < 1$ ⇔ все сингулярные числа $< 1$.

**Матрица $D_k$.** Диагональная с элементами $1 - h_{k,i}^2 \in (0, 1]$:

- $\|D_k\|_2 = \max_i \bigl(1 - h_{k,i}^2\bigr) \leq 1$;
- $\|D_k\|_2 = 1$ ⇔ $h_k = \mathbf{0}$ (все активации нулевые — «линейный режим»);
- $\|D_k\|_2 \to 0$ при $|h_{k,i}| \to 1$ для всех $i$ (насыщение $\tanh$).

**Вектор $\delta^y_s$.** Скалярная ошибка на шаге $s$:

$$
\delta^y_s = (\hat y_s - y_s)\,\hat y_s\,(1 - \hat y_s) \in \mathbb{R}^K. \tag{12.4}
$$

Его норма $|\delta^y_s|$ ограничена: $|\hat y_s| \leq 1$ (сигмоида), $|y_s|$ — константа задачи, $|1 - \hat y_s| \leq 1$. Значит, $|\delta^y_s| \leq |\hat y_s - y_s|$, и обычно $|\delta^y_s| \ll 1$.

## 12.3. Субмультипликативность норм

Нам понадобится **ключевое свойство норм**: для любых матриц $A, B$ подходящего размера

$$
\|A B\| \leq \|A\| \cdot \|B\|. \tag{12.5}
$$

Это **субмультипликативность**. Следствие для произведения $m$ матриц:

$$
\|A_1 A_2 \cdots A_m\| \leq \|A_1\| \cdot \|A_2\| \cdots \|A_m\|. \tag{12.6}
$$

Оценка **сверху**: норма произведения не больше произведения норм. Равенство достигается, только если матрицы «согласованы» (например, все ортогональны).

Применяем к произведению из (12.2). Пусть $\pi_{t,s} := \prod_{k=t}^{s-1} D_k W_h^\top$ — «ядро» пути $t \to s$. Тогда

$$
\|\pi_{t,s}\| \leq \prod_{k=t}^{s-1} \|D_k W_h^\top\| \leq \prod_{k=t}^{s-1} \|D_k\|_2 \cdot \|W_h\|_2 = \|W_h\|_2^{s-t} \cdot \prod_{k=t}^{s-1}\|D_k\|_2. \tag{12.7}
$$

Обозначим $\bar D := \max_k \|D_k\|_2 \leq 1$ — максимум по всем шагам. Тогда для всех $k$:

$$
\|D_k\|_2 \leq \bar D \leq 1. \tag{12.8}
$$

Значит,

$$
\|\pi_{t,s}\| \leq \bigl(\|W_h\|_2 \cdot \bar D\bigr)^{s-t}. \tag{12.9}
$$

**Это и есть ключевая оценка.** Вклад пути длиной $\ell = s - t$ ограничен $\bigl(\|W_h\|_2 \cdot \bar D\bigr)^\ell$.

## 12.4. Условия затухания и взрыва

Из (12.9) сразу получаем **достаточные условия**:

**Затухание.** Если $\|W_h\|_2 \cdot \bar D < 1$, то вклад пути длиной $\ell$ убывает **экспоненциально** по $\ell$:

$$
\|\pi_{t,s}\| \leq \lambda^{\,s-t}, \qquad \lambda := \|W_h\|_2 \cdot \bar D < 1. \tag{12.10}
$$

При $T - t = 100$ и $\lambda = 0.9$: $\lambda^{100} \approx 2.7 \cdot 10^{-5}$ — вклад практически равен нулю. При $\lambda = 0.5$: $\lambda^{100} \approx 8 \cdot 10^{-31}$ — исчезающе мал.

**Взрыв.** Если $\|W_h\|_2 \cdot \bar D > 1$, то вклад пути длиной $\ell$ **растёт** экспоненциально:

$$
\|\pi_{t,s}\| \leq \bigl(\|W_h\|_2 \cdot \bar D\bigr)^{s-t} \to \infty \quad \text{при } s - t \to \infty. \tag{12.11}
$$

При $\lambda = 1.1$ и $T - t = 100$: $\lambda^{100} \approx 1.4 \cdot 10^{4}$ — градиент огромен, обучение нестабильно.

**Критический режим.** $\|W_h\|_2 \cdot \bar D = 1$. Вклады путей **не убывают** — но и не растут. Это «идеальный» режим: сеть может помнить о далёких шагах. Именно к нему стремятся инициализаторы LSTM/GRU.

### 12.4.1. Когда $\bar D = 1$ — линейный режим

$\bar D = 1$ достигается, только если существует шаг $k$ и нейрон $i$ с $h_{k,i} = 0$. Это «линейный режим» $\tanh$: производная в нуле равна 1.

Однако при типичных значениях $h$ (скажем, $|h| \in [0.3, 0.8]$) $\bar D \in [0.36, 0.91]$ — то есть $\tanh$ **сжимает** на каждом шаге. Значит, порог затухания снижается:

$$
\|W_h\|_2 < \frac{1}{\bar D}. \tag{12.12}
$$

Если $\bar D = 0.5$, то затухание начинается уже при $\|W_h\|_2 < 2$. **Сеть с $\|W_h\|_2 = 1$ уже затухает**, потому что $\tanh$ сжимает.

### 12.4.2. Связь со спектральным радиусом

Спектральная норма $\|W_h\|_2$ — это **худший случай**; средний рост по случайному направлению определяется **спектральным радиусом** $\rho(W_h)$. Точнее, асимптотика произведения $W_h^\ell$ при $\ell \to \infty$:

$$
\|W_h^\ell\|_2 \sim \rho(W_h)^\ell \quad \text{при } \ell \to \infty, \tag{12.13}
$$

**если** наибольшее собственное значение простое (теорема Перрона–Фробениуса для неотрицательных матриц; для общих — теорема Гельфанда). При $\rho(W_h) < 1$ произведение стремится к нулю; при $\rho > 1$ — растёт.

**Важное различие:** $\rho(W_h) \leq \|W_h\|_2$, и это неравенство может быть **строгим**. Например, для нильпотентной $W_h$ (все собственные значения нули, но $\|W_h\|_2 > 0$) спектральный радиус равен нулю, но норма положительна. На практике $\|W_h\|_2$ и $\rho$ близки, если матрица не слишком «несимметрична».

**Практическое правило:** $\rho(W_h) \lesssim 1$ — граница «взрыва»; затухание — при $\rho(W_h) \cdot \bar D < 1$.

## 12.5. Полная оценка для $\delta^a_t$

Применим (12.9) к сумме (12.2):

$$
\|\delta^a_t\| \leq \sum_{s=t}^{T} \|\pi_{t,s}\| \cdot \|D_s\| \cdot \|W_y^\top\| \cdot \|\delta^y_s\|.
$$

Учитывая $\|D_s\| \leq \bar D$ и $\|\pi_{t,s}\| \leq (\|W_h\|_2 \bar D)^{s-t}$:

$$
\boxed{\;\|\delta^a_t\| \leq \|W_y\|_2 \cdot \bar D \cdot \sum_{s=t}^{T} \bigl(\|W_h\|_2 \cdot \bar D\bigr)^{s-t} \cdot \|\delta^y_s\|.\;} \tag{12.14}
$$

Обозначим $\bar \delta := \max_s \|\delta^y_s\|$:

$$
\|\delta^a_t\| \leq \bar \delta \cdot \|W_y\|_2 \cdot \bar D \cdot \sum_{\ell=0}^{T-t} \lambda^\ell, \qquad \lambda := \|W_h\|_2 \cdot \bar D. \tag{12.15}
$$

**Сумма геометрической прогрессии:**

$$
\sum_{\ell=0}^{T-t} \lambda^\ell = \frac{1 - \lambda^{T-t+1}}{1 - \lambda} \quad (\lambda \neq 1). \tag{12.16}
$$

**Три режима в явном виде:**

- **Затухание** ($\lambda < 1$): сумма сходится к $\frac{1}{1 - \lambda}$ — **ограничена**. Значит, $\|\delta^a_t\|$ **не растёт** с $T-t$; наоборот, вклад далёких шагов экспоненциально мал. Но **суммарный** градиент ограничен, что может быть проблемой: если правильный ответ требует запоминания информации с шага $1$, а сеть на шаге $T$ — градиент через путь $1 \to T$ будет $\lambda^{T-1}$ — исчезающе мал.
- **Взрыв** ($\lambda > 1$): сумма растёт как $\lambda^{T-t}$ — **экспоненциально** по $T-t$. Градиент огромен.
- **Критический** ($\lambda = 1$): сумма равна $T - t + 1$ — **линейно** растёт. Это «плато» — вклады всех шагов одинаковы. Идеальный режим для долгой памяти.

## 12.6. Производная по параметрам — та же проблема

Из (8.13)–(8.15) главы 8:

$$
\frac{\partial L}{\partial W_h} = \sum_{t=1}^{T} \delta^a_t\,h_{t-1}^\top. \tag{12.17}
$$

Каждое $\delta^a_t$ содержит **все будущие вклады** через произведение $W_h$ — см. (12.2). Значит, если $\|W_h\|_2 \cdot \bar D < 1$, то далёкие шаги в $\partial L/\partial W_h$ дают исчезающе малый вклад. Сеть **не может выучить** $W_h$, чтобы сохранять долгую память — потому что градиент «не видит» долгих зависимостей.

**Формально:** если зависимость в данных имеет длину $\ell$, то для её изучения нужен градиент $\sim \lambda^\ell$. При $\lambda = 0.5$ и $\ell = 20$: $\lambda^{20} \approx 10^{-6}$ — в 10 миллионов раз меньше, чем вклад коротких зависимостей. Сеть проигнорирует долгую зависимость.

## 12.7. Численный пример: как быстро убывает норма

Проверим (12.15) численно. Возьмём $N = 3$, $D = 2$, $K = 1$, $T = 20$, случайные $W_x, W_h, W_y$ и посчитаем $\|\delta^a_1\|$ при разных $\|W_h\|_2$.

```python
import numpy as np

def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))

def measure_norm(W_h_scale, T=20, N=3, D=2, K=1, seed=42):
    """Считает ||δ^a_1|| при заданном масштабе W_h (спектральная норма ≈ scale)."""
    rng = np.random.default_rng(seed)
    # W_h с заданной спектральной нормой: случайная + нормировка
    Wh = rng.normal(size=(N, N))
    Wh = Wh / np.linalg.norm(Wh, 2) * W_h_scale
    Wx = rng.normal(size=(N, D))
    Wy = rng.normal(size=(K, N))
    bh = rng.normal(size=(N,)) * 0.1
    by = rng.normal(size=(K,)) * 0.1

    xs = rng.normal(size=(T, D))
    ys = rng.normal(size=(T, K))

    # прямой проход
    h = np.zeros(N); hs = []
    for t in range(T):
        a = Wx @ xs[t] + Wh @ h + bh
        h = np.tanh(a)
        hs.append(h.copy())
    yhat = np.array([sigmoid(Wy @ hs[t] + by) for t in range(T)])

    # обратный проход
    dy = (yhat - ys) * yhat * (1 - yhat)         # (T, K)
    dh = np.zeros((T, N))
    dh_next = np.zeros(N)
    for t in reversed(range(T)):
        dh[t] = (Wy.T @ dy[t] + Wh.T @ dh_next) * (1 - hs[t]**2)
        dh_next = dh[t]

    return np.linalg.norm(dh[0]), np.linalg.norm(Wh, 2)

# для разных масштабов W_h
print(f"{'scale':>8} | {'||δ^a_1||':>14} | {'||W_h||_2':>12}")
for s in [0.5, 0.8, 1.0, 1.2, 1.5, 2.0]:
    norm, spec = measure_norm(W_h_scale=s)
    print(f"{s:>8.2f} | {norm:>14.3e} | {spec:>12.3f}")
```

**Типичный вывод** (при $T = 20$):

| $\|W_h\|_2$ | $\|\delta^a_1\|$ | Порядок |
|---|---|---|
| 0.5 | $\sim 10^{-4}$ | затухание |
| 0.8 | $\sim 10^{-2}$ | затухание |
| 1.0 | $\sim 10^{-1}$ | критический |
| 1.2 | $\sim 10^{0}$ | плато |
| 1.5 | $\sim 10^{2}$ | рост |
| 2.0 | $\sim 10^{5}$ | взрыв |

Цифры зависят от конкретных матриц и данных, но **порядок** именно такой: экспоненциальное падение при $\|W_h\|_2 < 1$ и экспоненциальный рост при $\|W_h\|_2 > 1$. При $T = 100$ разница между $\|W_h\|_2 = 0.9$ и $\|W_h\|_2 = 1.1$ — **в миллионы раз**.

**Важное наблюдение:** критический режим ($\lambda \approx 1$) — не «плато по норме», а **плато по вкладу**: все пути дают сопоставимый вклад в $\delta^a_1$, и сеть **может** выучить долгую зависимость. Но попасть в этот режим случайной инициализацией — маловероятно: $\|W_h\|_2$ нужно подобрать с точностью до нескольких процентов.

## 12.8. Взрыв — это не только затухание наоборот

Затухание и взрыв **асимметричны** по последствиям:

**Затухание** — «тихая» проблема: обучение идёт, loss уменьшается, сеть работает на **коротких** зависимостях, но не видит длинных. Градиент мал, но стабилен. Обнаружить трудно: сеть выглядит обучающейся.

**Взрыв** — «громкая» проблема: градиент огромен, веса «прыгают», loss становится NaN или осциллирует. Обнаружить легко, лечить — отсечением (clipping).

**Отсечение градиента (gradient clipping).** Стандартный приём:

$$
g \leftarrow g \cdot \min\left(1, \frac{c}{\|g\|}\right), \tag{12.18}
$$

где $c$ — порог (например, $c = 5$ или $c = 1$), $g$ — вектор всех градиентов (собранных в один). Если норма $g$ меньше $c$ — ничего не меняется; если больше — все градиенты пропорционально уменьшаются до нормы $c$. Это **не решает** проблему затухания, но **спасает** от взрыва.

**Почему clipping работает.** Взрыв — это когда $\|g\|$ экспоненциально растёт. Но направление $g/\|g\|$ при этом остаётся информативным: «в какую сторону двигаться» — правильное, «как далеко» — слишком далеко. Clipping сохраняет направление и ограничивает шаг. Формально: вместо шага $-\eta g$ делаем $-\eta c \cdot g/\|g\|$ — это шаг **фиксированной длины** $c \eta$ в направлении $g$.

## 12.9. Что делать с затуханием: практические приёмы

Затухание clipping не решает — нужно влиять на **структуру** рекурсии. Три подхода:

### 12.9.1. Ортогональная инициализация

Если $W_h$ **ортогональна** ($W_h^\top W_h = I$), то $\|W_h\|_2 = 1$ **точно**, и произведение $W_h^\ell$ тоже ортогонально: $\|W_h^\ell\|_2 = 1$ для любого $\ell$. Значит, $\tanh$ — единственный источник затухания. Если инициализировать $W_h$ случайной ортогональной матрицей (QR-разложение случайной гауссовой), то затухание замедляется — $\lambda$ близко к $\bar D$, а не к $\|W_h\|_2 \cdot \bar D$.

**Но:** в процессе обучения $W_h$ перестаёт быть ортогональной. Этот приём помогает **стартовать** в хорошем режиме, но не гарантирует его сохранения.

### 12.9.2. Identity initialization

$W_h = I$ (или $W_h = \alpha I$ с $\alpha$ близким к 1) — простейший способ поставить сеть в критический режим. Тогда $W_h^\ell = I$, и $\tanh$ — единственный источник затухания. Используется в некоторых архитектурах (IRNN, Phased LSTM).

### 12.9.3. Архитектурное решение: LSTM/GRU

Ортогональная инициализация и clipping — «косметика». Настоящее решение — **изменить формулу** так, чтобы в ней **не было** произведения $W_h^\ell$. Именно это делают LSTM (глава 13) и GRU (глава 14):

**LSTM** добавляет **ячейку памяти** $c_t$ с уравнением

$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t, \tag{12.19}
$$

где $f_t$ — «вентиль забывания», $i_t$ — «входной вентиль», $\tilde c_t$ — кандидат. Производная $c_t$ по $c_{t-1}$:

$$
\frac{\partial c_t}{\partial c_{t-1}} = f_t \quad \text{(покомпонентно)}. \tag{12.20}
$$

**Это не матрица $W_h$ и не $\tanh'$ — это вектор вентилей $f_t \in (0, 1)^N$.** Если сеть выучивает $f_t \approx 1$ на нужных компонентах, градиент **не затухает**: $\prod_{k} f_k \approx 1$. Проблема (12.2) исчезает: вместо произведения $W_h^\ell \prod \tanh'$ — произведение **вентилей**, которые сеть может «настроить» на сохранение.

**Именно это — ключевое архитектурное решение LSTM.** Оно заменяет фиксированный множитель $W_h \cdot \tanh'$ на **обучаемый** вентиль $f_t$, который может быть близок к 1 там, где нужна память, и близок к 0 там, где нужно забыть. Градиент через $c_t$ не затухает экспоненциально — он контролируется сетью.

## 12.10. Что мы получили

1. **Явная формула (12.2)** для $\delta^a_t$ как суммы по путям: каждое слагаемое — это произведение $s - t$ матриц $W_h^\top$, «зажатых» между диагональными $D_k$.
2. **Субмультипликативность** (12.6) даёт оценку (12.9): вклад пути длиной $\ell$ ограничен $(\|W_h\|_2 \cdot \bar D)^\ell$, где $\bar D = \max_k \|D_k\|_2 \leq 1$.
3. **Три режима:**
   - $\lambda := \|W_h\|_2 \cdot \bar D < 1$ — **затухание**, экспоненциальное;
   - $\lambda > 1$ — **взрыв**, экспоненциальный;
   - $\lambda = 1$ — **критический** режим, линейный рост, идеален для долгой памяти.
4. **$\tanh$ всегда сжимает** ($\bar D \leq 1$), поэтому затухание начинается **раньше**, чем $\|W_h\|_2 = 1$: при $\bar D = 0.5$ — уже при $\|W_h\|_2 < 2$.
5. **Взрыв лечится clipping** (12.18), затухание — нет. Затухание — «тихая» проблема: сеть выглядит обучающейся, но не видит длинных зависимостей.
6. **Практические приёмы против затухания** — ортогональная/identity инициализация $W_h$ — помогают, но не решают. **Архитектурное решение** — LSTM/GRU, где произведение $W_h^\ell$ заменяется произведением **обучаемых вентилей** $f_t \in (0, 1)^N$.
7. **Формально:** в LSTM $\partial c_t/\partial c_{t-1} = f_t$ (покомпонентно), и $\prod_k f_k$ может быть близко к 1 при $f_k \approx 1$. Сеть сама «настраивает» память, вместо того чтобы бороться с фиксированным $W_h$.

**Что дальше.** Глава 13 — LSTM: полный вывод прямого и обратного прохода. Ключевая формула — (12.19)–(12.20): вместо (12.1) будет **несколько** рекуррентных формул (для $f_t, i_t, o_t, c_t$), но структура «рекуррентное соотношение + сумма по путям» останется той же. В формуле для $\partial L/\partial c_t$ произведение $W_h^\top$ **исчезнет** — вместо него будет просто умножение на $f_t$. Именно это решает проблему (12.2) — и именно это делает LSTM рабочей архитектурой для длинных последовательностей.

# Глава 13. LSTM: ячейка долгой краткосрочной памяти

В главах 7–12 мы построили полную схему BPTT для RNN с произвольным числом нейронов, слоёв и направлений. И в главе 12 увидели, что у этой схемы есть **фундаментальный дефект**: вклад далёкого шага в градиент содержит произведение $W_h^{\,s-t}$, которое экспоненциально мало при $\|W_h\| < 1$ и экспоненциально велико при $\|W_h\| > 1$. Ни clipping, ни ортогональная инициализация не решают проблему — они лишь смягчают её. Нужно **изменить архитектуру** так, чтобы произведение $W_h^{\,s-t}$ **исчезло** из градиента.

LSTM (Long Short-Term Memory, Hochreiter & Schmidhuber, 1997) — первая архитектура, которая это делает. Ключевая идея: добавить **ячейку памяти** $c_t$, которая обновляется **аддитивно**:

$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t,
$$

где $f_t$ — «вентиль забывания» (forget gate), $i_t$ — «входной вентиль» (input gate), $\tilde c_t$ — кандидат. Производная $c_t$ по $c_{t-1}$ — **не матрица** $W_h$, а **вектор** $f_t \in (0,1)^N$:

$$
\frac{\partial c_t}{\partial c_{t-1}} = \operatorname{diag}(f_t).
$$

Если сеть выучивает $f_t \approx 1$ на нужных компонентах, градиент **не затухает**: $\prod_k f_k \approx 1$. Проблема (12.2) заменяется управляемым произведением **обучаемых вентилей**. Именно это делает LSTM способной запоминать зависимости на сотни шагов.

## 13.1. Зачем нужна LSTM — краткое напоминание главы 12

В главе 12 мы получили явную формулу для $\delta^a_t$ в простой RNN:

$$
\delta^a_t = \sum_{s=t}^{T} \left(\prod_{k=t}^{s-1} D_k\,W_h^\top\right) D_s\,W_y^\top \delta^y_s, \qquad D_k = \operatorname{diag}(1 - h_k \odot h_k). \tag{13.1}
$$

Норма вклада от шага $s$:

$$
\|\text{вклад от } s\| \lesssim \bigl(\|W_h\|_2 \cdot \bar D\bigr)^{s-t}, \qquad \bar D \leq 1. \tag{13.2}
$$

При $\|W_h\|_2 \cdot \bar D < 1$ — **затухание**, при $> 1$ — **взрыв**, при $= 1$ — критический режим.

**Что нужно изменить.** В (13.1) есть матрица $W_h^\top$ на каждом шаге. Чтобы градиент не затухал, нужно, чтобы **доминирующий путь** градиента шёл через **диагональный** множитель, который сеть может настраивать. LSTM делает это через ячейку $c_t$: путь $c_t \to c_{t+1} \to c_{t+2} \to \cdots$ имеет производную $\prod_k f_k$, и если $f_k \approx 1$, произведение тоже $\approx 1$.

**Сравнение с простой RNN:**

| | Простая RNN (гл. 8) | LSTM (эта глава) |
|---|---|---|
| Состояние | $h_t$ | $h_t$ и $c_t$ |
| Обновление памяти | $h_t = \tanh(W_h h_{t-1} + \dots)$ | $c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t$ |
| Градиент через память | $W_h^\top$ на каждом шаге | $f_t$ (поэлементно) на каждом шаге |
| Управление памятью | нет | вентили $f_t, i_t, o_t$ |

## 13.2. Архитектура и обозначения

Один слой LSTM, $T$ шагов. Вход $x_t \in \mathbb{R}^D$, скрытое состояние $h_t \in \mathbb{R}^N$, ячейка памяти $c_t \in \mathbb{R}^N$, выход $K$-мерный.

| Символ | Размерность | Смысл |
|---|---|---|
| $x_t$ | $\mathbb{R}^D$ | вход на шаге $t$ |
| $a^f_t, a^i_t, a^o_t$ | $\mathbb{R}^N$ | преактивации вентилей |
| $a^g_t$ | $\mathbb{R}^N$ | преактивация кандидата $\tilde c_t$ |
| $f_t, i_t, o_t$ | $\mathbb{R}^N$ | вентили: forget, input, output |
| $\tilde c_t$ | $\mathbb{R}^N$ | кандидат для ячейки |
| $c_t$ | $\mathbb{R}^N$ | ячейка памяти |
| $h_t$ | $\mathbb{R}^N$ | скрытое состояние |
| $z_t$ | $\mathbb{R}^K$ | преактивация выхода |
| $\hat y_t, y_t$ | $\mathbb{R}^K$ | предсказание, ответ |
| $W_f, W_i, W_o, W_g$ | $\mathbb{R}^{N \times N}$ | веса «$h_{t-1} \to$ вентиль/кандидат» |
| $U_f, U_i, U_o, U_g$ | $\mathbb{R}^{N \times D}$ | веса «$x_t \to$ вентиль/кандидат» |
| $b_f, b_i, b_o, b_g$ | $\mathbb{R}^N$ | смещения вентилей |
| $W_y$ | $\mathbb{R}^{K \times N}$ | веса выхода |
| $b_y$ | $\mathbb{R}^K$ | смещение выхода |
| $h_0 = \mathbf{0}$, $c_0 = \mathbf{0}$ | | начальные состояния |
| $\odot$ | | поэлементное умножение (Адамар) |

**Уравнения прямого прохода** (для $t = 1, \dots, T$):

$$
\boxed{
\begin{aligned}
a^f_t &= W_f\,h_{t-1} + U_f\,x_t + b_f, & f_t &= \sigma(a^f_t) &&\in \mathbb{R}^N, \\
a^i_t &= W_i\,h_{t-1} + U_i\,x_t + b_i, & i_t &= \sigma(a^i_t) &&\in \mathbb{R}^N, \\
a^o_t &= W_o\,h_{t-1} + U_o\,x_t + b_o, & o_t &= \sigma(a^o_t) &&\in \mathbb{R}^N, \\
a^g_t &= W_g\,h_{t-1} + U_g\,x_t + b_g, & \tilde c_t &= \tanh(a^g_t) &&\in \mathbb{R}^N, \\[4pt]
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde c_t &&\in \mathbb{R}^N, \\
h_t &= o_t \odot \tanh(c_t) &&\in \mathbb{R}^N, \\
z_t &= W_y\,h_t + b_y &&\in \mathbb{R}^K, \\
\hat y_t &= \sigma(z_t) &&\in \mathbb{R}^K.
\end{aligned}}
\tag{13.3}
$$

**Покомпонентная запись обновления ячейки** (чтобы видеть, откуда «аддитивность»):

$$
c_{t,i} = f_{t,i}\,c_{t-1,i} + i_{t,i}\,\tilde c_{t,i}, \qquad i = 1, \dots, N. \tag{13.3'}
$$

**Покомпонентная запись скрытого состояния**:

$$
h_{t,i} = o_{t,i}\,\tanh(c_{t,i}). \tag{13.3''}
$$

**Число обучаемых параметров:**

$$
\underbrace{4(N^2 + ND + N)}_{\text{4 вентиля/кандидат}} + \underbrace{KN + K}_{W_y, b_y} = 4N^2 + 4ND + 4N + KN + K. \tag{13.4}
$$

При $N=1$, $D=1$, $K=1$ — $4 + 4 + 4 + 1 + 1 = 14$. Простая RNN с теми же $N, D, K$ имела $1 + 1 + 1 + 1 + 1 = 5$ параметров. LSTM почти в 3 раза больше — за счёт трёх дополнительных вентилей.

**Замечание о нотации.** В литературе встречаются две эквивалентные записи: $(W, U)$ (наш выбор) и $(W_h, W_x)$. Первая короче и подчёркивает, что все четыре вентиля устроены одинаково; вторая сохраняет аналогию с главой 8. Мы используем $(W, U)$.

**Замечание о $\tilde c_t$.** Иногда обозначают $g_t$ (от «gate state»). Это **не вентиль**, а **кандидат** — новое значение, которое ячейка может «принять». Все три вентиля — $f, i, o$ — управляют тем, **что делать** с этим кандидатом.

На рис. 13.1 — архитектура одного шага LSTM: три вентиля (сигмоиды), кандидат (tanh), ячейка $c_t$ (аддитивное обновление), скрытое состояние $h_t = o_t \odot \tanh(c_t)$.

```python
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch

def draw_lstm_cell():
    """Рис. 13.1: одна ячейка LSTM на шаге t."""
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_xlim(0, 14); ax.set_ylim(0, 8)
    ax.axis('off'); ax.set_aspect('equal')

    def node(cx, cy, label, color, r=0.35):
        ax.add_patch(Circle((cx, cy), r, facecolor=color, edgecolor='black', lw=1.4, zorder=3))
        ax.text(cx, cy, label, ha='center', va='center', fontsize=10, zorder=4)

    def box(cx, cy, label, w=1.2, h=0.5, color='#E8E8E8'):
        ax.add_patch(FancyBboxPatch((cx-w/2, cy-h/2), w, h,
                                     boxstyle="round,pad=0.05",
                                     facecolor=color, edgecolor='black', lw=1.4, zorder=3))
        ax.text(cx, cy, label, ha='center', va='center', fontsize=10, zorder=4)

    def arrow(p1, p2, color='black', ls='-', lw=1.4):
        ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle='-|>', mutation_scale=12,
                                     lw=lw, color=color, linestyle=ls,
                                     shrinkA=0, shrinkB=0, zorder=2))

    # вход
    ax.text(0.5, 4, r'$x_t$', fontsize=12)
    ax.text(0.5, 6.5, r'$h_{t-1}$', fontsize=12)
    ax.text(0.5, 1.5, r'$c_{t-1}$', fontsize=12)

    # вентили
    box(3.5, 6.5, r'$\sigma$', color='#FFE29A')     # forget
    box(3.5, 4.0, r'$\sigma$', color='#FFE29A')     # input
    box(3.5, 2.0, r'$\sigma$', color='#FFE29A')     # output
    box(3.5, 5.3, r'$\tanh$', color='#F5C6A5')     # candidate

    ax.text(5.0, 6.5, r'$f_t$', fontsize=11)
    ax.text(5.0, 5.3, r'$\tilde c_t$', fontsize=11)
    ax.text(5.0, 4.0, r'$i_t$', fontsize=11)
    ax.text(5.0, 2.0, r'$o_t$', fontsize=11)

    # операции
    node(7.0, 5.3, r'$\times$', '#FFFFFF')
    node(9.0, 3.0, r'$+$', '#FFFFFF')
    node(7.0, 1.5, r'$\times$', '#FFFFFF')
    node(9.5, 4.5, r'$\tanh$', '#F5C6A5')
    node(9.0, 1.5, r'$\times$', '#FFFFFF')

    # стрелки входов к вентилям
    for y in [6.5, 5.3, 4.0, 2.0]:
        arrow((1.0, 6.5), (2.9, y))
    for y in [6.5, 5.3, 4.0, 2.0]:
        arrow((1.0, 4.0), (2.9, y))
    # на самом деле x и h идут к каждому вентилю - нарисуем проще
    ax.text(2.0, 3.2, '...', fontsize=10)

    # вентили к операциям
    arrow((4.1, 6.5), (6.65, 5.3))       # f_t к первой ×
    arrow((4.1, 5.3), (6.65, 5.3))       # c~ к первой ×
    arrow((4.1, 4.0), (8.65, 3.0))       # i_t ко +
    arrow((7.35, 5.3), (8.65, 3.0))       # (f×c~) ко +

    # c_{t-1} к первой ×
    arrow((1.0, 1.5), (6.65, 5.3))
    # + к c_t
    node(10.5, 3.0, r'$c_t$', '#B8E0B8')
    arrow((9.35, 3.0), (10.15, 3.0))

    # tanh от c_t
    arrow((10.5, 3.35), (9.5, 4.15))
    # o_t × tanh(c_t)
    arrow((4.1, 2.0), (8.65, 1.5))
    arrow((9.5, 4.85), (6.65, 1.5))       # tanh(c_t) ко второй ×

    # h_t
    node(10.5, 1.5, r'$h_t$', '#B8E0B8')
    arrow((9.35, 1.5), (10.15, 1.5))

    ax.text(7.0, 7.5, 'Рис. 13.1. Одна ячейка LSTM на шаге $t$',
            ha='center', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

draw_lstm_cell()
```

**Интуиция вентилей:**

- **$f_t$ (forget)** — «сколько $c_{t-1}$ сохранить». $f_t \approx 1$ — память сохраняется, $f_t \approx 0$ — память стирается.
- **$i_t$ (input)** — «сколько нового кандидата $\tilde c_t$ принять в ячейку».
- **$o_t$ (output)** — «какую часть $\tanh(c_t)$ выдать в $h_t$».

Именно **$f_t$** — ключевой для решения проблемы затухания: если сеть выучивает $f_t \approx 1$ на нужных компонентах, память сохраняется, и градиент через $c_t$ не затухает.

## 13.3. Прямой проход

Применяем (13.3) для $t = 1, \dots, T$. **Порядок важен:** сначала вентили, потом кандидат, потом ячейка, потом скрытое состояние, потом выход.

### Шаг 1. Вентили

Три сигмоиды — одинаковые по форме, разные по весам:

$$
f_t = \sigma\bigl(W_f h_{t-1} + U_f x_t + b_f\bigr), \quad i_t = \sigma\bigl(W_i h_{t-1} + U_i x_t + b_i\bigr), \quad o_t = \sigma\bigl(W_o h_{t-1} + U_o x_t + b_o\bigr). \tag{13.5}
$$

Каждый вентиль $\in (0,1)^N$ — «мягкий переключатель»: $1$ — открыт, $0$ — закрыт, промежуточные — частично.

### Шаг 2. Кандидат

$$
\tilde c_t = \tanh\bigl(W_g h_{t-1} + U_g x_t + b_g\bigr) \in (-1, 1)^N. \tag{13.6}
$$

Это **возможное новое значение** ячейки. В отличие от $h_t$ в простой RNN, $\tilde c_t$ **не становится** ячейкой сразу — сначала проходит через вентиль $i_t$.

### Шаг 3. Обновление ячейки памяти

$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t. \tag{13.7}
$$

**Ключевая формула.** В **покомпонентном виде:**

$$
c_{t,i} = \underbrace{f_{t,i}\,c_{t-1,i}}_{\text{сохранить старое}} + \underbrace{i_{t,i}\,\tilde c_{t,i}}_{\text{добавить новое}}.
$$

Каждая компонента ячейки **аддитивно** получает вклад: часть старого (взвешенная $f_t$) плюс часть нового (взвешенная $i_t$). Именно **аддитивность** (а не умножение на матрицу $W_h$) — ключ к решению проблемы затухания: производная $\partial c_t/\partial c_{t-1}$ — диагональная матрица $\operatorname{diag}(f_t)$, а не $W_h$.

### Шаг 4. Скрытое состояние и выход

$$
h_t = o_t \odot \tanh(c_t), \qquad z_t = W_y h_t + b_y, \qquad \hat y_t = \sigma(z_t). \tag{13.8}
$$

Вентиль $o_t$ управляет тем, **сколько ячейки выдать наружу**. Это важно: ячейка может хранить информацию, не «выплёвывая» её в выход — что позволяет сети копить контекст, не портя текущее предсказание.

### Подстановка чисел: $N=1$, $D=1$, $K=1$, $T=2$

Для простоты все параметры — скаляры:

$$
W_f = W_i = W_o = W_g = 0.5, \quad U_f = U_i = U_o = U_g = 0.5, \quad b_f = b_i = b_o = b_g = 0,
$$
$$
W_y = 0.5, \quad b_y = 0.
$$

Входы $x_1 = 1.0$, $x_2 = 1.0$. Ответы $y_1 = 0.0$, $y_2 = 1.0$. Начальные $h_0 = 0$, $c_0 = 0$.

**Шаг $t = 1$:**

$$
a^f_1 = 0.5 \cdot 0 + 0.5 \cdot 1.0 + 0 = 0.5, \qquad f_1 = \sigma(0.5) \approx 0.62246,
$$
$$
a^i_1 = 0.5, \qquad i_1 \approx 0.62246,
$$
$$
a^o_1 = 0.5, \qquad o_1 \approx 0.62246,
$$
$$
a^g_1 = 0.5, \qquad \tilde c_1 = \tanh(0.5) \approx 0.46212.
$$

**Ячейка:**

$$
c_1 = f_1 \cdot c_0 + i_1 \cdot \tilde c_1 = 0.62246 \cdot 0 + 0.62246 \cdot 0.46212 \approx 0.28769.
$$

**Скрытое состояние:**

$$
\tanh(c_1) = \tanh(0.28769) \approx 0.28011, \qquad h_1 = o_1 \cdot \tanh(c_1) \approx 0.62246 \cdot 0.28011 \approx 0.17434.
$$

**Выход:**

$$
z_1 = 0.5 \cdot 0.17434 + 0 \approx 0.08717, \qquad \hat y_1 = \sigma(0.08717) \approx 0.52178.
$$

**Шаг $t = 2$:**

$$
a^f_2 = 0.5 \cdot 0.17434 + 0.5 \cdot 1.0 + 0 \approx 0.58717, \qquad f_2 \approx 0.64277,
$$
$$
a^i_2 = 0.58717, \qquad i_2 \approx 0.64277,
$$
$$
a^o_2 = 0.58717, \qquad o_2 \approx 0.64277,
$$
$$
a^g_2 = 0.58717, \qquad \tilde c_2 = \tanh(0.58717) \approx 0.52786.
$$

**Ячейка** — здесь впервые $c_1 \neq 0$, и $f_2$ **умножается** на него:

$$
c_2 = f_2 \cdot c_1 + i_2 \cdot \tilde c_2 = 0.64277 \cdot 0.28769 + 0.64277 \cdot 0.52786 \approx 0.18492 + 0.33934 \approx 0.52426.
$$

**Скрытое состояние:**

$$
\tanh(c_2) = \tanh(0.52426) \approx 0.48089, \qquad h_2 = o_2 \cdot \tanh(c_2) \approx 0.64277 \cdot 0.48089 \approx 0.30911.
$$

**Выход:**

$$
z_2 = 0.5 \cdot 0.30911 \approx 0.15456, \qquad \hat y_2 = \sigma(0.15456) \approx 0.53856.
$$

### Таблица локальных производных

Все «кирпичи» для одного шага. **Проверьте на размерности** — это лучший способ убедиться в правильности.

**Производные преактиваций вентилей** (одинаковая структура для всех четырёх):

| Величина | Форма | Компонентно |
|---|---|---|
| $\partial a^f_t/\partial h_{t-1}$ | $N \times N$ | $(W_f)_{ij}$ |
| $\partial a^f_t/\partial x_t$ | $N \times D$ | $(U_f)_{ij}$ |
| $\partial a^f_t/\partial b_f$ | $N \times N$ | $\delta_{ij}$ |
| аналогично для $i, o, g$ | | |

**Производные активаций:**

$$
\frac{\partial f_t}{\partial a^f_t} = \operatorname{diag}(f_t \odot (1 - f_t)), \quad \frac{\partial i_t}{\partial a^i_t} = \operatorname{diag}(i_t \odot (1 - i_t)), \tag{13.9}
$$
$$
\frac{\partial o_t}{\partial a^o_t} = \operatorname{diag}(o_t \odot (1 - o_t)), \quad \frac{\partial \tilde c_t}{\partial a^g_t} = \operatorname{diag}(1 - \tilde c_t \odot \tilde c_t). \tag{13.10}
$$

**Производные ячейки $c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t$:**

| Величина | Форма | Компонентно |
|---|---|---|
| $\partial c_t/\partial f_t$ | $\mathbb{R}^{N \times N}$ | $\delta_{ij}\,c_{t-1,j}$ |
| $\partial c_t/\partial i_t$ | $\mathbb{R}^{N \times N}$ | $\delta_{ij}\,\tilde c_{t,j}$ |
| $\partial c_t/\partial \tilde c_t$ | $\mathbb{R}^{N \times N}$ | $\delta_{ij}\,i_{t,j}$ |
| $\partial c_t/\partial c_{t-1}$ | $\mathbb{R}^{N \times N}$ | $\delta_{ij}\,f_{t,j}$ |

**Ключевая строка:** $\partial c_t/\partial c_{t-1} = \operatorname{diag}(f_t)$. **Диагональная**, не $W_h$. Это и есть решение проблемы затухания.

**Производные скрытого состояния $h_t = o_t \odot \tanh(c_t)$:**

| Величина | Форма | Компонентно |
|---|---|---|
| $\partial h_t/\partial o_t$ | $\mathbb{R}^{N \times N}$ | $\delta_{ij}\,\tanh(c_{t,j})$ |
| $\partial h_t/\partial c_t$ | $\mathbb{R}^{N \times N}$ | $\delta_{ij}\,o_{t,j}\,(1 - \tanh^2(c_{t,j}))$ |

**Выход:** $\partial z_t/\partial h_t = W_y$, $\partial \hat y_t/\partial z_t = \operatorname{diag}(\hat y_t \odot (1 - \hat y_t))$ — как в главе 9.

## 13.4. Функция потерь

Как в главе 9:

$$
L = \sum_{t=1}^{T} \frac{1}{2}\,\|\hat y_t - y_t\|^2, \qquad \frac{\partial L}{\partial \hat y_t} = \hat y_t - y_t. \tag{13.11}
$$

## 13.5. Обратный проход

### 13.5.1. Что нового по сравнению с простой RNN

В простой RNN (глава 8) было **одно** скрытое состояние $h_t$ и **одна** дельта $\delta^a_t = \partial L/\partial a_t$. В LSTM **два** состояния — $h_t$ и $c_t$ — и **две** дельты:

$$
\delta^h_t := \frac{\partial L}{\partial h_t} \in \mathbb{R}^N, \qquad \delta^c_t := \frac{\partial L}{\partial c_t} \in \mathbb{R}^N. \tag{13.12}
$$

Кроме того, на каждом шаге есть **четыре** преактивации — $a^f_t, a^i_t, a^o_t, a^g_t$ — и **четыре** дельты:

$$
\delta^{af}_t := \frac{\partial L}{\partial a^f_t}, \quad \delta^{ai}_t := \frac{\partial L}{\partial a^i_t}, \quad \delta^{ao}_t := \frac{\partial L}{\partial a^o_t}, \quad \delta^{ag}_t := \frac{\partial L}{\partial a^g_t}. \tag{13.13}
$$

**Итого шесть дельт на шаг.** Каждая — вектор $\in \mathbb{R}^N$.

**Кто чьи потребители:**

| Состояние | Потребители | Следствие |
|---|---|---|
| $c_t$ | $h_t$ (через $\tanh$) и $c_{t+1}$ (через $f_{t+1}$) | $\delta^c_t$ зависит от $\delta^h_t$ **и** $\delta^c_{t+1}$ |
| $h_t$ | $z_t$ (выход) и $a^f_{t+1}, a^i_{t+1}, a^o_{t+1}, a^g_{t+1}$ (вентили шага $t+1$) | $\delta^h_t$ зависит от $\delta^y_t$ **и** от четырёх дельт шага $t+1$ |
| $f_t$ | $c_t$ | $\delta^{af}_t = (\delta^c_t \odot c_{t-1}) \odot f_t(1-f_t)$ |
| $i_t$ | $c_t$ | $\delta^{ai}_t = (\delta^c_t \odot \tilde c_t) \odot i_t(1-i_t)$ |
| $\tilde c_t$ | $c_t$ | $\delta^{ag}_t = (\delta^c_t \odot i_t) \odot (1-\tilde c_t^2)$ |
| $o_t$ | $h_t$ | $\delta^{ao}_t = (\delta^h_t \odot \tanh(c_t)) \odot o_t(1-o_t)$ |

### 13.5.2. Дельта выхода

Как в главе 9:

$$
\delta^y_t = (\hat y_t - y_t) \odot \hat y_t \odot (1 - \hat y_t) \in \mathbb{R}^K. \tag{13.14}
$$

### 13.5.3. Порядок вычислений на шаге $t$ (справа налево)

На каждом шаге $t$ (начиная с $t = T$) выполняем **в этом порядке**:

**Шаг A. $\delta^h_t$ — скрытое состояние.**

$h_t$ имеет **пять** потребителей: $z_t$ (выход) и четыре вентиля шага $t+1$ ($a^f_{t+1}, a^i_{t+1}, a^o_{t+1}, a^g_{t+1}$). При $t = T$ остаётся только $z_t$.

$$
\boxed{\;\delta^h_t = W_y^\top \delta^y_t + W_f^\top \delta^{af}_{t+1} + W_i^\top \delta^{ai}_{t+1} + W_o^\top \delta^{ao}_{t+1} + W_g^\top \delta^{ag}_{t+1}.\;} \tag{13.15}
$$

При $t = T$: $\delta^h_T = W_y^\top \delta^y_T$. (Всё, что с индексом $t+1$, отсутствует.)

**Шаг B. $\delta^o_t$ — выходной вентиль.**

$o_t$ входит только в $h_t = o_t \odot \tanh(c_t)$:

$$
\boxed{\;\delta^{ao}_t = \Bigl(\delta^h_t \odot \tanh(c_t)\Bigr) \odot o_t \odot (1 - o_t) \in \mathbb{R}^N.\;} \tag{13.16}
$$

Здесь произведение $\delta^h_t \odot \tanh(c_t)$ — это $\partial L/\partial o_t$; умножение на $o_t(1-o_t)$ — переход через сигмоиду к преактивации $a^o_t$.

**Шаг C. $\delta^c_t$ — ячейка памяти. КЛЮЧЕВАЯ ФОРМУЛА.**

$c_t$ входит в два места:

1. в $h_t = o_t \odot \tanh(c_t)$ — свой шаг;
2. в $c_{t+1} = f_{t+1} \odot c_t + i_{t+1} \odot \tilde c_{t+1}$ — следующий шаг.

При $t = T$ второго нет. Значит:

$$
\boxed{\;\delta^c_t = \underbrace{\delta^h_t \odot o_t \odot \bigl(1 - \tanh^2(c_t)\bigr)}_{\text{вклад через } h_t} \;+\; \underbrace{f_{t+1} \odot \delta^c_{t+1}}_{\text{вклад через } c_{t+1}}.\;} \tag{13.17}
$$

**Это — главная формула главы.** Сравните с (12.1) — простой RNN:

| | Простая RNN (12.1) | LSTM (13.17) |
|---|---|---|
| Путь через память | $W_h^\top \delta^a_{t+1}$ (матрица × вектор) | $f_{t+1} \odot \delta^c_{t+1}$ (поэлементно) |
| Что даёт затухание | $\|W_h\|$ на каждом шаге | $f_{t+1} \in (0,1)$ на каждом шаге |
| Управление | фиксированное | обучаемое (сеть может выучить $f \approx 1$) |

**Именно поэтому LSTM решает проблему затухания.** Вклад далёкого шага $s$ в $\delta^c_t$ содержит произведение $f_{t+1} \odot f_{t+2} \odot \cdots \odot f_s$ — **поэлементное**, и каждая компонента $\in (0,1)$ **обучаема**. Сеть может выучить $f_i \approx 1$ на нужных компонентах — тогда градиент проходит без затухания.

**Шаг D. Дельты вентилей и кандидата (до применения $\sigma/\tanh$).**

$f_t$ входит в $c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t$; значит, $\partial c_t/\partial f_t = \operatorname{diag}(c_{t-1})$, и

$$
\frac{\partial L}{\partial f_t} = \delta^c_t \odot c_{t-1} \in \mathbb{R}^N. \tag{13.18}
$$

Переходим через сигмоиду:

$$
\boxed{\;\delta^{af}_t = \Bigl(\delta^c_t \odot c_{t-1}\Bigr) \odot f_t \odot (1 - f_t).\;} \tag{13.19}
$$

Аналогично для $i_t$ и $\tilde c_t$:

$$
\boxed{\;\delta^{ai}_t = \Bigl(\delta^c_t \odot \tilde c_t\Bigr) \odot i_t \odot (1 - i_t),\;} \tag{13.20}
$$
$$
\boxed{\;\delta^{ag}_t = \Bigl(\delta^c_t \odot i_t\Bigr) \odot \bigl(1 - \tilde c_t \odot \tilde c_t\bigr).\;} \tag{13.21}
$$

Для $o_t$ — это уже было в шаге B, формула (13.16).

### 13.5.4. Сводка обратного прохода

$$
\boxed{
\begin{aligned}
&\text{1. Для всех } t: \quad \delta^y_t = (\hat y_t - y_t) \odot \hat y_t \odot (1-\hat y_t); \\[4pt]
&\text{2. Для } t = T, T-1, \dots, 1: \\
&\quad \text{(a) } \delta^h_t = W_y^\top \delta^y_t + W_f^\top \delta^{af}_{t+1} + W_i^\top \delta^{ai}_{t+1} + W_o^\top \delta^{ao}_{t+1} + W_g^\top \delta^{ag}_{t+1}; \\
&\quad \text{(b) } \delta^{ao}_t = \bigl(\delta^h_t \odot \tanh(c_t)\bigr) \odot o_t \odot (1-o_t); \\
&\quad \text{(c) } \delta^c_t = \delta^h_t \odot o_t \odot (1 - \tanh^2(c_t)) + f_{t+1} \odot \delta^c_{t+1}; \\
&\quad \text{(d) } \delta^{af}_t = \bigl(\delta^c_t \odot c_{t-1}\bigr) \odot f_t \odot (1-f_t); \\
&\quad \text{(e) } \delta^{ai}_t = \bigl(\delta^c_t \odot \tilde c_t\bigr) \odot i_t \odot (1-i_t); \\
&\quad \text{(f) } \delta^{ag}_t = \bigl(\delta^c_t \odot i_t\bigr) \odot (1 - \tilde c_t^2); \\[4pt]
&\text{3. } \frac{\partial L}{\partial W_y} = \sum_t \delta^y_t\,h_t^\top, \quad \frac{\partial L}{\partial b_y} = \sum_t \delta^y_t; \\
&\text{4. Для каждого вентиля } \alpha \in \{f, i, o, g\}: \\
&\quad \frac{\partial L}{\partial W_\alpha} = \sum_t \delta^{a\alpha}_t\,h_{t-1}^\top, \quad \frac{\partial L}{\partial U_\alpha} = \sum_t \delta^{a\alpha}_t\,x_t^\top, \quad \frac{\partial L}{\partial b_\alpha} = \sum_t \delta^{a\alpha}_t.
\end{aligned}}
\tag{13.22}
$$

**Граничные условия:** $\delta^c_{T+1} = \mathbf{0}$, $\delta^{af}_{T+1} = \delta^{ai}_{T+1} = \delta^{ao}_{T+1} = \delta^{ag}_{T+1} = \mathbf{0}$, $c_0 = \mathbf{0}$, $h_0 = \mathbf{0}$.

### 13.5.5. Покомпонентная проверка ключевой формулы (13.17)

Разложим $\delta^c_t$ покомпонентно, чтобы убедиться, что нет потерянного множителя. Для компоненты $i$:

$$
(\delta^c_t)_i = \frac{\partial L}{\partial c_{t,i}} = \sum_j \frac{\partial L}{\partial h_{t,j}}\frac{\partial h_{t,j}}{\partial c_{t,i}} + \sum_j \frac{\partial L}{\partial c_{t+1,j}}\frac{\partial c_{t+1,j}}{\partial c_{t,i}}.
$$

Первый член: $\partial h_{t,j}/\partial c_{t,i} = \delta_{ji}\,o_{t,j}\,(1-\tanh^2(c_{t,j}))$, значит, первый член = $(\delta^h_t)_i\,o_{t,i}\,(1-\tanh^2(c_{t,i}))$.

Второй член: $\partial c_{t+1,j}/\partial c_{t,i} = \delta_{ji}\,f_{t+1,j}$, значит, второй член = $(\delta^c_{t+1})_i\,f_{t+1,i}$.

Итого: $(\delta^c_t)_i = (\delta^h_t)_i\,o_{t,i}\,(1-\tanh^2(c_{t,i})) + f_{t+1,i}\,(\delta^c_{t+1})_i$ — ровно (13.17) покомпонентно. $\checkmark$

## 13.6. Численный пример: $N=1$, $D=1$, $K=1$, $T=2$

Продолжим пример из §13.3. Все параметры — скаляры, $W_f = W_i = W_o = W_g = U_f = U_i = U_o = U_g = 0.5$, $b_f = b_i = b_o = b_g = 0$, $W_y = 0.5$, $b_y = 0$.

### Прямой проход (числа из §13.3)

$$
f_1 = i_1 = o_1 \approx 0.62246, \quad \tilde c_1 \approx 0.46212, \quad c_1 \approx 0.28769, \quad h_1 \approx 0.17434, \quad \hat y_1 \approx 0.52178.
$$
$$
f_2 = i_2 = o_2 \approx 0.64277, \quad \tilde c_2 \approx 0.52786, \quad c_2 \approx 0.52426, \quad h_2 \approx 0.30911, \quad \hat y_2 \approx 0.53856.
$$

### Потеря

$$
L = \tfrac12(0.52178)^2 + \tfrac12(0.53856 - 1)^2 \approx 0.13613 + 0.10650 = 0.24263.
$$

### Дельты выходов

$$
\delta^y_1 = 0.52178 \cdot 0.52178 \cdot (1 - 0.52178) \approx 0.13020,
$$
$$
\delta^y_2 = (0.53856 - 1) \cdot 0.53856 \cdot (1 - 0.53856) \approx -0.11466.
$$

### Шаг $t = 2$ (справа налево)

**$\delta^h_2$** (граница, только выход):

$$
\delta^h_2 = W_y \cdot \delta^y_2 = 0.5 \cdot (-0.11466) \approx -0.05733.
$$

**$\delta^{ao}_2$:**

$$
\tanh(c_2) \approx 0.48089, \qquad \delta^{ao}_2 = (\delta^h_2 \cdot \tanh(c_2)) \cdot o_2 \cdot (1-o_2) \approx (-0.05733 \cdot 0.48089) \cdot 0.64277 \cdot 0.35723 \approx -0.00633.
$$

**$\delta^c_2$** (граница, $\delta^c_3 = 0$):

$$
1 - \tanh^2(c_2) = 1 - 0.48089^2 \approx 0.76874,
$$
$$
\delta^c_2 = \delta^h_2 \cdot o_2 \cdot (1 - \tanh^2(c_2)) + f_3 \cdot \delta^c_3 = (-0.05733) \cdot 0.64277 \cdot 0.76874 + 0 \approx -0.02833.
$$

**$\delta^{af}_2, \delta^{ai}_2, \delta^{ag}_2$:**

$$
\delta^{af}_2 = (\delta^c_2 \cdot c_1) \cdot f_2 \cdot (1-f_2) = (-0.02833 \cdot 0.28769) \cdot 0.64277 \cdot 0.35723 \approx -0.00187,
$$
$$
\delta^{ai}_2 = (\delta^c_2 \cdot \tilde c_2) \cdot i_2 \cdot (1-i_2) = (-0.02833 \cdot 0.52786) \cdot 0.64277 \cdot 0.35723 \approx -0.00344,
$$
$$
\delta^{ag}_2 = (\delta^c_2 \cdot i_2) \cdot (1 - \tilde c_2^2) = (-0.02833 \cdot 0.64277) \cdot (1 - 0.52786^2) \approx -0.01317.
$$

### Шаг $t = 1$

**$\delta^h_1$** (пять потребителей, но только шаг 2 в будущем):

$$
\delta^h_1 = W_y \delta^y_1 + W_f \delta^{af}_2 + W_i \delta^{ai}_2 + W_o \delta^{ao}_2 + W_g \delta^{ag}_2
$$
$$
= 0.5 \cdot 0.13020 + 0.5 \cdot (-0.00187) + 0.5 \cdot (-0.00344) + 0.5 \cdot (-0.00633) + 0.5 \cdot (-0.01317)
$$
$$
\approx 0.06510 + 0.5 \cdot (-0.02481) \approx 0.06510 - 0.01241 \approx 0.05269.
$$

**$\delta^{ao}_1$:**

$$
\tanh(c_1) \approx 0.28011, \qquad \delta^{ao}_1 = (0.05269 \cdot 0.28011) \cdot 0.62246 \cdot 0.37754 \approx 0.00347.
$$

**$\delta^c_1$** (ключевая формула — здесь виден вклад пути через $c_2$):

$$
1 - \tanh^2(c_1) = 1 - 0.28011^2 \approx 0.92154,
$$
$$
\delta^c_1 = \delta^h_1 \cdot o_1 \cdot (1 - \tanh^2(c_1)) + f_2 \cdot \delta^c_2
$$
$$
= 0.05269 \cdot 0.62246 \cdot 0.92154 + 0.64277 \cdot (-0.02833) \approx 0.03023 - 0.01821 \approx 0.01202.
$$

Обратите внимание: **первый член — «свой»**, второй — путь через $c_2$. Второй член **больше по модулю** первого — «будущее» доминирует, потому что $f_2 \approx 0.64$ (не близко к нулю).

**$\delta^{af}_1$** — здесь $c_0 = 0$, поэтому вклад нулевой:

$$
\delta^{af}_1 = (\delta^c_1 \cdot c_0) \cdot f_1(1-f_1) = 0.
$$

Это важное наблюдение: **вентиль забывания на первом шаге не влияет на потерю**, потому что нечего забывать. Градиент по $W_f$ будет определяться только шагом $t = 2$.

**$\delta^{ai}_1, \delta^{ag}_1$:**

$$
\delta^{ai}_1 = (\delta^c_1 \cdot \tilde c_1) \cdot i_1(1-i_1) = (0.01202 \cdot 0.46212) \cdot 0.62246 \cdot 0.37754 \approx 0.00130,
$$
$$
\delta^{ag}_1 = (\delta^c_1 \cdot i_1) \cdot (1-\tilde c_1^2) = (0.01202 \cdot 0.62246) \cdot (1 - 0.46212^2) \approx 0.00578.
$$

### Градиенты

**Выход:**

$$
\frac{\partial L}{\partial W_y} = \delta^y_1 h_1 + \delta^y_2 h_2 = 0.13020 \cdot 0.17434 + (-0.11466) \cdot 0.30911 \approx 0.02270 - 0.03544 \approx -0.01274,
$$
$$
\frac{\partial L}{\partial b_y} = 0.13020 - 0.11466 = 0.01554.
$$

**Вентили** (все $W_\alpha, U_\alpha, b_\alpha$):

| Вентиль | $\partial L/\partial W_\alpha$ | $\partial L/\partial U_\alpha = \partial L/\partial b_\alpha$ |
|---|---|---|
| $f$ | $0 \cdot 0 + (-0.00187)\cdot 0.17434 \approx -0.000326$ | $0 + (-0.00187) = -0.00187$ |
| $i$ | $0.00130\cdot 0 + (-0.00344)\cdot 0.17434 \approx -0.000600$ | $0.00130 - 0.00344 = -0.00214$ |
| $o$ | $0.00347\cdot 0 + (-0.00633)\cdot 0.17434 \approx -0.00110$ | $0.00347 - 0.00633 = -0.00286$ |
| $g$ | $0.00578\cdot 0 + (-0.01317)\cdot 0.17434 \approx -0.00230$ | $0.00578 - 0.01317 = -0.00739$ |

(В $\partial L/\partial W_\alpha = \sum_t \delta^{a\alpha}_t h_{t-1}$ первое слагаемое $t=1$ равно нулю, потому что $h_0 = 0$.)

**Проверка на вентиле забывания.** $\partial L/\partial W_f$ **мал** ($-0.000326$), а $\partial L/\partial U_f$ и $\partial L/\partial b_f$ тоже малы — потому что $f$ влияет на потерю только через $c_2$, а $c_2$ даёт малый вклад в $\delta^h_1$ (из-за $W_f = 0.5$). Это **не** проблема: на длинных последовательностях $f$ становится критически важным, и градиенты по нему растут.

## 13.7. Почему LSTM решает проблему затухания

Вернёмся к главе 12. Там было: в простой RNN вклад шага $s$ в $\delta^a_t$ содержит $\prod_{k=t}^{s-1} W_h^\top$, и норма этого произведения ~ $\|W_h\|^{s-t}$ — экспоненциально.

В LSTM вклад шага $s$ в $\delta^c_t$ содержит:

$$
\prod_{k=t+1}^{s} f_k \quad \text{(покомпонентно)}.
$$

**Ключевые свойства:**

1. **$f_k \in (0,1)^N$ — вектор, а не матрица.** Произведение — **поэлементное**. Каждая компонента памяти имеет **свой** вентиль, и затухание по одной компоненте не влияет на другие.
2. **$f_k$ — обучаемый.** Сеть может выучить $f_k \approx 1$ на компонентах, которые нужно помнить, и $f_k \approx 0$ на тех, что нужно стирать.
3. **Нет матричного умножения $W_h^\top$ на каждом шаге.** В $\delta^c_t$ матрицы $W_f, W_i, W_o, W_g$ появляются только при переходе к **преактивациям** (через $\delta^{a\alpha}_t$), а не в «магистральном пути» $c_t \to c_{t+1}$.

**Следствие.** Если сеть выучивает $f_k = 1$ на всех шагах, $\prod f_k = 1$ — градиент **не затухает**. Память сохраняется произвольно долго. Это и есть «constant error carousel» (CEC) — идея оригинальной статьи Hochreiter & Schmidhuber 1997 года.

**Сравнение путей градиента:**

| Путь | Простая RNN | LSTM |
|---|---|---|
| «Магистральный» (через память) | $W_h^\top$ на каждом шаге | $f_{t+1}$ на каждом шаге |
| Взрыв | возможно ($\|W_h\| > 1$) | **невозможен** ($f < 1$) |
| Затухание | $\|W_h\| \cdot \bar D < 1$ | $f_k$ — обучаемо |
| Другие пути | все через $W_h$ | через вентили $\sigma/\tanh$ (затухают, но **дополнительно** к магистральному) |

**Важно понимать:** LSTM **не устраняет** затухание полностью. Путь через $h_t \to$ вентили шага $t+1$ → $\delta^{a\alpha}$ → ... всё равно содержит $\sigma' \leq 1/4$ на каждом шаге и затухает. Но **магистральный** путь через $c_t$ **не затухает** — а именно он несёт долгую память. LSTM даёт сети **выбор**: использовать затухающий путь для коротких зависимостей, а незатухающий — для длинных.

## 13.8. Полная программа

```python
import numpy as np

def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))

class LSTMCell:
    def __init__(self, N, D, K, seed=0, eta=0.1):
        rng = np.random.default_rng(seed)
        self.N, self.D, self.K = N, D, K
        self.eta = eta
        # веса «h_{t-1} -> вентиль»
        self.Wf = rng.normal(scale=0.5, size=(N, N))
        self.Wi = rng.normal(scale=0.5, size=(N, N))
        self.Wo = rng.normal(scale=0.5, size=(N, N))
        self.Wg = rng.normal(scale=0.5, size=(N, N))
        # веса «x_t -> вентиль»
        self.Uf = rng.normal(scale=0.5, size=(N, D))
        self.Ui = rng.normal(scale=0.5, size=(N, D))
        self.Uo = rng.normal(scale=0.5, size=(N, D))
        self.Ug = rng.normal(scale=0.5, size=(N, D))
        # смещения
        self.bf = np.zeros(N); self.bi = np.zeros(N)
        self.bo = np.zeros(N); self.bg = np.zeros(N)
        # выход
        self.Wy = rng.normal(scale=0.5, size=(K, N))
        self.by = np.zeros(K)

    def forward(self, xs):
        T = len(xs)
        h = np.zeros(self.N); c = np.zeros(self.N)
        # кэш для backward
        cache = dict(h=[], c=[], f=[], i=[], o=[], g=[], tanh_c=[], x=[])
        hs, ys = [], []
        for t in range(T):
            af = self.Wf @ h + self.Uf @ xs[t] + self.bf
            ai = self.Wi @ h + self.Ui @ xs[t] + self.bi
            ao = self.Wo @ h + self.Uo @ xs[t] + self.bo
            ag = self.Wg @ h + self.Ug @ xs[t] + self.bg
            f, i, o, g = sigmoid(af), sigmoid(ai), sigmoid(ao), np.tanh(ag)
            c = f * c + i * g
            tc = np.tanh(c)
            h = o * tc
            z = self.Wy @ h + self.by
            ys.append(sigmoid(z))
            # кэш
            cache['h'].append(h.copy()); cache['c'].append(c.copy())
            cache['f'].append(f); cache['i'].append(i); cache['o'].append(o)
            cache['g'].append(g); cache['tanh_c'].append(tc)
            cache['x'].append(xs[t])
            hs.append(h.copy())
        return np.array(hs), np.array(ys), cache

    def loss(self, ys, targets):
        return 0.5 * np.sum((ys - targets)**2)

    def backward(self, ys, targets, cache):
        T = len(ys)
        dy = (ys - targets) * ys * (1 - ys)
        dh = np.zeros((T, self.N))
        dc = np.zeros((T, self.N))
        daf = np.zeros((T, self.N)); dai = np.zeros((T, self.N))
        dao = np.zeros((T, self.N)); dag = np.zeros((T, self.N))
        # обратный проход по времени
        dh_next = np.zeros(self.N); dc_next = np.zeros(self.N)
        daf_next = np.zeros(self.N); dai_next = np.zeros(self.N)
        dao_next = np.zeros(self.N); dag_next = np.zeros(self.N)
        for t in reversed(range(T)):
            # (a) δ^h_t
            dh[t] = self.Wy.T @ dy[t] + self.Wf.T @ daf_next + self.Wi.T @ dai_next \
                    + self.Wo.T @ dao_next + self.Wg.T @ dag_next
            # (b) δ^ao_t
            dao[t] = (dh[t] * cache['tanh_c'][t]) * cache['o'][t] * (1 - cache['o'][t])
            # (c) δ^c_t — ключевая формула
            c_prev = cache['c'][t-1] if t > 0 else np.zeros(self.N)
            f_next = cache['f'][t+1] if t+1 < T else np.zeros(self.N)
            dc[t] = dh[t] * cache['o'][t] * (1 - cache['tanh_c'][t]**2) + f_next * dc_next
            # (d,e,f) δ^af, δ^ai, δ^ag
            daf[t] = (dc[t] * c_prev) * cache['f'][t] * (1 - cache['f'][t])
            dai[t] = (dc[t] * cache['g'][t]) * cache['i'][t] * (1 - cache['i'][t])
            dag[t] = (dc[t] * cache['i'][t]) * (1 - cache['g'][t]**2)
            # подготовка следующей итерации
            daf_next, dai_next, dao_next, dag_next = daf[t], dai[t], dao[t], dag[t]
            dc_next = dc[t]
        # градиенты
        h_prev = np.vstack([np.zeros(self.N), np.array(cache['h'][:-1])])
        xs = np.array(cache['x'])
        gWf = sum(np.outer(daf[t], h_prev[t]) for t in range(T))
        gWi = sum(np.outer(dai[t], h_prev[t]) for t in range(T))
        gWo = sum(np.outer(dao[t], h_prev[t]) for t in range(T))
        gWg = sum(np.outer(dag[t], h_prev[t]) for t in range(T))
        gUf = sum(np.outer(daf[t], xs[t]) for t in range(T))
        gUi = sum(np.outer(dai[t], xs[t]) for t in range(T))
        gUo = sum(np.outer(dao[t], xs[t]) for t in range(T))
        gUg = sum(np.outer(dag[t], xs[t]) for t in range(T))
        gbf = daf.sum(axis=0); gbi = dai.sum(axis=0)
        gbo = dao.sum(axis=0); gbg = dag.sum(axis=0)
        gWy = sum(np.outer(dy[t], cache['h'][t]) for t in range(T))
        gby = dy.sum(axis=0)
        return dict(Wf=gWf, Wi=gWi, Wo=gWo, Wg=gWg,
                    Uf=gUf, Ui=gUi, Uo=gUo, Ug=gUg,
                    bf=gbf, bi=gbi, bo=gbo, bg=gbg, Wy=gWy, by=gby)

    def step(self, xs, targets):
        hs, ys, cache = self.forward(xs)
        L = self.loss(ys, targets)
        g = self.backward(ys, targets, cache)
        for k, v in g.items():
            setattr(self, k, getattr(self, k) - self.eta * v)
        return L, ys


# Проверка на примере §13.6
net = LSTMCell(N=1, D=1, K=1, eta=0.1)
net.Wf = np.array([[0.5]]); net.Wi = np.array([[0.5]])
net.Wo = np.array([[0.5]]); net.Wg = np.array([[0.5]])
net.Uf = np.array([[0.5]]); net.Ui = np.array([[0.5]])
net.Uo = np.array([[0.5]]); net.Ug = np.array([[0.5]])
net.bf = np.array([0.0]); net.bi = np.array([0.0])
net.bo = np.array([0.0]); net.bg = np.array([0.0])
net.Wy = np.array([[0.5]]); net.by = np.array([0.0])

xs = [np.array([1.0]), np.array([1.0])]
ys = np.array([[0.0], [1.0]])
L, yhat = net.step(xs, ys)
print("L =", L)              # ~0.2426
print("y^ =", yhat.ravel())  # [0.52178, 0.53856]
```

Вывод совпадает с §13.6. Если положить $f_t = 1$ (не обучать), $i_t = 1$, $o_t = 1$, LSTM вырождается в «кумулятивную сумму» $c_t = c_{t-1} + \tilde c_t$ — и проблема затухания исчезает полностью. Именно эту идею и реализует вентиль забывания, когда сеть выучивает $f \approx 1$.

## 13.9. Что мы получили

1. **Архитектура LSTM** (13.3) — четыре «преактивации» ($a^f, a^i, a^o, a^g$), три вентиля ($f, i, o$), кандидат $\tilde c_t$, ячейка $c_t$, скрытое состояние $h_t = o_t \odot \tanh(c_t)$.
2. **Ключевая идея** — **аддитивное обновление** ячейки: $c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t$. Производная $\partial c_t/\partial c_{t-1} = \operatorname{diag}(f_t)$ — **диагональная**, а не $W_h$.
3. **Шесть дельт на шаг**: $\delta^y, \delta^h, \delta^c, \delta^{af}, \delta^{ai}, \delta^{ao}, \delta^{ag}$ (последних четырёх — четыре; всего шесть **разных**).
4. **Порядок обратного прохода на шаге $t$:**
   - $\delta^h_t$ — из пяти потребителей (выход + четыре вентиля шага $t+1$);
   - $\delta^{ao}_t$ — из $\delta^h_t$ и $\tanh(c_t)$;
   - **$\delta^c_t$ — ключевая формула (13.17): $\delta^c_t = \delta^h_t \odot o_t \odot (1-\tanh^2(c_t)) + f_{t+1} \odot \delta^c_{t+1}$**;
   - $\delta^{af}_t, \delta^{ai}_t, \delta^{ag}_t$ — из $\delta^c_t$, $c_{t-1}$, $\tilde c_t$, $i_t$ и производных $\sigma/\tanh$.
5. **Градиенты параметров** — стандартные суммы по $t$: $\partial L/\partial W_\alpha = \sum_t \delta^{a\alpha}_t h_{t-1}^\top$ и аналогично для $U_\alpha, b_\alpha$.
6. **Решение проблемы затухания** — вклад шага $s$ в $\delta^c_t$ содержит $\prod_{k=t+1}^s f_k$ (**поэлементно**, а не $\prod W_h^\top$). Сеть может выучить $f \approx 1$ — тогда градиент не затухает. Взрыв **невозможен**: $f < 1$.
7. **Проверка при $T = 2$, $N = 1$** — все формулы дают совпадающие числа при ручном и программном расчёте; код в §13.8 воспроизводит §13.6.

**Что дальше.** Глава 14 — **GRU** (Gated Recurrent Unit): упрощение LSTM, в котором **два** вентиля вместо трёх и **нет отдельной ячейки** $c_t$ — её роль играет $h_t$. Формулы обратного прохода структурно те же, что (13.15)–(13.21), но короче: четыре дельты вместо шести, и ключевая формула $\partial h_t/\partial h_{t-1}$ содержит **вентиль обновления** $z_t$ вместо $f_t$ — с тем же эффектом решения проблемы затухания.

# Глава 14. GRU: вентильная рекуррентная единица

LSTM (глава 13) решает проблему затухания, но ценой усложнения: три вентиля, отдельная ячейка $c_t$, шесть дельт на шаг. GRU (Gated Recurrent Unit, Cho et al., 2014) — **упрощение**: два вентиля вместо трёх, **нет отдельной ячейки** $c_t$ (её роль играет $h_t$), четыре дельты вместо шести. Ключевая идея та же, что в LSTM: **аддитивное обновление** состояния, в котором «магистральный путь» градиента идёт через **вентиль**, а не через матрицу $W_h$. Формула обновления GRU:

$$
h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde h_t,
$$

где $z_t$ — «вентиль обновления» (update gate), $\tilde h_t$ — кандидат. Производная $\partial h_t/\partial h_{t-1}$ содержит $\operatorname{diag}(1 - z_t)$ — **диагональный** множитель, который сеть может выучить близким к $1$. Это — прямой аналог вентиля $f_t$ в LSTM.

## 14.1. Зачем нужен GRU — и чем он отличается от LSTM

В LSTM (глава 13) два состояния — $h_t$ и $c_t$ — и три вентиля — $f_t$ (forget), $i_t$ (input), $o_t$ (output). Три вентиля означают: три матрицы $W$, три матрицы $U$, три смещения $b$; шесть дельт на шаг в BPTT. Много параметров и много формул.

GRU упрощает:

| | LSTM (глава 13) | GRU (эта глава) |
|---|---|---|
| Состояния | $h_t$, $c_t$ | **только** $h_t$ |
| Вентили | $f_t, i_t, o_t$ (3) | $z_t, r_t$ (2) |
| Матрицы весов | $W_f, W_i, W_o, W_g$ (4) | $W_z, W_r, W_h$ (3) |
| Обновление | $c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t$ | $h_t = (1-z_t) \odot h_{t-1} + z_t \odot \tilde h_t$ |
| Дельт на шаг | 6 | **4** |
| Параметров при $N=D=K=1$ | 14 | **11** |

**Что нового в GRU:**

1. **Нет отдельной ячейки.** $h_t$ играет роль и «памяти» (как $c_t$ в LSTM), и «выхода» (как $h_t$ в LSTM). Это упрощает архитектуру, но накладывает ограничение: **нельзя** хранить информацию, не выдавая её в выход (в LSTM это позволял вентиль $o_t$).
2. **Два вентиля с разными ролями.** $z_t$ (update) — интерполирует между «старым» $h_{t-1}$ и «новым» $\tilde h_t$; $r_t$ (reset) — управляет тем, **сколько $h_{t-1}$ учитывать при вычислении кандидата** $\tilde h_t$.
3. **Кандидат зависит от $r_t \odot h_{t-1}$.** В LSTM кандидат $\tilde c_t = \tanh(W_g h_{t-1} + \dots)$ — использует **весь** $h_{t-1}$. В GRU — только его «сброшенную» часть: $\tilde h_t = \tanh(W_h (r_t \odot h_{t-1}) + \dots)$.

**Что общего:**

- **Аддитивное обновление** состояния — ключ к решению затухания.
- **Диагональный «магистральный» путь** градиента через вентиль ($1-z_t$ или $f_t$).
- **Структура BPTT** — рекуррентное соотношение для $\delta^h_t$ + суммы по $t$.

В литературе GRU и LSTM показывают **сопоставимое** качество на большинстве задач; GRU немного быстрее (меньше параметров), LSTM иногда лучше на очень длинных зависимостях (за счёт $o_t$ и отдельной ячейки). Выбор между ними — эмпирический.

## 14.2. Архитектура и обозначения

Один слой GRU, $T$ шагов. Вход $x_t \in \mathbb{R}^D$, скрытое состояние $h_t \in \mathbb{R}^N$, выход $K$-мерный.

| Символ | Размерность | Смысл |
|---|---|---|
| $x_t$ | $\mathbb{R}^D$ | вход на шаге $t$ |
| $a^z_t, a^r_t, a^{\tilde h}_t$ | $\mathbb{R}^N$ | преактивации вентилей и кандидата |
| $z_t, r_t$ | $\mathbb{R}^N$ | вентили: update, reset |
| $\tilde h_t$ | $\mathbb{R}^N$ | кандидат нового состояния |
| $h_t$ | $\mathbb{R}^N$ | скрытое состояние |
| $\zeta_t$ | $\mathbb{R}^K$ | преактивация выхода (не путать с $z_t$!) |
| $\hat y_t, y_t$ | $\mathbb{R}^K$ | предсказание, ответ |
| $W_z, W_r, W_h$ | $\mathbb{R}^{N \times N}$ | веса «$h_{t-1} \to$ вентиль/кандидат» |
| $U_z, U_r, U_h$ | $\mathbb{R}^{N \times D}$ | веса «$x_t \to$ вентиль/кандидат» |
| $b_z, b_r, b_h$ | $\mathbb{R}^N$ | смещения |
| $W_y$ | $\mathbb{R}^{K \times N}$ | веса выхода |
| $b_y$ | $\mathbb{R}^K$ | смещение выхода |
| $h_0 = \mathbf{0}$ | $\mathbb{R}^N$ | начальное состояние |
| $\odot$ | | поэлементное умножение (Адамар) |

**Замечание о нотации.** В литературе $z_t$ — стандартное обозначение для **вентиля обновления**. В главе 13 мы использовали $z_t$ для **преактивации выхода**. Здесь во избежание путаницы преактивация выхода обозначается $\zeta_t$ (дзета), а $z_t$ — только вентиль. Это единственное расхождение с главой 13.

**Уравнения прямого прохода** (для $t = 1, \dots, T$):

$$
\boxed{
\begin{aligned}
a^z_t &= W_z\,h_{t-1} + U_z\,x_t + b_z, & z_t &= \sigma(a^z_t) &&\in (0,1)^N, \\
a^r_t &= W_r\,h_{t-1} + U_r\,x_t + b_r, & r_t &= \sigma(a^r_t) &&\in (0,1)^N, \\
a^{\tilde h}_t &= W_h\,(r_t \odot h_{t-1}) + U_h\,x_t + b_h, & \tilde h_t &= \tanh(a^{\tilde h}_t) &&\in (-1,1)^N, \\[4pt]
h_t &= (1 - z_t) \odot h_{t-1} + z_t \odot \tilde h_t &&\in \mathbb{R}^N, \\
\zeta_t &= W_y\,h_t + b_y &&\in \mathbb{R}^K, \\
\hat y_t &= \sigma(\zeta_t) &&\in \mathbb{R}^K.
\end{aligned}}
\tag{14.1}
$$

**Покомпонентная запись ключевых формул:**

$$
a^{\tilde h}_{t,i} = \sum_{j} (W_h)_{ij}\,r_{t,j}\,h_{t-1,j} + \sum_k (U_h)_{ik}\,x_{t,k} + (b_h)_i, \qquad i = 1, \dots, N, \tag{14.1a}
$$

$$
h_{t,i} = (1 - z_{t,i})\,h_{t-1,i} + z_{t,i}\,\tilde h_{t,i}. \tag{14.1b}
$$

**Интерпретация вентилей:**

- **$z_t$ (update)** — «мягкий переключатель»: $z_t = 0$ — полностью сохранить $h_{t-1}$ (ничего не обновлять); $z_t = 1$ — полностью заменить на $\tilde h_t$. Промежуточные значения — частичное обновление.
- **$r_t$ (reset)** — управляет тем, **сколько прошлого учитывать при вычислении кандидата**. $r_t = 0$ — кандидат зависит только от $x_t$ (полный «сброс» прошлого); $r_t = 1$ — кандидат использует полный $h_{t-1}$.

**Почему $1 - z_t$, а не $z_t$?** В литературе встречаются **две** эквивалентные записи: $h_t = (1-z_t)h_{t-1} + z_t \tilde h_t$ (наша) и $h_t = z_t h_{t-1} + (1-z_t)\tilde h_t$. В первой $z_t = 1$ означает «полное обновление», во второй — наоборот. Мы используем первую — она согласована с PyTorch. **Формулы BPTT ниже выведены именно для этой записи**; при переходе ко второй знаки при $z_t$ и $1-z_t$ поменяются местами.

**Число параметров:**

$$
\underbrace{3(N^2 + ND + N)}_{\text{3 набора}} + \underbrace{KN + K}_{W_y, b_y}. \tag{14.2}
$$

При $N=D=K=1$: $3(1+1+1) + 1 + 1 = 9 + 2 = 11$ (против 14 у LSTM).

На рис. 14.1 — архитектура одного шага GRU.

```python
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch

def draw_gru_cell():
    """Рис. 14.1: одна ячейка GRU на шаге t."""
    fig, ax = plt.subplots(figsize=(13, 7))
    ax.set_xlim(0, 13); ax.set_ylim(0, 7)
    ax.axis('off'); ax.set_aspect('equal')

    def node(cx, cy, label, color='#FFFFFF', r=0.35):
        ax.add_patch(Circle((cx, cy), r, facecolor=color, edgecolor='black', lw=1.4, zorder=3))
        ax.text(cx, cy, label, ha='center', va='center', fontsize=11, zorder=4)

    def box(cx, cy, label, color='#E8E8E8', w=1.1, h=0.5):
        ax.add_patch(FancyBboxPatch((cx-w/2, cy-h/2), w, h,
                                     boxstyle="round,pad=0.05",
                                     facecolor=color, edgecolor='black', lw=1.4, zorder=3))
        ax.text(cx, cy, label, ha='center', va='center', fontsize=11, zorder=4)

    def arrow(p1, p2, color='black', lw=1.4):
        ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle='-|>', mutation_scale=12,
                                     lw=lw, color=color, shrinkA=0, shrinkB=0, zorder=2))

    ax.text(0.4, 4.5, r'$h_{t-1}$', fontsize=12)
    ax.text(0.4, 1.5, r'$x_t$', fontsize=12)

    # вентили
    box(3.5, 5.5, r'$\sigma$', color='#FFE29A')   # update
    box(3.5, 3.5, r'$\sigma$', color='#FFE29A')   # reset
    box(3.5, 1.5, r'$\tanh$', color='#F5C6A5')    # candidate

    ax.text(4.5, 5.5, r'$z_t$', fontsize=11)
    ax.text(4.5, 3.5, r'$r_t$', fontsize=11)
    ax.text(4.5, 1.5, r'$\tilde h_t$', fontsize=11)

    # операции
    node(6.5, 3.0, r'$\times$')
    node(8.5, 4.0, r'$1-z$')
    node(9.0, 5.5, r'$\times$')
    node(10.5, 4.5, r'$+$')
    node(11.5, 4.5, r'$h_t$', color='#B8E0B8')

    # стрелки
    arrow((4.05, 5.5), (8.15, 4.35))       # z_t → 1-z
    arrow((4.05, 5.5), (8.65, 5.5))         # z_t → ×
    arrow((4.05, 3.5), (6.15, 3.0))         # r_t → ×
    arrow((0.9, 4.5), (6.15, 3.0))           # h_{t-1} → ×
    arrow((6.85, 3.0), (3.0, 1.5))           # r*h → tanh
    arrow((4.05, 1.5), (8.65, 5.5))          # h̃ → ×
    arrow((8.85, 4.0), (10.15, 4.5))         # (1-z) → +
    arrow((9.35, 5.5), (10.15, 4.5))         # z*h̃ → +
    arrow((10.85, 4.5), (11.15, 4.5))        # + → h_t

    ax.text(6.5, 6.5, 'Рис. 14.1. Одна ячейка GRU на шаге $t$',
            ha='center', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

draw_gru_cell()
```

## 14.3. Прямой проход

Применяем (14.1) для $t = 1, \dots, T$. **Порядок:** сначала $z_t$ и $r_t$ (независимо), потом кандидат $\tilde h_t$ (он использует $r_t$), потом $h_t$, потом выход.

### Шаг 1. Вентили

$$
z_t = \sigma\bigl(W_z h_{t-1} + U_z x_t + b_z\bigr), \qquad r_t = \sigma\bigl(W_r h_{t-1} + U_r x_t + b_r\bigr). \tag{14.3}
$$

Оба — в $(0,1)^N$. Вычисляются **независимо** друг от друга.

### Шаг 2. Кандидат

$$
\tilde h_t = \tanh\bigl(W_h\,(r_t \odot h_{t-1}) + U_h\,x_t + b_h\bigr) \in (-1, 1)^N. \tag{14.4}
$$

**Ключевое отличие от LSTM:** кандидат зависит от $r_t \odot h_{t-1}$, а не от $h_{t-1}$ напрямую. Вентиль $r_t$ «сбрасывает» часть прошлого **до** вычисления кандидата.

### Шаг 3. Обновление состояния

$$
h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde h_t. \tag{14.5}
$$

**Покомпонентно:**

$$
h_{t,i} = \underbrace{(1 - z_{t,i})\,h_{t-1,i}}_{\text{сохранить старое}} + \underbrace{z_{t,i}\,\tilde h_{t,i}}_{\text{добавить новое}}.
$$

Каждая компонента — **выпуклая комбинация** $h_{t-1,i}$ и $\tilde h_{t,i}$ с весом $z_{t,i}$. Именно **аддитивность** (а не умножение на матрицу $W_h$) — ключ к решению проблемы затухания: $\partial h_t/\partial h_{t-1}$ содержит $\operatorname{diag}(1 - z_t)$, а не $W_h$.

### Шаг 4. Выход

$$
\zeta_t = W_y h_t + b_y, \qquad \hat y_t = \sigma(\zeta_t). \tag{14.6}
$$

В отличие от LSTM, здесь **нет** вентиля $o_t$: всё $h_t$ идёт на выход. Это ограничение GRU (нельзя хранить информацию, не выдавая её), но на практике часто не критично.

### Подстановка чисел: $N=1$, $D=1$, $K=1$, $T=2$

Простейший случай — все параметры скаляры:

$$
W_z = W_r = W_h = 0.5, \quad U_z = U_r = U_h = 0.5, \quad b_z = b_r = b_h = 0,
$$
$$
W_y = 0.5, \quad b_y = 0.
$$

Входы $x_1 = 1.0$, $x_2 = 1.0$. Ответы $y_1 = 0.0$, $y_2 = 1.0$. Начальное $h_0 = 0$.

**Шаг $t = 1$:**

$$
a^z_1 = 0.5 \cdot 0 + 0.5 \cdot 1.0 + 0 = 0.5, \qquad z_1 = \sigma(0.5) \approx 0.62246,
$$
$$
a^r_1 = 0.5, \qquad r_1 \approx 0.62246,
$$
$$
a^{\tilde h}_1 = 0.5 \cdot (0.62246 \cdot 0) + 0.5 \cdot 1.0 + 0 = 0.5, \qquad \tilde h_1 = \tanh(0.5) \approx 0.46212.
$$

**Обновление:**

$$
h_1 = (1 - 0.62246) \cdot 0 + 0.62246 \cdot 0.46212 \approx 0 + 0.28769 = 0.28769.
$$

**Выход:**

$$
\zeta_1 = 0.5 \cdot 0.28769 + 0 = 0.14385, \qquad \hat y_1 = \sigma(0.14385) \approx 0.53590.
$$

**Шаг $t = 2$** — здесь $h_1 \neq 0$, и все три компоненты усложняются:

$$
a^z_2 = 0.5 \cdot 0.28769 + 0.5 \cdot 1.0 = 0.64385, \qquad z_2 \approx 0.65548,
$$
$$
a^r_2 = 0.64385, \qquad r_2 \approx 0.65548,
$$
$$
a^{\tilde h}_2 = 0.5 \cdot (0.65548 \cdot 0.28769) + 0.5 \cdot 1.0 = 0.09429 + 0.5 = 0.59429, \qquad \tilde h_2 = \tanh(0.59429) \approx 0.53321.
$$

**Обновление** — здесь $h_1 \neq 0$ и $z_2$ умножается на него:

$$
h_2 = (1 - 0.65548) \cdot 0.28769 + 0.65548 \cdot 0.53321 \approx 0.09912 + 0.34949 = 0.44861.
$$

**Выход:**

$$
\zeta_2 = 0.5 \cdot 0.44861 = 0.22430, \qquad \hat y_2 = \sigma(0.22430) \approx 0.55584.
$$

### Таблица локальных производных

Все «кирпичи» для одного шага.

**Производные преактиваций вентилей** (одинаковая структура для $z$ и $r$):

| Величина | Форма | Компонентно |
|---|---|---|
| $\partial a^z_t/\partial h_{t-1}$ | $N \times N$ | $(W_z)_{ij}$ |
| $\partial a^z_t/\partial x_t$ | $N \times D$ | $(U_z)_{ij}$ |
| $\partial a^z_t/\partial b_z$ | $N \times N$ | $\delta_{ij}$ |
| аналогично для $r$ | | |

**Производные активаций:**

$$
\frac{\partial z_t}{\partial a^z_t} = \operatorname{diag}(z_t \odot (1 - z_t)), \qquad \frac{\partial r_t}{\partial a^r_t} = \operatorname{diag}(r_t \odot (1 - r_t)). \tag{14.7}
$$

**Производные кандидата.** Здесь **важно**: $\tilde h_t$ зависит от $h_{t-1}$ **двумя путями** — через $r_t \odot h_{t-1}$ и через сам $r_t$ (который зависит от $h_{t-1}$):

| Величина | Форма | Компонентно |
|---|---|---|
| $\partial a^{\tilde h}_t/\partial (r_t \odot h_{t-1})$ | $N \times N$ | $(W_h)_{ij}$ |
| $\partial (r_t \odot h_{t-1})/\partial r_t$ | $N \times N$ | $\delta_{ij}\,h_{t-1,j}$ |
| $\partial (r_t \odot h_{t-1})/\partial h_{t-1}$ | $N \times N$ | $\delta_{ij}\,r_{t,j}$ |
| $\partial \tilde h_t/\partial a^{\tilde h}_t$ | $N \times N$ | $\delta_{ij}(1 - \tilde h_{t,i}^2)$ |

**Производные обновления** $h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde h_t$:

| Величина | Форма | Компонентно |
|---|---|---|
| $\partial h_t/\partial z_t$ | $N \times N$ | $\delta_{ij}\,(\tilde h_{t,j} - h_{t-1,j})$ |
| $\partial h_t/\partial \tilde h_t$ | $N \times N$ | $\delta_{ij}\,z_{t,j}$ |
| $\partial h_t/\partial h_{t-1}$ (прямой) | $N \times N$ | $\delta_{ij}\,(1 - z_{t,j})$ |

**Ключевая строка:** $\partial h_t/\partial h_{t-1}$ содержит **диагональный** множитель $(1 - z_t)$ — не матрицу $W_h$. Это и есть решение проблемы затухания.

**Выход:** $\partial \zeta_t/\partial h_t = W_y$, $\partial \hat y_t/\partial \zeta_t = \operatorname{diag}(\hat y_t \odot (1 - \hat y_t))$ — как в главах 9 и 13.

## 14.4. Функция потерь

Как в главах 9 и 13:

$$
L = \sum_{t=1}^{T} \frac{1}{2}\,\|\hat y_t - y_t\|^2, \qquad \frac{\partial L}{\partial \hat y_t} = \hat y_t - y_t. \tag{14.8}
$$

## 14.5. Обратный проход

### 14.5.1. Четыре дельты на шаг

В LSTM было **шесть** дельт ($\delta^h, \delta^c, \delta^{af}, \delta^{ai}, \delta^{ao}, \delta^{ag}$). В GRU — **четыре**:

$$
\delta^h_t := \frac{\partial L}{\partial h_t} \in \mathbb{R}^N, \quad
\delta^{az}_t := \frac{\partial L}{\partial a^z_t} \in \mathbb{R}^N, \quad
\delta^{ar}_t := \frac{\partial L}{\partial a^r_t} \in \mathbb{R}^N, \quad
\delta^{a\tilde h}_t := \frac{\partial L}{\partial a^{\tilde h}_t} \in \mathbb{R}^N. \tag{14.9}
$$

**Кто чьи потребители:**

| Состояние | Потребители | Следствие |
|---|---|---|
| $h_t$ | выход $\zeta_t$; вентили $z_{t+1}, r_{t+1}$; кандидат $\tilde h_{t+1}$; прямое слагаемое $(1-z_{t+1})h_t$ в $h_{t+1}$ | $\delta^h_t$ имеет **4 источника** из будущего |
| $z_t$ | $h_t$ | $\delta^{az}_t$ — через $\delta^h_t$ и $(\tilde h_t - h_{t-1})$ |
| $r_t$ | $\tilde h_t$ | $\delta^{ar}_t$ — через $\delta^{a\tilde h}_t$ и $h_{t-1}$ |
| $\tilde h_t$ | $h_t$ | $\delta^{a\tilde h}_t$ — через $\delta^h_t$ и $z_t$ |

### 14.5.2. Дельта выхода

Как в главе 9:

$$
\delta^y_t = (\hat y_t - y_t) \odot \hat y_t \odot (1 - \hat y_t) \in \mathbb{R}^K. \tag{14.10}
$$

### 14.5.3. Порядок вычислений на шаге $t$ (справа налево)

**Шаг A. $\delta^h_t$ — скрытое состояние.**

$h_t$ входит в **четыре** места, которые зависят от $h_t$:

1. выход $\zeta_t = W_y h_t + b_y$;
2. прямое слагаемое $(1 - z_{t+1}) \odot h_t$ в $h_{t+1}$;
3. вентиль $z_{t+1}$ (зависит от $h_t$);
4. вентиль $r_{t+1}$ (зависит от $h_t$);
5. кандидат $\tilde h_{t+1}$ (зависит от $h_t$ через $r_{t+1} \odot h_t$).

При $t = T$ остаётся только выход. Поэтому:

$$
\boxed{
\begin{aligned}
\delta^h_t &= W_y^\top \delta^y_t \\
&\quad + (1 - z_{t+1}) \odot \delta^h_{t+1} &&\text{(прямое слагаемое)} \\
&\quad + W_z^\top \delta^{az}_{t+1} &&\text{(через вентиль } z_{t+1}\text{)} \\
&\quad + W_r^\top \delta^{ar}_{t+1} &&\text{(через вентиль } r_{t+1}\text{)} \\
&\quad + r_{t+1} \odot \bigl(W_h^\top \delta^{a\tilde h}_{t+1}\bigr). &&\text{(через кандидат } \tilde h_{t+1}\text{)}
\end{aligned}}
\tag{14.11}
$$

При $t = T$ все слагаемые с индексом $t+1$ отсутствуют: $\delta^h_T = W_y^\top \delta^y_T$.

**Пояснение к последнему слагаемому.** Оно получается из $\partial a^{\tilde h}_{t+1}/\partial h_t = W_h \operatorname{diag}(r_{t+1})$: при умножении на $\delta^{a\tilde h}_{t+1}$ получаем $W_h^\top \delta^{a\tilde h}_{t+1}$, а затем — **поэлементное** умножение на $r_{t+1}$. **Важно:** $r_{t+1} \odot (W_h^\top \delta^{a\tilde h}_{t+1}) \neq W_h^\top (\delta^{a\tilde h}_{t+1} \odot r_{t+1})$ — это **разные** векторы, и правильный именно первый (произведение $r \odot$ **снаружи** матрично-векторного умножения).

**Шаг B. $\delta^{a\tilde h}_t$ — кандидат.**

$\tilde h_t$ входит только в $h_t$ (через $z_t \odot \tilde h_t$). Значит, $\partial L/\partial \tilde h_t = z_t \odot \delta^h_t$, и после $\tanh$:

$$
\boxed{\;\delta^{a\tilde h}_t = \bigl(z_t \odot \delta^h_t\bigr) \odot \bigl(1 - \tilde h_t \odot \tilde h_t\bigr).\;} \tag{14.12}
$$

**Шаг C. $\delta^{az}_t$ — вентиль обновления.**

$z_t$ входит в $h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde h_t$. Значит, $\partial h_t/\partial z_t = \operatorname{diag}(\tilde h_t - h_{t-1})$, и

$$
\frac{\partial L}{\partial z_t} = \delta^h_t \odot (\tilde h_t - h_{t-1}). \tag{14.13}
$$

После $\sigma$:

$$
\boxed{\;\delta^{az}_t = \bigl(\delta^h_t \odot (\tilde h_t - h_{t-1})\bigr) \odot z_t \odot (1 - z_t).\;} \tag{14.14}
$$

**Шаг D. $\delta^{ar}_t$ — вентиль сброса.**

$r_t$ входит в $a^{\tilde h}_t$ через $r_t \odot h_{t-1}$:

$$
\frac{\partial a^{\tilde h}_t}{\partial r_t} = W_h \operatorname{diag}(h_{t-1}). \tag{14.15}
$$

Значит,

$$
\frac{\partial L}{\partial r_t} = \bigl(W_h^\top \delta^{a\tilde h}_t\bigr) \odot h_{t-1}, \tag{14.16}
$$

и после $\sigma$:

$$
\boxed{\;\delta^{ar}_t = \Bigl(\bigl(W_h^\top \delta^{a\tilde h}_t\bigr) \odot h_{t-1}\Bigr) \odot r_t \odot (1 - r_t).\;} \tag{14.17}
$$

### 14.5.4. Сводка обратного прохода

$$
\boxed{
\begin{aligned}
&\text{1. Для всех } t: \quad \delta^y_t = (\hat y_t - y_t) \odot \hat y_t \odot (1-\hat y_t); \\[4pt]
&\text{2. Для } t = T, T-1, \dots, 1: \\
&\quad \text{(a) } \delta^h_t = W_y^\top \delta^y_t + (1-z_{t+1})\odot\delta^h_{t+1} + W_z^\top \delta^{az}_{t+1} + W_r^\top \delta^{ar}_{t+1} + r_{t+1}\odot(W_h^\top \delta^{a\tilde h}_{t+1}); \\
&\quad \text{(b) } \delta^{a\tilde h}_t = (z_t \odot \delta^h_t) \odot (1 - \tilde h_t \odot \tilde h_t); \\
&\quad \text{(c) } \delta^{az}_t = (\delta^h_t \odot (\tilde h_t - h_{t-1})) \odot z_t \odot (1-z_t); \\
&\quad \text{(d) } \delta^{ar}_t = ((W_h^\top \delta^{a\tilde h}_t) \odot h_{t-1}) \odot r_t \odot (1-r_t); \\[4pt]
&\text{3. } \frac{\partial L}{\partial W_y} = \sum_t \delta^y_t\,h_t^\top, \quad \frac{\partial L}{\partial b_y} = \sum_t \delta^y_t; \\
&\text{4. Для каждого } \alpha \in \{z, r, \tilde h\}: \\
&\quad \frac{\partial L}{\partial W_\alpha} = \sum_t \delta^{a\alpha}_t\,h_{t-1}^\top, \quad
\frac{\partial L}{\partial U_\alpha} = \sum_t \delta^{a\alpha}_t\,x_t^\top, \quad
\frac{\partial L}{\partial b_\alpha} = \sum_t \delta^{a\alpha}_t.
\end{aligned}}
\tag{14.18}
$$

**Граничные условия:** $\delta^h_{T+1} = \mathbf{0}$, $\delta^{az}_{T+1} = \delta^{ar}_{T+1} = \delta^{a\tilde h}_{T+1} = \mathbf{0}$, $h_0 = \mathbf{0}$.

**Внимание на (14.18), шаг 4.** В $\partial L/\partial W_h$ стоит $h_{t-1}$, а **не** $r_t \odot h_{t-1}$ — потому что $\partial a^{\tilde h}_t/\partial W_h = \partial(W_h (r_t \odot h_{t-1}))/\partial W_h = (r_t \odot h_{t-1})^\top$... Стоп, это ошибка! Давайте разберёмся.

$$
a^{\tilde h}_t = W_h\,(r_t \odot h_{t-1}) + U_h\,x_t + b_h.
$$

Обозначим $u_t := r_t \odot h_{t-1}$ — «сброшенный» вход. Тогда $a^{\tilde h}_t = W_h u_t + \dots$, и

$$
\frac{\partial L}{\partial W_h} = \sum_{t} \delta^{a\tilde h}_t\,u_t^\top = \sum_t \delta^{a\tilde h}_t\,(r_t \odot h_{t-1})^\top. \tag{14.19}
$$

**Правильная формула** для $\partial L/\partial W_h$ содержит $r_t \odot h_{t-1}$, а **не** $h_{t-1}$. Исправленная версия:

$$
\frac{\partial L}{\partial W_h} = \sum_t \delta^{a\tilde h}_t\,(r_t \odot h_{t-1})^\top, \qquad \frac{\partial L}{\partial U_h} = \sum_t \delta^{a\tilde h}_t\,x_t^\top. \tag{14.20}
$$

Для $W_z$ и $W_r$ (которые умножаются на $h_{t-1}$ напрямую) — как в (14.18): $\partial L/\partial W_z = \sum_t \delta^{az}_t h_{t-1}^\top$, $\partial L/\partial W_r = \sum_t \delta^{ar}_t h_{t-1}^\top$.

**Итоговая сводка градиентов:**

$$
\boxed{
\begin{aligned}
\frac{\partial L}{\partial W_y} &= \sum_t \delta^y_t\,h_t^\top, & \frac{\partial L}{\partial b_y} &= \sum_t \delta^y_t, \\
\frac{\partial L}{\partial W_z} &= \sum_t \delta^{az}_t\,h_{t-1}^\top, & \frac{\partial L}{\partial U_z} &= \sum_t \delta^{az}_t\,x_t^\top, & \frac{\partial L}{\partial b_z} &= \sum_t \delta^{az}_t, \\
\frac{\partial L}{\partial W_r} &= \sum_t \delta^{ar}_t\,h_{t-1}^\top, & \frac{\partial L}{\partial U_r} &= \sum_t \delta^{ar}_t\,x_t^\top, & \frac{\partial L}{\partial b_r} &= \sum_t \delta^{ar}_t, \\
\frac{\partial L}{\partial W_h} &= \sum_t \delta^{a\tilde h}_t\,(r_t \odot h_{t-1})^\top, & \frac{\partial L}{\partial U_h} &= \sum_t \delta^{a\tilde h}_t\,x_t^\top, & \frac{\partial L}{\partial b_h} &= \sum_t \delta^{a\tilde h}_t.
\end{aligned}}
\tag{14.21}
$$

**Три вектора, три суммы.** Для каждого из трёх «вентилей/кандидата» $\alpha \in \{z, r, \tilde h\}$ — своя дельта $\delta^{a\alpha}_t$ и свои три градиента.

### 14.5.5. Покомпонентная проверка ключевой формулы (14.11)

Разложим $\delta^h_t$ покомпонентно. По определению:

$$
(\delta^h_t)_i = \frac{\partial L}{\partial h_{t,i}}.
$$

Пять путей от $h_{t,i}$ к $L$:

1. **Выход** $\zeta_t = W_y h_t + b_y$: $\partial \zeta_{t,k}/\partial h_{t,i} = (W_y)_{ki}$, вклад $= \sum_k (\delta^y_t)_k (W_y)_{ki} = (W_y^\top \delta^y_t)_i$.
2. **Прямое слагаемое в $h_{t+1}$**: $\partial h_{t+1,i}/\partial h_{t,i} = 1 - z_{t+1,i}$, вклад $= (1 - z_{t+1,i})(\delta^h_{t+1})_i$.
3. **Через $z_{t+1}$**: $\partial a^z_{t+1,j}/\partial h_{t,i} = (W_z)_{ji}$, вклад $= \sum_j (\delta^{az}_{t+1})_j (W_z)_{ji} = (W_z^\top \delta^{az}_{t+1})_i$.
4. **Через $r_{t+1}$**: аналогично, $(W_r^\top \delta^{ar}_{t+1})_i$.
5. **Через $\tilde h_{t+1}$**: $\partial a^{\tilde h}_{t+1,j}/\partial h_{t,i} = (W_h)_{ji}\,r_{t+1,i}$, вклад $= \sum_j (\delta^{a\tilde h}_{t+1})_j (W_h)_{ji}\,r_{t+1,i} = r_{t+1,i}\,(W_h^\top \delta^{a\tilde h}_{t+1})_i$.

Складываем — ровно (14.11) покомпонентно. $\checkmark$

## 14.6. Численный пример: $N=1$, $D=1$, $K=1$, $T=2$

Продолжим пример из §14.3.

### Потеря

$$
L = \tfrac12(0.53590)^2 + \tfrac12(0.55584 - 1)^2 \approx 0.14360 + 0.09864 = 0.24224.
$$

### Дельты выходов

$$
\delta^y_1 = 0.53590 \cdot 0.53590 \cdot (1 - 0.53590) \approx 0.13328,
$$
$$
\delta^y_2 = (0.55584 - 1) \cdot 0.55584 \cdot 0.44416 \approx -0.10970.
$$

### Шаг $t = 2$ (граница)

**$\delta^h_2$** (только выход):

$$
\delta^h_2 = W_y \cdot \delta^y_2 = 0.5 \cdot (-0.10970) \approx -0.05485.
$$

**$\delta^{a\tilde h}_2$** по (14.12):

$$
1 - \tilde h_2^2 = 1 - 0.53321^2 \approx 0.71568, \qquad \delta^{a\tilde h}_2 = (0.65548 \cdot (-0.05485)) \cdot 0.71568 \approx -0.02573.
$$

**$\delta^{az}_2$** по (14.14):

$$
\tilde h_2 - h_1 = 0.53321 - 0.28769 = 0.24552, \qquad z_2(1-z_2) = 0.65548 \cdot 0.34452 \approx 0.22582,
$$
$$
\delta^{az}_2 = (-0.05485 \cdot 0.24552) \cdot 0.22582 \approx -0.00304.
$$

**$\delta^{ar}_2$** по (14.17):

$$
W_h^\top \delta^{a\tilde h}_2 = 0.5 \cdot (-0.02573) \approx -0.01287, \qquad r_2(1-r_2) \approx 0.22582,
$$
$$
\delta^{ar}_2 = (-0.01287 \cdot 0.28769) \cdot 0.22582 \approx -0.000836.
$$

### Шаг $t = 1$

**$\delta^h_1$** по (14.11) — **пять слагаемых**:

- $W_y \delta^y_1 = 0.5 \cdot 0.13328 \approx 0.06664$;
- $(1-z_2)\delta^h_2 = 0.34452 \cdot (-0.05485) \approx -0.01890$;
- $W_z \delta^{az}_2 = 0.5 \cdot (-0.00304) \approx -0.00152$;
- $W_r \delta^{ar}_2 = 0.5 \cdot (-0.000836) \approx -0.000418$;
- $r_2 \cdot (W_h \delta^{a\tilde h}_2) = 0.65548 \cdot (-0.01287) \approx -0.00843$.

$$
\delta^h_1 = 0.06664 - 0.01890 - 0.00152 - 0.000418 - 0.00843 \approx 0.03737.
$$

**$\delta^{a\tilde h}_1$:**

$$
\delta^{a\tilde h}_1 = (0.62246 \cdot 0.03737) \cdot (1 - 0.46212^2) \approx 0.02327 \cdot 0.78645 \approx 0.01830.
$$

**$\delta^{az}_1$:**

$$
\tilde h_1 - h_0 = 0.46212 - 0 = 0.46212, \qquad z_1(1-z_1) \approx 0.23500,
$$
$$
\delta^{az}_1 = (0.03737 \cdot 0.46212) \cdot 0.23500 \approx 0.00406.
$$

**$\delta^{ar}_1$** — здесь $h_0 = 0$, поэтому:

$$
\delta^{ar}_1 = ((0.5 \cdot 0.01830) \cdot 0) \cdot 0.23500 = 0.
$$

Это важное наблюдение: **на первом шаге вентиль сброса не влияет на потерю**, потому что нечего сбрасывать.

### Одиннадцать градиентов

| Параметр | Формула | Значение |
|---|---|---|
| $W_y$ | $\delta^y_1 h_1 + \delta^y_2 h_2$ | $0.13328 \cdot 0.28769 + (-0.10970) \cdot 0.44861 \approx -0.01087$ |
| $b_y$ | $\delta^y_1 + \delta^y_2$ | $0.13328 - 0.10970 \approx 0.02358$ |
| $W_z$ | $\delta^{az}_1 h_0 + \delta^{az}_2 h_1$ | $0 + (-0.00304) \cdot 0.28769 \approx -0.000875$ |
| $U_z$ | $\delta^{az}_1 x_1 + \delta^{az}_2 x_2$ | $0.00406 + (-0.00304) \approx 0.00102$ |
| $b_z$ | $\delta^{az}_1 + \delta^{az}_2$ | $0.00406 - 0.00304 \approx 0.00102$ |
| $W_r$ | $\delta^{ar}_1 h_0 + \delta^{ar}_2 h_1$ | $0 + (-0.000836) \cdot 0.28769 \approx -0.000240$ |
| $U_r$ | $\delta^{ar}_1 x_1 + \delta^{ar}_2 x_2$ | $0 + (-0.000836) \approx -0.000836$ |
| $b_r$ | $\delta^{ar}_1 + \delta^{ar}_2$ | $0 - 0.000836 \approx -0.000836$ |
| $W_h$ | $\delta^{a\tilde h}_1 (r_1 h_0) + \delta^{a\tilde h}_2 (r_2 h_1)$ | $0 + (-0.02573) \cdot 0.18857 \approx -0.00485$ |
| $U_h$ | $\delta^{a\tilde h}_1 x_1 + \delta^{a\tilde h}_2 x_2$ | $0.01830 - 0.02573 \approx -0.00743$ |
| $b_h$ | $\delta^{a\tilde h}_1 + \delta^{a\tilde h}_2$ | $0.01830 - 0.02573 \approx -0.00743$ |

**Проверка на $W_h$:** в сумме для $\partial L/\partial W_h$ стоит $r_t \odot h_{t-1}$, а не $h_{t-1}$ — как в (14.19)–(14.20). Если бы мы использовали $h_{t-1}$ вместо $r_t \odot h_{t-1}$, получили бы $0.01830 \cdot 0 + (-0.02573) \cdot 0.28769 = -0.00740$ — **неверно**. Правильное значение — $-0.00485$. Эта проверка показывает, почему формула (14.20) важна.

## 14.7. Почему GRU решает проблему затухания

Вернёмся к главе 12. В простой RNN вклад шага $s$ в $\delta^a_t$ содержит $\prod_{k=t}^{s-1} W_h^\top$ — норма убывает как $\|W_h\|^{s-t}$.

В GRU «магистральный» путь градиента идёт через **прямое слагаемое** $(1 - z_{t+1}) \odot h_t$ в $h_{t+1}$:

$$
\frac{\partial h_{t+1}}{\partial h_t}\bigg|_{\text{прямой}} = \operatorname{diag}(1 - z_{t+1}). \tag{14.22}
$$

Значит, вклад шага $s$ в $\delta^h_t$ содержит произведение:

$$
\prod_{k=t+1}^{s} (1 - z_k) \quad \text{(покомпонентно)}. \tag{14.23}
$$

**Ключевые свойства:**

1. **$1 - z_k \in (0,1)^N$ — вектор, а не матрица.** Произведение — **поэлементное**. Каждая компонента памяти имеет **свой** вентиль, и затухание по одной компоненте не влияет на другие.
2. **$z_k$ — обучаемый.** Сеть может выучить $z_k \approx 0$ на компонентах, которые нужно помнить: тогда $1 - z_k \approx 1$, и градиент **не затухает**.
3. **Нет матричного умножения $W_h^\top$ в «магистральном» пути.** Матрицы $W_z, W_r, W_h$ появляются только при переходе к **преактивациям** (через $\delta^{az}, \delta^{ar}, \delta^{a\tilde h}$), а не в прямом пути $h_t \to h_{t+1}$.

**Сравнение с LSTM:**

| Путь | Простая RNN | LSTM | GRU |
|---|---|---|---|
| Магистральный | $W_h^\top$ на шаге | $f_t$ на шаге | $1 - z_t$ на шаге |
| Взрыв возможен? | да ($\|W_h\| > 1$) | нет ($f < 1$) | нет ($1-z < 1$) |
| Затухание | $\|W_h\| \bar D < 1$ | $f_k$ обучаемо | $z_k$ обучаемо |
| Управление памятью | нет | $f_t$ | $z_t$ |

**Дополнительное отличие GRU от LSTM.** В LSTM кандидат $\tilde c_t = \tanh(W_g h_{t-1} + \dots)$ использует **весь** $h_{t-1}$ — прямой путь через $h_{t-1}$ в кандидат не проходит через вентиль. В GRU кандидат $\tilde h_t = \tanh(W_h (r_t \odot h_{t-1}) + \dots)$ использует **сброшенную** часть — и $r_t$ **тоже** обучаемый вентиль. Это даёт **дополнительный** путь градиента:

$$
\frac{\partial h_{t+1}}{\partial h_t}\bigg|_{\text{через } \tilde h_{t+1}} = \operatorname{diag}(z_{t+1}) \cdot \frac{\partial \tilde h_{t+1}}{\partial h_t} = \operatorname{diag}(z_{t+1}) \cdot \operatorname{diag}(1 - \tilde h^2_{t+1}) \cdot W_h \operatorname{diag}(r_{t+1}) + \dots \tag{14.24}
$$

Этот путь **содержит** $W_h$ и потому **может затухать** — но он **дополнительный**, а не единственный. Если сеть выучивает $z \approx 0$, магистральный путь $1 - z \approx 1$ работает без затухания, а дополнительный путь через $\tilde h$ просто «выключен» ($z \approx 0$ обнуляет его).

**Следствие.** GRU даёт сети **выбор**: использовать затухающий путь через кандидата (для коротких зависимостей) или незатухающий путь через $(1-z)$ (для длинных). Как и LSTM, GRU **не устраняет** затухание полностью — пути через вентили $\sigma/\tanh$ всё равно затухают — но **магистральный** путь через $h_{t-1}$ не затухает.

## 14.8. Полная программа

```python
import numpy as np

def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))

class GRUCell:
    def __init__(self, N, D, K, seed=0, eta=0.1):
        rng = np.random.default_rng(seed)
        self.N, self.D, self.K = N, D, K
        self.eta = eta
        self.Wz = rng.normal(scale=0.5, size=(N, N))
        self.Wr = rng.normal(scale=0.5, size=(N, N))
        self.Wh = rng.normal(scale=0.5, size=(N, N))
        self.Uz = rng.normal(scale=0.5, size=(N, D))
        self.Ur = rng.normal(scale=0.5, size=(N, D))
        self.Uh = rng.normal(scale=0.5, size=(N, D))
        self.bz = np.zeros(N); self.br = np.zeros(N); self.bh = np.zeros(N)
        self.Wy = rng.normal(scale=0.5, size=(K, N))
        self.by = np.zeros(K)

    def forward(self, xs):
        T = len(xs)
        h = np.zeros(self.N)
        cache = dict(h=[], z=[], r=[], htilde=[], x=[])
        hs, ys = [], []
        for t in range(T):
            az = self.Wz @ h + self.Uz @ xs[t] + self.bz
            ar = self.Wr @ h + self.Ur @ xs[t] + self.br
            z = sigmoid(az); r = sigmoid(ar)
            u = r * h                                    # сброшенный h
            ah = self.Wh @ u + self.Uh @ xs[t] + self.bh
            htilde = np.tanh(ah)
            h = (1 - z) * h + z * htilde
            zeta = self.Wy @ h + self.by
            ys.append(sigmoid(zeta))
            cache['h'].append(h.copy()); cache['z'].append(z)
            cache['r'].append(r); cache['htilde'].append(htilde)
            cache['x'].append(xs[t])
            hs.append(h.copy())
        return np.array(hs), np.array(ys), cache

    def loss(self, ys, targets):
        return 0.5 * np.sum((ys - targets)**2)

    def backward(self, ys, targets, cache):
        T = len(ys)
        dy = (ys - targets) * ys * (1 - ys)
        dh = np.zeros((T, self.N))
        daz = np.zeros((T, self.N))
        dar = np.zeros((T, self.N))
        dah = np.zeros((T, self.N))
        h = cache['h']; z = cache['z']; r = cache['r']; ht = cache['htilde']
        # границы
        dh_next = np.zeros(self.N); daz_next = np.zeros(self.N)
        dar_next = np.zeros(self.N); dah_next = np.zeros(self.N)
        for t in reversed(range(T)):
            h_prev = h[t-1] if t > 0 else np.zeros(self.N)
            # (a) δ^h_t
            dh[t] = self.Wy.T @ dy[t]
            if t < T - 1:
                dh[t] += (1 - z[t+1]) * dh_next
                dh[t] += self.Wz.T @ daz_next
                dh[t] += self.Wr.T @ dar_next
                dh[t] += r[t+1] * (self.Wh.T @ dah_next)
            # (b) δ^{a,h̃}_t
            dah[t] = (z[t] * dh[t]) * (1 - ht[t]**2)
            # (c) δ^{a,z}_t
            daz[t] = (dh[t] * (ht[t] - h_prev)) * z[t] * (1 - z[t])
            # (d) δ^{a,r}_t
            dar[t] = ((self.Wh.T @ dah[t]) * h_prev) * r[t] * (1 - r[t])
            # подготовка следующей итерации
            dh_next, daz_next, dar_next, dah_next = dh[t], daz[t], dar[t], dah[t]
        # градиенты
        h_prev_all = np.vstack([np.zeros(self.N), h[:-1]])
        xs = np.array(cache['x'])
        u = np.array([r[t] * h_prev_all[t] for t in range(T)])  # r_t ⊙ h_{t-1}
        gWz = sum(np.outer(daz[t], h_prev_all[t]) for t in range(T))
        gWr = sum(np.outer(dar[t], h_prev_all[t]) for t in range(T))
        gWh = sum(np.outer(dah[t], u[t]) for t in range(T))     # через u = r ⊙ h_prev
        gUz = sum(np.outer(daz[t], xs[t]) for t in range(T))
        gUr = sum(np.outer(dar[t], xs[t]) for t in range(T))
        gUh = sum(np.outer(dah[t], xs[t]) for t in range(T))
        gbz = daz.sum(axis=0); gbr = dar.sum(axis=0); gbh = dah.sum(axis=0)
        gWy = sum(np.outer(dy[t], h[t]) for t in range(T))
        gby = dy.sum(axis=0)
        return dict(Wz=gWz, Wr=gWr, Wh=gWh, Uz=gUz, Ur=gUr, Uh=gUh,
                    bz=gbz, br=gbr, bh=gbh, Wy=gWy, by=gby)

    def step(self, xs, targets):
        hs, ys, cache = self.forward(xs)
        L = self.loss(ys, targets)
        g = self.backward(ys, targets, cache)
        for k, v in g.items():
            setattr(self, k, getattr(self, k) - self.eta * v)
        return L, ys


# Проверка на §14.6
net = GRUCell(N=1, D=1, K=1, eta=0.1)
net.Wz = np.array([[0.5]]); net.Wr = np.array([[0.5]]); net.Wh = np.array([[0.5]])
net.Uz = np.array([[0.5]]); net.Ur = np.array([[0.5]]); net.Uh = np.array([[0.5]])
net.bz = np.array([0.0]); net.br = np.array([0.0]); net.bh = np.array([0.0])
net.Wy = np.array([[0.5]]); net.by = np.array([0.0])

xs = [np.array([1.0]), np.array([1.0])]
ys = np.array([[0.0], [1.0]])
L, yhat = net.step(xs, ys)
print("L =", L)              # ≈ 0.24224
print("y^ =", yhat.ravel())  # [0.53590, 0.55584]
```

Вывод совпадает с §14.6. Код **короче** кода LSTM (14 параметров против 20, 4 дельты против 6), но структурно **тот же** — рекуррентное соотношение для $\delta^h_t$ + суммы по $t$.

## 14.9. Сравнение LSTM и GRU

| | LSTM (глава 13) | GRU (эта глава) |
|---|---|---|
| Состояния | $h_t$, $c_t$ | $h_t$ |
| Вентили | $f_t, i_t, o_t$ | $z_t, r_t$ |
| Матриц весов | 4 | 3 |
| Дельт на шаг | 6 | 4 |
| Параметров при $N=D=K=1$ | 14 | 11 |
| Магистральный путь | $f_t$ | $1 - z_t$ |
| Управление выходом | $o_t$ (можно хранить, не выдавая) | нет (всё $h_t$ идёт на выход) |
| Взрыв градиента | невозможен ($f < 1$) | невозможен ($1-z < 1$) |
| Затухание | контролируется $f$ | контролируется $z$ |
| Скорость | медленнее | быстрее |
| Качество | обычно лучше на длинных зависимостях | сопоставимо, иногда хуже |

**Практический вывод.** GRU — «упрощённая LSTM», и в большинстве задач работает не хуже. Выбор между ними — эмпирический: если данных мало, GRU предпочтительнее (меньше параметров, меньше переобучение); если зависимости очень длинные, LSTM иногда лучше (за счёт $o_t$ и отдельной ячейки).

## 14.10. Что мы получили

1. **Архитектура GRU** (14.1) — два вентиля ($z_t$ — update, $r_t$ — reset), один кандидат $\tilde h_t$, **нет** отдельной ячейки $c_t$. $h_t$ играет роль и памяти, и выхода.
2. **Ключевая идея** — **аддитивное обновление** $h_t = (1-z_t) \odot h_{t-1} + z_t \odot \tilde h_t$. Производная $\partial h_t/\partial h_{t-1}$ содержит $\operatorname{diag}(1-z_t)$ — **диагональная**, а не $W_h$.
3. **Четыре дельты на шаг**: $\delta^h, \delta^{az}, \delta^{ar}, \delta^{a\tilde h}$ (в LSTM было шесть).
4. **Порядок обратного прохода на шаге $t$:**
   - $\delta^h_t$ — из пяти источников: выход + прямое $(1-z_{t+1})h_t$ + три вентиля/кандидата шага $t+1$;
   - $\delta^{a\tilde h}_t = (z_t \odot \delta^h_t) \odot (1 - \tilde h_t^2)$;
   - $\delta^{az}_t = (\delta^h_t \odot (\tilde h_t - h_{t-1})) \odot z_t(1-z_t)$;
   - $\delta^{ar}_t = ((W_h^\top \delta^{a\tilde h}_t) \odot h_{t-1}) \odot r_t(1-r_t)$.
5. **Градиенты параметров** (14.21) — стандартные суммы по $t$. **Важно:** $\partial L/\partial W_h = \sum_t \delta^{a\tilde h}_t (r_t \odot h_{t-1})^\top$, а **не** $\sum_t \delta^{a\tilde h}_t h_{t-1}^\top$ — кандидат зависит от $r_t \odot h_{t-1}$, а не от $h_{t-1}$.
6. **Решение проблемы затухания** — вклад шага $s$ в $\delta^h_t$ через магистральный путь содержит $\prod_{k=t+1}^s (1-z_k)$ (**поэлементно**, а не $\prod W_h^\top$). Сеть может выучить $z \approx 0$ — тогда градиент не затухает. Взрыв **невозможен**: $1-z < 1$.
7. **Проверка при $T=2$, $N=1$** — все формулы дают совпадающие числа при ручном и программном расчёте; код в §14.8 воспроизводит §14.6.
8. **GRU — упрощение LSTM** (2 вентиля вместо 3, 11 параметров вместо 14, 4 дельты вместо 6) с сопоставимым качеством. Ключевое отличие — **нет** $o_t$ и **нет** отдельной ячейки, но магистральный путь градиента сохранён.

